# GLUCOSA EN DEPORTISTAS - CGMACROS

Este notebook reúne el flujo completo de carga, limpieza, análisis exploratorio y modelización del conjunto CGMacros.


## A. LIBRERÍAS

In [ ]:
# Matplotlib: permitir hacer zoom y desplazamiento en las gráficas. La alternativa por defecto es %matplotlib inline.
%matplotlib widget

# Librerías básicas para procesado de datos.
import pandas as pd
import os
import glob
import numpy as np
import re

# Librerías para visualización.
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib as mpl
from IPython.display import display
import matplotlib.colors as mcolors
import seaborn as sns
import mpl_toolkits.mplot3d  # Se importa para mantener compatibilidad con gráficas 3D.

# Librerías para análisis. 
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from scipy.fft import rfft, rfftfreq
from scipy.signal import welch
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller, kpss
from scipy.stats import spearmanr
from scipy.signal import correlate

# Librerías para modelización.
import io
import json
from typing import List, Dict, Union, Tuple, Any, Optional
# Si Numba está disponible, puede acelerar bucles numéricos muy costosos.
from numba import njit
import xgboost as xgb
import optuna
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
import pickle
from matplotlib.colors import ListedColormap
import shap
import shap.explainers._tree as shap_tree
from shap.explainers.other._ubjson import decode_ubjson_buffer


# Utilidad para mostrar una figura y liberarla justo después.
def mostrar_y_liberar_figura(fig):
    """
    Muestra una figura en el notebook y la cierra para liberar memoria.
    Esta utilidad evita que las gráficas acumulen memoria cuando se generan
    muchas figuras dentro de un bucle o de un proceso iterativo.

    Args:
        fig: Figura de Matplotlib ya construida.

    Returns:
        None

    Example:
        >>> mostrar_y_liberar_figura(fig)
    """
    display(fig)
    plt.close(fig)


# Función auxiliar: reconstruye la rejilla temporal real de una serie a partir de sus puntos válidos.
def _obtener_timestamps_reales(serie, paso_minutos):
    """
    Aísla los puntos reales reconstruyendo la rejilla temporal nativa teórica.
    Busca el primer valor numérico válido específico de la serie que recibe y
    reconstruye la secuencia temporal esperada a partir de ese punto inicial.

    Args:
        serie: Serie con índice temporal y valores numéricos.
        paso_minutos (int): Separación temporal original entre muestras.

    Returns:
        pd.DatetimeIndex: Rejilla temporal reconstruida a partir del primer y último dato válidos.

    Recommendations:
        - Se usa cuando la serie contiene ceros o NaNs que no representan mediciones reales.
    """
    serie_limpia = pd.to_numeric(serie, errors='coerce')
    # Eliminamos NaNs y ceros para encontrar el primer dato real del sensor.
    serie_limpia = serie_limpia[(serie_limpia.notna()) & (serie_limpia != 0)]
    if serie_limpia.empty:
        return pd.DatetimeIndex([])
        
    # Anclamos la rejilla al primer y último dato real exclusivo de esta serie.
    inicio = serie_limpia.index[0]
    fin = serie_limpia.index[-1]
    
    # Generamos la frecuencia nativa esperada (5 min para Dexcom, 15 min para Libre).
    return pd.date_range(start=inicio, end=fin, freq=f"{paso_minutos}min")


In [ ]:
##############################################################
#                    VARIABLES GLOBALES                      #
##############################################################

RAW_DIR = r"C:\Users\regin\Documents\UNIVERSIDAD\TFG\Datos Externos\CGMacros (PhysioNet)\Unzip\cgmacros-a-scientific-dataset-for-personalized-nutrition-and-diet-monitoring-1.0.0\CGMacros"
TRUSTED_DIR = r"C:\Users\regin\Documents\GitHub\TFG-pruebas\Data\CGMacros\Trusted"
EXTENDED_TRUSTED_DIR = ''

RESUMEN_RAW = r"C:\Users\regin\Documents\GitHub\TFG-pruebas\Data\CGMacros\resumen_diagnostico_raw.csv"
RESUMEN_TRUSTED = r"C:\Users\regin\Documents\GitHub\TFG-pruebas\Data\CGMacros\resumen_diagnostico_trusted.csv"

HISTOGRAMAS_DIR = r"C:\Users\regin\Documents\GitHub\TFG-pruebas\Analysis\CGMacros\Histogramas glucosa"

PERFIL_PROMEDIO_DIR = r"C:\Users\regin\Documents\GitHub\TFG-pruebas\Analysis\CGMacros\Perfil Promedio"

SCATTER_DIR = r"C:\Users\regin\Documents\GitHub\TFG-pruebas\Analysis\CGMacros\Dispersión"
SPEARMAN_CSV = os.path.join(SCATTER_DIR, "tabla_spearman_cgmacros.csv")
AGRUPACIONES_DIR = os.path.join(SCATTER_DIR, "Agrupaciones de gráficos")

CORRELACION_DIR = r'C:\Users\regin\Documents\GitHub\TFG-pruebas\Analysis\CGMacros\Correlación'
os.makedirs(CORRELACION_DIR, exist_ok=True) # ASEGURAMOS LAS CARPETAS DE SALIDA
CORRELACION_INDIVIDUAL_DIR = os.path.join(CORRELACION_DIR, "Individuales")
os.makedirs(CORRELACION_INDIVIDUAL_DIR, exist_ok=True)
for sub in ["Sensores", "Actividad", "Ayuno", "Postprandial"]: # Crear subcarpetas específicas para el análisis grupal jerárquico
    os.makedirs(os.path.join(CORRELACION_INDIVIDUAL_DIR, "Matriz", sub), exist_ok=True)
for sub in ["Ayuno", "Postprandial"]:
    os.makedirs(os.path.join(CORRELACION_INDIVIDUAL_DIR, "CCF", sub), exist_ok=True)
CORRELACION_GRUPO_DIR = os.path.join(CORRELACION_DIR, "Grupos")
os.makedirs(CORRELACION_GRUPO_DIR, exist_ok=True)
for sub in ["Sensores", "Actividad", "Ayuno", "Postprandial"]:
    os.makedirs(os.path.join(CORRELACION_GRUPO_DIR, "Matriz", sub), exist_ok=True)
for sub in ["Ayuno", "Postprandial"]:
    os.makedirs(os.path.join(CORRELACION_GRUPO_DIR, "CCF", sub), exist_ok=True)
FILTRO_NOMBRE = "CGMacros"
PLOT_REF_BANDS = True
# 1. Variables para la Matriz Global de Control
VARS_GLOBAL = [
    "glucose_libre", 
    "glucose_dexcom", 
    "heart_rate", 
    "calories_activity", 
    "mets"
]
# 2. Variables para la Matriz Postprandial (Estricta con Deltas, sin Absolutos)
VARS_POSTPRANDIAL = [
    "carbs", "protein", "fat", "fiber", "calories",
    "libre_delta_30", "libre_delta_60",
    "dexcom_delta_30", "dexcom_delta_60"
]
# 3. Variables para la Matriz de Ayuno Unificada (Absolutos + Deltas de Fitbit)
VARS_AYUNO = [
    "glucose_dexcom",  # Usaremos este sensor de referencia para el ayuno puro
    "heart_rate", "heart_rate_delta_15",
    "calories_activity", "calories_accum_15",
    "mets", "mets_sustained_15"
]

ESTACIONARIEDAD_DIR = r"C:\Users\regin\Documents\GitHub\TFG-pruebas\Analysis\CGMacros\Estacionariedad"

STL_DIR = r"C:\Users\regin\Documents\GitHub\TFG-pruebas\Analysis\CGMacros\Descomposición STL"
STL_CSV_DIRS = {
    "tendencia": os.path.join(STL_DIR, "tendencia"),
    "estacionalidad": os.path.join(STL_DIR, "estacionalidad"),
    "residuo": os.path.join(STL_DIR, "residuo"),
}

RESIDUOS_DIR = r"C:\Users\regin\Documents\GitHub\TFG-pruebas\Analysis\CGMacros\Descomposición STL\residuo"
RESIDUOS_ESTACIONARIEDAD_DIR = r"C:\Users\regin\Documents\GitHub\TFG-pruebas\Analysis\CGMacros\Análisis Residuos\Estacionariedad"
RESIDUOS_ACF_DIR = r"C:\Users\regin\Documents\GitHub\TFG-pruebas\Analysis\CGMacros\Análisis Residuos\Autocorrelación"
RESIDUOS_PACF_DIR = r"C:\Users\regin\Documents\GitHub\TFG-pruebas\Analysis\CGMacros\Análisis Residuos\Autocorrelación Parcial"

RUTA_BIO_CSV = r"C:\Users\regin\Documents\UNIVERSIDAD\TFG\Datos Externos\CGMacros (PhysioNet)\Unzip\cgmacros-a-scientific-dataset-for-personalized-nutrition-and-diet-monitoring-1.0.0\CGMacros\bio.csv"

# Contenedor global unificado para almacenar los resultados de test de todos los experimentos
# El enfoque 'metabolico' nace como un subdiccionario listo para alojar las tres cohortes
REGISTRO_MODELOS = {
    'general': None,
    'individual': None,
    'metabolico': {
        'No Diabético': None,
        'Prediabético': None,
        'Diabético': None
    }
}
MODELIZACION_DIR = r"C:\Users\regin\Documents\GitHub\TFG-pruebas\Modeling\CGMacros\Modelos"
PREDICCIONES_DIR = r"C:\Users\regin\Documents\GitHub\TFG-pruebas\Modeling\CGMacros\Gráficas\Predicciones"
CLARKE_DIR = r"C:\Users\regin\Documents\GitHub\TFG-pruebas\Modeling\CGMacros\Gráficas\Clarke"

# Definición de rutas binarias para el experimento general poblacional
METRICAS_POB_PKL_DIR = os.path.join(MODELIZACION_DIR, "reportes_loso_poblacional.pkl")
CURVAS_POB_PKL_DIR = os.path.join(MODELIZACION_DIR, "curvas_loso_poblacional.pkl")
MODELOS_POB_PKL_DIR = os.path.join(MODELIZACION_DIR, "modelos_xgboost_poblacional.pkl")
MATRICES_POB_PKL_DIR = os.path.join(MODELIZACION_DIR, "matrices_train_poblacional.pkl")

METRICAS_PKL_DIR = r"C:\Users\regin\Documents\GitHub\TFG-pruebas\Modeling\CGMacros\Modelos\metricas_individuales.pkl"
CURVAS_PKL_DIR = r"C:\Users\regin\Documents\GitHub\TFG-pruebas\Modeling\CGMacros\Modelos\curvas_individuales.pkl"
MODELOS_PKL_DIR = r"C:\Users\regin\Documents\GitHub\TFG-pruebas\Modeling\CGMacros\Modelos\modelos_xgboost_individuales.pkl"
MATRICES_PKL_DIR = r"C:\Users\regin\Documents\GitHub\TFG-pruebas\Modeling\CGMacros\Modelos\matrices_train_individuales.pkl"

SHAP_DIR = r"C:\Users\regin\Documents\GitHub\TFG-pruebas\Modeling\CGMacros\SHAP"

In [ ]:
# =====================================
# DICCIONARIO DE ETIQUETADO METABÓLICO 'Prediabético' 'Diabético'
# =====================================

MAPEO_DIABETES = {
    'CGMacros-001': 'No Diabético',
    'CGMacros-002': 'No Diabético',
    'CGMacros-003': 'Diabético',
    'CGMacros-004': 'No Diabético',
    'CGMacros-005': 'Diabético',
    'CGMacros-006': 'No Diabético',
    'CGMacros-007': 'Prediabético',
    'CGMacros-008': 'Prediabético',
    'CGMacros-009': 'Prediabético',
    'CGMacros-010': 'Prediabético',
    'CGMacros-011': 'Prediabético',
    'CGMacros-012': 'Diabético',
    'CGMacros-013': 'Prediabético',
    'CGMacros-014': 'Diabético',
    'CGMacros-015': 'No Diabético',
    'CGMacros-016': 'Prediabético',
    'CGMacros-017': 'No Diabético',
    'CGMacros-018': 'No Diabético',
    'CGMacros-019': 'No Diabético',
    'CGMacros-020': 'Prediabético',
    'CGMacros-021': 'No Diabético',
    'CGMacros-022': 'Prediabético',
    'CGMacros-023': 'Prediabético',
    
    
    'CGMacros-026': 'Prediabético',
    'CGMacros-027': 'No Diabético',
    'CGMacros-028': 'Diabético',
    'CGMacros-029': 'Prediabético',
    'CGMacros-030': 'Diabético',
    'CGMacros-031': 'No Diabético',
    'CGMacros-032': 'No Diabético',
    'CGMacros-033': 'No Diabético',
    'CGMacros-034': 'No Diabético',
    'CGMacros-035': 'Diabético',
    'CGMacros-036': 'Diabético',

    'CGMacros-038': 'Diabético',
    'CGMacros-039': 'Diabético',

    'CGMacros-041': 'Prediabético',
    'CGMacros-042': 'Diabético',
    'CGMacros-043': 'Prediabético',
    'CGMacros-044': 'Prediabético',
    'CGMacros-045': 'Prediabético',
    'CGMacros-046': 'Diabético',
    'CGMacros-047': 'Diabético',
    'CGMacros-048': 'No Diabético',
    'CGMacros-049': 'Diabético'
}


## B. CARGA Y PREPROCESADO DE DATOS

Esta sección recoge la preparación de los datos desde su forma cruda hasta una estructura lista para el análisis. Aquí se define cómo se leen los archivos, cómo se normalizan las series temporales y cómo se preparan los resúmenes de calidad.


Contaremos con tres fases de procesado de datos:

- Raw: datos crudos sin ninguna alteración según los descargamos de las fuentes correspondientes.
- Staging: datos preprocesados para mantener una misma estructura y formato, pasados por un proceso estándar de limpieza y preprocesado, pero aún sin unirlos a datos de otras fuentes distintas. En este caso no será necesaria porque únicamente contamos con una fuente conjunta.
- Trusted: datos procesados y listos para el consumo, agrupando todas las fuentes y archivos de cada participante en una estructura única y reutilizable.


### B1. Cargar datos raw

Los datos de glucosa de cada participante fueron tomados mediante sensores cuya medición se realiza cada 15 minutos en el caso de Libre GL y cada 5 minutos en el caso de Dexcom GL. Sin embargo, los datos descargados vienen ya interpolados cada minuto.

En esta parte del notebook se cargan los archivos raw y se procesan para cada usuario con una referencia temporal uniforme. Posteriormente se prepara una lógica flexible que permite adaptar el intervalo de tiempo si en un futuro se requiere una mayor granularidad.

También se generan resúmenes diagnósticos antes y después de cada procesado, con métricas de calidad como el número de ceros y NaNs en cada fase, para facilitar el control del flujo de datos.


#### B1.1 Datos cada minuto

Dependiendo de cada participante encontramos diferencias en los datos en crudo. Realizamos una separación en subseries para que participantes que tengan muestras de datos en períodos temporales muy separados se consideren como subseries independientes.

Llevamos a cabo un proceso de normalización de variables estándar e incluimos algunas variables como el "user_id" de cada participante.

Para darle sentido fisiológico a los datos, convertimos los valores cero de variables como la glucosa o el ritmo cardiaco a NaNs, e imputamos pequeños gaps sin datos mediante interpolación lineal, imponiendo un máximo de registros faltantes para ser imputados sin alterar el comportamiento de las series originales. Para el resto de variables, cuando el significado fisiológico no se vea comprometido, imputamos ceros en los valores faltantes.


In [ ]:
##############################################################
#                      MAPEO COLUMNAS                        #
##############################################################

# Mapeo canónico de columnas
column_map = {
    'timestamp': 'time',
    'libre gl': 'glucose_libre',
    'dexcom gl': 'glucose_dexcom',
    'hr': 'heart_rate',
    'calories (activity)': 'calories_activity',
    'intensity': 'mets',
    'meal type': 'meal_type',
    'amount consumed': 'amount_consumed',
    'image path': 'image_path'
}

In [ ]:
##############################################################
#                     CARGA DE ARCHIVOS                      #
##############################################################

def cargar_csv(file_path):
    """
    Carga un archivo CSV y mantiene Timestamp como columna.

    Args:
        file_path (str): Ruta completa al archivo CSV.

    Returns:
        pd.DataFrame: DataFrame con los datos cargados, índice temporal.

    Ejemplo:
        >>> df = cargar_csv("Data/Raw/Glucosa/CIS-UCM0001P.csv")

    Recomendaciones:
        - El archivo debe tener columna 'Timestamp'.
    """
    df = pd.read_csv(file_path)
    df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
    if 'Timestamp' in df.columns:
        df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
    return df


##############################################################
#                 NORMALIZACIÓN Y FORMATEO                   #
##############################################################

def unificar_columnas(df):
    """
    Normaliza nombres de columnas: minúsculas, sin espacios, renombra según 'column_map'.
    Convierte Intensity a METs con un factor de conversión.

    Args:
        df (pd.DataFrame): DataFrame de entrada.

    Returns:
        pd.DataFrame: DataFrame con columnas unificadas.

    Ejemplo:
        >>> df = pd.DataFrame({' Glucemias ': [100, 110], 'Time': ['2024-01-01 00:00:00', '2024-01-01 01:00:00']})
        >>> df = unificar_columnas(df)
        >>> print(df.columns)
        Index(['glucose', 'time'], dtype='object')

    Recomendaciones:
        - Llamar siempre tras la carga para evitar errores por nombres inconsistentes.
    """
    df = df.copy()
    df.columns = [str(col).lower() for col in df.columns]

    if 'intensity' in df.columns:
        intensidad_a_mets = {0: 0, 1: 15, 2: 40, 3: 80}
        df['intensity'] = pd.to_numeric(df['intensity'], errors='coerce').replace(intensidad_a_mets)

    df = df.rename(columns=column_map)
    df = df.loc[:, ~df.columns.str.contains('^unnamed', case=False, regex=True)]
    return df


def normalizar_formato_time(df):
    """Convierte la columna time en DatetimeIndex."""
    df = df.copy()
    if 'time' in df.columns:
        df['time'] = pd.to_datetime(df['time'], errors='coerce')
        df = df.dropna(subset=['time']).set_index('time')
    elif not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("DataFrame debe tener 'time' como columna o DatetimeIndex.")

    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)

    df.index.name = 'time'
    df.index = df.index.round('1ms')
    return df

def normalizar_formato_time(df):
    """
    Asegura que el índice temporal sea tipo DatetimeIndex y lo normaliza a milisegundos.

    Args:
        df (pd.DataFrame): DataFrame con o sin columna 'time'.

    Returns:
        pd.DataFrame: DataFrame con índice temporal normalizado.

    Ejemplo:
        >>> df = pd.DataFrame({'time': ['2024-01-01 00:00:00.12345', '2024-01-01 01:00:00.6789'], 'glucose': [100, 110]})
        >>> df = normalizar_formato_time(df)
        >>> print(df.index)
        DatetimeIndex(['2024-01-01 00:00:00.123000', '2024-01-01 01:00:00.679000'], dtype='datetime64[ns]', name='time', freq=None)

    Recomendaciones:
        - Verifica que la columna 'time' existe antes de llamar a esta función.
    """
    if 'time' in df.columns:
        df['time'] = pd.to_datetime(df['time'], errors='coerce')
        df.set_index('time', inplace=True)
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError("El índice del DataFrame no es un DatetimeIndex.")
    df.index = pd.to_datetime(df.index)
    df.index = df.index.round('1ms')
    return df


##############################################################
#                     AÑADIR VARIABLES                       #
##############################################################

def extraer_user_id_de_nombre(file_path):
    """
    Extrae el identificador único de usuario a partir de la ruta o nombre de un archivo.

    Busca patrones de texto específicos mediante expresiones regulares dentro del 
    nombre del archivo (sin extensión). Si no encuentra ningún patrón conocido, 
    devuelve el nombre base completo del archivo como identificador por defecto.

    Args:
        file_path (str): Ruta completa o nombre del archivo a procesar.

    Returns:
        str: El identificador de usuario extraído o el nombre del archivo sin extensión.

    Ejemplos:
        >>> extraer_user_id_de_nombre("/datos/CGMacros-123_registro.csv")
        'CGMacros-123'
        >>> extraer_user_id_de_nombre("CIS-UCM2026P_glucosa.xlsx")
        'CIS-UCM2026P'
        >>> extraer_user_id_de_nombre("U54321_archivo.txt")
        'U54321'

    Recomendaciones:
        - Asegurar que los nombres de los archivos sigan estrictamente la nomenclatura 
          del proyecto para evitar identificadores incorrectos.
    """
    # === 1. Aislamiento del Nombre del Archivo ===
    # Extrae el nombre del archivo con su extensión eliminando la ruta de carpetas
    base_name = os.path.basename(file_path)
    
    # Separa el nombre de la extensión y se queda únicamente con el nombre base
    base_name_no_ext = os.path.splitext(base_name)[0]
    
    # === 2. Búsqueda de Patrones con Expresión Regular ===
    # El patrón busca tres formatos posibles aislados por el operador '|' (O):
    # - CGMacros- seguido de exactamente 3 dígitos
    # - CIS-UCM seguido de exactamente 4 dígitos y una letra 'P'
    # - Una letra 'U' seguida de uno o más dígitos
    match = re.search(r'(CGMacros-\d{3}|CIS-UCM\d{4}P|U\d+)', base_name_no_ext)
    
    # === 3. Resolución del Identificador ===
    if match:
        # Devuelve el texto exacto que coincidió con el patrón
        return match.group(1)
        
    # Si no hay coincidencia, se asume el nombre del archivo como ID temporal
    return base_name_no_ext


def añadir_user_id(df, file_path):
    """
    Añade una nueva columna al DataFrame con el identificador de usuario extraído.

    Esta función delega la extracción del ID a 'extraer_user_id_de_nombre' y genera 
    una copia del DataFrame original para insertar la nueva columna sin alterar 
    los datos fuente por referencia.

    Args:
        df (pd.DataFrame): Matriz de datos original a la que se vinculará el usuario.
        file_path (str): Ruta o nombre del archivo del cual se extraerá el identificador.

    Returns:
        pd.DataFrame: Un nuevo DataFrame que incluye la columna adicional 'user_id'.

    Ejemplos:
        >>> import pandas as pd
        >>> df_inicial = pd.DataFrame({'glucose': [95, 102]})
        >>> df_final = añadir_user_id(df_inicial, "U123_data.csv")
        >>> print(df_final)
           glucose user_id
        0       95    U123
        1      102    U123

    Recomendaciones:
        - Utilizar esta función justo después de leer el archivo para mantener la 
          trazabilidad de los datos desde el inicio del script.
        - Evitar pasar estructuras de datos vacías para prevenir incoherencias en el índice.
    """
    # === 1. Obtención del Identificador ===
    user_id = extraer_user_id_de_nombre(file_path)
    
    # === 2. Clonación y Asignación Segura ===
    # Se crea una copia explícita para proteger el DataFrame original de modificaciones
    df = df.copy()
    
    # Se crea o sobrescribe la columna 'user_id' con el identificador obtenido
    df['user_id'] = user_id
    
    return df


##############################################################
#            CEROS A NAN EN VARIABLES FISIOLÓGICAS           #
##############################################################

def ceros_a_nan(df, columnas):
    """
    Convierte los valores 0 en las columnas indicadas a NaN.

    Args:
        df (pd.DataFrame): DataFrame de entrada.
        columnas (list): Lista de columnas a limpiar.

    Returns:
        pd.DataFrame: DataFrame con ceros convertidos a NaN en las columnas indicadas.

    Ejemplo:
        >>> df = pd.DataFrame({'glucose': [0, 100, 0, 120], 'heart_rate': [0, 70, 80, 0]})
        >>> df = ceros_a_nan(df, ['glucose', 'heart_rate'])
        >>> print(df)
           glucose  heart_rate
        0      NaN         NaN
        1    100.0        70.0
        2      NaN        80.0
        3    120.0         NaN

    Recomendaciones:
        - Útil para limpiar ceros imposibles en variables fisiológicas.
    """
    df = df.copy()
    for col in columnas:
        if col in df.columns:
            df[col] = df[col].replace([0], [np.nan])
    return df


##############################################################
#                   INTERPOLACIÓN (LINEAL)                   #
##############################################################

def interpolar_lineal_limited(df, col, max_gap=3):
    """
    Imputa huecos consecutivos (hasta max_gap) con interpolación lineal.

    Args:
        df (pd.DataFrame): DataFrame de entrada.
        col (str): Nombre de la columna a imputar.
        max_gap (int): Tamaño máximo de gap a rellenar.

    Returns:
        pd.Series: Serie con huecos pequeños interpolados.

    Ejemplo:
        >>> df = pd.DataFrame({'glucose': [100, None, None, 130, 140]})
        >>> df['glucose'] = interpolar_lineal_limited(df, 'glucose', max_gap=2)
        >>> print(df['glucose'].tolist())
        [100.0, 110.0, 120.0, 130.0, 140.0]
    """
    return df[col].interpolate(method='linear', limit=max_gap, limit_direction='both')


def imputar_valores_faltantes(df):
    """
    Imputa valores faltantes:
    - 'glucose_dexcom', 'glucose_libre' y 'heart_rate': interpolación lineal para huecos pequeños.
    - Otras variables se podrían imputar con cero.

    Args:
        df (pd.DataFrame): DataFrame de entrada.

    Returns:
        pd.DataFrame: DataFrame con valores imputados.

    Ejemplo:
        >>> df = pd.DataFrame({'glucose': [100, None, None, 130], 'steps': [None, 2, 3, None]})
        >>> df = imputar_valores_faltantes(df)
        >>> print(df)
           glucose  steps
        0    100.0    0.0
        1    110.0    2.0
        2    120.0    3.0
        3    130.0    0.0
    """
    df = df.copy()
    for col in ['heart_rate']:
        if col in df.columns:
            df[col] = interpolar_lineal_limited(df, col, max_gap=3)
    for col in ['glucose_dexcom']:
        if col in df.columns:
            df[col] = interpolar_lineal_limited(df, col, max_gap=15)
    for col in ['glucose_libre']:
        if col in df.columns:
            df[col] = interpolar_lineal_limited(df, col, max_gap=45)
    for col in []:
        if col in df.columns:
            df[col] = df[col].fillna(0)
    return df


##############################################################
#                 SEPARACIÓN EN SUBSERIES                    #
##############################################################

def dividir_por_faltantes(df, columnas, max_gap, min_valid_period):
    """
    Divide una serie temporal en subseries, cortando donde hay gaps mayores a max_gap en una lista de columnas.

    Args:
        df (pd.DataFrame): DataFrame de entrada.
        columnas (list): Lista de columnas a analizar para gaps.
        max_gap (int): Máximo gap permitido antes de dividir.
        min_valid_period (int): Mínimo de registros válidos para aceptar una subserie.

    Returns:
        dict: Diccionario {nombre_subserie: DataFrame_subserie}.

    Ejemplo:
        >>> df = pd.DataFrame({'glucose': [100, None, None, 130, None, None, None, 140, 150]})
        >>> subseries = dividir_por_faltantes(df, ['glucose'], max_gap=2, min_valid_period=2)
        >>> for k, v in subseries.items():
        ...     print(k, v['glucose'].tolist())
        serie1 [100.0, nan, nan, 130.0]
        serie2 [140.0, 150.0]

    Recomendaciones:
        - Usar tras normalizar e imputar los huecos pequeños.
        - Ajustar min_valid_period para evitar subseries demasiado cortas.
    """
    if isinstance(columnas, str):
        columnas = [columnas]
    columnas = [col for col in columnas if col in df.columns]
    if not columnas:
        return {'serie1': df.copy()}

    df = df.sort_index().copy()
    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("dividir_por_faltantes requiere un DatetimeIndex en 'time'.")

    gap = df.index.to_series().diff().gt(pd.Timedelta(minutes=max_gap)).fillna(False)
    nan_mask = df[columnas].isna().any(axis=1)
    corte = (gap & nan_mask).fillna(False)
    grupos = corte.cumsum()

    subseries = {}
    serie_id = 1
    for _, bloque in df.groupby(grupos):
        if bloque[columnas].notna().all(axis=1).sum() >= min_valid_period:
            subseries[f'serie{serie_id}'] = bloque.copy()
            serie_id += 1
    if not subseries:
        subseries['serie1'] = df.copy()
    return subseries


def dividir_por_usuario_y_faltantes(df, user_col, data_cols, max_gap, min_valid_period):
    """
    Divide el DataFrame en subseries por usuario y por gaps en una columna.

    Args:
        df (pd.DataFrame): DataFrame original.
        user_col (str): Columna de usuario.
        data_cols (list): Columnas a analizar para gaps.
        max_gap (int): Máximo gap permitido antes de dividir.
        min_valid_period (int): Mínimo registros válidos para nueva subserie.

    Returns:
        dict: Diccionario {nombre_subserie: DataFrame_subserie}.

    Ejemplo:
        >>> df = pd.DataFrame({'user_id': ['A', 'A', 'A', 'B', 'B'], 'glucose': [1, None, 3, 4, None]})
        >>> subseries = dividir_por_usuario_y_faltantes(df, 'user_id', ['glucose'], 1, 1)
        >>> print(list(subseries.keys()))
        ['A_serie1', 'B_serie1']

    Recomendaciones:
        - Útil para analizar la continuidad de datos por usuario.
    """
    all_subseries = {}
    users = df[user_col].dropna().unique() if user_col in df.columns else []
    for user_id in users:
        user_data = df[df[user_col] == user_id]
        subseries = dividir_por_faltantes(user_data, data_cols, max_gap, min_valid_period)
        for sub_name, sub_df in subseries.items():
            all_subseries[f'{user_id}_{sub_name}'] = sub_df
    return all_subseries


def dividir_subseries(df, file_path, data_cols=None, max_gap=12, min_valid=24):
    """
    Divide el DataFrame en subseries. Si solo hay una subserie, usa el nombre base.

    Args:
        df (pd.DataFrame): DataFrame procesado.
        file_path (str): Ruta del archivo original.
        data_cols (list): Lista de columnas para detectar huecos.
        max_gap (int): Gap máximo para dividir.
        min_valid (int): Mínimo registros válidos para subserie.

    Returns:
        dict: Diccionario {nombre_subserie: DataFrame_subserie}.

    Ejemplo:
        >>> df = pd.DataFrame({'user_id': ['A']*5, 'glucose': [1, None, None, 4, 5]})
        >>> subseries = dividir_subseries(df, "A.csv", data_cols=["glucose"], max_gap=1, min_valid=2)
        >>> print(list(subseries.keys()))
        ['A']  # si solo hay una subserie

    Recomendaciones:
        - Así, si solo hay una subserie, el archivo guardado será el nombre base.
    """
    user_id = df['user_id'].iloc[0] if 'user_id' in df.columns else extraer_user_id_de_nombre(file_path)
    if data_cols is None:
        data_cols = [col for col in ['glucose_dexcom', 'glucose_libre'] if col in df.columns]
        if not data_cols and 'glucose' in df.columns:
            data_cols = ['glucose']

    subseries = dividir_por_usuario_y_faltantes(
        df,
        'user_id' if 'user_id' in df.columns else df.index.name or 'user_id',
        data_cols,
        max_gap,
        min_valid,
    )

    if len(subseries) == 1:
        return {user_id: list(subseries.values())[0]}
    if len(subseries) > 1:
        return subseries
    return {user_id: df}


##############################################################
#                 RESÚMENES DIAGNÓSTICOS                     #
##############################################################

def generar_resumen_diagnostico(df, file_path):
    """
    Genera un resumen diagnóstico del DataFrame.

    Args:
        df (pd.DataFrame): DataFrame a resumir.
        file_path (str): Ruta o nombre de archivo (para identificar).

    Returns:
        dict: Resumen con métricas de calidad de datos.

    Ejemplo:
        >>> df = pd.DataFrame({'glucose': [100, None, 0], 'os': [1, 1, 1], 'steps': [0, 2, 0]})
        >>> resumen = generar_resumen_diagnostico(df, "U10.csv")
        >>> print(resumen)
        {'archivo': 'U10.csv', 'os': 1, 'rows': 3, 'nulos_glucose': 1, 'ceros_glucose': 1, ...}

    Recomendaciones:
        - Útil para comparar la calidad de los datos antes y después de la limpieza.
    """
    resultado = {
        'archivo': os.path.basename(file_path),
        'filas': len(df),
    }
    for col in df.columns:
        resultado[f'{col}_nans'] = df[col].isna().sum()
    cols_fisio = ['glucose_libre', 'glucose_dexcom', 'heart_rate', 'calories_activity', 'mets']
    for col in cols_fisio:
        if col in df.columns:
            resultado[f'{col}_ceros'] = (df[col] == 0).sum()
    return resultado


##############################################################
#              FUNCIÓN PRINCIPAL DE PROCESADO                #
##############################################################

def procesar_archivo_csv(file_path):
    """
    Procesa un archivo CSV:
      1. Carga y normaliza datos.
      2. Añade metadatos y genera resumen inicial.
      3. Divide en subseries (si aplica).
      4. Convierte ceros a NaN en variables fisiológicas en cada subserie.
      5. Imputa/interpola en cada subserie.
      6. Guarda cada subserie limpia e imputada.
      7. Genera resúmenes diagnósticos finales.

    Args:
        file_path (str): Ruta del archivo CSV original.

    Returns:
        tuple: (resumen_pre, resumenes_final)

    Ejemplo:
        >>> resumen_pre, resumenes_final = procesar_archivo_csv("Data/Raw/Glucosa/U10.csv")
        >>> print(resumen_pre)
        >>> print(resumenes_final)

    Recomendaciones:
        - Llama esta función desde el pipeline principal sobre cada archivo raw.
        - Los archivos de salida estarán ya limpios y listos para trusted/extensión.
    """
    # 1. Cargar y normalizar
    data = cargar_csv(file_path)
    data = unificar_columnas(data)
    data = normalizar_formato_time(data)
    data = añadir_user_id(data, file_path)

    # 2. Resumen diagnóstico inicial (raw, antes de imputar)
    resumen_pre = generar_resumen_diagnostico(data, file_path)

    # 3. Dividir en subseries (usando solo los NaN originales)
    gap_threshold = 240
    min_valid_duration = 120
    data_cols = [col for col in ['glucose_dexcom', 'glucose_libre'] if col in data.columns]

    subseries = dividir_subseries(
        df=data,
        file_path=file_path,
        data_cols=data_cols,
        max_gap=gap_threshold,
        min_valid=min_valid_duration,
    )

    resumenes_final = []
    os.makedirs(TRUSTED_DIR, exist_ok=True)

    # 4. Para cada subserie: convertir ceros a NaN, imputar/interpolar, guardar y resumir
    for nombre, sub_df in subseries.items():
        sub_df = ceros_a_nan(sub_df, ['glucose_libre', 'glucose_dexcom', 'heart_rate', 'calories_activity', 'mets'])
        sub_df = imputar_valores_faltantes(sub_df)
        user_id = sub_df['user_id'].iloc[0] if 'user_id' in sub_df.columns else extraer_user_id_de_nombre(file_path)
        nombre_archivo = f'{user_id}.csv' if nombre == user_id else f'{nombre}.csv'
        salida_path = os.path.join(TRUSTED_DIR, nombre_archivo)
        sub_df.to_csv(salida_path, index=True, date_format='%Y-%m-%d %H:%M:%S.%f', na_rep='NaN')
        resumenes_final.append(generar_resumen_diagnostico(sub_df, nombre))

    return resumen_pre, resumenes_final


##############################################################
#                       PIPELINE PRINCIPAL                   #
##############################################################

def main():
    """
    Ejecuta el pipeline completo:
      - Procesa todos los archivos CSV de la carpeta RAW.
      - Genera y guarda resúmenes diagnósticos iniciales y finales.

    Returns:
        None

    Ejemplo:
        >>> main()

    Recomendaciones:
        - Los archivos procesados se guardan en la carpeta de trusted.
        - El resumen de diagnóstico permite auditar la calidad de los datos.
    """
    csv_files = []
    for name in sorted(os.listdir(RAW_DIR)):
        participant_dir = os.path.join(RAW_DIR, name)
        if os.path.isdir(participant_dir):
            files = glob.glob(os.path.join(participant_dir, '*.csv'))
            if not files:
                continue
            for f in sorted(files):
                csv_files.append(f)

    resumenes_pre = []
    resumenes_post = []
    for file_path in csv_files:
        try:
            resumen_pre, resumenes_final = procesar_archivo_csv(file_path)
            resumenes_pre.append(resumen_pre)
            resumenes_post.extend(resumenes_final)
            user_id = os.path.splitext(os.path.basename(file_path))[0]
            print(f'[OK] Usuario {user_id} - csv cargado y procesado')
        except Exception as e:
            print(f'Error procesando {file_path}: {e}')

    pd.DataFrame(resumenes_pre).to_csv(RESUMEN_RAW, index=False)
    print(f'\nResumen de diagnóstico de datos raw guardado en: {RESUMEN_RAW}')
    pd.DataFrame(resumenes_post).to_csv(RESUMEN_TRUSTED, index=False)
    print(f'Resumen de diagnóstico de datos trusted guardado en: {RESUMEN_TRUSTED}')


In [ ]:
if __name__ == "__main__":
    main()

Comprobamos los resúmenes diagnósticos tanto de los datos raw como de los datos en trusted para ver que los procesos anteriores se han realizado correctamente.

In [ ]:
try:
    resumen_raw = pd.read_csv(RESUMEN_RAW)
except (FileNotFoundError, pd.errors.EmptyDataError):
    print(f"El fichero de resumen raw está vacío o no existe: {RESUMEN_RAW}")
    resumen_raw = pd.DataFrame()

resumen_raw

In [ ]:
try:
    RESUMEN_TRUSTED = pd.read_csv(RESUMEN_TRUSTED)
except (FileNotFoundError, pd.errors.EmptyDataError):
    print(f"El fichero de resumen trusted está vacío o no existe: {RESUMEN_TRUSTED}")
    RESUMEN_TRUSTED = pd.DataFrame()

RESUMEN_TRUSTED

Vemos que los archivos en trusted ya no presentan ceros en variables fisiológicas importantes como la glucosa o el ritmo cardiaco. Sin embargo, algunas de estas variables sí que tienen NaNs, que es algo que en adelante tendremos que tener en cuenta al realizar diferentes procesos y análisis.

Vemos que en 'mets' el número de datos por participante tiene gran variabilidad: en algunos participantes es muy bajo, por lo que probablemente no serán datos que podamos utilizar como base en nuestros análisis posteriores, y en otros participantes parece tener un registro completo.

#### B1.2 Gráficas raw vs trusted

Vamos a ver gráficamente los datos en crudo y la version de datos en fase trusted, ambas cada minuto.

In [ ]:
##############################################################
#                  GRÁFICAS RAW VS TRUSTED                   #
##############################################################

def graficar_glucosa_participante(user_id, variable="glucose_dexcom", show_extended=False):
    """
    Compara una variable entre raw, trusted y, opcionalmente, trusted extendido para un participante.

    Args:
        user_id (str): Identificador del participante.
        variable (str): Nombre de la variable a graficar.
        show_extended (bool): Si True, intenta cargar y graficar los CSV extendidos.
    """
    fig, ax = plt.subplots(figsize=(12, 6))

    graficado = False

    def cargar_csv_temporal(ruta_csv):
        """Carga un CSV usando la columna temporal disponible ('time' o 'Timestamp')."""
        df = pd.read_csv(ruta_csv)
        if "time" in df.columns:
            df["time"] = pd.to_datetime(df["time"], errors="coerce")
            df = df.dropna(subset=["time"]).set_index("time")
        elif "Timestamp" in df.columns:
            df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")
            df = df.dropna(subset=["Timestamp"]).set_index("Timestamp")
        else:
            raise ValueError("No se encontró una columna temporal compatible ('time' o 'Timestamp').")
        return df

    # 1. RAW: buscar el CSV del participante dentro de su carpeta
    raw_dir_participante = os.path.join(RAW_DIR, user_id)
    raw_files = sorted(glob.glob(os.path.join(raw_dir_participante, "*.csv")))
    for raw_file in raw_files:
        try:
            df_raw = cargar_csv(raw_file)
            df_raw = unificar_columnas(df_raw)
            df_raw = normalizar_formato_time(df_raw)
            if variable in df_raw.columns and not df_raw[variable].isnull().all():
                ax.plot(
                    df_raw.index,
                    df_raw[variable],
                    label=f"Raw: {os.path.basename(raw_file)}",
                    linewidth=3,
                    alpha=0.7,
                )
                graficado = True
        except Exception as e:
            print(f"No se pudo graficar RAW {raw_file}: {e}")

    # 2. TRUSTED: una o varias subseries del participante
    trusted_files = sorted(glob.glob(os.path.join(TRUSTED_DIR, f"{user_id}*.csv")))
    for trusted_file in trusted_files:
        try:
            df_trusted = cargar_csv_temporal(trusted_file)
            if variable in df_trusted.columns and not df_trusted[variable].isnull().all():
                ax.plot(
                    df_trusted.index,
                    df_trusted[variable],
                    label=f"Trusted: {os.path.basename(trusted_file)}",
                    linewidth=2,
                    alpha=0.8,
                )
                graficado = True
        except Exception as e:
            print(f"No se pudo graficar trusted {trusted_file}: {e}")

    # 3. EXTENDIDO (opcional): solo si show_extended=True
    if show_extended:
        extendidos_files = sorted(glob.glob(os.path.join(EXTENDED_TRUSTED_DIR, f"{user_id}*.csv")))
        colores = ["green", "red", "purple", "brown", "gray"]
        for i, ext_file in enumerate(extendidos_files):
            try:
                df_ext = cargar_csv_temporal(ext_file)
                if variable in df_ext.columns and not df_ext[variable].isnull().all():
                    nombre = os.path.basename(ext_file).replace(".csv", "")
                    ax.plot(
                        df_ext.index,
                        df_ext[variable],
                        label=f"Extendida: {nombre}",
                        linewidth=1.8,
                        linestyle="--",
                        color=colores[i % len(colores)],
                        alpha=0.75,
                    )
                    graficado = True
            except Exception as e:
                print(f"No se pudo graficar EXTENDIDO {ext_file}: {e}")

    if not graficado:
        print(f"No se encontraron datos válidos para graficar: {user_id} - {variable}")
        plt.close(fig)
        return

    ax.set_title(f"{user_id} - {variable}")
    ax.set_xlabel("Tiempo")
    ax.set_ylabel(variable)
    ax.legend()
    fig.tight_layout()
    mostrar_y_liberar_figura(fig)

# Ejemplo de uso:
# graficar_glucosa_participante("CGMacros-001", variable="glucose_dexcom")
# graficar_glucosa_participante("CGMacros-002", variable="glucose_dexcom")
# Para mostrar extendida: graficar_glucosa_participante("CGMacros-001", show_extended=True)

Graficamos el caso concreto de uno de los participantes (mejor si antes tenía un elevado número de ceros y/o NaNs), y vemos las diferencias entre las versiones de los datos.

In [ ]:
user_id = "CGMacros-001"

variables = ['glucose_dexcom', 'glucose_libre', 'heart_rate']
for variable in variables:
    g = graficar_glucosa_participante(user_id, variable=variable)
    plt.title(f"{user_id} - {variable}")

In [ ]:
plt.close('all')

### B2. Unificar por fenotipo metabólico

In [ ]:
def _cargar_y_unificar_dataset(input_dir):
    """
    Carga todos los archivos CSV de un directorio unificándolos en un único DataFrame etiquetado.

    Escanea el directorio de entrada, procesa individualmente cada archivo aplicando 
    limpieza temporal, inyecta el identificador de usuario basado en el nombre del fichero 
    y asigna la etiqueta del grupo metabólico correspondiente mediante una consulta global.

    Args:
        input_dir (str): Ruta del directorio del sistema donde se encuentran los archivos CSV.

    Returns:
        pd.DataFrame: Conjunto de datos consolidado, ordenado cronológicamente y configurado 
            con la columna 'time' como índice principal.

    Raises:
        FileNotFoundError: Si el directorio especificado no contiene ningún archivo con extensión .csv.

    Recomendaciones:
        - Asegurar que el diccionario global 'MAPEO_DIABETES' esté definido previamente en el 
          Notebook para evitar fallos de mapeo o asignaciones por defecto ('No Etiquetado').
        - Verificar que los nombres de los archivos CSV coincidan exactamente con las llaves 
          de identificación utilizadas en el diccionario de mapeo.
    """
    # === 1. Búsqueda y Validación de Archivos ===
    archivos_csv = glob.glob(os.path.join(input_dir, "*.csv"))
    if not archivos_csv:
        raise FileNotFoundError(f"No se encontraron archivos CSV en {input_dir}")
        
    lista_dfs = []
    
    # === 2. Bucle de Procesamiento Individual ===
    for ruta in archivos_csv:
        nombre_completo = os.path.basename(ruta)
        
        # Aislamiento del identificador de usuario: se extrae la cadena de texto 
        # base eliminando la extensión '.csv' mediante el índice de splitext
        user_id = os.path.splitext(nombre_completo)[0]
        
        # Lectura y depuración de la serie temporal del fichero actual
        df_individual = pd.read_csv(ruta)
        df_individual['time'] = pd.to_datetime(df_individual['time'], errors='coerce')
        df_individual = df_individual.dropna(subset=['time']).sort_values('time')
        
        # === 3. Inyección de Metadatos y Clasificación ===
        # Propagación del string 'user_id' en todas las filas del sub-DataFrame
        df_individual['user_id'] = user_id
        
        # Asociación del estado clínico consultando el diccionario global
        df_individual['grupo_metabolico'] = MAPEO_DIABETES.get(user_id, 'No Etiquetado')
        lista_dfs.append(df_individual)
        
    # === 4. Consolidación e Indexación Final ===
    # Fusión de todas las estructuras limpias en una matriz de datos global
    df_global = pd.concat(lista_dfs, ignore_index=True)
    
    return df_global.set_index('time').sort_index()

## C. ANÁLISIS DE DATOS

Esta parte del notebook transforma los datos procesados en descripciones, visualizaciones y comparaciones útiles para entender el comportamiento de la glucosa, la actividad y la alimentación.


### C1. Análisis Exploratorio (EDA)

El objetivo de esta sección es describir el comportamiento de las variables principales antes de pasar a modelos o inferencias más complejas.


#### C1.1 Glucosa y ritmo cardíaco

##### C1.1.1 Gráficas trusted

Este bloque revisa de forma visual las series temporales ya procesadas para detectar patrones claros y valorar la calidad del preprocesado.


Para graficar adecuadamente los datos debemos aplicamos un filtro sobre ceros y los NaNs, para evitar problemas con ventanas temporales que puedan quedarse sin graficar.

Incluiremos un sombreado que muestre los momentos de ejercicio (heart_rate superior a 160) y una ventana posterior (8 horas por defecto). Otra opción a desarrollar es usar para la definición de ejercicio las calories_activity o mets.

In [ ]:
# VERSIÓN CON DOS EJES NO ALINEADOS
'''
def graficar_glucosa_hr_trusted(
    archivo_master,
    output_folder,
    titulo=None,
    sombrear_ejercicio=True,
    sombrear_postejercicio_horas=8,
    figsize=(10, 6)
):
    """
    Grafica glucose_dexcom, glucose_libre y heart_rate de un archivo trusted, filtrando ceros y NaNs,
    y sombreando los bloques continuos de ejercicio (gris) y el post-ejercicio (gris claro).
    El nivel cero de ambos ejes queda alineado.
    
    Args:
        archivo_master (str): Ruta al archivo trusted del usuario.
        titulo (str, opcional): Título del gráfico. Si None, usa el nombre del archivo.
        sombrear_ejercicio (bool, opcional): Si True, añade sombreado en periodos de ejercicio.
        sombrear_postejercicio_horas (int, opcional): Horas a sombrear tras cada bloque de ejercicio.
        figsize (tuple, opcional): Tamaño de la figura.

    Returns:
        None
    """
    master = pd.read_csv(archivo_master, parse_dates=['time'])

    for col in ['glucose_dexcom', 'glucose_libre', 'heart_rate']:
        if col in master.columns:
            master[col] = master[col].astype(float)

    fig, ax1 = plt.subplots(figsize=figsize)

    # Sombrear bloques de ejercicio y post-ejercicio, y guardar handles para la leyenda
    legend_handles = []
    heart_rate_threshold = 160  # Umbral para detectar ejercicio
    if sombrear_ejercicio and 'heart_rate' in master.columns:
        master = master.sort_values('time')
        mask = (master['heart_rate'] > heart_rate_threshold)  
        mask_shift = mask.shift(1, fill_value=False)
        start_blocks = master['time'][(mask) & (~mask_shift)].reset_index(drop=True)
        end_blocks = master['time'][(mask_shift) & (~mask)].reset_index(drop=True)
        if len(end_blocks) < len(start_blocks):
            end_blocks = pd.concat([end_blocks, pd.Series(master['time'].iloc[-1:])]).reset_index(drop=True)
        for start, end in zip(start_blocks, end_blocks):
            ax1.axvspan(start, end, color='gray', alpha=0.2)
            post_end = end + pd.Timedelta(hours=sombrear_postejercicio_horas)
            ax1.axvspan(end, post_end, color='lightgray', alpha=0.15)
        legend_handles.append(mpatches.Patch(color='gray', alpha=0.2, label='Ejercicio'))
        legend_handles.append(mpatches.Patch(color='lightgray', alpha=0.15, label=f'Post-ejercicio: {sombrear_postejercicio_horas}h'))

    # Glucosa Dexcom y Libre (sin ceros ni NaN)
    if 'glucose_dexcom' in master.columns:
        glucose_dexcom_filtered = master[(master['glucose_dexcom'] != 0) & (~master['glucose_dexcom'].isna())]
        line_glucose_dexcom, = ax1.plot(glucose_dexcom_filtered['time'], glucose_dexcom_filtered['glucose_dexcom'], color='tab:blue', label='Glucosa Dexcom')
        ax1.tick_params(axis='y', labelcolor='tab:blue')
    if 'glucose_libre' in master.columns:
        glucose_libre_filtered = master[(master['glucose_libre'] != 0) & (~master['glucose_libre'].isna())]
        line_glucose_libre, = ax1.plot(glucose_libre_filtered['time'], glucose_libre_filtered['glucose_libre'], color='tab:cyan', label='Glucosa Libre')
        ax1.tick_params(axis='y', labelcolor='tab:cyan')
    ax1.set_xlabel('Tiempo')

    # --- FIJAR EL LÍMITE INFERIOR PARA EL EJE IZQUIERDO ---
    ax1.set_ylim(bottom=20)

    # Heart Rate (sin ceros ni NaN)
    ax2 = ax1.twinx()
    lines_hr = []
    if 'heart_rate' in master.columns:
        heart_rate_filtered = master[(master['heart_rate'] != 0) & (~master['heart_rate'].isna())]
        line_hr, = ax2.plot(heart_rate_filtered['time'], heart_rate_filtered['heart_rate'], color='tab:red', alpha=0.6, label='Heart Rate')
        lines_hr.append(line_hr)
    ax2.set_ylabel('Heart Rate', color='tab:red')
    ax2.tick_params(axis='y', labelcolor='tab:red')

    # --- FIJAR EL LÍMITE INFERIOR PARA EL EJE DERECHO ---
    ax2.set_ylim(bottom=20)

    # Leyenda combinada
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    handles = lines1 + lines_hr + legend_handles
    labels = labels1 + labels2 + [h.get_label() for h in legend_handles]
    ax2.legend(handles, labels, loc='upper left')

    # Título con el nombre del archivo si no se pasa explícito
    if titulo is None:
        nombre_archivo = os.path.basename(archivo_master).replace('.csv', '')
        titulo = f"{nombre_archivo} - Glucosa Dexcom, Glucosa Libre y Heart Rate vs Tiempo"
    plt.title(titulo)
    fig.tight_layout()
    plt.show()

    # Guardado de la gráfica generada
    os.makedirs(output_folder, exist_ok=True)  # Crea la carpeta si no existe
    nombre_salida = f"{nombre_archivo}_grafico_glucosa_hr.png"
    ruta_salida = os.path.join(output_folder, nombre_salida)
    plt.savefig(ruta_salida, dpi=300)  
    print(f" Gráfica guardada con éxito en: {ruta_salida}")
    '''


In [ ]:
def graficar_glucosa_hr_trusted(
    archivo_master,
    output_folder,
    titulo=None,
    sombrear_ejercicio=True,
    sombrear_postejercicio_horas=8,
    figsize=(10, 6)
):
    """
    Grafica glucose_dexcom, glucose_libre y heart_rate compartiendo un único
    eje vertical a la izquierda, ajustando dinámicamente los límites verticales.

    Args:
        archivo_master (str): Ruta al archivo trusted del usuario.
        titulo (str, opcional): Título del gráfico. Si None, usa el nombre del archivo.
        sombrear_ejercicio (bool, opcional): Si True, añade sombreado en periodos de ejercicio.
        sombrear_postejercicio_horas (int, opcional): Horas a sombrear tras cada bloque de ejercicio.
        figsize (tuple, opcional): Tamaño de la figura.

    Returns:
        None
    """
    master = pd.read_csv(archivo_master, parse_dates=['time'])

    for col in ['glucose_dexcom', 'glucose_libre', 'heart_rate']:
        if col in master.columns:
            master[col] = master[col].astype(float)

    fig, ax1 = plt.subplots(figsize=figsize)

    # Sombrear bloques de ejercicio y post-ejercicio
    legend_handles = []
    heart_rate_threshold = 160
    if sombrear_ejercicio and 'heart_rate' in master.columns:
        master = master.sort_values('time')
        mask = (master['heart_rate'] > heart_rate_threshold)
        mask_shift = mask.shift(1, fill_value=False)
        start_blocks = master['time'][(mask) & (~mask_shift)].reset_index(drop=True)
        end_blocks = master['time'][(mask_shift) & (~mask)].reset_index(drop=True)
        if len(end_blocks) < len(start_blocks):
            end_blocks = pd.concat([end_blocks, pd.Series(master['time'].iloc[-1:])]).reset_index(drop=True)
        for start, end in zip(start_blocks, end_blocks):
            ax1.axvspan(start, end, color='gray', alpha=0.2)
            post_end = end + pd.Timedelta(hours=sombrear_postejercicio_horas)
            ax1.axvspan(end, post_end, color='lightgray', alpha=0.15)
        legend_handles.append(mpatches.Patch(color='gray', alpha=0.2, label='Ejercicio'))
        legend_handles.append(mpatches.Patch(color='lightgray', alpha=0.15, label=f'Post-ejercicio: {sombrear_postejercicio_horas}h'))

    # --- AJUSTAR LÍMITES DINÁMICOS PARA UN SOLO EJE ---
    valores_y = []
    for col in ['glucose_dexcom', 'glucose_libre', 'heart_rate']:
        if col in master.columns:
            valores_y.extend(master.loc[master[col] > 0, col].dropna().tolist())

    if valores_y:
        y_min = min(valores_y) * 0.95
        y_max = max(valores_y) * 1.05
        ax1.set_ylim(y_min, y_max)

    # Graficar Glucosa Dexcom y Libre en ax1
    if 'glucose_dexcom' in master.columns:
        glucose_dexcom_filtered = master[(master['glucose_dexcom'] != 0) & (~master['glucose_dexcom'].isna())]
        ax1.plot(glucose_dexcom_filtered['time'], glucose_dexcom_filtered['glucose_dexcom'], color='tab:blue', label='Glucosa Dexcom')
    if 'glucose_libre' in master.columns:
        glucose_libre_filtered = master[(master['glucose_libre'] != 0) & (~master['glucose_libre'].isna())]
        ax1.plot(glucose_libre_filtered['time'], glucose_libre_filtered['glucose_libre'], color='tab:cyan', label='Glucosa Libre')

    # Graficar Heart Rate también en ax1 (mismo eje)
    if 'heart_rate' in master.columns:
        heart_rate_filtered = master[(master['heart_rate'] != 0) & (~master['heart_rate'].isna())]
        ax1.plot(heart_rate_filtered['time'], heart_rate_filtered['heart_rate'], color='tab:red', alpha=0.6, label='Heart Rate')

    # Configuración de etiquetas del eje único
    ax1.set_xlabel('Tiempo')
    ax1.set_ylabel('Unidades (mg/dL y bpm)', color='black')
    ax1.tick_params(axis='y', labelcolor='black')
    ax1.grid(True, linestyle='--', alpha=0.5)

    # Leyenda unificada
    lines, labels = ax1.get_legend_handles_labels()
    handles = lines + legend_handles
    all_labels = labels + [h.get_label() for h in legend_handles]
    ax1.legend(handles, all_labels, loc='upper left')

    # Título dinámico
    nombre_archivo = os.path.basename(archivo_master).replace('.csv', '')
    if titulo is None:
        titulo = f"{nombre_archivo} - Glucosa Dexcom, Glucosa Libre y Heart Rate vs Tiempo"
    ax1.set_title(titulo)
    fig.tight_layout()

    # Guardado de la gráfica generada
    os.makedirs(output_folder, exist_ok=True)
    nombre_salida = f"{nombre_archivo}_glucosa_hr.png"
    ruta_salida = os.path.join(output_folder, nombre_salida)
    fig.savefig(ruta_salida, dpi=300)
    print(f" Gráfica guardada en: {ruta_salida}")

    # Mostrar en notebook y liberar memoria
    mostrar_y_liberar_figura(fig)

    ###########################################################
    # VERSIÓN ALTERNATIVA: DOS EJES ALINEADOS POR ABAJO
    ###########################################################
    # Si en algún momento quieres volver a dos ejes (glucosa y HR separados),
    # usar ax2 = ax1.twinx() y fijar ambos límites inferiores de forma consistente:
    #    ax1.set_ylim(bottom=20)
    #    ax2.set_ylim(bottom=20)
    # Mantener ese enfoque solo si interesa separar unidades visualmente.


In [ ]:
def main(folder_path, output_folder):
    """
    Busca y procesa todos los archivos CSV en una carpeta, graficando y guardando los datos de glucosa de cada uno.

    Args:
        folder_path (str): Ruta a la carpeta donde se encuentran los archivos CSV.
        output_folder (str): Carpeta donde se guardarán las imágenes generadas.

    Returns:
        None. Guarda un archivo de imagen (.png) por cada CSV procesado.
    """
    for filename in os.listdir(folder_path):
        if filename.endswith('.csv'):
            file_path = os.path.join(folder_path, filename)
            graficar_glucosa_hr_trusted(file_path, output_folder)
            

if __name__ == "__main__":
    folder_path = TRUSTED_DIR
    output_folder = r'C:\Users\regin\Documents\GitHub\TFG-pruebas\Analysis\CGMacros\Gráficas glucosa y HR'
    main(folder_path, output_folder)


In [ ]:
plt.close('all')

Aunque hemos puesto un filtro de NaNs y 0 a estas variables, aún puede haber datos con valor numérico muy bajo (1.5, 1...) que pasan este filtro y que pueden aparecer en las gráficas.

##### C1.1.2 Histograma de glucosa

Este bloque compara glucosa Dexcom y Libre con bins de ancho fijo de 2.5 mg/dL.

Un log-transform o Box–Cox puede estabilizar la varianza si el histograma y la dispersión muestran asimetrías o heterocedasticidad.

In [ ]:
def histograma_glucosa_trusted(archivo_master, output_folder, bin_width=2.5, titulo=None, figsize=(10, 6)):
    """
    Genera un histograma con glucosa dexcom y libre superpuestas.

    Args:
        archivo_master (str): Ruta al archivo trusted del usuario.
        output_folder (str): Carpeta donde guardar la imagen.
        bin_width (float, opcional): Ancho de cada bin en mg/dL.
        titulo (str, opcional): Título del gráfico. Si None, usa el nombre del archivo.
        figsize (tuple, opcional): Tamaño de la figura.

    Returns:
        output_path (str): Ruta de la imagen guardada.
    """
    master = pd.read_csv(archivo_master)

    # Extraer datos
    glucosa_dexcom = master.get('glucose_dexcom', pd.Series(dtype=float))
    glucosa_libre = master.get('glucose_libre', pd.Series(dtype=float))

    # Filtrar ceros y NaNs
    glucosa_dexcom = glucosa_dexcom[(glucosa_dexcom != 0) & (~glucosa_dexcom.isna())]
    glucosa_libre = glucosa_libre[(glucosa_libre != 0) & (~glucosa_libre.isna())]

    if glucosa_dexcom.empty and glucosa_libre.empty:
        print(f"No hay datos validos en {archivo_master}.")
        return None

    # Bins de ancho fijo para que todos los sujetos sean comparables
    all_values = pd.concat([glucosa_dexcom, glucosa_libre])
    min_val = all_values.min()
    max_val = all_values.max()
    bins = np.arange(min_val, max_val + bin_width, bin_width)

    # Crear figura
    plt.figure(figsize=figsize)

    # Histogramas superpuestos
    if not glucosa_dexcom.empty:
        plt.hist(glucosa_dexcom, bins=bins, label='Dexcom', color='blue', alpha=0.6, edgecolor='black')
    if not glucosa_libre.empty:
        plt.hist(glucosa_libre, bins=bins, label='Libre', color='cyan', alpha=0.6, edgecolor='black')

    plt.xlabel('Glucosa (mg/dL)')
    plt.ylabel('Frecuencia')

    if titulo is None:
        nombre_archivo = os.path.basename(archivo_master).replace('.csv', '')
        titulo = f"{nombre_archivo} - Histograma de Glucosa"
    plt.title(titulo)

    plt.legend()
    plt.grid(axis='y', linestyle='--', alpha=0.4)
    plt.tight_layout()

    ###########################################################
    # VERSIÓN CON DOS HISTOGRAMAS EN SUBPLOTS (NO SUPERPUESTAS)
    ###########################################################

    # Crear figura con subplots
    #fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)

    # Histograma Dexcom
    #if not glucosa_dexcom.empty:
    #    ax1.hist(glucosa_dexcom, bins=bins, color='blue', alpha=0.7, edgecolor='black')
    #    ax1.set_xlabel('Glucosa (mg/dL)')
    #    ax1.set_ylabel('Frecuencia')
    #    ax1.set_title('Glucosa Dexcom')
    #    ax1.grid(axis='y', linestyle='--', alpha=0.4)

    # Histograma Libre
    #if not glucosa_libre.empty:
    #    ax2.hist(glucosa_libre, bins=bins, color='cyan', alpha=0.7, edgecolor='black')
    #    ax2.set_xlabel('Glucosa (mg/dL)')
    #    ax2.set_ylabel('Frecuencia')
    #    ax2.set_title('Glucosa Libre')
    #    ax2.grid(axis='y', linestyle='--', alpha=0.4)

    # Título general
    #if titulo is None:
    #    nombre_archivo = os.path.basename(archivo_master).replace('.csv', '')
    #    titulo = f"{nombre_archivo} - Distribución de Glucosa"
    #fig.suptitle(titulo, fontsize=14, fontweight='bold')

    #plt.tight_layout()

    # Guardar
    os.makedirs(output_folder, exist_ok=True)
    output_filename = os.path.splitext(os.path.basename(archivo_master))[0] + '_hist_glucosa.png'
    output_path = os.path.join(output_folder, output_filename)
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.show()

    return output_path

In [ ]:
def main_histogramas(folder_path, output_folder):
    """
    Procesa todos los archivos CSV en una carpeta, generando y guardando el histograma de glucosa de cada uno.

    Args:
        folder_path (str): Ruta a la carpeta donde se encuentran los archivos CSV.
        output_folder (str): Carpeta donde se guardarán las imágenes generadas.

    Returns:
        None. Guarda un archivo de imagen (.png) por cada CSV procesado.
    """
    for filename in os.listdir(folder_path):
        if filename.endswith('.csv'):
            file_path = os.path.join(folder_path, filename)
            output_path = histograma_glucosa_trusted(file_path, output_folder)
            if output_path:
                print(f"Imagen de {filename} guardada en: {output_path}")


In [ ]:
if __name__ == "__main__":
    main_histogramas(TRUSTED_DIR, HISTOGRAMAS_DIR)

In [ ]:
plt.close('all')

##### C1.1.3 Proceso estocástico

Supongamos que tenemos un proceso estocástico con variables que representan los valores de la glucosa cada cinco minutos.
Si tomamos como una realización el intervalo de 24 horas correspondiente a un día, podemos observar la serie temporal como distintas realizaciones de un proceso estocástico.

In [ ]:
def plot_glucose_by_day(file_path, output_folder):
    """
    Grafica los datos de glucosa Dexcom por día a partir de un archivo CSV y guarda la imagen en la carpeta especificada.

    Argumentos:
        file_path (str): Ruta al archivo CSV que contiene las columnas 'time' y 'glucose_dexcom'.
        output_folder (str): Carpeta donde se guardará la imagen generada.

    Retorna:
        output_path (str): Ruta completa de la imagen guardada.

    Recomendaciones:
        - El archivo debe tener al menos las columnas 'time' y 'glucose_dexcom'.
        - 'time' debe estar en un formato reconocible por pandas.
        - La carpeta de salida debe existir; si no, se creará automáticamente.
    """
    df = pd.read_csv(file_path, parse_dates=['time'])
    df = df[['time', 'glucose_dexcom']].dropna()
    df['date'] = df['time'].dt.date

    days = list(df['date'].unique())
    cmap = mpl.colormaps['magma'].resampled(len(days))

    fig, ax = plt.subplots(figsize=(10, 6))
    for i, day in enumerate(days):
        group = df[df['date'] == day]
        hour = group['time'].dt.hour + group['time'].dt.minute / 60
        ax.plot(hour, group['glucose_dexcom'], label=str(day), color=cmap(i))

    ax.set_xlabel('Hora del día')
    ax.set_ylabel('Glucosa Dexcom')
    ax.set_title(f'Procesos estocásticos de glucosa Dexcom (cada día es una realización)\nArchivo: {os.path.basename(file_path)}')
    ax.legend(loc='upper right', fontsize='small', ncol=2)
    ax.grid(True)
    fig.tight_layout()

    os.makedirs(output_folder, exist_ok=True)
    output_filename = os.path.splitext(os.path.basename(file_path))[0] + '.png'
    output_path = os.path.join(output_folder, output_filename)
    fig.savefig(output_path)
    mostrar_y_liberar_figura(fig)
    return output_path


In [ ]:
def main(folder_path, output_folder):
    """
    Busca y procesa todos los archivos CSV en una carpeta, graficando y guardando los datos de glucosa de cada uno.

    Args:
        folder_path (str): Ruta a la carpeta donde se encuentran los archivos CSV.
        output_folder (str): Carpeta donde se guardarán las imágenes generadas.

    Returns:
        None. Guarda un archivo de imagen (.png) por cada CSV procesado.

    Recomendaciones:
        - Verifica que la ruta de salida tenga permisos de escritura.
        - Si hay muchos archivos, asegúrate de tener suficiente espacio en disco.
    """
    for filename in os.listdir(folder_path):
        if filename.endswith('.csv'):
            file_path = os.path.join(folder_path, filename)
            output_path = plot_glucose_by_day(file_path, output_folder)
            print(f"Imagen guardada en: {output_path}")

if __name__ == "__main__":
    carpeta = TRUSTED_DIR
    salida = r'C:\Users\regin\Documents\GitHub\TFG-pruebas\Analysis\CGMacros\Procesos estocásticos'
    main(carpeta, salida)

In [ ]:
plt.close('all')

Esta visualización por días trata cada jornada como una realización del mismo proceso estocástico y muestra cierta variabilidad interdiaria sin un patrón circadiano robusto, lo que refuerza la preferencia por modelos con estructura principalmente de corto plazo.

###### C.1.3.1 Perfil promedio

Primero haremos una breve auditoría para ver por sujeto qué tipos de comida tienen y cuántas se corresponden con desayunos, para luego crear perfiles postprandiales promedio generales y tras el desayuno.

In [ ]:
# =============================================================================
# AUDITORÍA DE DATOS DE COMIDAS POR SUJETO Y TIPO DE COMIDA
# =============================================================================

def audit_case_insensitive_meals(folder_path):
    """
    Audita el dataset unificando las categorías de comida sin importar mayúsculas

    o minúsculas, y cuenta el total de desayunos corregidos.

    Descripción:
        Carga la columna 'meal_type', elimina nulos y aplica transformaciones de texto:
        pasa todo a minúsculas (`.str.lower()`) y remueve espacios en blanco. De esta forma,
        asocia de forma segura tanto 'Breakfast' como 'breakfast' bajo un mismo conteo.

    Args:
        folder_path (str): Ruta a la carpeta que contiene los archivos CSV de los sujetos.

    Returns:
        None. Imprime el reporte corregido directamente en la terminal.

    Recomendaciones:
        - Utiliza este reporte para validar que todos los sujetos (salvo excepciones reales)
          ahora recuperan sus datos de desayuno correspondientes.
    """
    if not os.path.exists(folder_path):
        print(f"Error: La ruta no existe -> {folder_path}")
        return

    print(f"{'Archivo':<30} | {'¿Tiene Desayuno?':<18} | {'Valores Originales':<40} | {'Total Breakfast'}")
    print("-" * 105)

    for filename in sorted(os.listdir(folder_path)):
        if filename.endswith(".csv"):
            file_path = os.path.join(folder_path, filename)

            try:
                # Leer únicamente la columna de comidas por eficiencia
                df_chunk = pd.read_csv(file_path, usecols=["meal_type"])
                
                # Filtrar valores nulos para el análisis de etiquetas
                clean_meals = df_chunk["meal_type"].dropna()

                if clean_meals.empty:
                    print(f"{filename:<30} | {'NO (Sin Comidas)':<18} | {'[Vacio]':<40} | 0")
                    continue

                # Guardar los nombres originales para visualización del usuario
                original_distinct = clean_meals.astype(str).str.strip().unique()
                original_str = ", ".join(original_distinct)

                # Estandarizar a minúsculas y limpiar espacios para el conteo real
                standardized_meals = clean_meals.astype(str).str.lower().str.strip()
                
                # Contar cualquier variante de desayuno ('breakfast')
                breakfast_count = (standardized_meals == "breakfast").sum()
                has_breakfast = "SÍ" if breakfast_count > 0 else "NO"

                print(f"{filename:<30} | {has_breakfast:<18} | {original_str:<40} | {breakfast_count}")

            except Exception as e:
                print(f"{filename:<30} | {'ERROR':<18} | No se pudo procesar: {str(e):<34} | -")


if __name__ == "__main__":
    # Ejecutar la auditoría estandarizada
    audit_case_insensitive_meals(TRUSTED_DIR)

In [ ]:
def extract_postprandial_windows(df, meal_filter=None, window_minutes=180):
    """
    Identifica las comidas de forma tolerante a mayúsculas/minúsculas y extrae

    matrices de ventanas postprandiales absolutas y normalizadas (deltas).

    Descripción:
        Estandariza la columna 'meal_type' convirtiéndola a minúsculas y eliminando
        espacios en blanco invisibles. Si se define un 'meal_filter' (ej. 'Breakfast'),
        lo busca de forma insensible a las mayúsculas (buscando 'breakfast'). Extrae 
        los siguientes `window_minutes` registros (3 horas) y gestiona de forma segura 
        los cortes o datos truncados al final del archivo usando valores NaN.

    Args:
        df (pd.DataFrame): Dataframe preprocesado del sujeto con columnas
            'meal_type' y 'glucose_dexcom'.
        meal_filter (str, opcional): Tipo de comida a filtrar ('Breakfast', 'Lunch', 'Dinner').
            Si es None, procesa todas las comidas. Por defecto es None.
        window_minutes (int, opcional): Duración del periodo postprandial en minutos.
            Por defecto es 180 (3 horas).

    Returns:
        tuple: Un par de matrices NumPy (np.ndarray) de dos dimensiones:
            - absolute_matrix: Filas = comidas, Columnas = minutos (0 a window_minutes). Valores absolutos.
            - delta_matrix: Filas = comidas, Columnas = minutos. Valores en Delta (Glucosa_t - Glucosa_0).

    Recomendaciones:
        - El dataframe debe venir preprocesado minuto a minuto y ordenado en el tiempo.
        - Gracias a la normalización de strings, procesa correctamente 'breakfast', 'Breakfast' o 'BREAKFAST'.
    """
    # Crear una serie temporal en minúsculas y limpia de espacios para indexar correctamente
    # Manteniendo los índices originales del DataFrame para poder extraer la glucosa después
    clean_meal_series = df["meal_type"].dropna().astype(str).str.lower().str.strip()

    # Identificar los índices de inicio de comida según el filtro estandarizado
    if meal_filter:
        target_meal = meal_filter.lower().strip()
        meal_indices = clean_meal_series[clean_meal_series == target_meal].index
    else:
        # Si no hay filtro, tomamos cualquier fila donde meal_type original no sea nulo
        meal_indices = df[df["meal_type"].notna()].index

    absolute_list = []
    delta_list = []

    # Iterar sobre cada evento de comida detectado con la nueva lógica robusta
    for idx in meal_indices:
        # Definir el rango de la ventana (+1 para incluir el minuto 180 completo)
        end_idx = idx + window_minutes + 1

        # Extraer el segmento de glucosa correspondiente desde el DataFrame original
        glucose_segment = df["glucose_dexcom"].iloc[idx:end_idx].values

        # Manejo seguro de desbordamiento: si la comida está muy cerca del final del archivo
        if len(glucose_segment) < (window_minutes + 1):
            padded_segment = np.empty(window_minutes + 1)
            padded_segment[:] = np.nan
            padded_segment[: len(glucose_segment)] = glucose_segment
            glucose_segment = padded_segment

        # Calcular la glucosa base en el minuto cero de la ingesta
        base_glucose = glucose_segment[0]

        # Evitar errores si el minuto cero resultara ser un valor nulo inesperado
        if not np.isnan(base_glucose):
            absolute_list.append(glucose_segment)
            # Restamos el valor inicial para forzar a la curva a iniciar en Y = 0
            delta_list.append(glucose_segment - base_glucose)

    # Convertir las listas acumuladas en matrices estructuradas de NumPy
    if absolute_list:
        absolute_matrix = np.array(absolute_list)
        delta_matrix = np.array(delta_list)
    else:
        absolute_matrix = np.empty((0, window_minutes + 1))
        delta_matrix = np.empty((0, window_minutes + 1))

    return absolute_matrix, delta_matrix


def generate_and_save_plot(
    matrix, output_path, title, ylabel, is_delta=False, color_theme="purple"
):
    """
    Genera y guarda una gráfica de perfil postprandial promedio (media +- desviación estándar).

    Descripción:
        Calcula la media y la desviación estándar minuto a minuto ignorando los
        valores NaN (mediante np.nanmean y np.nanstd). Utiliza matplotlib para
        dibujar la línea de tendencia central y un área sombreada transparente
        para la dispersión.

    Args:
        matrix (np.ndarray): Matriz bidimensional con los perfiles de glucosa.
        output_path (str): Ruta completa del archivo de imagen a guardar (.png).
        title (str): Título principal de la gráfica.
        ylabel (str): Etiqueta del eje Y (p. ej., 'Glucosa Dexcom (mg/dL)').
        is_delta (bool, opcional): Si es True, fuerza a que la línea de la cuadrícula
            en Y=0 sea más visible. Por defecto es False.
        color_theme (str, opcional): Paleta de colores básica ('purple' para general, 'orange' para desayuno).
            Por defecto es 'purple'.
    """
    # Validar de forma segura si la matriz tiene datos que graficar
    if matrix.size == 0 or matrix.shape[0] == 0:
        print(
            f"   [Aviso] Matriz vacía para {os.path.basename(output_path)}. Gráfica omitida."
        )
        return

    # Calcular estadísticas ignorando registros truncados (NaNs) al final del archivo
    mean_profile = np.nanmean(matrix, axis=0)
    std_profile = np.nanstd(matrix, axis=0)
    total_meals = matrix.shape[0]

    # Configurar el eje X temporal (de 0 a 180 minutos)
    time_axis = np.arange(len(mean_profile))

    # Definir estilos y colores según el tipo de comida analizada
    fig, ax = plt.subplots(figsize=(9, 5.5))
    main_color = "#6A1B9A" if color_theme == "purple" else "#E65100"
    shade_color = "#E1BEE7" if color_theme == "purple" else "#FFE0B2"

    # Pintar la región de la desviación estándar (±1 SD)
    ax.fill_between(
        time_axis,
        mean_profile - std_profile,
        mean_profile + std_profile,
        color=shade_color,
        alpha=0.5,
        label="Desviación Estándar (±1 SD)",
    )

    # Pintar la línea de la media promedio postprandial
    ax.plot(
        time_axis,
        mean_profile,
        color=main_color,
        linewidth=2.5,
        label=f"Media Promedio (n={total_meals} comidas)",
    )

    # Añadir una línea guía horizontal en cero si es una gráfica de deltas
    if is_delta:
        ax.axhline(0, color="gray", linestyle="--", linewidth=1)

    # Ajustes estéticos finales de la gráfica
    ax.set_xlabel("Tiempo postprandial (minutos)", fontsize=11, fontweight="bold")
    ax.set_ylabel(ylabel, fontsize=11, fontweight="bold")
    ax.set_title(title, fontsize=12, fontweight="bold", pad=12)

    # Configurar marcas en el eje X cada 30 minutos de forma fija
    ax.set_xticks(np.arange(0, 181, 30))
    ax.grid(True, linestyle=":", alpha=0.6)
    ax.legend(loc="upper right", frameon=True, facecolor="white", edgecolor="none")

    fig.tight_layout()

    # Guardar en disco a alta resolución y cerrar figura para liberar RAM de inmediato
    fig.savefig(output_path, dpi=150)
    plt.close(fig)


def process_subject_profiles(file_path, output_folder):
    """
    Procesa el archivo de datos de un sujeto para generar y guardar cuatro gráficas postprandiales.

    Carga los datos del fichero, extrae el identificador del sujeto y ejecuta de forma 
    secuencial el aislamiento de ventanas temporales para calcular tanto los perfiles 
    absolutos como las variaciones (deltas). El proceso se divide en análisis general 
    (todas las comidas) y análisis específico (solo desayunos).

    Args:
        file_path (str): Ruta del archivo CSV individual que contiene los registros del sujeto.
        output_folder (str): Ruta del directorio del sistema donde se exportarán los gráficos.

    Returns:
        None: La función no retorna ningún objeto, exporta los archivos visuales (.png) directamente.

    Recomendaciones:
        - Asegurar que las funciones auxiliares 'extract_postprandial_windows' y 
          'generate_and_save_plot' estén cargadas previamente en el entorno.
        - Verificar los permisos de escritura en la carpeta de destino antes de iniciar.
    """
    # === 1. Ingesta de Datos y Extracción de Identificador ===
    df = pd.read_csv(file_path, parse_dates=["time"])
    subject_id = os.path.splitext(os.path.basename(file_path))[0]
    print(f"Procesando perfiles para el sujeto: {subject_id}...")

    # Creación del directorio de salida si no existiera previamente
    os.makedirs(output_folder, exist_ok=True)

    # =========================================================================
    # FASE 1: Perfiles Postprandiales Generales (Cualquier comida)
    # =========================================================================
    # Obtención de matrices analíticas sin aplicar filtros por tipo de ingesta
    abs_matrix_gen, delta_matrix_gen = extract_postprandial_windows(
        df, meal_filter=None
    )

    # Generación y exportación del Perfil General Absoluto
    out_gen_abs = os.path.join(
        output_folder, f"{subject_id}_perfil_postprandial_promedio.png"
    )
    title_gen_abs = f"Perfil Postprandial Promedio - Todas las Comidas\nSujeto: {subject_id}"
    generate_and_save_plot(
        matrix=abs_matrix_gen,
        output_path=out_gen_abs,
        title=title_gen_abs,
        ylabel="Glucosa Dexcom (mg/dL)",
        is_delta=False,
        color_theme="purple",
    )

    # Generación y exportación del Perfil General Deltas (Variación)
    out_gen_del = os.path.join(
        output_folder, f"{subject_id}_perfil_postprandial_promedio_deltas.png"
    )
    title_gen_del = f"Variación de Glucosa Postprandial (Delta) - Todas las Comidas\nSujeto: {subject_id}"
    generate_and_save_plot(
        matrix=delta_matrix_gen,
        output_path=out_gen_del,
        title=title_gen_del,
        ylabel="Δ Glucosa Dexcom (mg/dL)",
        is_delta=True,
        color_theme="purple",
    )

    # =========================================================================
    # FASE 2: Perfiles Postprandiales Específicos (Solo Desayuno)
    # =========================================================================
    # Obtención de matrices aplicando el filtro por cadena de texto para desayunos
    abs_matrix_bf, delta_matrix_bf = extract_postprandial_windows(
        df, meal_filter="Breakfast"
    )

    # Generación y exportación del Perfil Desayuno Absoluto
    out_bf_abs = os.path.join(
        output_folder, f"{subject_id}_perfil_postprandial_promedio_desayuno.png"
    )
    title_bf_abs = (f"Perfil Postprandial Promedio - Solo Desayunos\nSujeto: {subject_id}")
    generate_and_save_plot(
        matrix=abs_matrix_bf,
        output_path=out_bf_abs,
        title=title_bf_abs,
        ylabel="Glucosa Dexcom (mg/dL)",
        is_delta=False,
        color_theme="orange"
    )
        
    # Generación y exportación del Perfil Desayuno Deltas (Variación)
    out_bf_del = os.path.join(output_folder, f"{subject_id}_perfil_postprandial_promedio_desayuno_deltas.png")
    title_bf_del = f"Variación de Glucosa Postprandial (Delta) - Solo Desayunos\nSujeto: {subject_id}"
    generate_and_save_plot(
        matrix=delta_matrix_bf,
        output_path=out_bf_del,
        title=title_bf_del,
        ylabel="Δ Glucosa Dexcom (mg/dL)",
        is_delta=True,
        color_theme="orange"
    )

In [ ]:
def main(folder_path, output_folder):
    """
    Función orquestadora principal para la ejecución por lotes.
    """
    if not os.path.exists(folder_path):
        print(f"Error: La carpeta de origen no existe: {folder_path}")
        return

    for filename in sorted(os.listdir(folder_path)):
        if filename.endswith(".csv"):
            file_path = os.path.join(folder_path, filename)
            process_subject_profiles(file_path, output_folder)

    print("\n[¡Proceso Completado Exitosamente! Perfiles generados para todos los casos]")

if __name__ == "__main__":
    main(TRUSTED_DIR, PERFIL_PROMEDIO_DIR)

Además incluimos un análisis poblacional y por grupsos metabólicos.

In [ ]:
def extract_postprandial_windows(df, meal_filter=None, window_minutes=180):
    """
    Identifica las comidas de forma tolerante a mayúsculas/minúsculas y extrae
    matrices de ventanas postprandiales absolutas y normalizadas (deltas).

    Descripción:
        Estandariza la columna 'meal_type' a minúsculas y elimina espacios.
        Extrae los siguientes `window_minutes` registros (3 horas) tras el inicio
        de la ingesta y gestiona de forma segura los finales de archivo con NaNs.

    Args:
        df (pd.DataFrame): Dataframe del sujeto con columnas 'meal_type' y 'glucose_dexcom'.
        meal_filter (str, opcional): Tipo de comida ('Breakfast', 'Lunch', 'Dinner').
            Si es None, procesa todas las comidas. Por defecto es None.
        window_minutes (int, opcional): Duración del periodo en minutos. Por defecto es 180.

    Returns:
        tuple: (absolute_matrix, delta_matrix) como np.ndarrays de dos dimensiones (filas=comidas, columnas=minutos).
    """
    clean_meal_series = df["meal_type"].dropna().astype(str).str.lower().str.strip()

    if meal_filter:
        target_meal = meal_filter.lower().strip()
        meal_indices = clean_meal_series[clean_meal_series == target_meal].index
    else:
        meal_indices = df[df["meal_type"].notna()].index

    absolute_list = []
    delta_list = []

    for idx in meal_indices:
        end_idx = idx + window_minutes + 1
        glucose_segment = df["glucose_dexcom"].iloc[idx:end_idx].values

        if len(glucose_segment) < (window_minutes + 1):
            padded_segment = np.empty(window_minutes + 1)
            padded_segment[:] = np.nan
            padded_segment[: len(glucose_segment)] = glucose_segment
            glucose_segment = padded_segment

        base_glucose = glucose_segment[0]

        if not np.isnan(base_glucose):
            absolute_list.append(glucose_segment)
            delta_list.append(glucose_segment - base_glucose)

    abs_matrix = (
        np.array(absolute_list)
        if absolute_list
        else np.empty((0, window_minutes + 1))
    )
    del_matrix = (
        np.array(delta_list) if delta_list else np.empty((0, window_minutes + 1))
    )

    return abs_matrix, del_matrix


def plot_single_group(matrix, output_path, title, ylabel, is_delta=False):
    """
    Genera y guarda la gráfica independiente para el Grupo Poblacional Global.

    Descripción:
        Calcula y dibuja una única línea de media central con su sombreado de
        desviación estándar correspondiente utilizando una paleta grisácea neutra.

    Args:
        matrix (np.ndarray): Matriz bidimensional de datos agrupados.
        output_path (str): Ruta completa del archivo de imagen de salida (.png).
        title (str): Título principal de la gráfica.
        ylabel (str): Etiqueta para el eje Y.
        is_delta (bool, opcional): Si es True, añade línea de referencia en Y=0.
    """
    if matrix.size == 0 or matrix.shape[0] == 0:
        return

    mean_profile = np.nanmean(matrix, axis=0)
    std_profile = np.nanstd(matrix, axis=0)
    time_axis = np.arange(len(mean_profile))

    fig, ax = plt.subplots(figsize=(9, 5.5))

    ax.fill_between(
        time_axis,
        mean_profile - std_profile,
        mean_profile + std_profile,
        color="#B0BEC5",
        alpha=0.4,
        label="Desviación Estándar (±1 SD)",
    )
    ax.plot(
        time_axis,
        mean_profile,
        color="#37474F",
        linewidth=2.5,
        label=f"Media Poblacional (n={matrix.shape[0]} comidas)",
    )

    if is_delta:
        ax.axhline(0, color="gray", linestyle="--", linewidth=1)

    ax.set_xlabel("Tiempo postprandial (minutos)", fontsize=11, fontweight="bold")
    ax.set_ylabel(ylabel, fontsize=11, fontweight="bold")
    ax.set_title(title, fontsize=12, fontweight="bold", pad=12)
    ax.set_xticks(np.arange(0, 181, 30))
    ax.grid(True, linestyle=":", alpha=0.6)
    ax.legend(loc="upper right", frameon=True, facecolor="white")

    fig.tight_layout()
    fig.savefig(output_path, dpi=150)
    plt.close(fig)


def plot_overlay_groups(groups_dict, output_path, title, ylabel, is_delta=False):
    """
    Genera y guarda una gráfica en formato Grid (1x3) con límites del eje Y optimizados

    y corrección del parámetro de opacidad en la leyenda (framealpha).

    Descripción:
        Crea una fila con tres subplots compartiendo escala vertical estricta.
        Ajusta los límites del eje Y para centrar las curvas, mueve las leyendas 
        a la esquina inferior derecha y aplica de forma correcta el parámetro
        'framealpha' para la transparencia de la caja.

    Args:
        groups_dict (dict): Diccionario con las matrices de glucosa por subgrupo.
        output_path (str): Ruta completa de destino para guardar la imagen (.png).
        title (str): Título principal superior de la figura completa.
        ylabel (str): Etiqueta para el eje Y (solo dibujada en el panel izquierdo).
        is_delta (bool, opcional): Si es True, añade la línea guía horizontal en Y=0.
    """
    # Orden de paneles de izquierda a derecha con sus estilos clínicos estables
    group_order = ["No Diabético", "Prediabético", "Diabético"]
    styles = {
        "No Diabético": {"line": "#2E7D32", "shade": "#C8E6C9", "title": "No Diabético"},
        "Prediabético": {"line": "#EF6C00", "shade": "#FFE0B2", "title": "Prediabético"},
        "Diabético": {"line": "#C62828", "shade": "#FFCDD2", "title": "Diabético"},
    }

    # 1. Escaneo dinámico min/max para fijar la escala compartida del eje Y
    global_min = 100.0
    global_max = 200.0  # Valores base de seguridad

    for group_name in group_order:
        matrix = groups_dict.get(group_name)
        if matrix is not None and matrix.size > 0:
            mean_p = np.nanmean(matrix, axis=0)
            std_p = np.nanstd(matrix, axis=0)
            
            local_min = np.nanmin(mean_p - std_p)
            local_max = np.nanmax(mean_p + std_p)
            
            if local_min < global_min:
                global_min = local_min
            if local_max > global_max:
                global_max = local_max

    # Ajuste fino de márgenes verticales
    y_range = global_max - global_min
    y_limit_bottom = global_min - (y_range * 0.03)  # Margen inferior ceñido al 3%
    y_limit_top = global_max + (y_range * 0.04)     # Margen superior ceñido al 4%

    # Ajustar límite inferior si estamos analizando deltas
    if is_delta and y_limit_bottom > -30:
        y_limit_bottom = -30

    # 2. Configurar el lienzo (1 fila, 3 columnas de subplots)
    fig, axes = plt.subplots(1, 3, figsize=(15, 5.2), sharey=True)
    time_axis = np.arange(181)
    has_any_data = False

    for i, group_name in enumerate(group_order):
        ax = axes[i]
        matrix = groups_dict.get(group_name)
        color_cfg = styles[group_name]

        ax.set_title(color_cfg["title"], fontsize=11, fontweight="bold")

        if matrix is not None and matrix.size > 0:
            has_any_data = True
            mean_profile = np.nanmean(matrix, axis=0)
            std_profile = np.nanstd(matrix, axis=0)
            total_meals = matrix.shape

            # Dibujar área de dispersión (±1 SD)
            ax.fill_between(
                time_axis,
                mean_profile - std_profile,
                mean_profile + std_profile,
                color=color_cfg["shade"],
                alpha=0.5,
                label="Desviación Estándar (±1 SD)"
            )

            # Dibujar la línea de tendencia central de la media
            ax.plot(
                time_axis,
                mean_profile,
                color=color_cfg["line"],
                linewidth=2.5,
                label=f"Media (n={matrix.shape[0]} comidas)",  
            )
            
            ax.legend(
                loc="upper right", 
                fontsize="small", 
                frameon=True, 
                facecolor="white", 
                framealpha=0.9
            )

        if is_delta:
            ax.axhline(0, color="gray", linestyle="--", linewidth=1)

        # Configuración homogénea de ejes y cuadrícula
        ax.set_xlim(-5, 185)
        ax.set_xticks(np.arange(0, 181, 30))
        ax.set_xlabel("Tiempo postprandial (min)", fontsize=10)
        ax.grid(True, linestyle=":", alpha=0.6)

        # Forzar la escala vertical optimizada
        ax.set_ylim(y_limit_bottom, y_limit_top)

        if i == 0:
            ax.set_ylabel(ylabel, fontsize=11, fontweight="bold")

    if not has_any_data:
        plt.close(fig)
        return

    # Añadir título superior centralizado
    fig.suptitle(title, fontsize=13, fontweight="bold", y=0.98)
    
    # Ajuste de cajas compacto
    fig.tight_layout()
    
    # Exportar imagen final en alta resolución
    fig.savefig(output_path, dpi=150)
    plt.close(fig)


def generate_pipeline_plots(
    all_absolute_matrices, all_delta_matrices, meal_label, output_folder
):
    """
    Orquesta la creación de las gráficas individuales poblacionales y las

    comparativas superpuestas traduciendo las etiquetas al español correcto.

    Args:
        all_absolute_matrices (dict): Diccionario de matrices de glucosa absoluta por grupo.
        all_delta_matrices (dict): Diccionario de matrices de deltas de glucosa por grupo.
        meal_label (str): Etiqueta descriptiva del tipo de comida ('General' o 'Breakfast').
        output_folder (str): Carpeta donde se exportarán las imágenes resultantes.
    """
    # Traducción y formateo estricto para el TFG (evitando el spanglish)
    if meal_label == "Breakfast":
        suffix = "_desayuno"
        title_suffix = " - Sólo desayunos"
    else:
        suffix = ""
        title_suffix = ""

    # 1. Exportar bloque independiente del Grupo Poblacional Global
    pob_abs_path = os.path.join(
        output_folder, f"Grupo_Poblacional_perfil_postprandial{suffix}.png"
    )
    plot_single_group(
        matrix=all_absolute_matrices["Poblacional"],
        output_path=pob_abs_path,
        title=f"Perfil Postprandial Promedio Poblacional{title_suffix}",
        ylabel="Glucosa Dexcom (mg/dL)",
        is_delta=False,
    )

    pob_del_path = os.path.join(
        output_folder, f"Grupo_Poblacional_perfil_postprandial{suffix}_deltas.png"
    )
    plot_single_group(
        matrix=all_delta_matrices["Poblacional"],
        output_path=pob_del_path,
        title=f"Variación de Glucosa Postprandial Poblacional (Delta){title_suffix}",
        ylabel="Δ Glucosa Dexcom (mg/dL)",
        is_delta=True,
    )

    # 2. Exportar bloque comparativo de subgrupos superpuestos (el Grid 1x3)
    comp_abs_path = os.path.join(
        output_folder, f"Comparativa_Grupos_perfil_postprandial{suffix}.png"
    )
    plot_overlay_groups(
        groups_dict=all_absolute_matrices,
        output_path=comp_abs_path,
        title=f"Comparativa Postprandial por Estado Metabólico{title_suffix}",
        ylabel="Glucosa Dexcom (mg/dL)",
        is_delta=False,
    )

    comp_del_path = os.path.join(
        output_folder, f"Comparativa_Grupos_perfil_postprandial{suffix}_deltas.png"
    )
    plot_overlay_groups(
        groups_dict=all_delta_matrices,
        output_path=comp_del_path,
        title=f"Comparativa de Variación Glucémica (Delta){title_suffix}",
        ylabel="Δ Glucosa Dexcom (mg/dL)",
        is_delta=True,
    )

In [ ]:
def main(folder_path, output_folder):
    """
    Estructura la recolección por lotes acumulando datos en estructuras de grupos
    metabólicos según el diccionario de mapeo MAPEO_DIABETES y ejecuta los plots.

    Args:
        folder_path (str): Carpeta origen que contiene los archivos CSV de los sujetos.
        output_folder (str): Carpeta destino para almacenar los gráficos generados.
    """
    if not os.path.exists(folder_path):
        print(f"Error: Carpeta de origen no encontrada: {folder_path}")
        return

    os.makedirs(output_folder, exist_ok=True)

    categories = ["Poblacional", "No Diabético", "Prediabético", "Diabético"]
    general_abs_data = {cat: [] for cat in categories}
    general_del_data = {cat: [] for cat in categories}
    breakfast_abs_data = {cat: [] for cat in categories}
    breakfast_del_data = {cat: [] for cat in categories}

    print("Iniciando fase de recolección y asignación por grupos metabólicos...")

    for filename in sorted(os.listdir(folder_path)):
        if filename.endswith(".csv"):
            file_path = os.path.join(folder_path, filename)

            # Obtener el nombre base en texto limpio antes de segmentar series con split
            base_name_str = os.path.splitext(filename)[0]
            subject_key = base_name_str.split("_")[0]

            group_assignment = MAPEO_DIABETES.get(subject_key)

            if not group_assignment:
                print(
                    f"   [Aviso] {filename} omitido. '{subject_key}' no se encuentra en MAPEO_DIABETES."
                )
                continue

            try:
                df = pd.read_csv(file_path, parse_dates=["time"])

                # --- Recolección 1: Todas las comidas ---
                abs_gen, del_gen = extract_postprandial_windows(df, meal_filter=None)
                if abs_gen.shape[0] > 0:
                    general_abs_data["Poblacional"].extend(abs_gen)
                    general_del_data["Poblacional"].extend(del_gen)
                    general_abs_data[group_assignment].extend(abs_gen)
                    general_del_data[group_assignment].extend(del_gen)

                # --- Recolección 2: Solo desayunos ---
                abs_bf, del_bf = extract_postprandial_windows(
                    df, meal_filter="Breakfast"
                )
                if abs_bf.shape[0] > 0:
                    breakfast_abs_data["Poblacional"].extend(abs_bf)
                    breakfast_del_data["Poblacional"].extend(del_bf)
                    breakfast_abs_data[group_assignment].extend(abs_bf)
                    breakfast_del_data[group_assignment].extend(del_bf)

            except Exception as e:
                print(f"   [Error] Fallo crítico al procesar {filename}: {str(e)}")

    # Consolidación rigurosa de estructuras evitando errores booleanos de NumPy
    final_gen_abs = {
        cat: (
            np.array(general_abs_data[cat])
            if general_abs_data[cat]
            else np.empty((0, 181))
        )
        for cat in categories
    }
    final_gen_del = {
        cat: (
            np.array(general_del_data[cat])
            if general_del_data[cat]
            else np.empty((0, 181))
        )
        for cat in categories
    }
    final_bf_abs = {
        cat: (
            np.array(breakfast_abs_data[cat])
            if breakfast_abs_data[cat]
            else np.empty((0, 181))
        )
        for cat in categories
    }
    final_bf_del = {
        cat: (
            np.array(breakfast_del_data[cat])
            if breakfast_del_data[cat]
            else np.empty((0, 181))
        )
        for cat in categories
    }

    print("\nGenerando y exportando colecciones de gráficos finales...")
    generate_pipeline_plots(final_gen_abs, final_gen_del, "General", output_folder)
    generate_pipeline_plots(final_bf_abs, final_bf_del, "Breakfast", output_folder)
    print(f"\n[¡Pipeline Completado! Encontrará los gráficos en: {output_folder}]")


if __name__ == "__main__":
    main(TRUSTED_DIR, PERFIL_PROMEDIO_DIR)

#### C1.2 Calorías y METs: actividad

Los METs (Equivalentes Metabólicos) puede considerarse esencialmente la misma variable que la intensidad del ejercicio, ya que los METs son una medida estándar de la intendsidad absoluta de la actividad física. Es una variable numérica continua que representa el coste energético de una actividad, donde 1 MET equivale al gasto energético en reposo, y 4 METs, por ejemplo, significaría que la actividad consume cuatro veces más energía que estar en reposo. Como variable categórica tendríamos los siguientes rangos aproximados:
- Sedentario: <1.5 METs
- Ligero: 1.6-3.0 METs
- Moderado: 3.0-5.9 METs
- Vigoroso: >6.0 METs

Nuestros datos contienen los datos METs multiplicados x10.

En nuestra base de datos hay sujetos que en vez de usar la variable 'METS' usan 'Intensity', que toma valores 1,2 y 3. Para adaptarlo de la mejor forma a los datos de METs del resto de participantes, que oscilan entre 10 y 129, hemos hecho un mapeo 0->0, 1->15, 2->40 y 3->80.

In [ ]:
def explorar_calorias_mets(file_path, output_folder):
    """
    Carga y visualiza gráficamente un archivo trusted.
    Incluye: señales temporales de actividad, con calorías y mets (intensidad).
    Guarda la imagen en la carpeta especificada.

    Args:
        file_path (str): Ruta al archivo CSV del usuario.
        output_folder (str): Carpeta donde guardar la imagen.

    Returns:
        output_path (str): Ruta de la imagen guardada.
    """
    print(f"\n📂 Explorando archivo: {file_path}")

    df = pd.read_csv(file_path, parse_dates=['time'])
    df.set_index('time', inplace=True)

    user_id = os.path.splitext(os.path.basename(file_path))[0]

    fig, ax = plt.subplots(figsize=(10, 6))
    plotted = False

    if 'mets' in df.columns:
        df['mets'].plot(ax=ax, label='METS', color='purple')
        plotted = True
    if 'calories_activity' in df.columns:
        df['calories_activity'].plot(ax=ax, label='Calories', color='red')
        plotted = True

    ax.set_title(f"Calorías y METs - {user_id}")
    ax.set_ylabel('Unidades')
    ax.legend()
    ax.grid(True)
    fig.tight_layout()

    if plotted:
        os.makedirs(output_folder, exist_ok=True)
        output_filename = f"{user_id}_actividad_fisica.png"
        output_path = os.path.join(output_folder, output_filename)
        fig.savefig(output_path)
        print(f"Imagen guardada en: {output_path}")
        mostrar_y_liberar_figura(fig)
        return output_path
    else:
        plt.close(fig)
        print(f"[ADVERTENCIA] No hay columnas 'mets' o 'calories' en {file_path}")
        return None


In [ ]:
def main(folder_path, output_folder):
    """
    Procesa todos los archivos CSV en una carpeta, graficando pasos y calorías de cada uno y guardando las imágenes.

    Args:
        folder_path (str): Ruta a la carpeta donde se encuentran los archivos CSV.
        output_folder (str): Carpeta donde se guardarán las imágenes generadas.

    Returns:
        None.
    """
    for filename in os.listdir(folder_path):
        if filename.endswith('.csv'):
            file_path = os.path.join(folder_path, filename)
            explorar_calorias_mets(file_path, output_folder)

if __name__ == "__main__":
    carpeta = TRUSTED_DIR
    salida = r'C:\Users\regin\Documents\GitHub\TFG-pruebas\Analysis\CGMacros\Gráficas calorías y METs'
    main(carpeta, salida)

In [ ]:
plt.close('all')

#### C1.3 Macronutrientes y calorías: alimentación

Este bloque genera una gráfica por participante con barras apiladas de macronutrientes y una linea de calorías superpuesta.

In [ ]:
def graficar_macros_por_evento(archivo_master, output_folder):
    """
    Genera un gráfico combinado de barras apiladas y líneas para visualizar macronutrientes y calorías.

    Carga un archivo maestro, identifica las columnas de macronutrientes y energía disponibles, 
    y filtra los eventos con datos válidos. Muestra de forma simultánea los componentes 
    nutricionales (eje primario) y la ingesta calórica (eje secundario) a lo largo del tiempo, 
    limitando la visualización a los últimos 50 eventos para mantener la legibilidad.

    Args:
        archivo_master (str): Ruta del archivo CSV que contiene el histórico de eventos nutricionales.
        output_folder (str): Ruta del directorio del sistema donde se exportará la imagen resultante.

    Returns:
        str o None: Retorna la ruta absoluta del archivo gráfico generado (.png) o 
            None si el archivo de entrada no cumple con los requisitos mínimos de datos.

    Ejemplos:
        >>> path_grafico = graficar_macros_por_evento("Data/U101_events.csv", "Analysis/Plots/")
        Grafica de macros guardada en: Analysis/Plots/U101_events_macros.png

    Recomendaciones:
        - Asegurar que la función auxiliar 'mostrar_y_liberar_figura' esté definida previamente 
          en el entorno para evitar errores al cierre del gráfico.
        - Verificar la nomenclatura de las columnas del CSV de entrada si se observa 
          la omisión sistemática de algún macronutriente en las barras.
    """
    # === 1. Ingesta de Datos y Validación Inicial ===
    df = pd.read_csv(archivo_master)

    # Identificación dinámica de la columna calórica y de los macronutrientes presentes
    posibles_calorias = ['calories', 'calories', 'calories_activity']
    calories_col = next((col for col in posibles_calorias if col in df.columns), None)
    macro_cols = [col for col in ['carbs', 'protein', 'fat', 'fiber'] if col in df.columns]
    
    if calories_col is None or not macro_cols:
        print(f"Archivo {archivo_master} sin columnas suficientes para graficar macros.")
        return None

    # === 2. Filtrado de Eventos de Ingesta ===
    # Aislamiento de registros que aporten valor energético
    events = df[df[calories_col] > 0].copy()
    if events.empty:
        print(f"No hay eventos con calorias en {archivo_master}.")
        return None

    # Depuración y ordenación cronológica de las marcas temporales
    events['time'] = pd.to_datetime(events['time'], errors='coerce') if 'time' in events.columns else pd.NaT
    events = events.dropna(subset=['time']).sort_values('time')
    if events.empty:
        print(f"No hay marcas temporales validas en {archivo_master}.")
        return None

    # Acotación de la muestra a los últimos 50 eventos para optimizar el espacio visual
    if len(events) > 50:
        events = events.tail(50)

    # === 3. Configuración y Muestreo del Eje Temporal ===
    labels = events['time'].dt.strftime('%d-%m %H:%M').tolist()
    x_idx = np.arange(len(events))
    
    # Cálculo del salto (step) para evitar la superposición de textos en el eje X
    max_ticks = 12
    step = max(1, len(x_idx) // max_ticks)
    xtick_positions = x_idx[::step]
    xtick_labels = [labels[i] for i in xtick_positions]

    # === 4. Construcción del Gráfico Base (Eje Primario: Macronutrientes) ===
    fig, ax1 = plt.subplots(figsize=(14, 6))
    bottom = np.zeros(len(events), dtype=float)
    colors = {
        'protein': '#1f77b4',
        'carbs': '#ff7f0e',
        'fiber': '#2ca02c',
        'fat': 'red',
    }

    # Renderizado secuencial de barras para estructurar el formato apilado
    for col in ['carbs', 'protein', 'fat', 'fiber']:
        if col in events.columns:
            ax1.bar(x_idx, events[col].fillna(0), bottom=bottom, color=colors[col], label=col.title(), width=0.8)
            bottom += events[col].fillna(0).to_numpy(dtype=float)

    # === 5. Construcción del Eje Secundario (Eje Doble: Calorías) ===
    # Creación de un eje X compartido con escala métrica independiente en el eje Y
    ax2 = ax1.twinx()
    ax2.plot(x_idx, events[calories_col].fillna(0), color='purple', linewidth=2.0, marker='o', markersize=3, label='Calories')
    ax2.set_ylabel('Calories', color='purple')
    ax2.tick_params(axis='y', labelcolor='purple')

    # === 6. Personalización de Estética y Leyendas ===
    ax1.set_xticks(xtick_positions)
    ax1.set_xticklabels(xtick_labels, rotation=45, ha='right')
    ax1.set_xlabel('Tiempo')
    ax1.set_ylabel('Macronutrientes')
    ax1.grid(axis='y', linestyle='--', alpha=0.3)

    nombre_archivo = os.path.basename(archivo_master).replace('.csv', '')
    ax1.set_title(f'{nombre_archivo} - Macronutrientes por evento')

    # Fusión manual de los elementos de leyenda de ambos ejes para mostrar una única caja
    handles1, labels1 = ax1.get_legend_handles_labels()
    handles2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(handles1 + handles2, labels1 + labels2, loc='upper left')

    # === 7. Exportación del Artefacto Visual ===
    fig.tight_layout()
    os.makedirs(output_folder, exist_ok=True)
    output_filename = os.path.splitext(os.path.basename(archivo_master))[0] + '_macros.png'
    output_path = os.path.join(output_folder, output_filename)
    fig.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f'Grafica de macros guardada en: {output_path}')
    
    # Liberación de recursos de memoria gráfica asignados al objeto figura
    mostrar_y_liberar_figura(fig)
    return output_path


In [ ]:
def main(folder_path, output_folder):
    """
    Genera las graficas de macros para todos los archivos trusted de CGMacros.
    
    Args:
        folder_path (str): Ruta a la carpeta donde se encuentran los archivos CSV.
        output_folder (str): Carpeta donde se guardarán las imágenes generadas.

    Returns:
        None.
    """
    for filename in os.listdir(folder_path):
        if filename.endswith('.csv'):
            file_path = os.path.join(folder_path, filename)
            graficar_macros_por_evento(str(file_path), output_folder)
            

if __name__ == "__main__":
    folder_path = TRUSTED_DIR
    output_folder = r'C:\Users\regin\Documents\GitHub\TFG-pruebas\Analysis\CGMacros\Gráficas macronutrientes'
    main(folder_path, output_folder)

### C2. Análisis Multivariante

#### C2.1 Scatter plot - Gráficos de dispersión

Explicación de las Soluciones Incorporadas

1. **Gestión de Carpetas Automatizada**: En cada una de las 4 funciones específicas se genera una variable `sub_dir` uniendo la ruta base con el nombre exacto de la prueba (ej: `Glucosa Dexcom vs Libre`). El script crea la carpeta automáticamente si no existe gracias a `os.makedirs(sub_dir, exist_ok=True)`.
2. **`pd.merge_asof` para Datos Reales**: En el análisis *Dexcom vs Libre*, el código aísla las muestras válidas e independientes de cada sensor y las cruza buscando proximidad temporal con un límite de tolerancia de 3 minutos, esquivando las falsas tendencias que provoca la interpolación.
3. **Control del *Shift* según el remuestreo**: 
   * Para el gráfico de ejercicio se usan datos cada 5 minutos reales, haciendo que `shift(-6)` equivalga fisiológicamente a adelantar la glucosa exactamente 30 minutos.
   * Para las comidas se aprovecha la continuidad lineal minuto a minuto para no perder la hora exacta del evento de ingesta, por lo que el retraso se aplica directamente con `shift(-30)`.
4. **Degradados Multi-Paleta**: El gráfico de macronutrientes calcula la relación de grasas + proteínas frente a los hidratos. Luego, mediante los mapas `Purples` y `Oranges`, genera intensidades visuales de color proporcionales a dicha complejidad alimentaria, mostrando dos leyendas de barras independientes al costado derecho del lienzo.

In [ ]:
def _cargar_y_preparar_tiempo(ruta_archivo):
    """
    Carga un archivo CSV y establece la columna 'time' como el índice temporal principal.

    Realiza la lectura del fichero, valida la presencia obligatoria del campo temporal, 
    transforma los registros a tipo datetime eliminando valores inválidos o nulos, y 
    devuelve la estructura ordenada cronológicamente por su nuevo índice.

    Args:
        ruta_archivo (str): Ruta del sistema del archivo CSV que se va a procesar.

    Returns:
        pd.DataFrame: Estructura de datos indexada por tiempo (DatetimeIndex) y 
            ordenada de forma ascendente.

    Raises:
        ValueError: Si la columna 'time' no se encuentra presente dentro del archivo.

    Ejemplos:
        >>> df_preparado = _cargar_y_preparar_tiempo("Data/U101_glucose.csv")
        >>> print(isinstance(df_preparado.index, pd.DatetimeIndex))
        True

    Recomendaciones:
        - Asegurar que el archivo de entrada contenga marcas temporales legibles por 
          los analizadores estándar de Pandas para evitar la pérdida masiva de filas nulas.
        - Utilizar este método como paso inicial de ingesta en flujos de series temporales.
    """
    # === 1. Ingesta de Datos y Validación Estructural ===
    df = pd.read_csv(ruta_archivo)
    if 'time' not in df.columns:
        raise ValueError(f"El archivo {ruta_archivo} no contiene la columna 'time'.")
    
    # === 2. Conversión y Limpieza Temporal ===
    # Transforma las cadenas a objetos datetime; los errores de formato se convierten en NaT
    df['time'] = pd.to_datetime(df['time'], errors='coerce')
    
    # Elimina filas sin fecha válida, ordena cronológicamente y transfiere el campo al índice
    df = df.dropna(subset=['time']).sort_values('time').set_index('time')
    
    return df


def _obtener_timestamps_reales(serie, paso_minutos):
    """
    Genera una cuadrícula temporal regular equidistante basada en los límites de la serie.

    Filtra los valores no numéricos y nulos de la serie temporal para establecer los puntos 
    de inicio y fin reales. A partir de estos extremos, construye un índice temporal regular 
    con la frecuencia en minutos especificada, sirviendo como rejilla base para identificar 
    registros originales frente a procesos de interpolación.

    Args:
        serie (pd.Series): Serie de datos indexada por tiempo (DatetimeIndex) que se va a analizar.
        paso_minutos (int): Intervalo de tiempo en minutos para establecer la frecuencia de la rejilla.

    Returns:
        pd.DatetimeIndex: Índice temporal regular generado o un DatetimeIndex vacío si la serie 
            no contiene datos válidos tras la limpieza.

    Ejemplos:
        >>> # Asumiendo una serie con datos cada 5 minutos
        >>> indices = _obtener_timestamps_reales(serie_glucosa, paso_minutos=5)
        >>> print(indices)
        DatetimeIndex(['2026-01-01 00:00:00', ...], dtype='datetime64[ns]', freq='5min')

    Recomendaciones:
        - Asegurar que el índice de la serie de entrada sea estrictamente un DatetimeIndex 
          para evitar errores de atributos al extraer los extremos temporales.
        - Utilizar un valor entero positivo para 'paso_minutos' que se ajuste a la tasa de 
          muestreo teórica del dispositivo de medición (ej. 5 para CGM Dexcom).
    """
    # === 1. Limpieza y Depuración de la Serie ===
    # Forzado de conversión a tipo numérico; los elementos no válidos se transforman en NaN
    serie_limpia = pd.to_numeric(serie, errors='coerce')
    
    # Exclusión de registros nulos y valores iguales a cero
    serie_limpia = serie_limpia[(serie_limpia.notna()) & (serie_limpia != 0)]
    
    if serie_limpia.empty:
        return pd.DatetimeIndex([])
        
    # === 2. Extracción de Límites Temporales ===
    # Aislamiento del primer y último Timestamp exactos del índice para acotar la rejilla
    inicio = serie_limpia.index[0]
    fin = serie_limpia.index[-1]
    
    # === 3. Generación de la Rejilla Temporal ===
    # Construcción del rango de fechas regular con la frecuencia muestral indicada
    return pd.date_range(start=inicio, end=fin, freq=f"{paso_minutos}min")


# =========================================================================
# 1. ANÁLISIS DE SENSORES (DEXCOM VS LIBRE)
# =========================================================================

def graficar_dexcom_vs_libre(df, user_id, out_dir):
    """
    Realiza una validación y comparación estadística directa entre sensores Dexcom y Libre.

    Aísla las muestras temporales reales de ambos dispositivos mediante rejillas teóricas, 
    cruza los datos utilizando un criterio de proximidad temporal estricto (tolerancia máxima 
    de 3 minutos) y genera un análisis de correlación de Spearman junto con un gráfico de 
    dispersión que incluye la línea de identidad.

    Args:
        df (pd.DataFrame): DataFrame indexado por tiempo con las lecturas de glucosa de ambos sensores.
        user_id (str): Identificador único del sujeto bajo análisis.
        out_dir (str): Directorio raíz del sistema donde se exportará la subcarpeta de resultados.

    Returns:
        None: La función no retorna ningún objeto, exporta el gráfico (.png) directamente si 
            se cumple el número mínimo de registros emparejados.

    Ejemplos:
        >>> graficar_dexcom_vs_libre(df_usuario, "U101", "Analysis/Plots/")

    Recomendaciones:
        - Asegurar la carga de 'spearmanr' desde scipy.stats y de 'mostrar_y_liberar_figura' 
          en el entorno antes de la ejecución.
        - Verificar que las columnas 'glucose_dexcom' y 'glucose_libre' existan en el DataFrame.
    """
    # === 1. Configuración de Directorios y Rejillas Temporales ===
    sub_dir = os.path.join(out_dir, "Glucosa Dexcom vs Libre")
    os.makedirs(sub_dir, exist_ok=True)
    
    # Generación de rejillas teóricas (5 min para Dexcom, 15 min para Libre) para aislar datos reales
    rejilla_dex = _obtener_timestamps_reales(df['glucose_dexcom'], 5)
    rejilla_lib = _obtener_timestamps_reales(df['glucose_libre'], 15)
    
    if rejilla_dex.empty or rejilla_lib.empty:
        print(f"[{user_id}] Datos nativos insuficientes para Dexcom vs Libre.")
        return
        
    # Reindexación para eliminar valores interpolados y mantener únicamente lecturas nativas
    dexcom_real = df[['glucose_dexcom']].reindex(rejilla_dex).dropna()
    libre_real = df[['glucose_libre']].reindex(rejilla_lib).dropna()
    
    # === 2. Cruce de Datos por Proximidad Temporal ===
    # Emparejamiento asincrónico buscando el registro más cercano en un entorno de +/- 3 minutos
    df_cruces = pd.merge_asof(
        dexcom_real, libre_real, left_index=True, right_index=True,
        direction='nearest', tolerance=pd.Timedelta('3min')
    ).dropna()

    # Validación del volumen mínimo de muestras emparejadas para el cálculo estadístico
    if len(df_cruces) < 5:
        return

    # === 3. Cálculo de Correlación No Paramétrica ===
    rho, p_val = spearmanr(df_cruces['glucose_dexcom'], df_cruces['glucose_libre'])

    # === 4. Construcción del Gráfico de Dispersión ===
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.scatter(df_cruces['glucose_dexcom'], df_cruces['glucose_libre'], alpha=0.5, color='tab:purple')
    
    # Definición de límites dinámicos garantizando un área visual mínima (50 a 300 mg/dL)
    lims = [min(df_cruces.min().min(), 50), max(df_cruces.max().max(), 300)]
    ax.plot(lims, lims, 'k--', alpha=0.5, label='Línea de Identidad (Y=X)')
    
    # Inyección de métricas estadísticas en un cuadro de texto flotante
    ax.text(0.05, 0.90, f"Spearman r: {rho:.3f}\np-val: {p_val:.3e}\nn = {len(df_cruces)}", 
            transform=ax.transAxes, fontsize=10, bbox=dict(boxstyle="round", alpha=0.1))

    # Personalización estética del gráfico
    ax.set_title(f"Validación de Sensores (Datos Reales): {user_id}")
    ax.set_xlabel("Glucosa Dexcom Real (mg/dL)")
    ax.set_ylabel("Glucosa Libre Real (mg/dL)")
    ax.set_xlim(lims)
    ax.set_ylim(lims)
    ax.grid(True, linestyle=':', alpha=0.6)
    ax.legend(loc='lower right')
    plt.tight_layout()
    
    # === 5. Almacenamiento y Cierre ===
    filename = os.path.join(sub_dir, f"{user_id}_dexcom_vs_libre.png")
    plt.savefig(filename, dpi=300)
    mostrar_y_liberar_figura(fig)


# =========================================================================
# 2. ANÁLISIS DE WEARABLE (HR VS CALORÍAS)
# =========================================================================

def graficar_hr_vs_calorias(df, user_id, out_dir):
    """
    Evalúa la correlación minuto a minuto entre la frecuencia cardíaca y el gasto calórico del wearable.

    Aísla las variables de ritmo cardíaco y calorías activas, descarta registros nulos o 
    iguales a cero para evitar sesgos por inactividad o desconexión del dispositivo, y calcula 
    el coeficiente de Spearman junto con un gráfico de dispersión de alta densidad.

    Args:
        df (pd.DataFrame): DataFrame indexado por tiempo con las métricas del wearable.
        user_id (str): Identificador único del sujeto bajo análisis.
        out_dir (str): Directorio raíz del sistema donde se exportará la subcarpeta de resultados.

    Returns:
        None: La función no retorna ningún objeto, exporta el gráfico (.png) directamente si 
            se cumple el número mínimo de registros válidos.

    Ejemplos:
        >>> graficar_hr_vs_calorias(df_wearable, "U102", "Analysis/Plots/")

    Recomendaciones:
        - Asegurar la carga previa de 'spearmanr' desde scipy.stats en el entorno de ejecución.
        - Utilizar esta función preferiblemente con datos de resolución minutal para capturar 
          correctamente la dinámica de la actividad física.
    """
    # === 1. Configuración de Directorios y Limpieza de Datos ===
    sub_dir = os.path.join(out_dir, "HR vs Calorías Actividad")
    os.makedirs(sub_dir, exist_ok=True)
    
    # Selección de variables y eliminación de valores nulos
    df_plot = df[['heart_rate', 'calories_activity']].dropna()
    
    # Exclusión de ceros para conservar únicamente periodos con actividad registrada y medición real
    df_plot = df_plot[(df_plot['calories_activity'] > 0) & (df_plot['heart_rate'] > 0)]
    
    # Validación del volumen mínimo de muestras para el cálculo de la correlación
    if len(df_plot) < 5:
        return
        
    # === 2. Análisis Estadístico de Correlación ===
    rho, p_val = spearmanr(df_plot['heart_rate'], df_plot['calories_activity'])

    # === 3. Construcción Gráfica de la Dispersión ===
    fig, ax = plt.subplots(figsize=(9, 5.5))
    
    # Uso de alfa bajo (0.2) y tamaño reducido (s=15) debido a la alta densidad de puntos minutales
    ax.scatter(df_plot['heart_rate'], df_plot['calories_activity'], alpha=0.2, color='tab:red', s=15)
    
    # Inyección del coeficiente de correlación y tamaño muestral en el plano visual
    ax.text(0.05, 0.90, f"Spearman r: {rho:.3f}\nn = {len(df_plot)}", 
            transform=ax.transAxes, fontsize=10, bbox=dict(boxstyle="round", alpha=0.1))

    # Personalización estética y rejillas del gráfico
    ax.set_title(f"Sincronización de Actividad Minuto a Minuto: {user_id}")
    ax.set_xlabel("Frecuencia Cardíaca (bpm)")
    ax.set_ylabel("Calorías de Actividad (kcal/min)")
    ax.grid(True, linestyle=':', alpha=0.6)
    plt.tight_layout()
    
    # === 4. Almacenamiento y Cierre de Recursos ===
    filename = os.path.join(sub_dir, f"{user_id}_hr_vs_calorias.png")
    plt.savefig(filename, dpi=300)
    mostrar_y_liberar_figura(fig)


# =========================================================================
# 3. ANÁLISIS EJERCICIO VS GLUCOSA (DEXCOM + SHIFTS)
# =========================================================================

def graficar_dexcom_vs_actividad_shift(df, user_id, out_dir, minutos_shift):
    """
    Analiza el efecto diferido del ejercicio sobre los niveles de glucosa aplicando un desfase temporal.

    Desplaza la serie de glucosa hacia el pasado el número de minutos especificado para evaluar 
    el impacto rezagado de las calorías de actividad. Filtra los puntos correspondientes a registros 
    reales del sensor (frecuencia de 5 minutos) y exporta un gráfico de dispersión con el cálculo 
    de correlación de Spearman.

    Args:
        df (pd.DataFrame): DataFrame indexado por tiempo con las lecturas del sensor y del wearable.
        user_id (str): Identificador único del sujeto bajo estudio.
        out_dir (str): Carpeta raíz del sistema dedicada a la salida de resultados.
        minutos_shift (int): Cantidad de minutos a desplazar la serie (valor negativo para retrasar glucosa).

    Returns:
        None: La función guarda la figura directamente en el almacenamiento local.

    Ejemplos:
        >>> graficar_dexcom_vs_actividad_shift(df_sujeto, "U101", "Analysis/Plots/", 30)

    Recomendaciones:
        - Verificar que la serie temporal contenga suficientes periodos con gasto calórico activo 
          mayor a cero para poder realizar un análisis de dispersión representativo.
        - Asegurar el alineamiento y consistencia minutal del DataFrame antes de aplicar el método shift.
    """
    # === 1. Configuración de Directorios y Desfase Temporal ===
    sub_dir = os.path.join(out_dir, f"Glucosa Dexcom vs Actividad Shift {minutos_shift}min")
    os.makedirs(sub_dir, exist_ok=True)
    
    # Aplicación del desplazamiento sobre el eje temporal continuo minuto a minuto
    df_temp = df.copy()
    df_temp['glucose_shifted'] = df_temp['glucose_dexcom'].shift(-minutos_shift)
    
    # === 2. Filtrado e Indexación de Datos Reales ===
    # Recuperación de marcas de tiempo teóricas para descartar puntos interpolados artificialmente
    rejilla_dex = _obtener_timestamps_reales(df['glucose_dexcom'], 5)
    if rejilla_dex.empty:
        return
        
    # Reindexación sobre la rejilla nativa y exclusión de fases en reposo (calorías = 0)
    df_reales = df_temp[['calories_activity', 'glucose_shifted']].reindex(rejilla_dex).dropna()
    df_plot = df_reales[df_reales['calories_activity'] > 0]
    
    # Validación del tamaño muestral mínimo requerido para el cálculo
    if len(df_plot) < 5:
        print(f"[{user_id}] Puntos insuficientes para Actividad con Shift de {minutos_shift}min.")
        return
        
    # === 3. Evaluación de Correlación Estadísticas ===
    rho, p_val = spearmanr(df_plot['calories_activity'], df_plot['glucose_shifted'])

    # === 4. Exportación del Gráfico Individual ===
    fig_save, ax_save = plt.subplots(figsize=(10, 6))
    ax_save.scatter(df_plot['calories_activity'], df_plot['glucose_shifted'], alpha=0.6, color='tab:blue', s=30)
    
    # Inyección de las métricas de correlación calculadas
    ax_save.text(0.05, 0.90, f"Spearman r: {rho:.3f}\np-val: {p_val:.3f}", transform=ax_save.transAxes, fontsize=10, bbox=dict(boxstyle="round", alpha=0.1))
    
    # Configuración de etiquetas y formato visual
    ax_save.set_title(f"Efecto del Ejercicio (Glucosa retrasada +{minutos_shift} min): {user_id}")
    ax_save.set_xlabel("Calorías de Actividad (kcal/min)")
    ax_save.set_ylabel("Glucosa Dexcom Posterior (mg/dL)")
    ax_save.grid(True, linestyle=':', alpha=0.6)
    plt.tight_layout()
    
    # Grabado físico en el disco y liberación del objeto gráfico
    fig_save.savefig(os.path.join(sub_dir, f"{user_id}_dexcom_vs_actividad_{minutos_shift}min.png"), dpi=300)
    plt.close(fig_save)


def _dibujar_subplot_actividad(df, ax, minutos_shift, user_id, out_dir):
    """
    Helper interno para renderizar el gráfico de dispersión de actividad en un eje compartido.

    Calcula de forma idéntica el desfase y filtrado de datos para un sujeto, dibuja la 
    distribución sobre un objeto Axes preexistente (para composición de figuras múltiples), 
    y paralelamente exporta la imagen individual limpia en su directorio correspondiente.

    Args:
        df (pd.DataFrame): DataFrame indexado por tiempo con las lecturas del sensor y del wearable.
        ax (matplotlib.axes.Axes): Región o subgráfico donde se pintará la dispersión actual.
        minutos_shift (int): Cantidad de minutos a desplazar la serie temporal.
        user_id (str): Identificador único del sujeto analizado.
        out_dir (str): Directorio raíz para la exportación de ficheros gráficos.

    Returns:
        bool: Retorna True si el procesamiento y dibujo fueron exitosos, o False si 
            los datos resultaron insuficientes.

    Recomendaciones:
        - Utilizar esta función dentro de estructuras de bucles que configuren cuadrículas (subplots) 
          para la comparación simultánea de múltiples desfases temporales.
    """
    # === 1. Inicialización y Simulación del Desfase ===
    sub_dir = os.path.join(out_dir, f"Glucosa Dexcom vs Actividad Shift {minutos_shift}min")
    os.makedirs(sub_dir, exist_ok=True)
    
    df_temp = df.copy()
    df_temp['glucose_shifted'] = df_temp['glucose_dexcom'].shift(-minutos_shift)
    
    # === 2. Extracción de Puntos Reales y Filtrado de Actividad ===
    rejilla_dex = _obtener_timestamps_reales(df['glucose_dexcom'], 5)
    if rejilla_dex.empty:
        return False
        
    df_reales = df_temp[['calories_activity', 'glucose_shifted']].reindex(rejilla_dex).dropna()
    df_plot = df_reales[df_reales['calories_activity'] > 0]
    
    if len(df_plot) < 5:
        return False
        
    # === 3. Renderizado sobre el Eje Compartido (Subplot) ===
    rho, p_val = spearmanr(df_plot['calories_activity'], df_plot['glucose_shifted'])
    ax.scatter(df_plot['calories_activity'], df_plot['glucose_shifted'], alpha=0.6, color='tab:blue', s=30)
    
    ax.text(0.05, 0.90, f"Spearman r: {rho:.3f}\np-val: {p_val:.3f}", 
            transform=ax.transAxes, fontsize=9, bbox=dict(boxstyle="round", alpha=0.1))
    
    ax.set_title(f"Shift +{minutos_shift} min")
    ax.set_xlabel("Calorías de Actividad (kcal/min)")
    ax.set_ylabel("Glucosa Dexcom Posterior (mg/dL)")
    ax.grid(True, linestyle=':', alpha=0.6)
    
    # === 4. Exportación Independiente del Gráfico Limpio ===
    fig_save, ax_save = plt.subplots(figsize=(10, 6))
    ax_save.scatter(df_plot['calories_activity'], df_plot['glucose_shifted'], alpha=0.6, color='tab:blue', s=30)
    ax_save.text(0.05, 0.90, f"Spearman r: {rho:.3f}\np-val: {p_val:.3f}", transform=ax_save.transAxes, fontsize=10, bbox=dict(boxstyle="round", alpha=0.1))
    ax_save.set_title(f"Efecto del Ejercicio (Glucosa retrasada +{minutos_shift} min): {user_id}")
    ax_save.set_xlabel("Calorías de Actividad (kcal/min)")
    ax_save.set_ylabel("Glucosa Dexcom Posterior (mg/dL)")
    ax_save.grid(True, linestyle=':', alpha=0.6)
    plt.tight_layout()
    
    fig_save.savefig(os.path.join(sub_dir, f"{user_id}_dexcom_vs_actividad_{minutos_shift}min.png"), dpi=300)
    plt.close(fig_save)
    
    return True


# =========================================================================================
# 4. ANÁLISIS DE MACRONUTRIENTES ABSOLUTOS VS GLUCOSA (+ SHIFTS)
# =========================================================================================

def graficar_glucosa_vs_macronutrientes(df, ax, fig_padre, minutos_shift, user_id, out_dir, cbar_pos, sensor_a_graficar):
    """
    Representa el impacto de los carbohidratos en la glucosa posprandial sobre un eje específico y exporta la figura.

    Aplica un desfase temporal a la glucosa para analizar la respuesta metabólica retardada tras la ingesta. 
    Mide la complejidad nutricional de las comidas mediante el ratio grasas+proteínas frente a carbohidratos, 
    proyecta la dispersión usando mapas de color según el tipo de sensor seleccionado (Dexcom/Libre) 
    y genera en paralelo un archivo gráfico de forma individual con su propia barra de color matemática.

    Args:
        df (pd.DataFrame): DataFrame indexado por tiempo con registros de sensores y macronutrientes.
        ax (matplotlib.axes.Axes): Subgráfico o eje específico de una matriz mayor donde se pintará la dispersión.
        fig_padre (matplotlib.figure.Figure): Objeto figura principal que actúa como contenedor de la composición.
        minutos_shift (int): Minutos de desplazamiento temporal aplicados a las series de glucosa.
        user_id (str): Identificador único del sujeto bajo análisis.
        out_dir (str): Directorio raíz del sistema donde se estructurarán las carpetas de salida.
        cbar_pos (list): Lista de coordenadas [left, bottom, width, height] para ubicar la barra de color.
        sensor_a_graficar (str): Cadena de texto selectora ('dexcom' o cualquier otro valor para 'libre').

    Returns:
        bool: Retorna True si la visualización y la exportación se completaron correctamente, o False 
            si el conjunto de datos resultante tras el filtrado contiene menos de 3 registros válidos.

    Recomendaciones:
        - Asegurar el mapeo de color correcto mediante la importación previa de matplotlib.colors como mcolors.
        - Utilizar esta función como un componente modular dentro de bucles diseñados para matrices de subplots.
    """
    # === 1. Configuración de Directorios y Desplazamiento Temporal ===
    sub_dir = os.path.join(out_dir, f"Glucosa vs Carbs Shift {minutos_shift}min")
    os.makedirs(sub_dir, exist_ok=True)
    
    # Simulación de las series desfasadas minuto a minuto de forma continua
    df_shifted = df.copy()
    df_shifted['dexcom_shifted'] = df_shifted['glucose_dexcom'].shift(-minutos_shift)
    df_shifted['libre_shifted'] = df_shifted['glucose_libre'].shift(-minutos_shift)
    
    # === 2. Filtrado Nutricional y Cálculo de Complejidad ===
    # Exclusión de registros sin ingesta registrada de carbohidratos
    df_comidas = df_shifted[df_shifted['carbs'] > 0].copy()
    if df_comidas.empty:
        return False
        
    # Estimación del índice de complejidad añadiendo un valor infinitesimal para evitar divisiones por cero
    df_comidas['ratio_complejidad'] = (df_comidas['fat'] + df_comidas['protein']) / (df_comidas['carbs'] + 1e-5)
    norm_cbar = mcolors.Normalize(vmin=0.0, vmax=1.5)
    
    # === 3. Configuración Estética Dinámica por Sensor ===
    if sensor_a_graficar == 'dexcom':
        cmap = plt.get_cmap('Purples_r')
        col_shifted = 'dexcom_shifted'
        label_sensor = 'Dexcom'
    else:
        cmap = plt.get_cmap('Oranges_r')
        col_shifted = 'libre_shifted'
        label_sensor = 'Libre'

    # Depuración de valores nulos en las variables que se van a graficar
    df_plot = df_comidas[['carbs', col_shifted, 'ratio_complejidad']].dropna()
    if len(df_plot) < 3:
        return False

    # Cálculo del coeficiente de correlación de Spearman
    rho, p_val = spearmanr(df_plot['carbs'], df_plot[col_shifted])
    
    # Ajuste de contraste para asegurar la legibilidad de los marcadores con tonalidades claras
    v_norm = norm_cbar(df_plot['ratio_complejidad'])
    colors = cmap(0.25 + v_norm * 0.65)  

    # === 4. Renderizado sobre el Eje de la Matriz General ===
    ax.scatter(df_plot['carbs'], df_plot[col_shifted], c=colors, 
               edgecolor='none', s=65, label=f"{label_sensor} (r: {rho:.2f}, p: {p_val:.1e})", alpha=0.9)

    ax.set_title(f"{label_sensor} | Shift +{minutos_shift} min", fontsize=10, weight='bold')
    ax.set_xlabel("Carbohidratos Ingeridos (g)", fontsize=9)
    ax.set_ylabel("Glucosa Postprandial (mg/dL)", fontsize=9)
    ax.grid(True, linestyle=':', alpha=0.5)
    ax.legend(loc='upper left', frameon=True, facecolor="white", framealpha=0.9, fontsize=8.5)
    ax.margins(y=0.15)
    
    # Adición de la barra de color vinculada al lienzo maestro compartido
    texto_definicion = r"Ratio: $\frac{\mathrm{Grasas} + \mathrm{Proteínas}}{\mathrm{Carbohidratos}}$"
    cbar_ax = fig_padre.add_axes(cbar_pos)
    fig_padre.colorbar(plt.cm.ScalarMappable(norm=norm_cbar, cmap=cmap), cax=cbar_ax, label=f'Complejidad {label_sensor}.\n{texto_definicion}')

    # =========================================================================
    # EXPORTACIÓN FÍSICA INDIVIDUAL LIMPIA POR SENSOR (A su subcarpeta clásica)
    # =========================================================================
    fig_save, ax_save = plt.subplots(figsize=(11, 7))
    ax_save.scatter(df_plot['carbs'], df_plot[col_shifted], c=colors, edgecolor='none', s=80, alpha=0.9)
    ax_save.text(0.95, 0.95, f"Spearman r: {rho:.2f}\np: {p_val:.1e}", transform=ax_save.transAxes, fontsize=10, ha='right', va='top', bbox=dict(boxstyle="round", facecolor="white", alpha=0.8, edgecolor="gray"))
    ax_save.set_title(f"Respuesta {label_sensor} (+{minutos_shift}min) vs Carbohidratos: {user_id}")
    ax_save.set_xlabel("Carbohidratos Ingeridos (g)")
    ax_save.set_ylabel("Glucosa Postprandial (mg/dL)")
    ax_save.grid(True, linestyle=':', alpha=0.5)
    
    # Configuración de márgenes y anclaje de la barra de color independiente de la figura aislada
    plt.subplots_adjust(right=0.85)
    cbar_s = fig_save.add_axes([0.88, 0.20, 0.02, 0.6])
    fig_save.colorbar(plt.cm.ScalarMappable(norm=norm_cbar, cmap=cmap), cax=cbar_s, label=f"Complejidad Nutricional\n{texto_definicion}")
    
    # Grabado físico en disco y liberación del objeto gráfico
    fig_save.savefig(os.path.join(sub_dir, f"{user_id}_{sensor_a_graficar}_vs_macronutrientes_{minutos_shift}min.png"), dpi=300)
    plt.close(fig_save)
    
    return True


def _dibujar_subplot_macronutrientes(df, ax, fig_padre, minutos_shift, user_id, out_dir, cbar_pos, sensor_a_graficar):
    """
    Helper interno encargado de proyectar un scatter nutricional en composiciones de matrices 2x2.

    Replica de forma exacta el tratamiento analítico de desfases temporales y cálculo de ratios 
    de complejidad macro-nutricional. Pinta sobre la cuadrícula del lienzo compartido y exporta, 
    de forma aislada, el archivo gráfico correspondiente a la combinación actual sin mezclas.

    Args:
        df (pd.DataFrame): DataFrame indexado por tiempo con registros de sensores y macronutrientes.
        ax (matplotlib.axes.Axes): Eje o cuadrícula secundaria del lienzo donde se representará el scatter.
        fig_padre (matplotlib.figure.Figure): Objeto figura principal que actúa como contenedor de la matriz.
        minutos_shift (int): Minutos aplicados para retrasar o adelantar la respuesta glucémica.
        user_id (str): Identificador único del sujeto en análisis.
        out_dir (str): Directorio del sistema donde se guardarán las carpetas organizadas por turnos.
        cbar_pos (list): Posición específica para anclar la barra de color en la figura contenedora.
        sensor_a_graficar (str): Selector del sensor activo a evaluar ('dexcom' o 'libre').

    Returns:
        bool: Retorna True si el subgráfico se dibujó con éxito, o False si el volumen de datos 
            es menor al umbral mínimo tras limpiar nulos.

    Recomendaciones:
        - Mantener la correspondencia estricta en el paso del parámetro 'fig_padre' para evitar la 
          superposición accidental de ejes de colorbar sobre las áreas de dibujo de las distribuciones.
    """
    # === 1. Configuración de Directorios y Desfase de Series ===
    sub_dir = os.path.join(out_dir, f"Glucosa vs Carbs Shift {minutos_shift}min")
    os.makedirs(sub_dir, exist_ok=True)
    
    # Simulación de las series desplazadas en el tiempo minuto a minuto
    df_shifted = df.copy()
    df_shifted['dexcom_shifted'] = df_shifted['glucose_dexcom'].shift(-minutos_shift)
    df_shifted['libre_shifted'] = df_shifted['glucose_libre'].shift(-minutos_shift)
    
    # === 2. Extracción de Eventos de Comida y Proporciones ===
    df_comidas = df_shifted[df_shifted['carbs'] > 0].copy()
    if df_comidas.empty:
        return False
        
    # Cálculo del índice macro-nutricional añadiendo un epsilon para mitigar divisiones por cero
    df_comidas['ratio_complejidad'] = (df_comidas['fat'] + df_comidas['protein']) / (df_comidas['carbs'] + 1e-5)
    norm_cbar = mcolors.Normalize(vmin=0.0, vmax=1.5)
    
    # === 3. Configuración Dinámica de Mapas de Color ===
    if sensor_a_graficar == 'dexcom':
        cmap = plt.get_cmap('Purples_r')
        col_shifted = 'dexcom_shifted'
        label_sensor = 'Dexcom'
    else:
        cmap = plt.get_cmap('Oranges_r')
        col_shifted = 'libre_shifted'
        label_sensor = 'Libre'

    # Eliminación de nulos en las variables destinadas a la representación bidimensional
    df_plot = df_comidas[['carbs', col_shifted, 'ratio_complejidad']].dropna()
    if len(df_plot) < 3:
        return False

    rho, p_val = spearmanr(df_plot['carbs'], df_plot[col_shifted])
    
    # Ajuste para optimizar el rango cromático visible de los marcadores
    v_norm = norm_cbar(df_comidas['ratio_complejidad'])
    colors = cmap(0.25 + v_norm * 0.65)  

    # === 4. Representación en el Eje de la Matriz General ===
    ax.scatter(df_plot['carbs'], df_plot[col_shifted], c=colors, 
               edgecolor='none', s=65, label=f"{label_sensor} (r: {rho:.2f}, p: {p_val:.1e})", alpha=0.9)

    ax.set_title(f"{label_sensor} | Shift +{minutos_shift} min", fontsize=10, weight='bold')
    ax.set_xlabel("Carbohidratos Ingeridos (g)", fontsize=9)
    ax.set_ylabel("Glucosa Postprandial (mg/dL)", fontsize=9)
    ax.grid(True, linestyle=':', alpha=0.5)
    ax.legend(loc='upper left', frameon=True, facecolor="white", framealpha=0.9, fontsize=8.5)
    ax.margins(y=0.15)
    
    # Vinculación de la escala de color sobre el lienzo contenedor
    texto_definicion = r"Ratio: $\frac{\mathrm{Grasas} + \mathrm{Proteínas}}{\mathrm{Carbohidratos}}$"
    cbar_ax = fig_padre.add_axes(cbar_pos)
    fig_padre.colorbar(plt.cm.ScalarMappable(norm=norm_cbar, cmap=cmap), cax=cbar_ax, label=f'Complejidad {label_sensor}.\n{texto_definicion}')

    # === 5. Exportación Física del Subgráfico Aislado ===
    fig_save, ax_save = plt.subplots(figsize=(11, 7))
    ax_save.scatter(df_plot['carbs'], df_plot[col_shifted], c=colors, edgecolor='none', s=80, alpha=0.9)
    ax_save.text(0.95, 0.95, f"Spearman r: {rho:.2f}\np: {p_val:.1e}", transform=ax_save.transAxes, fontsize=10, ha='right', va='top', bbox=dict(boxstyle="round", facecolor="white", alpha=0.8, edgecolor="gray"))
    ax_save.set_title(f"Respuesta {label_sensor} (+{minutos_shift}min) vs Carbohidratos: {user_id}")
    ax_save.set_xlabel("Carbohidratos Ingeridos (g)")
    ax_save.set_ylabel("Glucosa Postprandial (mg/dL)")
    ax_save.grid(True, linestyle=':', alpha=0.5)
    
    plt.subplots_adjust(right=0.85)
    cbar_s = fig_save.add_axes([0.88, 0.20, 0.02, 0.6])
    fig_save.colorbar(plt.cm.ScalarMappable(norm=norm_cbar, cmap=cmap), cax=cbar_s, label=f"Complejidad Nutricional\n{texto_definicion}")
    
    # Grabado físico final y cierre del recurso del subgráfico
    fig_save.savefig(os.path.join(sub_dir, f"{user_id}_{sensor_a_graficar}_vs_macronutrientes_{minutos_shift}min.png"), dpi=300)
    plt.close(fig_save)
    
    return True


# ========================================================================
# 5. ANÁLISIS DE MACRONUTRIENTES ABSOLUTOS VS DELTAS GLUCOSA (+ SHIFTS)
# ========================================================================

def graficar_delta_glucosa_vs_macronutrientes(df, user_id, out_dir):
    """
    Genera una matriz comparativa 2x2 del incremento neto de glucosa y exporta gráficos individuales.

    Calcula la variación posprandial (Delta) a los 30 y 60 minutos del evento para ambos sensores 
    (Dexcom y Libre). Representa el impacto de la ingesta de carbohidratos en una figura unificada, 
    codificando la complejidad nutricional por color, y exporta en paralelo los cuatro gráficos 
    limpios a sus carpetas correspondientes junto con la composición integrada.

    Args:
        df (pd.DataFrame): DataFrame indexado por tiempo con las lecturas de sensores y variables nutricionales.
        user_id (str): Identificador único del sujeto bajo análisis.
        out_dir (str): Directorio raíz donde se estructurarán las subcarpetas de los ficheros de salida.

    Returns:
        None: La función no retorna ningún objeto, gestiona la persistencia de imágenes directamente.

    Recomendaciones:
        - Asegurar que la constante global 'AGRUPACIONES_DIR' esté previamente declarada en el Notebook.
        - Verificar la disponibilidad del paquete matplotlib.colors mapeado como mcolors antes de ejecutar.
    """
    # === 1. Configuración de Directorios de Almacenamiento ===
    sub_dir_30 = os.path.join(out_dir, "Delta Glucosa vs Carbs Shift 30min")
    sub_dir_60 = os.path.join(out_dir, "Delta Glucosa vs Carbs Shift 60min")
    os.makedirs(sub_dir_30, exist_ok=True)
    os.makedirs(sub_dir_60, exist_ok=True)
    os.makedirs(AGRUPACIONES_DIR, exist_ok=True) 
    
    # === 2. Simulación de Desfases y Cálculo de Variaciones Netas ===
    df_delta = df.copy()
    for shift in [30, 60]:
        df_delta[f'dexcom_delta_{shift}'] = df_delta['glucose_dexcom'].shift(-shift) - df_delta['glucose_dexcom']
        df_delta[f'libre_delta_{shift}'] = df_delta['glucose_libre'].shift(-shift) - df_delta['glucose_libre']
        
    # === 3. Aislamiento de Registros Nutricionales ===
    df_comidas = df_delta[df_delta['carbs'] > 0].copy()
    if df_comidas.empty:
        print(f"[{user_id}] Sin registros de comidas para el análisis de Delta Glucosa.")
        return

    df_comidas['ratio_complejidad'] = (df_comidas['fat'] + df_comidas['protein']) / (df_comidas['carbs'] + 1e-5)
    norm_cbar = mcolors.Normalize(vmin=0.0, vmax=1.5)
    texto_definicion = r"Ratio de Complejidad: $\frac{\mathrm{Grasas\ (g)} + \mathrm{Proteínas\ (g)}}{\mathrm{Carbohidratos\ (g)}}$"
    
    # === 4. Inicialización del Lienzo de la Matriz 2x2 ===
    fig_del, axs_del = plt.subplots(2, 2, figsize=(17, 11), sharey='row')
    fig_mac_title = (f"Matriz de Variación Neta de Glucosa ($\Delta$Glucosa): {user_id}\n"
                     f"(Mide la diferencia de mg/dL desde el momento de la ingesta)")
    fig_del.suptitle(fig_mac_title, fontsize=12, weight='bold')
    
    cbar_positions = {
        ('dexcom', 30): [0.415, 0.56, 0.010, 0.30],
        ('dexcom', 60): [0.915, 0.56, 0.010, 0.30],
        ('libre', 30):  [0.415, 0.12, 0.010, 0.30],
        ('libre', 60):  [0.915, 0.12, 0.010, 0.30]
    }
    
    sensores = [('dexcom', 'Purples_r', 'Dexcom'), ('libre', 'Oranges_r', 'Libre')]
    shifts = [30, 60]
    
    # === 5. Bucle Estructural para el Mapeo de la Matriz ===
    for row_idx, (sensor, cmap_name, label_sensor) in enumerate(sensores):
        cmap = plt.get_cmap(cmap_name)
        for col_idx, shift in enumerate(shifts):
            ax = axs_del[row_idx, col_idx]
            col_delta = f"{sensor}_delta_{shift}"
            
            df_plot = df_comidas[['carbs', col_delta, 'ratio_complejidad']].dropna()
            if len(df_plot) < 3:
                continue
                
            rho, p_val = spearmanr(df_plot['carbs'], df_plot[col_delta])
            
            v_norm = norm_cbar(df_plot['ratio_complejidad'])
            colors = cmap(0.25 + v_norm * 0.65) 
            
            leyenda_texto = f"{label_sensor} (r: {rho:.2f}, p: {p_val:.1e})"
            
            ax.scatter(df_plot['carbs'], df_plot[col_delta], c=colors, 
                       edgecolor='none', s=65, label=leyenda_texto, alpha=0.9)
            
            ax.set_title(f"{label_sensor} | $\Delta$Glucosa a los +{shift} min", fontsize=10, weight='bold')
            ax.set_xlabel("Carbohidratos Ingeridos (g)", fontsize=9)
            ax.set_ylabel(f"$\Delta$ Glucosa (mg/dL desde el inicio)", fontsize=9)
            ax.grid(True, linestyle=':', alpha=0.5)
            ax.legend(loc='upper left', frameon=True, facecolor="white", framealpha=0.9, fontsize=8.5)
            ax.margins(y=0.15)
            ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
            
            cbar_pos = cbar_positions[(sensor, shift)]
            cbar_ax = fig_del.add_axes(cbar_pos)
            fig_del.colorbar(plt.cm.ScalarMappable(norm=norm_cbar, cmap=cmap), cax=cbar_ax, label=f'Complejidad {label_sensor}.\n{texto_definicion}')
            
            # =========================================================================
            # EXPORTACIÓN FÍSICA DEL GRÁFICO INDEPENDIENTE (PNG ÚNICO)
            # =========================================================================
            sub_dir_save = sub_dir_30 if shift == 30 else sub_dir_60
            fig_save, ax_save = plt.subplots(figsize=(11, 7))
            ax_save.scatter(df_plot['carbs'], df_plot[col_delta], c=colors, edgecolor='none', s=80, alpha=0.9)
            ax_save.axhline(0, color='gray', linestyle='--', alpha=0.5)
            ax_save.text(0.95, 0.95, f"Spearman r: {rho:.2f}\np: {p_val:.1e}", transform=ax_save.transAxes, fontsize=10, ha='right', va='top', bbox=dict(boxstyle="round", facecolor="white", alpha=0.8, edgecolor="gray"))
            ax_save.set_title(f"Variación Neta {label_sensor} (+{shift}min) vs Carbohidratos: {user_id}")
            ax_save.set_xlabel("Carbohidratos Ingeridos (g)")
            ax_save.set_ylabel("$\Delta$ Glucosa Postprandial (mg/dL)")
            ax_save.grid(True, linestyle=':', alpha=0.5)
            
            plt.subplots_adjust(right=0.85)
            cbar_s = fig_save.add_axes([0.88, 0.20, 0.02, 0.6])
            fig_save.colorbar(plt.cm.ScalarMappable(norm=norm_cbar, cmap=cmap), cax=cbar_s, label=f"Complejidad Nutricional\n{texto_definicion}")
            
            fig_save.savefig(os.path.join(sub_dir_save, f"{user_id}_{sensor}_delta_glucosa_{shift}min.png"), dpi=300)
            plt.close(fig_save)

    # === 6. Ajustes Visuales de la Matriz General y Grabado Final ===
    plt.subplots_adjust(left=0.06, right=0.88, top=0.88, bottom=0.08, hspace=0.38, wspace=0.45)
    
    filename_matriz = os.path.join(AGRUPACIONES_DIR, f"{user_id}_comparativa_deltas.png")
    fig_del.savefig(filename_matriz, dpi=300)
    
    print(f"[OK] Renderizando matriz de Delta Glucosa 2x2 para {user_id}...")
    mostrar_y_liberar_figura(fig_del)


# ====================================
# ORQUESTADOR DEL PIPELINE INDIVIDUAL
# ====================================

def procesar_analisis_dispersion(input_dir, output_dir):
    """
    Ejecuta el pipeline principal de análisis analítico y generador de gráficos por participante.

    Escanea el directorio de entrada para procesar secuencialmente cada archivo de datos. 
    Para cada sujeto, carga la serie temporal y gestiona de forma ordenada la invocación 
    de los módulos de validación de sensores, sincronización de wearables, construcción de 
    paneles de actividad con desfase temporal y matrices combinadas de macronutrientes.

    Args:
        input_dir (str): Directorio del sistema donde se alojan los archivos CSV de los sujetos.
        output_dir (str): Directorio raíz destino donde se estructurarán los subficheros analíticos.

    Returns:
        None: La función no retorna objetos, coordina la ejecución y exportación de los módulos.

    Recomendaciones:
        - Asegurar que todas las funciones de graficación individuales y auxiliares llamadas 
          estén cargadas en el entorno antes de iniciar el orquestador.
        - Definir la variable global 'AGRUPACIONES_DIR' con antelación para garantizar el correcto 
          almacenamiento de las composiciones multicuadrícula.
    """
    # === 1. Rastreo e Ingesta de Ficheros de Datos ===
    archivos_csv = glob.glob(os.path.join(input_dir, "*.csv"))
    if not archivos_csv:
        print(f"[ERROR] No se encontraron archivos CSV en: {input_dir}")
        return

    # === 2. Iteración Secuencial por Participante ===
    for ruta_archivo in archivos_csv:
        nombre_completo = os.path.basename(ruta_archivo)
        user_id = os.path.splitext(nombre_completo)[0]
        print(f"\n>>> Analizando Participante: {user_id}")

        try:
            df = _cargar_y_preparar_tiempo(ruta_archivo)

            graficar_dexcom_vs_libre(df, user_id, output_dir)
            graficar_hr_vs_calorias(df, user_id, output_dir)

            # ==========================================
            # PARALELIZACIÓN DE ACTIVIDAD INDIVIDUAL (1X2 HORIZONTAL)
            # ==========================================
            fig_act, axs_act = plt.subplots(1, 2, figsize=(16, 6))
            fig_act.suptitle(f"Comparativa del Efecto del Ejercicio en Glucosa: {user_id}", fontsize=13, weight='bold')
            
            p1 = _dibujar_subplot_actividad(df, axs_act[0], 30, user_id, output_dir)
            p2 = _dibujar_subplot_actividad(df, axs_act[1], 60, user_id, output_dir)
            
            if p1 or p2:
                plt.tight_layout()
                os.makedirs(AGRUPACIONES_DIR, exist_ok=True)
                filename_act = os.path.join(AGRUPACIONES_DIR, f"{user_id}_comparativa_actividad.png")
                fig_act.savefig(filename_act, dpi=300)
                
                print(f"[OK] Renderizando comparativa de Actividad para {user_id}...")
                mostrar_y_liberar_figura(fig_act)
            else:
                plt.close(fig_act)

            # =========================================================================
            # MATRIZ 2X2 DE MACRONUTRIENTES ABSOLUTOS INDIVIDUAL (BARRAS MÁS PEGADAS)
            # =========================================================================
            fig_mac, axs_mac = plt.subplots(2, 2, figsize=(17, 11), sharey='row')
            fig_mac.suptitle(f"Matriz de Comparativa Nutricional Absoluta: {user_id}\n(Puntos oscuros indican MENOR proporción de Grasas + Proteínas)", fontsize=12, weight='bold')
            
            cbar_dex_30 = [0.415, 0.56, 0.010, 0.30]
            cbar_dex_60 = [0.915, 0.56, 0.010, 0.30]
            cbar_lib_30 = [0.415, 0.12, 0.010, 0.30]
            cbar_lib_60 = [0.915, 0.12, 0.010, 0.30]
            
            m1 = _dibujar_subplot_macronutrientes(df, axs_mac[0, 0], fig_mac, 30, user_id, output_dir, cbar_dex_30, 'dexcom')
            m2 = _dibujar_subplot_macronutrientes(df, axs_mac[0, 1], fig_mac, 60, user_id, output_dir, cbar_dex_60, 'dexcom')
            
            m3 = _dibujar_subplot_macronutrientes(df, axs_mac[1, 0], fig_mac, 30, user_id, output_dir, cbar_lib_30, 'libre')
            m4 = _dibujar_subplot_macronutrientes(df, axs_mac[1, 1], fig_mac, 60, user_id, output_dir, cbar_lib_60, 'libre')
            
            if m1 or m2 or m3 or m4:
                plt.subplots_adjust(left=0.06, right=0.88, top=0.88, bottom=0.08, hspace=0.38, wspace=0.45)
                os.makedirs(AGRUPACIONES_DIR, exist_ok=True)
                filename_mac = os.path.join(AGRUPACIONES_DIR, f"{user_id}_comparativa_macronutrientes.png")
                fig_mac.savefig(filename_mac, dpi=300)
                
                print(f"[OK] Renderizando matriz de Macronutrientes 2x2 para {user_id}...")
                mostrar_y_liberar_figura(fig_mac)
            else:
                plt.close(fig_mac)

            # =========================================================================
            # MATRIZ 2X2 DE VARIACIÓN NETA (DELTAS) INDIVIDUAL
            # =========================================================================
            graficar_delta_glucosa_vs_macronutrientes(df, user_id, output_dir)

        except Exception as e:
            print(f"[ERROR] Falló el procesamiento completo en {nombre_completo}: {e}")


In [ ]:
if __name__ == "__main__":    
    procesar_analisis_dispersion(TRUSTED_DIR, SCATTER_DIR)

Creamos una tabla resumen con los resultados de los distintos coeficientes de Spearman, sus p-valores y el tamaño muestral por participante.

In [ ]:
def _obtener_timestamps_reales(serie, paso_minutos):
    """
    Genera una cuadrícula temporal regular equidistante basada en los límites de la serie.

    Filtra los valores no numéricos y nulos de la serie temporal para establecer los puntos 
    de inicio y fin reales. A partir de estos extremos, construye un índice temporal regular 
    con la frecuencia en minutos especificada, sirviendo como rejilla base para identificar 
    registros originales frente a procesos de interpolación.

    Args:
        serie (pd.Series): Serie de datos indexada por tiempo (DatetimeIndex) que se va a analizar.
        paso_minutos (int): Intervalo de tiempo en minutos para establecer la frecuencia de la rejilla.

    Returns:
        pd.DatetimeIndex: Índice temporal regular generado o un DatetimeIndex vacío si la serie 
            no contiene datos válidos tras la limpieza.

    Ejemplos:
        >>> # Asumiendo una serie con datos cada 5 minutos
        >>> indices = _obtener_timestamps_reales(serie_glucosa, paso_minutos=5)
        >>> print(indices)
        DatetimeIndex(['2026-01-01 00:00:00', ...], dtype='datetime64[ns]', freq='5min')

    Recomendaciones:
        - Asegurar que el índice de la serie de entrada sea estrictamente un DatetimeIndex 
          para evitar errores de atributos al extraer los extremos temporales.
        - Utilizar un valor entero positivo para 'paso_minutos' que se ajuste a la tasa de 
          muestreo teórica del dispositivo de medición (ej. 5 para CGM Dexcom).
    """
    # === 1. Limpieza y Depuración de la Serie ===
    # Forzado de conversión a tipo numérico; los elementos no válidos se transforman en NaN
    serie_limpia = pd.to_numeric(serie, errors='coerce')
    
    # Exclusión de registros nulos y valores iguales a cero
    serie_limpia = serie_limpia[(serie_limpia.notna()) & (serie_limpia != 0)]
    
    if serie_limpia.empty:
        return pd.DatetimeIndex([])
        
    # === 2. Extracción de Límites Temporales ===
    # Aislamiento del primer y último Timestamp exactos del índice para acotar la rejilla
    inicio = serie_limpia.index[0]
    fin = serie_limpia.index[-1]
    
    # === 3. Generación de la Rejilla Temporal ===
    # Construcción del rango de fechas regular con la frecuencia muestral indicada
    return pd.date_range(start=inicio, end=fin, freq=f"{paso_minutos}min")


def calcular_metricas_csv(input_dir, csv_out_path):
    """
    Genera un archivo CSV global con el consolidado de coeficientes de correlación del pipeline.

    Recorre secuencialmente los archivos de los participantes para estimar los coeficientes 
    de Spearman (r), p-valores y tamaños muestrales (n) asociados a cada análisis: validación 
    de sensores, sincronización de wearables, y los desfases temporales (30 y 60 min) de 
    actividad física, macronutrientes absolutos y variaciones netas (deltas).

    Args:
        input_dir (str): Directorio del sistema donde se alojan los archivos CSV de los sujetos.
        csv_out_path (str): Ruta completa del sistema donde se guardará la matriz consolidada (.csv).

    Returns:
        None: La función no retorna ningún objeto, exporta los datos directamente al disco duro.

    Ejemplos:
        >>> calcular_metricas_csv("Data/Raw/", "Analysis/Matriz_Estadistica.csv")

    Recomendaciones:
        - Asegurar la definición y accesibilidad de la función '_obtener_timestamps_reales' 
          antes de realizar la llamada al orquestador estadístico.
        - Verificar que el directorio destino de 'csv_out_path' cuente con permisos de escritura.
    """
    # === 1. Validación de Directorios e Ingesta Inicial ===
    archivos_csv = glob.glob(os.path.join(input_dir, "*.csv"))
    if not archivos_csv:
        print(f"[ERROR] No se encontraron archivos CSV en: {input_dir}")
        return

    # Lista donde acumularemos las filas de la tabla
    tabla_resumen = []

    print(f"Calculando matriz estadística para {len(archivos_csv)} archivos...")
    
    # === 2. Bucle de Procesamiento por Participante ===
    for ruta_archivo in archivos_csv:
        nombre_completo = os.path.basename(ruta_archivo)
        user_id = os.path.splitext(nombre_completo)[0]
        
        try:
            # Lectura y depuración del eje temporal cronológico del archivo
            df = pd.read_csv(ruta_archivo)
            df['time'] = pd.to_datetime(df['time'], errors='coerce')
            df = df.dropna(subset=['time']).sort_values('time').set_index('time')
            
            # -----------------------------------------------------------------
            # 1. MÉTRICA: VALIDACIÓN DE SENSORES (Dexcom vs Libre)
            # -----------------------------------------------------------------
            # Aislamiento de lecturas emparejadas en ventanas de proximidad
            rejilla_dex = _obtener_timestamps_reales(df['glucose_dexcom'], 5)
            rejilla_lib = _obtener_timestamps_reales(df['glucose_libre'], 15)
            
            if not rejilla_dex.empty and not rejilla_lib.empty:
                dexcom_real = df[['glucose_dexcom']].reindex(rejilla_dex).dropna()
                libre_real = df[['glucose_libre']].reindex(rejilla_lib).dropna()
                df_cruces = pd.merge_asof(
                    dexcom_real, libre_real, left_index=True, right_index=True,
                    direction='nearest', tolerance=pd.Timedelta('3min')
                ).dropna()
                
                # Almacenamiento estadístico de validación si se supera el umbral de puntos
                if len(df_cruces) >= 5:
                    rho, p_val = spearmanr(df_cruces['glucose_dexcom'], df_cruces['glucose_libre'])
                    tabla_resumen.append({
                        'Sujeto': user_id, 'Análisis': 'Validación Sensores', 
                        'Variable_X': 'Glucosa Dexcom Real', 'Variable_Y': 'Glucosa Libre Real',
                        'Ventana_Shift': '0 min', 'Spearman_r': rho, 'p_value': p_val, 'N_Muestras': len(df_cruces)
                    })

            # -----------------------------------------------------------------
            # 2. MÉTRICA: SINCRONIZACIÓN WEARABLE (HR vs Calorías)
            # -----------------------------------------------------------------
            # Depuración de registros minutales de actividad física activa
            df_hr = df[['heart_rate', 'calories_activity']].dropna()
            df_hr = df_hr[(df_hr['calories_activity'] > 0) & (df_hr['heart_rate'] > 0)]
            if len(df_hr) >= 5:
                rho, p_val = spearmanr(df_hr['heart_rate'], df_hr['calories_activity'])
                tabla_resumen.append({
                    'Sujeto': user_id, 'Análisis': 'Sincronización Wearable', 
                    'Variable_X': 'Frecuencia Cardíaca', 'Variable_Y': 'Calorías Actividad',
                    'Ventana_Shift': '0 min', 'Spearman_r': rho, 'p_value': p_val, 'N_Muestras': len(df_hr)
                })

            # -----------------------------------------------------------------
            # BUCLE PARA VENTANAS TEMPORALES (30 y 60 min)
            # -----------------------------------------------------------------
            for shift in [30, 60]:
                
                # 3. MÉTRICA: Glucosa Dexcom vs Actividad Física (Shift)
                # Estimación de la respuesta retardada al ejercicio sobre puntos de muestreo reales
                df_temp = df.copy()
                df_temp['glucose_shifted'] = df_temp['glucose_dexcom'].shift(-shift)
                rejilla_dex = _obtener_timestamps_reales(df['glucose_dexcom'], 5)
                
                if not rejilla_dex.empty:
                    df_reales = df_temp[['calories_activity', 'glucose_shifted']].reindex(rejilla_dex).dropna()
                    df_plot = df_reales[df_reales['calories_activity'] > 0]
                    if len(df_plot) >= 5:
                        rho, p_val = spearmanr(df_plot['calories_activity'], df_plot['glucose_shifted'])
                        tabla_resumen.append({
                            'Sujeto': user_id, 'Análisis': 'Glucosa vs Ejercicio', 
                            'Variable_X': 'Calorías Actividad', 'Variable_Y': f'Glucosa Dexcom (+{shift}min)',
                            'Ventana_Shift': f'{shift} min', 'Spearman_r': rho, 'p_value': p_val, 'N_Muestras': len(df_plot)
                        })

                # 4. MÉTRICA: Impacto de Macronutrientes Absoluto (Carbs vs Glucosa Shift)
                # Análisis de la ingesta alimentaria frente a la respuesta glucémica posprandial retardada
                df_shifted = df.copy()
                df_shifted['dexcom_shifted'] = df_shifted['glucose_dexcom'].shift(-shift)
                df_shifted['libre_shifted'] = df_shifted['glucose_libre'].shift(-shift)
                df_comidas = df_shifted[df_shifted['carbs'] > 0].copy()
                
                if not df_comidas.empty:
                    # Dexcom Absoluto
                    df_dex = df_comidas[['carbs', 'dexcom_shifted']].dropna()
                    if len(df_dex) >= 3:
                        rho, p_val = spearmanr(df_dex['carbs'], df_dex['dexcom_shifted'])
                        tabla_resumen.append({
                            'Sujeto': user_id, 'Análisis': 'Glucosa Absoluta vs Nutrición', 
                            'Variable_X': 'Carbohidratos Ingeridos', 'Variable_Y': f'Glucosa Dexcom (+{shift}min)',
                            'Ventana_Shift': f'{shift} min', 'Spearman_r': rho, 'p_value': p_val, 'N_Muestras': len(df_dex)
                        })
                    
                    # Libre Absoluto
                    df_lib = df_comidas[['carbs', 'libre_shifted']].dropna()
                    if len(df_lib) >= 3:
                        rho, p_val = spearmanr(df_lib['carbs'], df_lib['libre_shifted'])
                        tabla_resumen.append({
                            'Sujeto': user_id, 'Análisis': 'Glucosa Absoluta vs Nutrición', 
                            'Variable_X': 'Carbohidratos Ingeridos', 'Variable_Y': f'Glucosa Libre (+{shift}min)',
                            'Ventana_Shift': f'{shift} min', 'Spearman_r': rho, 'p_value': p_val, 'N_Muestras': len(df_lib)
                        })

                # 5. MÉTRICA: Variación Neta (Delta Glucosa vs Carbs)
                # Medición aritmética del cambio posprandial (mg/dL) desde el punto de inicio de la ingesta
                df_delta = df.copy()
                df_delta['dexcom_delta'] = df_delta['glucose_dexcom'].shift(-shift) - df_delta['glucose_dexcom']
                df_delta['libre_delta'] = df_delta['glucose_libre'].shift(-shift) - df_delta['glucose_libre']
                df_comidas_delta = df_delta[df_delta['carbs'] > 0].copy()
                
                if not df_comidas_delta.empty:
                    # Dexcom Delta
                    df_dex_d = df_comidas_delta[['carbs', 'dexcom_delta']].dropna()
                    if len(df_dex_d) >= 3:
                        rho, p_val = spearmanr(df_dex_d['carbs'], df_dex_d['dexcom_delta'])
                        tabla_resumen.append({
                            'Sujeto': user_id, 'Análisis': 'Delta Glucosa vs Nutrición', 
                            'Variable_X': 'Carbohidratos Ingeridos', 'Variable_Y': f'Delta Glucosa Dexcom (+{shift}min)',
                            'Ventana_Shift': f'{shift} min', 'Spearman_r': rho, 'p_value': p_val, 'N_Muestras': len(df_dex_d)
                        })
                    
                    # Libre Delta
                    df_lib_d = df_comidas_delta[['carbs', 'libre_delta']].dropna()
                    if len(df_lib_d) >= 3:
                        rho, p_val = spearmanr(df_lib_d['carbs'], df_lib_d['libre_delta'])
                        tabla_resumen.append({
                            'Sujeto': user_id, 'Análisis': 'Delta Glucosa vs Nutrición', 
                            'Variable_X': 'Carbohidratos Ingeridos', 'Variable_Y': f'Delta Glucosa Libre (+{shift}min)',
                            'Ventana_Shift': f'{shift} min', 'Spearman_r': rho, 'p_value': p_val, 'N_Muestras': len(df_lib_d)
                        })

        except Exception as e:
            # Captura de errores locales para evitar la interrupción del pipeline completo
            print(f"[ERROR] No se pudieron extraer métricas de {nombre_completo}: {e}")

    # === 3. Consolidación y Exportación de Resultados ===
    # Conversión de la lista de diccionarios en un DataFrame plano de Pandas
    df_global_resumen = pd.DataFrame(tabla_resumen)
    
    if not df_global_resumen.empty:
        # Guardamos el resultado en la raíz de la carpeta de dispersión
        df_global_resumen.to_csv(csv_out_path, index=False)
        print(f"\n[OK] Matriz global consolidada con éxito.")
        print(f"Archivo guardado en: {csv_out_path}")
        print(f"Total de registros estadísticos guardados: {len(df_global_resumen)}")
    else:
        print("[ADVERTENCIA] La tabla resumen está vacía. No se ha generado ningún CSV.")


In [ ]:
if __name__ == "__main__":
    # Ejecutamos la consolidación de métricas
    calcular_metricas_csv(TRUSTED_DIR, SPEARMAN_CSV)


Para el análisis correlacional de carácter poblacional, implementamos un algoritmo de segmentación por sujeto previo a la agregación de datos. 

Para cada individuo de forma aislada, se determina su marca de tiempo basal (origen) y se construye una rejilla temporal teórica basada en la frecuencia nativa de muestreo de cada dispositivo (5 minutos para Dexcom y 15 minutos para Libre). Mediante un proceso de reindexación exacta anclada a dicho origen individual, se eliminan los registros minutos a minuto derivados de la interpolación. 

Posteriormente, se realiza un acoplamiento por proximidad temporal (merge_asof, tolerancia < 3 min) a nivel de sujeto. La búsqueda por proximidad (merge_asof) se ejecuta únicamente en la función `graficar_poblacional_dexcom_vs_libre`. El resto de análisis de macros y actividad física utilizan cruces directos por índice de tiempo minuto a minuto (datos interpolados completos para macros y filtrado directo para actividad).

Finalmente, las muestras purificadas de toda la cohorte se concatenan para el cómputo de los coeficientes de Spearman globales.

En el caso del grupo 'Diabético' nos vemos forzados a eliminar registros que nos alteran nuestros coeficientes.

Fisiológicamente, comer por ejemplo 750g de carbohidratos en una sola ingesta es incompatible con la vida o un error de registro flagrante (puede ser que se hayan sumado las calorías totales de varios días, un error de tecleo al poner un cero de más, o un fallo en el wearable nutricional). Por ello aplicamos un filtro de calidad de datos para eliminar artefactos de entrada del usuario.

In [ ]:
# =========================================================================
# 1. ANÁLISIS POBLACIONAL DE SENSORES (DEXCOM VS LIBRE - PROXIMIDAD)
# =========================================================================

def graficar_poblacional_dexcom_vs_libre(df_global, out_dir):
    """
    Realiza un análisis de correlación colectiva entre los sensores Dexcom y Libre usando datos agregados.

    Agrupa la información por participante, aísla de forma independiente las marcas temporales 
    nativas de cada dispositivo mediante rejillas teóricas y consolida los cruces por proximidad 
    asincrónica en una única matriz poblacional para estimar el coeficiente de Spearman global.

    Args:
        df_global (pd.DataFrame): DataFrame unificado que contiene el histórico multisujeto 
            con las lecturas de ambos sensores y la columna 'user_id'.
        out_dir (str): Directorio raíz del sistema donde se localiza la carpeta de salidas analíticas.

    Returns:
        None: La función almacena el gráfico resultante (.png) directamente en el disco duro.

    Ejemplos:
        >>> graficar_poblacional_dexcom_vs_libre(df_consolidado, "Analysis/Plots/")

    Recomendaciones:
        - Asegurar que la función auxiliar '_obtener_timestamps_reales' se encuentre cargada 
          en el entorno para evitar interrupciones durante el bucle de agrupación.
        - Utilizar un valor de opacidad bajo (alpha=0.15) al graficar para prevenir la saturación 
          visual debido al elevado volumen de puntos distribuidos de la muestra colectiva.
    """
    # === 1. Configuración del Directorio de Salida ===
    sub_dir = os.path.join(out_dir, "Glucosa Dexcom vs Libre")
    os.makedirs(sub_dir, exist_ok=True)
    
    lista_cruces_reales = []
    
    # === 2. Bucle de Extracción e Intersección Temporal por Sujeto ===
    for user_id, df_sujeto in df_global.groupby('user_id', observed=False):
        df_sujeto = df_sujeto.copy()
        rejilla_dex = _obtener_timestamps_reales(df_sujeto['glucose_dexcom'], 5)
        rejilla_lib = _obtener_timestamps_reales(df_sujeto['glucose_libre'], 15)
        
        if rejilla_dex.empty or rejilla_lib.empty:
            continue
            
        # Filtrado de mediciones originales descartando valores interpolados
        dexcom_real = df_sujeto[['glucose_dexcom']].reindex(rejilla_dex).dropna()
        libre_real = df_sujeto[['glucose_libre']].reindex(rejilla_lib).dropna()
        
        if not dexcom_real.empty and not libre_real.empty:
            # Sincronización asincrónica basada en proximidad temporal con ventana de tolerancia
            cruces_sujeto = pd.merge_asof(
                dexcom_real, libre_real, left_index=True, right_index=True, 
                direction='nearest', tolerance=pd.Timedelta('3min')
            ).dropna()
            lista_cruces_reales.append(cruces_sujeto)
            
    # === 3. Validación y Concatenación de la Muestra Poblacional ===
    if not lista_cruces_reales:
        print("[ADVERTENCIA] Sin datos para el cruce poblacional de sensores.")
        return
        
    df_total = pd.concat(lista_cruces_reales)
    
    # Estimación estadística de correlación no paramétrica sobre el conjunto global
    rho, p_val = spearmanr(df_total.iloc[:, 0], df_total.iloc[:, 1])
    
    # === 4. Construcción Gráfica Colectiva ===
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.scatter(df_total.iloc[:, 0], df_total.iloc[:, 1], alpha=0.15, color='purple', s=10)
    
    # Delimitación de márgenes dinámicos asegurando un encuadre visual mínimo (50 a 300 mg/dL)
    lims = [min(df_total.min().min(), 50), max(df_total.max().max(), 300)]
    ax.plot(lims, lims, 'k--', alpha=0.5, label='Línea de Identidad (Y=X)')
    
    # Inyección de las métricas agrupadas en un cuadro de texto flotante
    ax.text(0.05, 0.90, f"Spearman Poblacional r: {rho:.3f}\np-val: {p_val:.3e}\nn = {len(df_total)}", 
            transform=ax.transAxes, fontsize=10, bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))
    
    # Ajustes estéticos de la distribución analítica
    ax.set_title("Validación Colectiva de Sensores (Datos Reales)")
    ax.set_xlabel("Glucosa Dexcom Real (mg/dL)")
    ax.set_ylabel("Glucosa Libre Real (mg/dL)")
    ax.set_xlim(lims)
    ax.set_ylim(lims)
    ax.grid(True, linestyle=':', alpha=0.5)
    plt.tight_layout()
    
    # === 5. Almacenamiento y Cierre de Recursos ===
    filename = os.path.join(sub_dir, "Poblacional_dexcom_vs_libre.png")
    fig.savefig(filename, dpi=300)
    mostrar_y_liberar_figura(fig)
    print(f"[OK] Guardado Poblacional: {filename}")


# =========================================================================
# 2. ANÁLISIS POBLACIONAL DE WEARABLE (HR VS CALORÍAS)
# =========================================================================

def graficar_poblacional_hr_vs_calorias(df_global, out_dir):
    """
    Evalúa la correlación de Spearman minuto a minuto a nivel colectivo para la telemetría del wearable.

    Consolida los registros de ritmo cardíaco y gasto energético por actividad de todos los 
    participantes en una única muestra global. Descarta periodos de inactividad o desconexión 
    (valores nulos o iguales a cero) y genera un diagrama de dispersión diseñado para alta densidad de puntos.

    Args:
        df_global (pd.DataFrame): DataFrame unificado que contiene la información combinada de 
            todos los sujetos del estudio.
        out_dir (str): Directorio raíz del sistema donde se exportará la subcarpeta de resultados.

    Returns:
        None: La función no retorna ningún objeto, exporta el gráfico (.png) directamente al disco.

    Ejemplos:
        >>> graficar_poblacional_hr_vs_calorias(df_consolidado, "Analysis/Plots/")

    Recomendaciones:
        - Asegurar la carga de 'spearmanr' desde scipy.stats en el entorno antes de la llamada.
        - Mantener un tamaño de marcador muy reducido (s=5) y una transparencia alta (alpha=0.05) 
          para mitigar el solapamiento de puntos debido al masivo volumen de registros minutales acumulados.
    """
    # === 1. Configuración del Directorio y Limpieza Colectiva ===
    # Guardamos en la misma carpeta que ya existía para el análisis individual
    sub_dir = os.path.join(out_dir, "HR vs Calorías Actividad")
    os.makedirs(sub_dir, exist_ok=True)
    
    # Aislamiento de métricas y eliminación de valores vacíos
    df_plot = df_global[['heart_rate', 'calories_activity']].dropna()
    
    # Exclusión de ceros para evaluar exclusivamente los periodos con ejercicio físico activo registrado
    df_plot = df_plot[(df_plot['calories_activity'] > 0) & (df_plot['heart_rate'] > 0)]
    
    # Validación del volumen mínimo de muestras agregadas para habilitar el cálculo estadístico
    if len(df_plot) < 10:
        return
        
    # === 2. Análisis Estadístico de Correlación Colectiva ===
    rho, p_val = spearmanr(df_plot['heart_rate'], df_plot['calories_activity'])
    
    # === 3. Construcción del Gráfico de Dispersión Poblacional ===
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Renderizado con opacidad ultra-baja para visualizar las zonas de mayor concentración de datos
    ax.scatter(df_plot['heart_rate'], df_plot['calories_activity'], alpha=0.05, color='tab:red', s=5)
    
    # Inyección de las métricas de correlación poblacional en un cuadro de texto flotante
    ax.text(0.05, 0.90, f"Spearman Poblacional r: {rho:.3f}\np-val: {p_val:.3e}\nn = {len(df_plot)} min", 
            transform=ax.transAxes, fontsize=10, bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))
            
    # Configuración de etiquetas y mallas estéticas
    ax.set_title("Sincronización Colectiva de Wearables Minuto a Minuto")
    ax.set_xlabel("Frecuencia Cardíaca (bpm)")
    ax.set_ylabel("Calorías de Actividad (kcal/min)")
    ax.grid(True, linestyle=':', alpha=0.5)
    plt.tight_layout()
    
    # === 4. Almacenamiento Físico del Artefacto Visual ===
    # NOMBRAMIENTO SOLICITADO: Guardado en la carpeta existente con nombre Poblacional
    filename = os.path.join(sub_dir, "Poblacional_hr_vs_calorias.png")
    fig.savefig(filename, dpi=300)
    mostrar_y_liberar_figura(fig)
    print(f"[OK] Guardado Poblacional: {filename}")


# =========================================================================
# 3. MATRIZ DE EJERCICIO POR GRUPOS METABÓLICOS (DEXCOM + SHIFTS)
# =========================================================================

def graficar_actividad_por_grupos_diabetes(df_global, out_dir):
    """
    Genera una matriz analítica poblacional 3x2 que evalúa el efecto del ejercicio según el fenotipo metabólico.

    Aplica desfases temporales (30 y 60 min) a la serie de glucosa Dexcom de todos los participantes 
    sobre marcas de tiempo reales. Segmenta la muestra agregada en tres subgrupos (No Diabético, 
    Prediabético y Diabético), computa las correlaciones de Spearman y exporta tanto los subgráficos 
    individuales limpios como la matriz integrada multisectorial.

    Args:
        df_global (pd.DataFrame): DataFrame consolidado con el histórico multisujeto que incluye las 
            lecturas de glucosa, calorías de actividad y la asignación del 'grupo_metabolico'.
        out_dir (str): Directorio raíz del sistema utilizado para estructurar las carpetas de salida.

    Returns:
        None: La función no retorna ningún objeto, gestiona la persistencia de imágenes directamente.

    Ejemplos:
        >>> graficar_actividad_por_grupos_diabetes(df_consolidado, "Analysis/Plots/")

    Recomendaciones:
        - Asegurar que la constante global 'AGRUPACIONES_DIR' esté previamente declarada en el Notebook 
          para evitar fallos de ruta al guardar el lienzo combinado.
        - Verificar que los strings del campo 'grupo_metabolico' coincidan exactamente con las llaves 
          de la lista 'grupos' ('No Diabético', 'Prediabético', 'Diabético') para prevenir matrices vacías.
    """
    # === 1. Inicialización de Carpetas y Estructuras de Desfase ===
    os.makedirs(AGRUPACIONES_DIR, exist_ok=True)
    
    lista_sujetos_shifted = []
    
    # === 2. Bucle de Simulación de Desplazamientos por Participante ===
    for user_id, df_sujeto in df_global.groupby('user_id', observed=False):
        df_sujeto = df_sujeto.copy()
        
        # Desplazamiento temporal continuo minuto a minuto para evaluar respuestas rezagadas
        df_sujeto['glucose_shifted_30'] = df_sujeto['glucose_dexcom'].shift(-30)
        df_sujeto['glucose_shifted_60'] = df_sujeto['glucose_dexcom'].shift(-60)
        
        # Filtrado basado en marcas de muestreo reales para descartar interpolaciones artificiales
        rejilla_dex = _obtener_timestamps_reales(df_sujeto['glucose_dexcom'], 5)
        if not rejilla_dex.empty:
            lista_sujetos_shifted.append(df_sujeto.reindex(rejilla_dex))
            
    if not lista_sujetos_shifted: 
        return
        
    df_completos = pd.concat(lista_sujetos_shifted)
    grupos = ['No Diabético', 'Prediabético', 'Diabético']
    shifts = [30, 60]

    # === 3. Configuración del Lienzo Multipanel de la Matriz ===
    fig, axs = plt.subplots(3, 2, figsize=(16, 15), sharey='row', sharex=True)
    fig.suptitle("Efecto Colectivo del Ejercicio Segregado por Fenotipo Metabólico", fontsize=12, weight='bold')
    
    # === 4. Bucle Estructural de Mapeo (Filas: Grupos Clínicos | Columnas: Desfases) ===
    for row_idx, grupo in enumerate(grupos):
        # Aislamiento del subconjunto correspondiente al fenotipo clínico actual
        df_grupo = df_completos[df_completos['grupo_metabolico'] == grupo]
        
        for col_idx, shift in enumerate(shifts):
            ax = axs[row_idx, col_idx]
            col_y = f'glucose_shifted_{shift}'
            
            # Limpieza local de nulos y exclusión de periodos sedentarios o en reposo (calorías = 0)
            df_plot = df_grupo[['calories_activity', col_y]].dropna()
            df_plot = df_plot[df_plot['calories_activity'] > 0]
            
            # Validación del umbral muestral mínimo para el subgráfico de la celda actual
            if len(df_plot) < 5:
                ax.text(0.5, 0.5, "Datos insuficientes", ha='center', va='center')
                continue
                
            # Estimación del coeficiente de correlación no paramétrica
            rho, p_val = spearmanr(df_plot['calories_activity'], df_plot[col_y])
            
            # Renderizado de la distribución de dispersión en el panel compartido
            ax.scatter(df_plot['calories_activity'], df_plot[col_y], alpha=0.4, color='tab:blue', s=15)
            ax.set_title(f"{grupo} | Shift +{shift} min", fontsize=10, weight='bold')
            ax.grid(True, linestyle=':', alpha=0.4)
            ax.text(0.95, 0.95, f"r: {rho:.2f}\np: {p_val:.2e}", transform=ax.transAxes, fontsize=8, ha='right', va='top', bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))
            
            # =========================================================================
            # EXPORTACIÓN LIMPIA INDIVIDUAL POR FENOTIPO (A las carpetas shift originales)
            # =========================================================================
            sub_dir_save = os.path.join(out_dir, f"Glucosa Dexcom vs Actividad Shift {shift}min")
            fig_save, ax_save = plt.subplots(figsize=(10, 6))
            ax_save.scatter(df_plot['calories_activity'], df_plot[col_y], alpha=0.4, color='tab:blue', s=30)
            ax_save.set_title(f"Efecto del Ejercicio (Glucosa retrasada +{shift} min): {grupo}")
            ax_save.grid(True, linestyle=':', alpha=0.6)
            plt.tight_layout()
            
            # Normalización del nombre del archivo eliminando caracteres especiales o acentos
            grupo_clean = grupo.replace('é', 'e').replace('á', 'a').replace(' ', '_') + "s"
            fig_save.savefig(os.path.join(sub_dir_save, f"{grupo_clean}_dexcom_vs_actividad_{shift}min.png"), dpi=300)
            plt.close(fig_save)

    # Ajustes geométricos finales para la correcta visualización espacial del panel unificado
    plt.subplots_adjust(left=0.06, right=0.95, top=0.92, bottom=0.06, hspace=0.35, wspace=0.25)
    
    # === 5. Redirección y Almacenamiento de la Matriz Agrupada ===
    filename_matriz = os.path.join(AGRUPACIONES_DIR, "Poblacional_matriz_ejercicio_por_grupos.png")
    fig.savefig(filename_matriz, dpi=300)
    mostrar_y_liberar_figura(fig)
    print(f"[OK] Matriz Agrupada Guardada: {filename_matriz}")


# =========================================================================================
# 4. MATRIZ DE MACRONUTRIENTES ABSOLUTOS POR GRUPOS METABÓLICOS (2X2 ANTERIOR -> AHORA 3X2)
# =========================================================================================

def graficar_macros_por_grupos_diabetes(df_global, out_dir):
    """
    Genera matrices de dispersión 1x2 por fenotipo y sensor para evaluar el impacto absoluto de los carbohidratos.

    Segmenta la población agregada según su grupo metabólico y recorre de forma cruzada cada 
    dispositivo (Dexcom/Libre) y desfase temporal (30 y 60 min). Codifica la complejidad nutricional 
    mediante una escala cromática basada en el ratio grasas+proteínas frente a carbohidratos, y exporta 
    tanto los subgráficos independientes como los paneles horizontales consolidados.

    Args:
        df_global (pd.DataFrame): DataFrame unificado con el histórico clínico de todos los 
            sujetos, incluyendo columnas de glucosa, macronutrientes y 'grupo_metabolico'.
        out_dir (str): Directorio raíz del sistema utilizado para estructurar las carpetas de salida.

    Returns:
        None: La función no retorna ningún objeto, gestiona la persistencia de imágenes directamente.

    Ejemplos:
        >>> graficar_macros_por_grupos_diabetes(df_consolidado, "Analysis/Plots/")

    Recomendaciones:
        - Asegurar que la constante global 'AGRUPACIONES_DIR' esté previamente declarada en el Notebook.
        - Verificar la disponibilidad del paquete matplotlib.colors mapeado como mcolors antes de ejecutar.
    """
    # === 1. Configuración de Directorios y Simulación de Desfases ===
    os.makedirs(AGRUPACIONES_DIR, exist_ok=True)
    
    lista_sujetos_shifted = []
    
    # Generación de series desplazadas minuto a minuto de manera continua para toda la población
    for user_id, df_sujeto in df_global.groupby('user_id', observed=False):
        df_sujeto = df_sujeto.copy()
        df_sujeto['dexcom_30'] = df_sujeto['glucose_dexcom'].shift(-30)
        df_sujeto['dexcom_60'] = df_sujeto['glucose_dexcom'].shift(-60)
        df_sujeto['libre_30'] = df_sujeto['glucose_libre'].shift(-30)
        df_sujeto['libre_60'] = df_sujeto['glucose_libre'].shift(-60)
        lista_sujetos_shifted.append(df_sujeto)
        
    df_global_shifted = pd.concat(lista_sujetos_shifted)
    grupos = ['No Diabético', 'Prediabético', 'Diabético']
    config_sensores = [('dexcom', 'Purples_r', 'Dexcom'), ('libre', 'Oranges_r', 'Libre')]
    shifts = [30, 60]
    norm_cbar = mcolors.Normalize(vmin=0.0, vmax=1.5)
    texto_definicion = r"Ratio de Complejidad: $\frac{\mathrm{Grasas\ (g)} + \mathrm{Proteínas\ (g)}}{\mathrm{Carbohidratos\ (g)}}$"

    # === 2. Bucle Estructural de Agrupación Fenotípica ===
    for grupo in grupos:
        df_grupo = df_global_shifted[df_global_shifted['grupo_metabolico'] == grupo]
        
        # Iteración secundaria cruzando por tipo de Sensor
        for sensor, cmap_name, label_sensor in config_sensores:
            fig, axs = plt.subplots(1, 2, figsize=(16, 7.5), sharey=True)
            fig.suptitle(f"Comparativa Nutricional: {grupo} ({label_sensor})", fontsize=11, weight='bold')
            plotted_any = False
            
            # === 3. Procesamiento y Mapeo Horizontal de Ventanas Temporales ===
            for col_idx, shift in enumerate(shifts):
                ax = axs[col_idx]
                col_y = f"{sensor}_{shift}"
                
                # Aislamiento exclusivo de registros correspondientes a eventos con ingesta
                df_comidas = df_grupo[df_grupo['carbs'] > 0].copy()
                df_plot = df_comidas[['carbs', col_y, 'fat', 'protein']].dropna()
                if df_plot.empty: 
                    continue
                
                # Estimación de la métrica macro-nutricional mitigando indeterminaciones por división
                df_plot['ratio_complejidad'] = (df_plot['fat'] + df_plot['protein']) / (df_plot['carbs'] + 1e-5)
                
                # Cálculo del coeficiente de correlación de Spearman sobre la muestra filtrada
                rho, p_val = spearmanr(df_plot['carbs'], df_plot[col_y])
                cmap = plt.get_cmap(cmap_name)
                
                # Ajuste dinámico de contraste visual para la escala cromática
                colors = cmap(0.25 + norm_cbar(df_plot['ratio_complejidad']) * 0.65)
                
                # Renderizado de la distribución sobre la cuadrícula del panel unificado
                ax.scatter(df_plot['carbs'], df_plot[col_y], c=colors, edgecolor='none', s=75, alpha=0.85, label=f"{label_sensor} (r: {rho:.2f}, p: {p_val:.1e})")
                ax.set_title(f"Shift +{shift} min")
                ax.grid(True, linestyle=':', alpha=0.5)
                ax.legend(loc='upper left', frameon=True, facecolor="white", framealpha=0.9, fontsize=9)
                plotted_any = True
                
                # Posicionamiento absoluto y renderizado de la escala cromática compartida en la matriz
                cbar_x_pos = 0.44 if col_idx == 0 else 0.93
                cbar_ax = fig.add_axes([cbar_x_pos, 0.12, 0.012, 0.65])
                fig.colorbar(plt.cm.ScalarMappable(norm=norm_cbar, cmap=cmap), cax=cbar_ax, label=f"Complejidad\n{texto_definicion}")

                # =========================================================================
                # EXPORTACIÓN INDIVIDUAL A CARPETA DE NUTRICIÓN ORIGINAL (Aislada por Sensor)
                # =========================================================================
                sub_dir_save = os.path.join(out_dir, f"Glucosa vs Carbs Shift {shift}min")
                fig_save, ax_save = plt.subplots(figsize=(11, 7))
                ax_save.scatter(df_plot['carbs'], df_plot[col_y], c=colors, edgecolor='none', s=80, alpha=0.85)
                ax_save.set_title(f"Respuesta {label_sensor} (+{shift}min): {grupo}")
                ax_save.grid(True, linestyle=':', alpha=0.5)
                
                plt.subplots_adjust(right=0.85)
                cbar_s1 = fig_save.add_axes([0.88, 0.15, 0.02, 0.6])
                fig_save.colorbar(plt.cm.ScalarMappable(norm=norm_cbar, cmap=cmap), cax=cbar_s1, label='Complejidad Nutricional')
                
                # Normalización del string eliminando caracteres especiales o acentos para el guardado físico
                grupo_clean = grupo.replace('é', 'e').replace('á', 'a').replace(' ', '_') + "s"
                fig_save.savefig(os.path.join(sub_dir_save, f"{grupo_clean}_{sensor}_vs_macronutrientes_{shift}min.png"), dpi=300)
                plt.close(fig_save)

            # === 4. Grabado y Cierre del Lienzo Consolidado Horizontal ===
            if plotted_any:
                plt.subplots_adjust(left=0.06, right=0.91, top=0.85, bottom=0.12, wspace=0.45)
                
                # REDIRECCIÓN: Guardamos el panel 1x2 agrupado en la nueva carpeta común
                grupo_fn = grupo.lower().replace(' ', '_')
                filename_agrupado = os.path.join(AGRUPACIONES_DIR, f"Poblacional_matriz_macros_absolutos_{grupo_fn}_{sensor}.png")
                fig.savefig(filename_agrupado, dpi=300)
                mostrar_y_liberar_figura(fig)
            else:
                plt.close(fig)


def _dibujar_subplot_macronutrientes(df, ax, fig_padre, minutos_shift, user_id, out_dir, sensor_a_graficar):
    """
    Helper interno encargado de proyectar la dispersión nutricional en una rejilla paralela 3x2.

    Replica el tratamiento analítico de desfases temporales y cálculo de proporciones de 
    complejidad macro-nutricional. Dibuja la distribución sobre un objeto Axes preexistente 
    optimizando el contraste cromático para lecturas continuas y exporta de forma aislada 
    el archivo gráfico correspondiente al sujeto en disco duro.

    Args:
        df (pd.DataFrame): DataFrame indexado por tiempo con registros de sensores y macronutrientes.
        ax (matplotlib.axes.Axes): Región o subgráfico de la rejilla donde se pintará la dispersión.
        fig_padre (matplotlib.figure.Figure): Objeto figura principal que actúa como contenedor de la matriz.
        minutos_shift (int): Cantidad de minutos aplicados para desplazar la serie temporal de glucosa.
        user_id (str): Identificador único del sujeto en análisis.
        out_dir (str): Directorio del sistema donde se guardarán las carpetas organizadas por turnos.
        sensor_a_graficar (str): Selector del dispositivo activo a evaluar ('dexcom' o 'libre').

    Returns:
        tuple (float, float) o tuple (None, None): Retorna una tupla con el coeficiente rho de Spearman 
            y su correspondiente p-valor, o valores nulos si los registros son insuficientes.

    Recomendaciones:
        - Asegurar el correcto mapeo de color mediante la importación previa de matplotlib.colors como mcolors.
        - Utilizar el retorno de métricas de esta función para registrar o imprimir resúmenes de rendimiento 
          de forma externa si se desea.
    """
    # === 1. Configuración de Directorios y Desfase de Series ===
    sub_dir = os.path.join(out_dir, f"Grupos_Glucosa vs Carbs Macronutrientes")
    os.makedirs(sub_dir, exist_ok=True)
    
    # Simulación de las series desfasadas de manera continua minuto a minuto
    df_shifted = df.copy()
    df_shifted['dexcom_shifted'] = df_shifted['glucose_dexcom'].shift(-minutos_shift)
    df_shifted['libre_shifted'] = df_shifted['glucose_libre'].shift(-minutos_shift)
    
    # === 2. Extracción de Eventos de Comida y Proporciones ===
    # Aislamiento exclusivo de registros correspondientes a ingestas con carbohidratos
    df_comidas = df_shifted[df_shifted['carbs'] > 0].copy()
    if df_comidas.empty:
        return None, None
        
    # Cálculo del índice macro-nutricional añadiendo un epsilon para mitigar divisiones por cero
    df_comidas['ratio_complejidad'] = (df_comidas['fat'] + df_comidas['protein']) / (df_comidas['carbs'] + 1e-5)
    norm_cbar = mcolors.Normalize(vmin=0.0, vmax=1.5)
    
    # === 3. Configuración Estética Dinámica según Sensor ===
    if sensor_a_graficar == 'dexcom':
        cmap = plt.get_cmap('Purples_r')
        col_shifted = 'dexcom_shifted'
        label_sensor = 'Dexcom'
    else:
        cmap = plt.get_cmap('Oranges_r')
        col_shifted = 'libre_shifted'
        label_sensor = 'Libre'

    # Eliminación de nulos en las variables destinadas a la representación bidimensional
    df_plot = df_comidas[['carbs', col_shifted, 'ratio_complejidad']].dropna()
    if len(df_plot) < 3:
        return None, None

    # Estimación del coeficiente de correlación de Spearman
    rho, p_val = spearmanr(df_plot['carbs'], df_plot[col_shifted])
    
    # Ajuste de contraste para asegurar la legibilidad de puntos de baja densidad cromática
    v_norm = norm_cbar(df_plot['ratio_complejidad'])
    colors = cmap(0.25 + v_norm * 0.65)  

    # === 4. Renderizado sobre el Eje de la Matriz General ===
    ax.scatter(df_plot['carbs'], df_plot[col_shifted], c=colors, 
               edgecolor='none', s=65, alpha=0.85)

    ax.grid(True, linestyle=':', alpha=0.4)
    ax.margins(y=0.12)
    
    # =========================================================================
    # GUARDADO INDEPENDIENTE EN DISCO
    # =========================================================================
    fig_save, ax_save = plt.subplots(figsize=(10, 6))
    ax_save.scatter(df_plot['carbs'], df_plot[col_shifted], c=colors, edgecolor='none', s=80, alpha=0.9)
    ax_save.text(0.95, 0.95, f"Spearman r: {rho:.2f}\np: {p_val:.1e}", transform=ax_save.transAxes, fontsize=10, ha='right', va='top', bbox=dict(boxstyle="round", facecolor="white", alpha=0.8, edgecolor="gray"))
    ax_save.set_title(f"Respuesta {label_sensor} (+{minutos_shift}min) vs Carbohidratos: {user_id}")
    ax_save.set_xlabel("Carbohidratos Ingeridos (g)")
    ax_save.set_ylabel("Glucosa Postprandial (mg/dL)")
    ax_save.grid(True, linestyle=':', alpha=0.5)
    
    # Posicionamiento manual y adición de la barra de color independiente de la figura aislada
    texto_definicion = r"Ratio: $\frac{\mathrm{Grasas} + \mathrm{Proteínas}}{\mathrm{Carbohidratos}}$"
    plt.subplots_adjust(right=0.85)
    cbar_s = fig_save.add_axes([0.88, 0.20, 0.02, 0.6])
    fig_save.colorbar(plt.cm.ScalarMappable(norm=norm_cbar, cmap=cmap), cax=cbar_s, label=f"Complejidad Nutricional\n{texto_definicion}")
    
    # Grabado en almacenamiento local y liberación de memoria
    fig_save.savefig(os.path.join(sub_dir, f"{user_id}_{sensor_a_graficar}_vs_macronutrientes_{minutos_shift}min.png"), dpi=300)
    plt.close(fig_save)
    
    return rho, p_val


# ========================================================================
# 5. MATRIZ DE RESPUESTA NETO (DELTA GLUCOSA) POR GRUPOS METABÓLICOS (3X2)
# ========================================================================

def graficar_delta_macros_por_grupos(df_global, out_dir):
    """
    Genera matrices de dispersión 1x2 por fenotipo y sensor para evaluar la variación neta (Delta).

    Segmenta la base de datos de forma colectiva según el grupo clínico y calcula el incremento 
    aritmético de mg/dL a los 30 y 60 minutos posteriores a la ingesta. Representa las distribuciones 
    mapeando la complejidad nutricional por color y exporta de forma paralela los gráficos aislados 
    y los paneles horizontales consolidados.

    Args:
        df_global (pd.DataFrame): DataFrame unificado con el histórico multisujeto que contiene 
            las lecturas de glucosa, variables nutricionales y la asignación del 'grupo_metabolico'.
        out_dir (str): Directorio raíz del sistema utilizado para estructurar las carpetas de salida.

    Returns:
        None: La función no retorna ningún objeto, gestiona la persistencia de imágenes directamente.

    Recomendaciones:
        - Asegurar que la constante global 'AGRUPACIONES_DIR' esté previamente declarada en el Notebook.
        - Verificar que las carpetas destino de deltas individuales existan o se creen previamente 
          para evitar fallos de escritura del sistema.
    """
    # === 1. Configuración de Directorios y Estimación de Variaciones ===
    os.makedirs(AGRUPACIONES_DIR, exist_ok=True)
    
    lista_sujetos_shifted = []
    
    # Simulación y cálculo aritmético de incrementos netos minuto a minuto para toda la población
    for user_id, df_sujeto in df_global.groupby('user_id', observed=False):
        df_sujeto = df_sujeto.copy()
        df_sujeto['dexcom_delta_30'] = df_sujeto['glucose_dexcom'].shift(-30) - df_sujeto['glucose_dexcom']
        df_sujeto['dexcom_delta_60'] = df_sujeto['glucose_dexcom'].shift(-60) - df_sujeto['glucose_dexcom']
        df_sujeto['libre_delta_30'] = df_sujeto['glucose_libre'].shift(-30) - df_sujeto['glucose_libre']
        df_sujeto['libre_delta_60'] = df_sujeto['glucose_libre'].shift(-60) - df_sujeto['glucose_libre']
        lista_sujetos_shifted.append(df_sujeto)
        
    df_global_shifted = pd.concat(lista_sujetos_shifted)
    grupos = ['No Diabético', 'Prediabético', 'Diabético']
    config_sensores = [('dexcom', 'Purples_r', 'Dexcom'), ('libre', 'Oranges_r', 'Libre')]
    shifts = [30, 60]
    norm_cbar = mcolors.Normalize(vmin=0.0, vmax=1.5)
    texto_definicion = r"Ratio de Complejidad: $\frac{\mathrm{Grasas\ (g)} + \mathrm{Proteínas\ (g)}}{\mathrm{Carbohidratos\ (g)}}$"

    # === 2. Bucle Estructural de Segmentación Clínica ===
    for grupo in grupos:
        df_grupo = df_global_shifted[df_global_shifted['grupo_metabolico'] == grupo]
        
        # Iteración secundaria basada en el tipo de Sensor
        for sensor, cmap_name, label_sensor in config_sensores:
            fig, axs = plt.subplots(1, 2, figsize=(16, 7.5), sharey=True)
            fig.suptitle(f"Matriz de Variación Neta ($\Delta$Glucosa): {grupo} ({label_sensor})", fontsize=11, weight='bold')
            plotted_any = False
            
            # === 3. Procesamiento y Mapeo Horizontal de Desfases (Columnas) ===
            for col_idx, shift in enumerate(shifts):
                ax = axs[col_idx]
                col_y = f"{sensor}_delta_{shift}"
                
                # Aislamiento analítico de eventos correspondientes a ingestas
                df_comidas = df_grupo[df_grupo['carbs'] > 0].copy()
                df_plot = df_comidas[['carbs', col_y, 'fat', 'protein']].dropna()
                if df_plot.empty: 
                    continue
                
                # Cálculo de la proporción nutricional añadiendo un epsilon infinitesimal
                df_plot['ratio_complejidad'] = (df_plot['fat'] + df_plot['protein']) / (df_plot['carbs'] + 1e-5)
                
                # Estimación del coeficiente de correlación de Spearman sobre la variación neta
                rho, p_val = spearmanr(df_plot['carbs'], df_plot[col_y])
                cmap = plt.get_cmap(cmap_name)
                
                # Configuración de contraste para optimizar la legibilidad cromática
                colors = cmap(0.25 + norm_cbar(df_plot['ratio_complejidad']) * 0.65)
                
                # Trazado de dispersión y adición de la línea basal neutra (Delta = 0)
                ax.scatter(df_plot['carbs'], df_plot[col_y], c=colors, edgecolor='none', s=75, alpha=0.85, label=f"{label_sensor} (r: {rho:.2f}, p: {p_val:.1e})")
                ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
                ax.set_title(f"Shift +{shift} min")
                ax.grid(True, linestyle=':', alpha=0.5)
                ax.legend(loc='upper left', frameon=True, facecolor="white", framealpha=0.9, fontsize=9)
                plotted_any = True
                
                # Dimensionamiento y adición de la escala de colores al panel de la matriz
                cbar_x_pos = 0.44 if col_idx == 0 else 0.93
                cbar_ax = fig.add_axes([cbar_x_pos, 0.12, 0.012, 0.65])
                fig.colorbar(plt.cm.ScalarMappable(norm=norm_cbar, cmap=cmap), cax=cbar_ax, label=f"Complejidad\n{texto_definicion}")

                # =========================================================================
                # EXPORTACIÓN INDIVIDUAL A CARPETA DELTA ORIGINAL (Aislada por Sensor)
                # =========================================================================
                sub_dir_save = os.path.join(out_dir, f"Delta Glucosa vs Carbs Shift {shift}min")
                fig_save, ax_save = plt.subplots(figsize=(11, 7))
                ax_save.scatter(df_plot['carbs'], df_plot[col_y], c=colors, edgecolor='none', s=80, alpha=0.85)
                ax_save.axhline(0, color='gray', linestyle='--', alpha=0.5)
                ax_save.set_title(f"Variación Neta {label_sensor} (+{shift}min): {grupo}")
                ax_save.grid(True, linestyle=':', alpha=0.5)
                
                plt.subplots_adjust(right=0.85)
                cbar_s = fig_save.add_axes([0.86, 0.20, 0.02, 0.6])
                fig_save.colorbar(plt.cm.ScalarMappable(norm=norm_cbar, cmap=cmap), cax=cbar_s, label='Complejidad Nutricional')
                
                # Depuración de caracteres especiales y acentos para el guardado físico del PNG
                grupo_clean = grupo.replace('é', 'e').replace('á', 'a').replace(' ', '_') + "s"
                fig_save.savefig(os.path.join(sub_dir_save, f"{grupo_clean}_{sensor}_delta_glucosa_{shift}min.png"), dpi=300)
                plt.close(fig_save)

            # === 4. Grabado y Cierre del Lienzo Consolidado de Deltas ===
            if plotted_any:
                plt.subplots_adjust(left=0.06, right=0.91, top=0.85, bottom=0.12, wspace=0.45)
                
                # REDIRECCIÓN: Guardamos la matriz delta 1x2 agrupada en la nueva carpeta unificada
                grupo_fn = grupo.lower().replace(' ', '_')
                filename_agrupado = os.path.join(AGRAPACIONES_DIR, f"Poblacional_matriz_deltas_{grupo_fn}_{sensor}.png")
                fig.savefig(filename_agrupado, dpi=300)
                mostrar_y_liberar_figura(fig)
            else:
                plt.close(fig)


def _dibujar_subplot_deltas(df, ax, fig_padre, minutos_shift, user_id, out_dir, sensor_a_graficar):
    """
    Helper interno encargado de proyectar la dispersión de deltas nutricionales en una rejilla paralela 3x2.

    Replica el tratamiento analítico de variaciones netas (Deltas) aritméticas y el cálculo de 
    proporciones de complejidad macro-nutricional. Dibuja la distribución sobre un objeto Axes 
    preexistente inyectando una línea basal neutra (Delta = 0), y exporta de forma aislada el 
    archivo gráfico correspondiente al sujeto en disco duro.

    Args:
        df (pd.DataFrame): DataFrame indexado por tiempo con registros de sensores y macronutrientes.
        ax (matplotlib.axes.Axes): Región o subgráfico de la rejilla donde se pintará la dispersión de deltas.
        fig_padre (matplotlib.figure.Figure): Objeto figura principal que actúa como contenedor de la matriz.
        minutos_shift (int): Cantidad de minutos aplicados para estimar la variación posprandial neta.
        user_id (str): Identificador único del sujeto en análisis.
        out_dir (str): Directorio del sistema donde se guardarán las carpetas organizadas por turnos.
        sensor_a_graficar (str): Selector del dispositivo activo a evaluar ('dexcom' o 'libre').

    Returns:
        tuple (float, float) o tuple (None, None): Retorna una tupla con el coeficiente rho de Spearman 
            y su correspondiente p-valor, o valores nulos si los registros son insuficientes.

    Recomendaciones:
        - Asegurar el correcto mapeo de color mediante la importación previa de matplotlib.colors como mcolors.
        - Utilizar el retorno de métricas de esta función para registrar o imprimir resúmenes de rendimiento 
          de forma externa si se desea.
    """
    # === 1. Configuración de Directorios de Almacenamiento ===
    sub_dir = os.path.join(out_dir, f"Grupos_Delta Glucosa vs Carbs")
    os.makedirs(sub_dir, exist_ok=True)
    
    df_delta = df.copy()
    col_delta = f"{sensor_a_graficar}_delta_{minutos_shift}"
    
    # === 2. Selección de Sensor y Cálculo Aritmético del Delta ===
    if sensor_a_graficar == 'dexcom':
        df_delta[col_delta] = df_delta['glucose_dexcom'].shift(-minutos_shift) - df_delta['glucose_dexcom']
        cmap = plt.get_cmap('Purples_r')
        label_sensor = 'Dexcom'
    else:
        df_delta[col_delta] = df_delta['glucose_libre'].shift(-minutos_shift) - df_delta['glucose_libre']
        cmap = plt.get_cmap('Oranges_r')
        label_sensor = 'Libre'
        
    # === 3. Extracción de Eventos de Comida y Proporciones ===
    df_comidas = df_delta[df_delta['carbs'] > 0].copy()
    if df_comidas.empty:
        return None, None
        
    # Cálculo del índice macro-nutricional añadiendo un epsilon para mitigar divisiones por cero
    df_comidas['ratio_complejidad'] = (df_comidas['fat'] + df_comidas['protein']) / (df_comidas['carbs'] + 1e-5)
    norm_cbar = mcolors.Normalize(vmin=0.0, vmax=1.5)
    
    # Eliminación de nulos en las variables destinadas a la representación bidimensional
    df_plot = df_comidas[['carbs', col_delta, 'ratio_complejidad']].dropna()
    if len(df_plot) < 3:
        return None, None

    # Estimación del coeficiente de correlación de Spearman sobre la variación neta
    rho, p_val = spearmanr(df_plot['carbs'], df_plot[col_delta])
    
    # Ajuste de contraste para asegurar la legibilidad de puntos de baja densidad cromática
    v_norm = norm_cbar(df_plot['ratio_complejidad'])
    colors = cmap(0.25 + v_norm * 0.65)  

    # === 4. Renderizado sobre el Eje de la Matriz General ===
    ax.scatter(df_plot['carbs'], df_plot[col_delta], c=colors, edgecolor='none', s=65, alpha=0.85)
    ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
    ax.grid(True, linestyle=':', alpha=0.4)
    ax.margins(y=0.12)
    
    # =========================================================================
    # GUARDADO INDEPENDIENTE EN DISCO
    # =========================================================================
    fig_save, ax_save = plt.subplots(figsize=(11, 7))
    ax_save.scatter(df_plot['carbs'], df_plot[col_delta], c=colors, edgecolor='none', s=80, alpha=0.9)
    ax_save.axhline(0, color='gray', linestyle='--', alpha=0.5)
    ax_save.text(0.95, 0.95, f"Spearman r: {rho:.2f}\np: {p_val:.1e}", transform=ax_save.transAxes, fontsize=10, ha='right', va='top', bbox=dict(boxstyle="round", facecolor="white", alpha=0.8, edgecolor="gray"))
    ax_save.set_title(f"Variación Neta {label_sensor} (+{minutos_shift}min) vs Carbohidratos: {user_id}")
    ax_save.set_xlabel("Carbohidratos Ingeridos (g)")
    ax_save.set_ylabel("$\Delta$ Glucosa Postprandial (mg/dL)")
    ax_save.grid(True, linestyle=':', alpha=0.5)
    
    # Posicionamiento manual y adición de la barra de color independiente de la figura aislada
    texto_definicion = r"Ratio: $\frac{\mathrm{Grasas} + \mathrm{Proteínas}}{\mathrm{Carbohidratos}}$"
    plt.subplots_adjust(right=0.82)
    cbar_s = fig_save.add_axes([0.86, 0.20, 0.02, 0.6])
    fig_save.colorbar(plt.cm.ScalarMappable(norm=norm_cbar, cmap=cmap), cax=cbar_s, label=f"Complejidad Nutricional\n{texto_definicion}")
    
    # Grabado en almacenamiento local y liberación de memoria
    fig_save.savefig(os.path.join(sub_dir, f"{user_id}_{sensor_a_graficar}_delta_{minutos_shift}min.png"), dpi=300)
    plt.close(fig_save)
    
    return rho, p_val


In [ ]:
# ====================================
# ORQUESTADOR DEL PIPELINE POBLACIONAL
# ====================================

def main_poblacional():
    """
    Orquestador principal que controla la carga unificada de la cohorte
    y ejecuta secuencialmente los 5 análisis poblacionales para el TFG.
    """
    print("=== INICIANDO PIPELINE DE ANÁLISIS POBLACIONAL PARA EL TFG ===")
    try:
        # 1. Carga unificada e inyección de fenotipos en memoria
        df_global = _cargar_y_unificar_dataset(TRUSTED_DIR)
        # FILTRO DE SEGURIDAD CLÍNICA: Excluimos errores de registro nutricional (> 300g de carbs, > 40g de fibra)
        # A. Excluye errores de registro de carbs (> 300g)
        # B. Excluye errores biológicamente imposibles de fibra (> 40g por comida)
        condicion_carbs = (df_global['carbs'] <= 300) | (df_global['carbs'].isna())
        condicion_fibra = (df_global['fiber'] <= 40) | (df_global['fiber'].isna())
        df_global = df_global[condicion_carbs & condicion_fibra]
        print(f"[OK] Dataset integrado en memoria. Registros totales: {len(df_global)}")
        
        # 2. Bloque 1/5: Validación de sensores independientes
        print("\n[1/5] Ejecutando Validación Poblacional de Sensores (merge_asof)...")
        graficar_poblacional_dexcom_vs_libre(df_global, SCATTER_DIR)
        
        # 3. Bloque 2/5: Sincronización telemétrica minuto a minuto
        print("\n[2/5] Ejecutando Sincronización Minutal del Wearable...")
        graficar_poblacional_hr_vs_calorias(df_global, SCATTER_DIR)
        
        # 4. Bloque 3/5: Matriz de ejercicio físico real segregado
        print("\n[3/5] Ejecutando Matriz de Actividad Física por Grupos...")
        graficar_actividad_por_grupos_diabetes(df_global, SCATTER_DIR)
        
        # 5. Bloque 4/5: Matriz de impacto nutricional absoluto masivo
        print("\n[4/5] Ejecutando Matriz de Macronutrientes Absolutos...")
        graficar_macros_por_grupos_diabetes(df_global, SCATTER_DIR)
        
        # 6. Bloque 5/5: Matriz de variaciones de ganancia neta (Deltas)
        print("\n[5/5] Ejecutando Matriz de Variación Neta (Deltas)...")
        graficar_delta_macros_por_grupos(df_global, SCATTER_DIR)
        
        print("\n=== PIPELINE POBLACIONAL COMPLETADO CON ÉXITO ===")
        
    except Exception as e:
        print(f"\n[ERROR CRÍTICO] El pipeline poblacional falló por completo: {e}")


if __name__ == "__main__":
    # Bloque de ejecución principal
    main_poblacional()


#### C2.2 Correlaciones cruzadas

- Control de Instrumentación: Comparamos la discrepancia de lecturas entre los dos sensores de manera rigurosa usando solo instantes donde ambos coincidieron físicamente (merge_asof).
- Matriz Postprandial Estática (Filtrada): Mide el impacto lineal-monótono (spearman) de la fibra, grasas, proteínas y carbohidratos aislando solo el momento en el que el plato entra al cuerpo, evaluando el incremento neto real (Deltas a 30 y 60 min).
- Matriz de Ayuno Unificada: Analiza la correlación entre tus variables basales continuas de Fitbit (como la frecuencia cardíaca o los bloques rodantes de METs) frente a los picos de estrés o las líneas base de glucosa.
- CCF Dinámica: Encuentra de forma automatizada los minutos exactos de retraso digestivo o fisiológico (Time-to-Peak) del cuerpo ante los carbohidratos o la actividad física sostenida.

Al analizar el archivo resumen_correlaciones_grupales.csv que vamos a  programar, debemos fijarnos muy bien en la diferencia entre: post_carbs_timeToPeak_dexcom vs post_carbs_timeToPeak_libre. 
Matemáticamente, la CCF nos va a dar el número exacto de minutos en el que reacciona cada uno. Lo habitual y biológicamente esperable es que el valor de libre sea significativamente mayor (unos 10-15 minutos más tarde) que el de dexcom, debido al retraso natural de la difusión de la glucosa desde la sangre capilar hacia el líquido intersticial.

---

**📊 GUÍA DE INTERPRETACIÓN DE GRÁFICOS CCF (CROSS-CORRELATION FUNCTION)**
________________________________________________________________________________________

Esta sección detalla la metodología analítica para interpretar los gráficos de la función de correlación cruzada generados en el pipeline, permitiendo descifrar las dinámicas temporales entre variables fisiológicas, nutricionales y de actividad física.

---

**🛠️ Anatomía Estructural de un Gráfico de CCF**
__________________________________________________

Al abrir cualquier imagen de correlación cruzada exportada por el sistema, se deben identificar tres componentes métricos fundamentales:

*   **Eje Horizontal (Eje X - Lags):** Mide la dimensión temporal expresada en minutos.
    *   *Punto de origen (0):* Representa el instante exacto del evento base.
    *   *Región derecha (+):* Muestreo de minutos hacia el futuro (+50, +100 min).
    *   *Región izquierda (-):* Muestreo de minutos hacia el pasado (-50, -100 min).
*   **Eje Vertical (Eje Y - Correlación):** Mide la magnitud y dirección de la fuerza asociativa.
    *   *Rango métrico:* Oscila estrictamente entre -1.0 y +1.0.
    *   *Límite +1.0:* Relación directa o positiva perfecta.
    *   *Límite -1.0:* Relación inversa o negativa perfecta.
    *   *Punto 0.0:* Ausencia absoluta de acoplamiento lineal entre las series.
*   **Bandas Horizontales Discontinuas (Naranjas):** Delimitan los umbrales de significancia estadística.
    *   *Criterio de validación:* Cualquier sección de la curva que sobresalga de estas bandas representa un efecto real y estadísticamente significativo, descartando fluctuaciones aleatorias o ruido de fondo.

---

**📐 Mecánica Matemática del Desfase (Lags)**
__________________________________________________

La función analítica se ejecuta bajo la estructura lógica:

`correlate(serie_1, serie_2)`

En el entorno matemático de Python, el algoritmo evalúa la concordancia desplazando secuencialmente la `serie_2` sobre el eje de la `serie_1`. La dirección del signo matemático resultante determina el orden de precedencia biológica:

**⏱️ Desfase Óptimo Positivo (`+`)**
*   *Mecánica:* Desplazamiento de la `serie_2` hacia el futuro para lograr el acoplamiento óptimo.
*   *Flujo Cronológico:* La `serie_1` precede en el tiempo a la `serie_2`.
*   *Esquema:* Serie 1 $\rightarrow$ Serie 2

**⏳ Desfase Óptimo Negativo (`-`)**
*   *Mecánica:* Desplazamiento de la `serie_2` hacia el pasado para lograr el acoplamiento óptimo.
*   *Flujo Cronológico:* La `serie_2` precede en el tiempo a la `serie_1`.
*   *Esquema:* Serie 2 $\rightarrow$ Serie 1

---

**🚀 Guía de Lectura de Gráficos Posprandiales (`carbs` vs `glucose_dexcom`)**
__________________________________________________

A continuación, se detalla el itinerario visual secuencial para interpretar correctamente la respuesta metabólica posprandial de un usuario evaluando carbohidratos frente a glucosa continua:

**Paso 1: Mirar el título (El "Resumen" del Reloj)**
*   Localizar el texto automatizado en la zona superior del gráfico (ej. `Desfase Óptimo (lag*) = 55 min`, `ρ* = 0.65`).
*   *Extracción de conclusiones:* Identificación directa del *Time-to-Peak* digestivo del usuario (55 minutos tras la ingesta) y la intensidad de la respuesta glucémica ($\rho = 0.65$, correlación positiva fuerte).

**Paso 2: Analizar el punto "Lag 0" (El momento de la ingesta)**
*   Fijar la vista en el marcador `Lag 0` del eje horizontal.
*   *Extracción de conclusiones:* Validar que la curva se encuentre próxima al valor 0.0 del eje vertical; esto confirma la consistencia de la lógica fisiológica, donde el azúcar en sangre no experimenta alteraciones en el minuto exacto de la ingesta.

**Paso 3: Seguir la curva hacia la derecha (El viaje de la digestión)**
*   Seguir visualmente la curva hacia el cuadrante derecho positivo del eje X (+15, +30, +45 min).
*   *Extracción de conclusiones:* Observar la progresión ascendente de la curva al cruzar la banda naranja de significancia (paso de nutrientes al torrente sanguíneo) hasta coronar el pico máximo (`lag*`), determinando la tasa de absorción neta.

**Paso 4: Observar la caída de la curva (La acción de la insulina)**
*   Continuar el recorrido visual hacia la derecha, superando el pico de los 55 minutos (+120, +180 min).
*   *Extracción de conclusiones:* Evaluar la pendiente descendente de la curva hacia la línea basal; representa dinámicamente la acción de la insulina endógena y la captura tisular de glucosa, indicando el fin del periodo de absorción de esa comida.

---

**🏃 Contraste Analítico: Dinámicas Inversas (Ejercicio en Ayuno)**
__________________________________________________

Para contrastar perfiles, el pipeline evalúa escenarios de correlación negativa, como el cruce de intensidad sustained frente a glucosa Dexcom en ayunas (`mets_sustained_15` vs `glucose_dexcom`):

*   *Morfología de la curva:* La traza no forma una elevación; se proyecta un valle pronunciado hacia el extremo inferior del eje vertical (ej. valores negativos de $\rho = -0.40$).
*   *Lectura del desfase:* Si el valle se localiza a los +20 o +30 minutos, se interpreta que sostener una intensidad física elevada genera un impacto inverso rezagado. Es decir, el usuario experimenta su máximo descenso de glucosa en sangre entre 20 y 30 minutos después de iniciar la actividad sostenida.


In [ ]:
# =========================================================================
# UTILIDADES DE VARIABLES DERIVADAS Y PROXIMIDAD
# =========================================================================

def infer_step_minutes(df, time_col="time"):
    """
    Infiere matemáticamente el paso de tiempo o frecuencia de muestreo en minutos de un DataFrame.

    Convierte la columna temporal especificada a tipo datetime, descarta valores nulos, ordena 
    los registros cronológicamente y calcula la mediana de las diferencias de tiempo entre filas 
    consecutivas para estimar el intervalo muestral característico del conjunto de datos.

    Args:
        df (pd.DataFrame): Conjunto de datos bajo análisis que contiene el histórico temporal.
        time_col (str, opcional): Nombre de la columna que registra las marcas de tiempo. 
            Por defecto es "time".

    Returns:
        float o None: Retorna el intervalo muestral calculado en minutos (tipo float) o 
            None si la columna no existe, contiene menos de dos filas o carece de varianza.

    Ejemplos:
        >>> frecuencia = infer_step_minutes(df_glucosa, time_col="time")
        >>> print(frecuencia)
        5.0

    Recomendaciones:
        - Asegurar que la columna temporal cuente con formatos legibles por el motor de Pandas 
          para prevenir pérdidas de datos masivas durante la conversión coercitiva.
    """
    # === 1. Validación Estructural de la Columna Temporal ===
    if time_col not in df.columns:
        return None
        
    # Conversión a datetime aislando los errores y ordenando la serie de forma ascendente
    t = pd.to_datetime(df[time_col], errors="coerce")
    t = t.dropna().sort_values()
    if len(t) < 2:
        return None
        
    # === 2. Estimación del Intervalo Muestral ===
    # Diferenciación consecutiva y extracción del delta total expresado en segundos
    dt = t.diff().dt.total_seconds().dropna()
    if len(dt) == 0:
        return None
        
    # Aplicación de la mediana estadística para mitigar el sesgo de posibles desfases aislados
    return np.median(dt) / 60.0


def calcular_variables_derivadas_metabolicas(df):
    """
    Calcula transformaciones temporales, acumulados y medias móviles sobre series de sensores y wearables.

    Asegura el alineamiento cronológico inicial y computa de forma secuencial los incrementos 
    netos (Deltas) de glucosa a futuro (30 y 60 min), la acumulación energética de calorías en ventanas 
    de 15 y 30 minutos, la intensidad media móvil de METs sostenidos y las variaciones bruscas de pulso.

    Args:
        df (pd.DataFrame): DataFrame original indexado o con la columna 'time' y métricas continuas.

    Returns:
        pd.DataFrame: Una copia modificada del DataFrame original enriquecida con las nuevas 
            variables metabólicas y de actividad física derivadas.

    Ejemplos:
        >>> df_enriquecido = calcular_variables_derivadas_metabolicas(df_crudo)

    Recomendaciones:
        - Garantizar un muestreo uniforme minuto a minuto para que las ventanas de filas equivalgan 
          exactamente a los intervalos nominales en minutos (ej. window=15 para 15 minutos).
        - Verificar que los datos no contengan saltos temporales masivos sin rellenar antes de 
          aplicar las operaciones de rezago (shift) o rolling.
    """
    # === 1. Clonación y Aseguramiento del Orden Cronológico ===
    df = df.copy()
    
    if "time" in df.columns:
        df['time_dt'] = pd.to_datetime(df['time'], errors='coerce')
        df = df.sort_values('time_dt').reset_index(drop=True)
    
    # === 2. Cálculo de Shifts y Deltas de Glucosa a Futuro ===
    for shift in [30, 60]:
        if "glucose_dexcom" in df.columns:
            df[f'dexcom_delta_{shift}'] = df['glucose_dexcom'].shift(-shift) - df['glucose_dexcom']
        if "glucose_libre" in df.columns:
            df[f'libre_delta_{shift}'] = df['glucose_libre'].shift(-shift) - df['glucose_libre']

    # === 3. Transformaciones Rodantes de Actividad y Telemetría ===
    # Sumatorios acumulados de energía sobre ventanas deslizantes de corta y mediana duración
    if "calories_activity" in df.columns:
        df["calories_accum_15"] = df["calories_activity"].rolling(window=15, min_periods=1).sum()
        df["calories_accum_30"] = df["calories_activity"].rolling(window=30, min_periods=1).sum()
        
    # Filtrado de picos de intensidad mediante la media móvil de METs sostenidos
    if "mets" in df.columns:
        df["mets_sustained_15"] = df["mets"].rolling(window=15, min_periods=1).mean()
        
    # Diferenciación del ritmo cardíaco respecto a un retardo base para estimar estrés físico
    if "heart_rate" in df.columns:
        df["heart_rate_delta_15"] = df["heart_rate"] - df["heart_rate"].shift(15)

    # === 4. Limpieza de Estructuras Temporales Auxiliares ===
    if 'time_dt' in df.columns:
        df = df.drop(columns=['time_dt'])
        
    return df


def comparar_sensores_proximidad(df, tolerancia_minutos=2):
    """
    Cruza y extrae lecturas coincidentes de ambos sensores de glucosa mediante un criterio de proximidad.

    Aísla las series de lecturas reales de los sensores descartando valores interpolados o nulos, 
    y ejecuta una unión asincrónica emparejando el instante temporal más cercano en ambas fuentes 
    dentro del margen de tolerancia en minutos especificado.

    Args:
        df (pd.DataFrame): Matriz de datos combinada que contiene las lecturas de glucosa de interés.
        tolerancia_minutos (int, opcional): Margen de tiempo máximo permitido para considerar válidos 
            dos puntos cercanos. Por defecto es 2.

    Returns:
        pd.DataFrame: Estructura de datos resultante emparejada sin nulos con las columnas 'time_dt', 
            'glucose_libre' y 'glucose_dexcom'. Retorna un DataFrame vacío si faltan variables clave.

    Ejemplos:
        >>> df_coincidencias = comparar_sensores_proximidad(df_sujeto, tolerancia_minutos=3)

    Recomendaciones:
        - Utilizar esta función para estudios de validación de hardware analizando exclusivamente las 
          fases donde coexistan mediciones nativas de ambos dispositivos.
        - Asegurar que el DataFrame de entrada no contenga duplicados en el eje temporal para evitar 
          distorsiones o cruces cartesianos inesperados en el merge_asof.
    """
    # === 1. Validación Estructural de Columnas de Entrada ===
    if "glucose_libre" not in df.columns or "glucose_dexcom" not in df.columns:
        return pd.DataFrame()
        
    # === 2. Preparación y Depuración Temporal del Dataset ===
    df_clean = df.copy()
    df_clean['time_dt'] = pd.to_datetime(df_clean['time'], errors='coerce')
    df_clean = df_clean.dropna(subset=['time_dt']).sort_values('time_dt')
    
    # Separación en estructuras independientes abstrayendo nulos para aislar mediciones nativas reales
    df_libre_real = df_clean[['time_dt', 'glucose_libre']].dropna().copy()
    df_dexcom_real = df_clean[['time_dt', 'glucose_dexcom']].dropna().copy()
    
    if df_libre_real.empty or df_dexcom_real.empty:
        return pd.DataFrame()
        
    # === 3. Unión Asincrónica por Cercanía Temporal ===
    # Emparejamiento bidireccional buscando el registro temporal más próximo dentro del intervalo acotado
    merged = pd.merge_asof(
        df_dexcom_real, 
        df_libre_real, 
        on='time_dt', 
        direction='nearest', 
        tolerance=pd.Timedelta(minutes=tolerancia_minutos)
    ).dropna()
    
    return merged


# =========================================================================
# UTILIDADES DE CORRELACIÓN (SPEARMAN DEFINITIVO) Y GRÁFICOS
# =========================================================================

def estandarizar(x: pd.Series) -> np.ndarray:
    """
    Estandariza estadísticamente una serie numérica centrando su media en cero y varianza unitaria.

    Elimina valores nulos, realiza una conversión explícita a tipo float y calcula la media 
    y la desviación estándar muestral. Si la desviación estándar es nula o inválida, devuelve 
    la serie únicamente centrada respecto a su media para evitar indeterminaciones.

    Args:
        x (pd.Series): Serie de datos numéricos que se va a transformar.

    Returns:
        np.ndarray: Vector de NumPy con los valores estandarizados o una matriz vacía si la 
            serie de entrada carece de registros válidos.

    Recomendaciones:
        - Utilizar este método de forma previa al cálculo de la función de correlación cruzada 
          para garantizar la correcta normalización del coeficiente escalar resultante.
    """
    # === 1. Depuración e Ingesta de Datos ===
    x = x.dropna().astype(float)
    if len(x) == 0:
        return np.array([])
        
    # === 2. Estimación de Parámetros Estadísticos ===
    mu = x.mean()
    sd = x.std(ddof=1)
    
    # === 3. Transformación Métrica ===
    if sd == 0 or np.isnan(sd):
        return (x - mu).values
    return ((x - mu) / sd).values


def sincronizar_longitud(a: np.ndarray, b: np.ndarray):
    """
    Ajusta y recorta de forma síncrona dos vectores numéricos para que tengan la misma dimensión.

    Determina la longitud mínima común entre ambos arrays de NumPy y segmenta ambas estructuras 
    hasta dicho límite para posibilitar operaciones algebraicas directas entre elementos.

    Args:
        a (np.ndarray): Primer vector numérico bajo análisis.
        b (np.ndarray): Segundo vector numérico bajo análisis.

    Returns:
        tuple (np.ndarray, np.ndarray): Una tupla con ambos vectores recortados a la misma longitud.

    Recomendaciones:
        - Aplicar esta función antes de operaciones vectoriales punto a punto en series temporales 
          donde se sospeche de desalineación por pérdidas en los extremos de la muestra.
    """
    n = min(len(a), len(b))
    return a[:n], b[:n]


def lags_vector(n: int) -> np.ndarray:
    """
    Construye el vector de desplazamientos temporales (lags) adecuado para una función de correlación continua.

    Genera un rango simétrico de números enteros equidistantes que representa el dominio completo de 
    desfases teóricos asociados a una serie temporal de longitud específica.

    Args:
        n (int): Longitud o cantidad de muestras síncronas de la serie.

    Returns:
        np.ndarray: Vector secuencial que abarca el intervalo cerrado desde -n + 1 hasta n - 1.

    Recomendaciones:
        - Utilizar este índice numérico como la referencia del eje horizontal en la graficación 
          de curvas analíticas de correlación cruzada.
    """
    return np.arange(-n + 1, n)


def parse_user_from_filename(fname: str):
    """
    Aísla el identificador único del participante partiendo de la ruta o nombre físico del archivo.

    Extrae el nombre base del fichero abstrayendo las rutas absolutas del sistema operativo y 
    descarta la extensión final del documento para devolver un string de identidad limpio.

    Args:
        fname (str): Cadena de texto con el nombre o ruta del archivo de entrada.

    Returns:
        str: El identificador único obtenido a partir del nombre base del documento.

    Recomendaciones:
        - Mantener la nomenclatura homogénea de los ficheros de datos en el almacenamiento local 
          para asegurar la correcta extracción automática de los metadatos del sujeto.
    """
    return os.path.splitext(os.path.basename(fname))[0]


def plot_corr_matrix(df, cols, title_suffix, tipo_matriz, user_id):
    """
    Calcula la matriz de correlación de Spearman de variables seleccionadas y exporta un mapa de calor.

    Filtra dinámicamente las columnas deseadas reteniendo únicamente variables numéricas sin nulos, 
    computa la matriz asociativa no paramétrica y exporta el gráfico de calor (.png) estructurando 
    de forma jerárquica los directorios de salida según la tipología del análisis.

    Args:
        df (pd.DataFrame): DataFrame con el histórico de registros del participante.
        cols (list): Lista con los nombres de las columnas que se desean incorporar en la matriz.
        title_suffix (str): Texto aclaratorio complementario que se inyectará en el título del gráfico.
        tipo_matriz (str): Nombre del subdirectorio específico para categorizar la salida (ej. 'ayuno').
        user_id (str): Identificador único del participante bajo estudio.

    Returns:
        pd.DataFrame o None: Matriz simétrica estructurada de coeficientes de correlación de Spearman, 
            o None si no coexisten suficientes columnas numéricas válidas para el cálculo.

    Recomendaciones:
        - Asegurar que la constante global 'CORRELACION_INDIVIDUAL_DIR' se encuentre previamente definida 
          en el Notebook para mitigar fallos al anclar rutas del sistema.
    """
    # === 1. Filtrado Estructural de Atributos Presentes ===
    cols_present = [c for c in cols if c in df.columns]
    if len(cols_present) < 2:
        return
        
    # Selección estricta de columnas numéricas eliminando registros totalmente vacíos
    mat = df[cols_present].select_dtypes(include=[np.number]).dropna(how='all')
    if mat.empty:
        return
        
    # === 2. Estimación Asociativa de Spearman ===
    corr = mat.corr(method='spearman')
    
    # === 3. Dimensionamiento y Configuración del Lienzo Gráfico ===
    plt.figure(figsize=(1.4 * len(corr.columns), 1.1 * len(corr.columns)))
    sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", vmin=-1, vmax=1, cbar_kws={"shrink": 0.8})
    plt.title(f"Matriz Spearman ({title_suffix}) - {user_id}", fontsize=11, weight='bold')
    plt.tight_layout()
    
    # ---- LÓGICA DE DIRECTORIO JERÁRQUICO ----
    # Ruta final: Analysis\Correlación\Individuales\Matriz\[tipo_matriz]
    folder_path = os.path.join(CORRELACION_INDIVIDUAL_DIR, "Matriz", tipo_matriz)
    os.makedirs(folder_path, exist_ok=True)
    
    # Depuración de caracteres especiales y acentos para el guardado físico del archivo
    suffix_clean = title_suffix.lower().replace(" ", "_")
    fname = os.path.join(folder_path, f"matriz_{suffix_clean}_{user_id}.png")
    
    # === 4. Almacenamiento y Cierre de Memoria ===
    plt.savefig(fname, dpi=300)
    plt.close()
    print(f"[OK] Matriz guardada en {tipo_matriz}: {fname}")
    return corr


def plot_ccf(series1: pd.Series, series2: pd.Series, user_id: str,
             var1: str, var2: str, tipo_ccf: str, max_lag=180):
    """
    Calcula la función de correlación cruzada continua, identifica el desfase óptimo y exporta la curva.

    Estandariza los vectores analíticos, sincroniza sus longitudes y estima los coeficientes 
    desplazando las series simétricamente. Identifica el pico de máxima magnitud absoluto (lag* y rho*), 
    añade bandas discontinuas opcionales de significancia estadística y graba el gráfico resultante 
    en su subcarpeta analítica correspondiente.

    Args:
        series1 (pd.Series): Serie temporal base que actuará como referencia fija en el eje temporal.
        series2 (pd.Series): Serie temporal desplazada de forma secuencial sobre la primera serie.
        user_id (str): Identificador único del sujeto bajo análisis.
        var1 (str): Nombre descriptivo de la variable base para el etiquetado del gráfico.
        var2 (str): Nombre descriptivo de la variable desplazada para el etiquetado del gráfico.
        tipo_ccf (str): Nombre de la subcarpeta jerárquica para segmentar las salidas (ej. 'Ayuno').
        max_lag (int, opcional): Límite máximo en minutos para acotar la visualización del eje X. 
            Por defecto es 180.

    Returns:
        tuple (int, float) o None: Una tupla con el desfasamiento óptimo en minutos (lag_star) 
            y su respectivo coeficiente de correlación máxima (rho_star), o None si los vectores iniciales son inválidos.

    Recomendaciones:
        - Garantizar la disponibilidad previa en el entorno de la constante analítica 'PLOT_REF_BANDS' 
          y de la función estándar 'correlate' para prevenir interrupciones al procesar las trazas.
    """
    # === 1. Normalización y Sincronización Dimensional ===
    x = estandarizar(series1)
    y = estandarizar(series2)
    if len(x) == 0 or len(y) == 0:
        return
        
    x, y = sincronizar_longitud(x, y)
    n = len(x)
    
    # === 2. Estimación Secuencial de Correlaciones Cruzadas ===
    cc = correlate(x, y, mode="full") / n
    lags = lags_vector(n)
    
    # Acotación estricta de los resultados según el margen máximo de desfase permitido
    mask = (lags >= -max_lag) & (lags <= max_lag)
    lags = lags[mask]
    cc = cc[mask]

    if len(cc) == 0:
        return

    # === 3. Identificación del Pico Óptimo Absoluto (Reloj del pipeline) ===
    idx = np.argmax(np.abs(cc))
    lag_star = lags[idx]
    rho_star = cc[idx]

    # === 4. Construcción Gráfica de la Curva Temporal ===
    plt.figure(figsize=(10, 5))
    plt.plot(lags, cc, color="tab:blue", linewidth=1.5, label=f"{var1} ↔ {var2}")
    plt.axhline(0, color="gray", linestyle="--", linewidth=0.8)
    
    # Adición opcional de las bandas de significancia estadística paramétricas
    if PLOT_REF_BANDS:
        band = 1.96 / np.sqrt(n)
        plt.axhline(band, color="tab:orange", linestyle="--", linewidth=0.9, label="±1.96/√n")
        plt.axhline(-band, color="tab:orange", linestyle="--", linewidth=0.9)
        
    # Personalización visual de los ejes y leyendas
    plt.xlabel("Lags (Desfase en minutos)", fontsize=10)
    plt.ylabel("Correlación cruzada", fontsize=10)
    plt.title(f"CCF ({tipo_ccf}) - {var1} vs {var2} - {user_id}\n"
              f"Reloj (lag*) = {lag_star} min, ρ* = {rho_star:.3f}", fontsize=11, weight='bold')
    plt.grid(True, alpha=0.3)
    plt.legend(loc="upper right")
    plt.tight_layout()
    
    # ---- LÓGICA DE DIRECTORIO JERÁRQUICO ----
    # Ruta final: Analysis\Correlación\Individuales\CCF\[tipo_ccf]
    folder_path = os.path.join(CORRELACION_INDIVIDUAL_DIR, "CCF", tipo_ccf)
    os.makedirs(folder_path, exist_ok=True)
    
    fname = os.path.join(folder_path, f"ccf_{var1}_vs_{var2}_{user_id}.png")
    
    # === 5. Grabado Físico y Liberación de Memoria ===
    plt.savefig(fname, dpi=300)
    plt.close()
    print(f"[OK] CCF guardado en {tipo_ccf}: {fname}")
    return lag_star, rho_star


# =========================================================================
# PIPELINE PRINCIPAL Y FILTROS DE ESTADO (AYUNO VS POSTPRANDIAL)
# =========================================================================

def process_file(csv_path: str):
    """
    Ejecuta el pipeline de procesamiento individual segmentando por estados biológicos autónomos.

    Carga el archivo del participante, calcula variables derivadas continuas y genera matrices 
    de control. Divide los registros en ventanas postprandiales (bloques de 3 horas desde la ingesta) 
    y de ayuno estricto (más de 4 horas sin ingesta), extrayendo coeficientes de correlación y 
    desfases analíticos dinámicos (CCF) de forma automatizada.

    Args:
        csv_path (str): Ruta completa del archivo CSV individual que se va a procesar.

    Returns:
        dict o None: Diccionario indexado con las métricas cuantitativas consolidadas del participante 
            (coeficientes rho y desfases óptimos), o None si ocurre una excepción de lectura en el archivo.

    Recomendaciones:
        - Asegurar la carga previa de las constantes globales 'VARS_POSTPRANDIAL', 'VARS_AYUNO' 
          y de todos los módulos de graficación auxiliares en el entorno antes de la llamada.
        - Verificar que el formato de entrada mantenga la resolución minutal continua para no 
          distorsionar las máscaras lógicas de tiempo acumulado.
    """
    # === 1. Ingesta, Validación Estructural e Inicialización de Métricas ===
    user_id = parse_user_from_filename(csv_path)
    try:
        df = pd.read_csv(csv_path)
    except Exception as e:
        print(f"Error procesando el archivo {csv_path}: {e}")
        return None
    
    # Diccionario para almacenar las métricas compactas de este usuario concreto
    metricas_usuario = {"user_id": user_id}

    # Cálculo secuencial de variables de desfases (Shifts/Deltas) y rodantes en serie continua
    df = calcular_variables_derivadas_metabolicas(df)
    
    # === 2. Construcción de Matrices de Control y Validación Estándar ===
    # 2.1 Comparación de sensores por proximidad temporal (Datos 100% reales)
    df_proximidad = comparar_sensores_proximidad(df, tolerance_minutos=2)
    if not df_proximidad.empty:
        corr_prox = plot_corr_matrix(df_proximidad, ["glucose_libre", "glucose_dexcom"], "Proximidad Sensores Real", "Sensores", user_id)
        if corr_prox is not None and "glucose_libre" in corr_prox.columns and "glucose_dexcom" in corr_prox.columns:
            metricas_usuario["corr_sensores_real"] = corr_prox.loc["glucose_libre", "glucose_dexcom"]
       
    # 2.2 Validación interna de Fitbit en la serie continua (HR, METS, Calories)
    plot_corr_matrix(df, ["heart_rate", "calories_activity", "mets"], "Control Fitbit Global", "Actividad", user_id)

    # === 3. Segmentación Algorítmica de Estados Metabólicos ===
    # Reset del índice para garantizar la correspondencia en operaciones posicionales
    df = df.reset_index(drop=True)
    indices_comidas = df[df['carbs'] > 0].index.tolist()
    
    # Inicialización de máscaras booleanas para clasificar cada minuto del día
    es_postprandial = np.zeros(len(df), dtype=bool)
    
    # Ventana Postprandial: Bloques de 3 horas (180 minutos) desde el inicio de cada comida
    for idx in indices_comidas:
        es_postprandial[idx : min(idx + 180, len(df))] = True
        
    # Ventana de Ayuno Estricto: Minutos donde NO hay comida Y han pasado más de 4 horas (240 min)
    # Estimación de minutos transcurridos acumulando los eventos de ingesta binarios
    es_comida_binario = (df['carbs'] > 0).astype(int)
    comida_acum = es_comida_binario.cumsum()
    
    # Agrupamos por el bloque de comida actual para contar los minutos transcurridos
    minutos_desde_comida = df.groupby(comida_acum).cumcount()
    # Si estamos en el bloque 0 (antes de la primera comida detectada), asumimos estado base
    minutos_desde_comida.loc[comida_acum == 0] = 9999 
    
    es_ayuno = (minutos_desde_comida >= 240) & (~es_postprandial)

    # Separación del DataFrame en los dos entornos biológicos autónomos filtrados
    df_postprandial_estado = df[es_postprandial].copy()
    df_ayuno_estado = df[es_ayuno].copy()

    # === 4. Análisis Estático Postprandial (Macros vs Deltas Glucémicos) ===
    if not df_postprandial_estado.empty:
        # Replicamos el filtro del scatter para cruzar solo el momento exacto del plato frente a su resultado futuro
        df_platos_exactos = df_postprandial_estado[df_postprandial_estado['carbs'] > 0]
        if not df_platos_exactos.empty:
            corr_post = plot_corr_matrix(df_platos_exactos, VARS_POSTPRANDIAL, "Postprandial Macros vs Deltas", "Postprandial", user_id)
            # Extraemos correlaciones clave del impacto de macros contra deltas a 60 min
            if corr_post is not None:
                for macro in ["carbs", "fiber", "fat", "protein"]:
                    # Extracción para Dexcom
                    if macro in corr_post.index and "dexcom_delta_60" in corr_post.columns:
                        metricas_usuario[f"corr_{macro}_vs_deltaDexcom60"] = corr_post.loc[macro, "dexcom_delta_60"]
                    # NUEVA ADICIÓN: Extracción idéntica para el sensor Libre
                    if macro in corr_post.index and "libre_delta_60" in corr_post.columns:
                        metricas_usuario[f"corr_{macro}_vs_deltaLibre60"] = corr_post.loc[macro, "libre_delta_60"]

    # === 5. Análisis Estático de Ayuno (Tono Basal vs Deltas Fitbit) ===
    # Evaluación exclusiva sobre registros reales nativos del sensor Dexcom (cada 5 min)
    if not df_ayuno_estado.empty:
        df_ayuno_real = df_ayuno_estado.dropna(subset=["glucose_dexcom"])
        if not df_ayuno_real.empty:
            corr_ayuno = plot_corr_matrix(df_ayuno_real, VARS_AYUNO, "Ayuno Absolutos y Deltas Fitbit", "Ayuno", user_id)
            if corr_ayuno is not None and "heart_rate" in corr_ayuno.index and "glucose_dexcom" in corr_ayuno.columns:
                metricas_usuario["corr_ayuno_HR_vs_GlucosaAbs"] = corr_ayuno.loc["heart_rate", "glucose_dexcom"]

    # === 6. Análisis de Correlaciones Cruzadas Dinámicas (CCF) ===
    # 6.1 CCF en Periodo de Ayuno (Frecuencia real Dexcom cada 5 min para evitar tendencias artificiales)
    if not df_ayuno_estado.empty:
        df_ayuno_ccf = df_ayuno_estado.dropna(subset=["glucose_dexcom"])
        if len(df_ayuno_ccf) > 10:
            # Variables rodantes (Suma de calorías a 15 y 30 min | Media de METS a 15 min)
            if "calories_accum_15" in df_ayuno_ccf.columns:
                res = plot_ccf(df_ayuno_ccf["calories_accum_15"], df_ayuno_ccf["glucose_dexcom"], user_id, "calories_accum_15", "glucose_dexcom", "Ayuno", max_lag=180)
                if res: metricas_usuario["ayuno_lag_calAcum15"], metricas_usuario["ayuno_rho_calAcum15"] = res
            if "calories_accum_30" in df_ayuno_ccf.columns:
                res = plot_ccf(df_ayuno_ccf["calories_accum_30"], df_ayuno_ccf["glucose_dexcom"], user_id, "calories_accum_30", "glucose_dexcom", "Ayuno", max_lag=180)
                if res: metricas_usuario["ayuno_lag_calAcum30"], metricas_usuario["ayuno_rho_calAcum30"] = res
            if "mets_sustained_15" in df_ayuno_ccf.columns:
                res = plot_ccf(df_ayuno_ccf["mets_sustained_15"], df_ayuno_ccf["glucose_dexcom"], user_id, "mets_sustained_15", "glucose_dexcom", "Ayuno", max_lag=180)
                if res: metricas_usuario["ayuno_lag_metsSust15"], metricas_usuario["ayuno_rho_metsSust15"] = res

    # 6.2 CCF en Periodo Postprandial (Señales interpoladas a 1 min para estimar el Time-to-Peak exacto)
    if not df_postprandial_estado.empty:
        # Evaluación de la señal continua de carbohidratos contra la evolución glucémica en la ventana de digestión
        if "carbs" in df_postprandial_estado.columns and "glucose_dexcom" in df_postprandial_estado.columns:
            res = plot_ccf(df_postprandial_estado["carbs"], df_postprandial_estado["glucose_dexcom"], user_id, "carbs", "glucose_dexcom", "Postprandial", max_lag=180)
            if res: metricas_usuario["post_carbs_timeToPeak_dexcom"], metricas_usuario["post_carbs_rho_dexcom"] = res
        if "carbs" in df_postprandial_estado.columns and "glucose_libre" in df_postprandial_estado.columns:
            res = plot_ccf(df_postprandial_estado["carbs"], df_postprandial_estado["glucose_libre"], user_id, "carbs", "glucose_libre", "Postprandial", max_lag=180)
            if res: metricas_usuario["post_carbs_timeToPeak_libre"], metricas_usuario["post_carbs_rho_libre"] = res
            
    return metricas_usuario

def main():
    patron = f"*{FILTRO_NOMBRE}*.csv" if FILTRO_NOMBRE else "*.csv"
    archivos = glob.glob(os.path.join(TRUSTED_DIR, patron))
    if not archivos:
        print(f"No se encontraron archivos CSV en '{TRUSTED_DIR}' con filtro '{FILTRO_NOMBRE}'")
        return
        
    print(f"Iniciando procesamiento metabólico de {len(archivos)} archivo(s)...")
    
    # Lista para ir acumulando las métricas de todos los pacientes procesados
    resultados_globales = []

    for archivo in archivos:
        res_usuario = process_file(archivo)
        if res_usuario is not None:
            resultados_globales.append(res_usuario)

    # Guardado del archivo CSV agregado final
    if resultados_globales:
        df_resumen = pd.DataFrame(resultados_globales)
        
        # Guardamos directamente en la raíz de la carpeta de correlación
        csv_salida = os.path.join(CORRELACION_DIR, "resumen_correlaciones_grupales.csv")
        df_resumen.to_csv(csv_salida, index=False, encoding="utf-8-sig")
        
        print("\n" + "="*50)
        print(f"[EXITO] Archivo agregado guardado en:\n{csv_salida}")
        print(f"Se consolidaron métricas clave de {len(df_resumen)} usuarios.")
        print("="*50)
    else:
        print("[WARN] No se extrajeron métricas válidas para consolidar el archivo grupal.")

    print("¡Análisis unificado completado con éxito!")


In [ ]:
if __name__ == "__main__":
    main()

Ahora realizaremos los mismos análisis pero agrupando por fenotipo metabólico.

In [ ]:
# =========================================================================
# UTILIDADES ESTADÍSTICAS BASE Y VARIABLES DERIVADAS
# =========================================================================

def estandarizar(x: pd.Series) -> np.ndarray:
    """
    Estandariza estadísticamente una serie numérica centrando su media en cero y varianza unitaria.

    Elimina valores nulos, realiza una conversión explícita a tipo float y calcula la media 
    y la desviación estándar muestral. Si la desviación estándar es nula o inválida, devuelve 
    la serie únicamente centrada respecto a su media para evitar indeterminaciones.

    Args:
        x (pd.Series): Serie de datos numéricos que se va a transformar.

    Returns:
        np.ndarray: Vector de NumPy con los valores estandarizados o una matriz vacía si la 
            serie de entrada carece de registros válidos.

    Recomendaciones:
        - Utilizar este método de forma previa al cálculo de la función de correlación cruzada 
          para garantizar la correcta normalización del coeficiente escalar resultante.
    """
    # === 1. Depuración e Ingesta de Datos ===
    x = x.dropna().astype(float)
    if len(x) == 0:
        return np.array([])
    
    # === 2. Estimación de Parámetros Estadísticos ===
    mu = x.mean()
    sd = x.std(ddof=1)
    
    # === 3. Transformación Métrica ===
    if sd == 0 or np.isnan(sd):
        return (x - mu).values
    return ((x - mu) / sd).values


def sincronizar_longitud(a: np.ndarray, b: np.ndarray):
    """
    Ajusta y recorta de forma síncrona dos vectores numéricos para que tengan la misma dimensión.

    Determina la longitud mínima común entre ambos arrays de NumPy y segmenta ambas estructuras 
    hasta dicho límite para posibilitar operaciones algebraicas directas entre elementos.

    Args:
        a (np.ndarray): Primer vector numérico bajo análisis.
        b (np.ndarray): Segundo vector numérico bajo análisis.

    Returns:
        tuple (np.ndarray, np.ndarray): Una tupla con ambos vectores recortados a la misma longitud.

    Recomendaciones:
        - Aplicar esta función antes de operaciones vectoriales punto a punto en series temporales 
          donde se sospeche de desalineación por pérdidas en los extremos de la muestra.
    """
    n = min(len(a), len(b))
    return a[:n], b[:n]


def lags_vector(n: int) -> np.ndarray:
    """
    Construye el vector de desplazamientos temporales (lags) adecuado para una función de correlación continua.

    Genera un rango simétrico de números enteros equidistantes que representa el dominio completo de 
    desfases teóricos asociados a una serie temporal de longitud específica.

    Args:
        n (int): Longitud o cantidad de muestras síncronas de la serie.

    Returns:
        np.ndarray: Vector secuencial que abarca el intervalo cerrado desde -n + 1 hasta n - 1.

    Recomendaciones:
        - Utilizar este índice numérico como la referencia del eje horizontal en la graficación 
          de curvas analíticas de correlación cruzada.
    """
    return np.arange(-n + 1, n)


def parse_user_from_filename(fname: str):
    """
    Aísla el identificador único del participante partiendo de la ruta o nombre físico del archivo.

    Extrae el nombre base del fichero abstrayendo las rutas absolutas del sistema operativo y 
    descarta la extensión final del documento. Si el nombre se corresponde con la nomenclatura 
    del proyecto, aplica un truncamiento estricto a los primeros 12 caracteres para dar soporte 
    a subseries y mantener la homogeneidad del ID base.

    Args:
        fname (str): Cadena de texto con el nombre o ruta del archivo de entrada.

    Returns:
        str: El identificador único obtenido y normalizado a partir del nombre base del documento.

    Recomendaciones:
        - Mantener el prefijo homogéneo en los ficheros de datos locales para garantizar el correcto 
          funcionamiento de las subseries durante el procesamiento por lotes.
    """
    # === 1. Aislamiento del Nombre Base ===
    nombre_base = os.path.splitext(os.path.basename(fname))[0]
    
    # === 2. Validación de Prefijo y Truncamiento de Subseries ===
    # Si el nombre empieza por CGMacros-, extraemos solo los 12 caracteres fijos (CGMacros-XXX)
    if nombre_base.startswith("CGMacros-"):
        return nombre_base[:12]
    return nombre_base


def infer_step_minutes(df, time_col="time"):
    """
    Infiere matemáticamente el paso de tiempo o frecuencia de muestreo en minutos de un DataFrame.

    Convierte la columna temporal especificada a tipo datetime, descarta valores nulos, ordena 
    los registros cronológicamente y calcula la mediana de las diferencias de tiempo entre filas 
    consecutivas para estimar el intervalo muestral característico del conjunto de datos.

    Args:
        df (pd.DataFrame): Conjunto de datos bajo análisis que contiene el histórico temporal.
        time_col (str, opcional): Nombre de la columna que registra las marcas de tiempo. 
            Por defecto es "time".

    Returns:
        float o None: Retorna el intervalo muestral calculado en minutos (tipo float) o 
            None si la columna no existe, contiene menos de dos filas o carece de varianza.

    Ejemplos:
        >>> frecuencia = infer_step_minutes(df_glucosa, time_col="time")
        >>> print(frecuencia)
        5.0

    Recomendaciones:
        - Asegurar que la columna temporal cuente con formatos legibles por el motor de Pandas 
          para prevenir pérdidas de datos masivas durante la conversión coercitiva.
    """
    # === 1. Validación Estructural de la Columna Temporal ===
    if time_col not in df.columns:
        return None
        
    # Conversión a datetime aislando los errores y ordenando la serie de forma ascendente
    t = pd.to_datetime(df[time_col], errors="coerce")
    t = t.dropna().sort_values()
    if len(t) < 2:
        return None
        
    # === 2. Estimación del Intervalo Muestral ===
    # Diferenciación consecutiva y extracción del delta total expresado en segundos
    dt = t.diff().dt.total_seconds().dropna()
    if len(dt) == 0:
        return None
        
    # Aplicación de la mediana estadística para mitigar el sesgo de posibles desfases aislados
    return np.median(dt) / 60.0


# =========================================================================
# FUNCIONES DE GRAFICADO PARA ANÁLISIS POBLACIONAL Y COHORTES (GRUPOS)
# =========================================================================

def plot_corr_matrix_grupal(df, cols, title_suffix, tipo_matriz):
    """
    Calcula la matriz de correlación de Spearman para una cohorte específica y exporta un mapa de calor.

    Filtra dinámicamente los atributos de interés en base a las columnas presentes en el DataFrame, 
    selecciona las variables numéricas descartando filas completamente vacías y computa la matriz 
    de asociación no paramétrica para exportar un gráfico de calor (.png) estructurado por grupos.

    Args:
        df (pd.DataFrame): DataFrame unificado que contiene los datos agregados del grupo o cohorte.
        cols (list): Lista con las cadenas de texto de las columnas que se integrarán en la matriz.
        title_suffix (str): Texto complementario indicativo del grupo que se inyectará en el título del gráfico.
        tipo_matriz (str): Nombre del subdirectorio específico para clasificar la salida (ej. 'ayuno').

    Returns:
        None: La función no retorna ningún objeto, exporta el gráfico (.png) directamente al disco.

    Recomendaciones:
        - Asegurar que la constante global 'CORRELACION_GRUPO_DIR' se encuentre previamente inicializada 
          en el entorno para evitar errores al anclar las rutas del sistema.
    """
    # === 1. Filtrado de Atributos Presentes y Selección Numérica ===
    cols_present = [c for c in cols if c in df.columns]
    if len(cols_present) < 2:
        return
        
    mat = df[cols_present].select_dtypes(include=[np.number]).dropna(how='all')
    if mat.empty:
        return
        
    # === 2. Estimación de Coeficientes de Spearman ===
    corr = mat.corr(method='spearman')
    
    # === 3. Inicialización Estética y Renderizado del Mapa de Calor ===
    plt.figure(figsize=(1.4 * len(corr.columns), 1.1 * len(corr.columns)))
    sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", vmin=-1, vmax=1, cbar_kws={"shrink": 0.8})
    plt.title(f"Matriz Spearman Grupal ({title_suffix})", fontsize=11, weight='bold')
    plt.tight_layout()
    
    # === 4. Configuración de Carpetas y Grabado Físico ===
    # Estructuración jerárquica: Analysis/CGMacros/Correlación/Grupos/Matriz/[tipo_matriz]
    folder_path = os.path.join(CORRELACION_GRUPO_DIR, "Matriz", tipo_matriz)
    os.makedirs(folder_path, exist_ok=True)
    
    # Normalización del string eliminando acentos y espacios para el fichero físico
    grupo_clean = title_suffix.lower().replace(" ", "_").replace('é', 'e').replace('á', 'a')
    fname = os.path.join(folder_path, f"grupo_matriz_{tipo_matriz.lower()}_{grupo_clean}.png")
    
    plt.savefig(fname, dpi=300)
    plt.close()
    print(f"[OK] Matriz Grupal guardada en {tipo_matriz}: {fname}")


def plot_ccf_grupal(series1: pd.Series, series2: pd.Series, var1: str, var2: str, 
                    label_grupo: str, tipo_ccf: str, max_lag=180):
    """
    Calcula la función de correlación cruzada continua a nivel de cohorte y exporta la curva temporal.

    Estandariza los vectores analíticos del grupo, sincroniza sus longitudes y estima las correlaciones 
    desplazando las series de forma simétrica. Identifica el punto de máxima magnitud absoluto (Reloj Colectivo), 
    añade bandas paramétricas de significancia estadística opcionales y graba la figura resultante.

    Args:
        series1 (pd.Series): Serie temporal base colectiva que actuará como referencia fija en el eje temporal.
        series2 (pd.Series): Serie temporal colectiva que se desplazará secuencialmente sobre la primera serie.
        var1 (str): Nombre descriptivo de la variable fija para el etiquetado del eje y leyendas.
        var2 (str): Nombre descriptivo de la variable móvil para el etiquetado del eje y leyendas.
        label_grupo (str): Identificador o etiqueta representativa de la cohorte analizada (ej. 'Diabéticos').
        tipo_ccf (str): Nombre de la subcarpeta jerárquica para segmentar las salidas (ej. 'Postprandial').
        max_lag (int, opcional): Límite máximo en minutos para acotar la visualización del eje X. 
            Por defecto es 180.

    Returns:
        tuple (int, float) o None: Una tupla con el desfasamiento óptimo colectivo (lag_star) 
            y su coeficiente de correlación máxima (rho_star), o None si los vectores numéricos son inválidos.

    Recomendaciones:
        - Verificar que las constantes globales 'PLOT_REF_BANDS' y la función 'correlate' se encuentren 
          definidas previamente en el Notebook antes de invocar este módulo grupal.
    """
    # === 1. Normalización Métrica y Alineamiento de Vectores ===
    x = estandarizar(series1)
    y = estandarizar(series2)
    if len(x) == 0 or len(y) == 0:
        return
        
    x, y = sincronizar_longitud(x, y)
    n = len(x)
    
    # === 2. Estimación Secuencial de Correlaciones Cruzadas ===
    cc = correlate(x, y, mode="full") / n
    lags = lags_vector(n)
    
    # Acotación estricta de los desfases según la ventana máxima permitida
    mask = (lags >= -max_lag) & (lags <= max_lag)
    lags = lags[mask]
    cc = cc[mask]

    if len(cc) == 0:
        return

    # === 3. Identificación del Pico Óptimo Colectivo (Reloj de Cohorte) ===
    idx = np.argmax(np.abs(cc))
    lag_star = lags[idx]
    rho_star = cc[idx]

    # === 4. Renderizado Gráfico del Perfil de Correlación Cruzada ===
    plt.figure(figsize=(10, 5))
    plt.plot(lags, cc, color="tab:blue", linewidth=1.5, label=f"{var1} ↔ {var2}")
    plt.axhline(0, color="gray", linestyle="--", linewidth=0.8)
    
    # Inyección de las bandas horizontales discontinuas de significancia estadística
    if PLOT_REF_BANDS:
        band = 1.96 / np.sqrt(n)
        plt.axhline(band, color="tab:orange", linestyle="--", linewidth=0.9, label="±1.96/√n")
        plt.axhline(-band, color="tab:orange", linestyle="--", linewidth=0.9)
        
    # Personalización visual de los ejes y anotaciones métricas
    plt.xlabel("Lags (Desfase en minutos)", fontsize=10)
    plt.ylabel("Correlación cruzada", fontsize=10)
    plt.title(f"CCF Grupal ({tipo_ccf}) - {label_grupo}\n{var1} vs {var2}\n"
              f"Reloj Colectivo (lag*) = {lag_star} min, ρ* = {rho_star:.3f}", fontsize=11, weight='bold')
    plt.grid(True, alpha=0.3)
    plt.legend(loc="upper right")
    plt.tight_layout()
    
    # === 5. Estructuración de Carpetas y Grabado en Disco ===
    # Ruta de destino: Analysis/CGMacros/Correlación/Grupos/CCF/[tipo_ccf]
    folder_path = os.path.join(CORRELACION_GRUPO_DIR, "CCF", tipo_ccf)
    os.makedirs(folder_path, exist_ok=True)
    
    # Depuración de caracteres especiales y acentos para el guardado del fichero físico
    grupo_clean = label_grupo.lower().replace(" ", "_").replace('é', 'e').replace('á', 'a')
    fname = os.path.join(folder_path, f"grupo_ccf_{var1}_vs_{var2}_{grupo_clean}.png")
    
    plt.savefig(fname, dpi=300)
    plt.close()
    print(f"[OK] CCF Grupal guardada en {tipo_ccf}: {fname}")
    return lag_star, rho_star


# =========================================================================
# MOTORES DE ANÁLISIS POBLACIONAL Y COHORTES METABÓLICAS
# =========================================================================

def generar_matrices_control_poblacional(df_global):
    """
    Consolida de forma poblacional masiva los datos de control para toda la cohorte del estudio.

    Agrupa la información por participante y ejecuta de manera secuencial la validación cruzada 
    de sensores mediante proximidad temporal asrónica (tolerancia máxima de 3 minutos) y la 
    validación interna de consistencia de la telemetría del wearable (Fitbit).

    Args:
        df_global (pd.DataFrame): DataFrame unificado que almacena el histórico colectivo de 
            todos los sujetos del estudio.

    Returns:
        None: La función no retorna ningún objeto, delega la exportación de las imágenes a 
            'plot_corr_matrix_grupal'.

    Recomendaciones:
        - Verificar la existencia y accesibilidad de las utilidades 'comparar_sensores_proximidad' 
          y 'plot_corr_matrix_grupal' en el entorno de ejecución.
        - Asegurar que las variables biométricas del wearable contengan mediciones mayores a cero 
          para evitar filtrados que dejen vacía la muestra.
    """
    # === 1. Validación de Ingesta Poblacional ===
    print("\n[INFO] Generando matrices de control poblacional global...")
    
    # --- 1.1 Dexcom vs Libre (Búsqueda por proximidad temporal en toda la población) ---
    lista_cruces_reales = []
    for user_id, df_sujeto in df_global.groupby('user_id', observed=False):
        # Reutilizamos tu función del Bloque 1 (tolerancia unificada a 3 min para el grupal)
        df_prox_sujeto = comparar_sensores_proximidad(df_sujeto, tolerancia_minutos=3)
        if not df_prox_sujeto.empty:
            lista_cruces_reales.append(df_prox_sujeto)
            
    if lista_cruces_reales:
        df_sensores_poblacional = pd.concat(lista_cruces_reales, ignore_index=True)
        # Invoca la función grupal para guardar en: Grupos / Matriz / Sensores
        plot_corr_matrix_grupal(df_sensores_poblacional, ["glucose_libre", "glucose_dexcom"], "Poblacional Todos los Sensores", "Sensores")
        
    # --- 1.2 Fitbit Wearable Global (Sincronización masiva de HR vs Calorías vs METs) ---
    df_fitbit_poblacional = df_global[["heart_rate", "calories_activity", "mets"]].dropna(how='all')
    # Filtro de actividad biológica mínima para evitar nulos masivos
    df_fitbit_poblacional = df_fitbit_poblacional[(df_fitbit_poblacional["heart_rate"] > 0) | (df_fitbit_poblacional["calories_activity"] > 0)]
    
    if len(df_fitbit_poblacional) > 10:
        # Guarda en: Grupos / Matriz / Actividad
        plot_corr_matrix_grupal(df_fitbit_poblacional, ["heart_rate", "calories_activity", "mets"], "Poblacional Wearable Telemetria", "Actividad")


def generar_analisis_por_grupos_diabetes(df_global):
    """
    Ejecuta el pipeline estadístico masivo para segmentar y analizar la población por cohortes clínicas.

    Genera las variables derivadas metabólicas para cada participante, concatena la información y 
    separa el conjunto global en tres fenotipos diagnósticos. Para cada subgrupo, aplica máscaras lógicas 
    temporales para aislar los estados biológicos autónomos (ayuno y postprandial), computa matrices 
    de Spearman estáticas, traza perfiles de correlación cruzada (CCF) y consolida un CSV resumen grupal.

    Args:
        df_global (pd.DataFrame): DataFrame centralizado con los registros multisujeto de todos los 
            participantes y sus respectivas asignaciones de grupo clínico.

    Returns:
        None: La función no retorna ningún objeto, exporta la matriz estructurada (.csv) y los gráficos a disco.

    Recomendaciones:
        - Asegurar que constantes como 'VARS_POSTPRANDIAL', 'VARS_AYUNO' y 'CORRELACION_DIR' se encuentren 
          inicializadas previamente en las celdas de configuración del Notebook.
        - Verificar que los métodos auxiliares 'calcular_variables_derivadas_metabolicas', 'plot_corr_matrix_grupal' 
          y 'plot_ccf_grupal' estén cargados para evitar excepciones en mitad de la iteración.
    """
    # === 1. Preprocesamiento General por Participante y Concatenación ===
    grupos_metabolicos = ['No Diabético', 'Prediabético', 'Diabético']
    
    lista_sujetos_procesados = []
    for user_id, df_sujeto in df_global.groupby('user_id', observed=False):
        df_sujeto_proc = calcular_variables_derivadas_metabolicas(df_sujeto)
        lista_sujetos_procesados.append(df_sujeto_proc)
        
    df_poblacion_total = pd.concat(lista_sujetos_procesados, ignore_index=True)
    
    # Lista para almacenar las métricas consolidadas de cada cohorte clínica
    metricas_grupales_csv = []
    
    # === 2. Bucle Estructural de Cohortes Clínicas ===
    for grupo in grupos_metabolicos:
        df_grupo = df_poblacion_total[df_poblacion_total['grupo_metabolico'] == grupo].copy()
        if df_grupo.empty:
            continue
            
        grupo_fn = grupo.lower().replace(' ', '_').replace('é', 'e').replace('á', 'a')
        print(f" -> Procesando Cohorte Clínica Grupal: {grupo}...")
        
        # Inicializamos el registro de este grupo para el CSV
        datos_grupo = {"grupo_metabolico": grupo}

        # -----------------------------------------------------------------
        # A. APLICACIÓN DE FILTROS DE ESTADO POBLACIONALES (Ayuno vs Postprandial)
        # -----------------------------------------------------------------
        # Forzado de orden posicional para alineamiento uniforme de ventanas de tiempo
        df_grupo = df_grupo.reset_index(drop=True)
        indices_comidas = df_grupo[df_grupo['carbs'] > 0].index.tolist()
        
        es_postprandial = np.zeros(len(df_grupo), dtype=bool)
        for idx in indices_comidas:
            es_postprandial[idx : min(idx + 180, len(df_grupo))] = True
            
        # Estimación de minutos transcurridos tras el último evento de ingesta registrado
        es_comida_binario = (df_grupo['carbs'] > 0).astype(int)
        comida_acum = es_comida_binario.cumsum()
        minutos_desde_comida = df_grupo.groupby(comida_acum).cumcount()
        minutos_desde_comida.loc[comida_acum == 0] = 9999 
        
        es_ayuno = (minutos_desde_comida >= 240) & (~es_postprandial)
        
        # Aislamiento de sub-DataFrames específicos según el entorno biológico
        df_postprandial_grupo = df_grupo[es_postprandial].copy()
        df_ayuno_grupo = df_grupo[es_ayuno].copy()

        # -----------------------------------------------------------------
        # B. COHORTES - ANÁLISIS MATRICIAL ESTÁTICO (Spearman)
        # -----------------------------------------------------------------
        # B.1 Matriz Postprandial Grupal (Macros vs Deltas de 30 y 60 min)
        if not df_postprandial_grupo.empty:
            df_platos_exactos = df_postprandial_grupo[df_postprandial_grupo['carbs'] > 0]
            if not df_platos_exactos.empty:
                plot_corr_matrix_grupal(df_platos_exactos, VARS_POSTPRANDIAL, f"Postprandial Macros {grupo}", "Postprandial")
                
                # Extraemos matemáticamente las fuerzas de correlación colectivas para el CSV
                mat_post = df_platos_exactos[VARS_POSTPRANDIAL].select_dtypes(include=[np.number]).dropna(how='all')
                if not mat_post.empty:
                    corr_post = mat_post.corr(method='spearman')
                    for macro in ["carbs", "fiber", "fat", "protein"]:
                        if macro in corr_post.index:
                            if "dexcom_delta_60" in corr_post.columns:
                                datos_grupo[f"corr_{macro}_vs_deltaDexcom60"] = corr_post.loc[macro, "dexcom_delta_60"]
                            if "libre_delta_60" in corr_post.columns:
                                datos_grupo[f"corr_{macro}_vs_deltaLibre60"] = corr_post.loc[macro, "libre_delta_60"]

        # B.2 Matriz de Ayuno Grupal (Tono Basal Fitbit vs Glucosa Absoluta Dexcom Real)
        if not df_ayuno_grupo.empty:
            df_ayuno_real = df_ayuno_grupo.dropna(subset=["glucose_dexcom"])
            if not df_ayuno_real.empty:
                plot_corr_matrix_grupal(df_ayuno_real, VARS_AYUNO, f"Ayuno Basal {grupo}", "Ayuno")
                
                mat_ayuno = df_ayuno_real[VARS_AYUNO].select_dtypes(include=[np.number]).dropna(how='all')
                if not mat_ayuno.empty:
                    corr_ayuno = mat_ayuno.corr(method='spearman')
                    if "heart_rate" in corr_ayuno.index and "glucose_dexcom" in corr_ayuno.columns:
                        datos_grupo["corr_ayuno_HR_vs_GlucosaAbs"] = corr_ayuno.loc["heart_rate", "glucose_dexcom"]

        # -----------------------------------------------------------------
        # C. COHORTES - ANÁLISIS TEMPORAL DINÁMICO (CCF)
        # -----------------------------------------------------------------
        # C.1 CCF de Ayuno (Frecuencia real Dexcom para mitigar tendencias)
        if not df_ayuno_grupo.empty:
            df_ayuno_ccf = df_ayuno_grupo.dropna(subset=["glucose_dexcom"])
            if len(df_ayuno_ccf) > 15:
                # Modificación para extraer los valores en base al return de plot_ccf_grupal
                for v1, v2, c_label in [("calories_accum_30", "glucose_dexcom", "calAcum30"), ("mets_sustained_15", "glucose_dexcom", "metsSust15")]:
                    if v1 in df_ayuno_ccf.columns:
                        res = plot_ccf_grupal(df_ayuno_ccf[v1], df_ayuno_ccf[v2], v1, v2, grupo, "Ayuno", max_lag=180)
                        if res: 
                            datos_grupo[f"ayuno_lag_{c_label}"], datos_grupo[f"ayuno_rho_{c_label}"] = res

        # C.2 CCF Postprandial (Señales continuas interpoladas a 1 min para buscar el Time-to-Peak de digestión)
        if not df_postprandial_grupo.empty:
            if "carbs" in df_postprandial_grupo.columns:
                if "glucose_dexcom" in df_postprandial_grupo.columns:
                    res = plot_ccf_grupal(df_postprandial_grupo["carbs"], df_postprandial_grupo["glucose_dexcom"], "carbs", "glucose_dexcom", grupo, "Postprandial", max_lag=180)
                    if res: 
                        datos_grupo["post_carbs_timeToPeak_dexcom"], datos_grupo["post_carbs_rho_dexcom"] = res
                if "glucose_libre" in df_postprandial_grupo.columns:
                    res = plot_ccf_grupal(df_postprandial_grupo["carbs"], df_postprandial_grupo["glucose_libre"], "carbs", "glucose_libre", grupo, "Postprandial", max_lag=180)
                    if res: 
                        datos_grupo["post_carbs_timeToPeak_libre"], datos_grupo["post_carbs_rho_libre"] = res

        # Acumulación de las métricas de la cohorte actual para la tabla consolidada
        metricas_grupales_csv.append(datos_grupo)

    # === 3. Exportación Automática del CSV Colectivo Consolidado ===
    if metricas_grupales_csv:
        df_resumen_grupos = pd.DataFrame(metricas_grupales_csv)
        csv_salida = os.path.join(CORRELACION_DIR, "resumen_correlaciones_grupales_cohortes.csv")
        df_resumen_grupos.to_csv(csv_salida, index=False, encoding="utf-8-sig")
        print(f"\n[EXITO] Guardado CSV de Métricas por Cohortes en: {csv_salida}")


# =========================================================================
# ORQUESTADOR DEL PIPELINE POBLACIONAL Y COHORTES (GRUPOS)
# =========================================================================

def main_poblacional():
    """
    Orquestador principal que controla la carga unificada de la cohorte,
    inyecta las categorías del diccionario MAPEO_DIABETES, calcula variables
    derivadas agregadas y ejecuta secuencialmente los análisis grupales para el TFG.
    """
    print("\n" + "="*60)
    print("=== INICIANDO PIPELINE DE ANÁLISIS POBLACIONAL (GRUPOS) ===")
    print("="*60)
    
    patron = f"*{FILTRO_NOMBRE}*.csv" if FILTRO_NOMBRE else "*.csv"
    archivos = glob.glob(os.path.join(TRUSTED_DIR, patron))
    if not archivos:
        print(f"[ERROR] No se encontraron archivos CSV en '{TRUSTED_DIR}' con filtro '{FILTRO_NOMBRE}'")
        return
        
    print(f"Cargando e integrando en memoria {len(archivos)} archivo(s) de pacientes...")
    
    lista_dfs_poblacion = []
    
    # 1. Bucle de carga masiva, inyección de ID e inyección de Fenotipo Médico
    for archivo in archivos:
        user_id = parse_user_from_filename(archivo)
        try:
            temp_df = pd.read_csv(archivo)
            temp_df['user_id'] = user_id
            
            # --- INYECCIÓN DINÁMICA DE TU DICCIONARIO MAPEO_DIABETES ---
            # Si el sujeto existe en tu diccionario, le asigna su grupo clínico. 
            # Si no existe, lo clasifica temporalmente como "No Asignado" para evitar caídas del script.
            if user_id in MAPEO_DIABETES:
                temp_df['grupo_metabolico'] = MAPEO_DIABETES[user_id]
            else:
                temp_df['grupo_metabolico'] = "No Asignado"
                print(f"[WARN] El sujeto '{user_id}' no se encontró en MAPEO_DIABETES. Clasificado como 'No Asignado'.")
            
            # Calculamos las variables rodantes y de desfase individuales antes de concatenar 
            # para asegurar que los rollings operen estrictamente sobre la línea de tiempo pura de cada paciente.
            temp_df_proc = calcular_variables_derivadas_metabolicas(temp_df)
            lista_dfs_poblacion.append(temp_df_proc)
            
        except Exception as e:
            print(f"[ERROR] No se pudo leer el archivo de '{user_id}': {e}")
            continue
            
    if not lista_dfs_poblacion:
        print("[ERROR CRÍTICO] Ningún archivo pudo ser cargado de forma válida en memoria.")
        return
        
    # Concatenamos todos los datos individuales transformados en un super DataFrame de cohorte
    df_global = pd.concat(lista_dfs_poblacion, ignore_index=True)
    
    # --- FILTRO DE SEGURIDAD CLÍNICA (LÓGICA DE TU SCATTER GRUPAL) ---
    # Excluye picos imposibles o errores humanos de registro nutricional (> 300g de carbohidratos, > 40g de fibra en una sola ingesta)
    condicion_carbs = (df_global['carbs'] <= 300) | (df_global['carbs'].isna())
    condicion_fibra = (df_global['fiber'] <= 40) | (df_global['fiber'].isna())
    df_global = df_global[condicion_carbs & condicion_fibra]
    print(f"[OK] Integración finalizada con éxito. Registros colectivos totales: {len(df_global)} minutos.")
    
    try:
        # 2. Análisis Poblacional Global (Sin filtros de estado: Gráficos 1.1 y 1.2)
        print("\n[Fase 1/2] Ejecutando Validación Poblacional de Instrumentos (Sensores y Wearable)...")
        generar_matrices_control_poblacional(df_global)
        
        # 3. Análisis por Cohortes Clínicas (Con filtros de estado: Gráficos 1.a, 1.b, 2.1 y 2.2)
        print("\n[Fase 2/2] Ejecutando Segmentación y Matrices/CCF por Grupos Clínicos de Diabetes...")
        generar_analisis_por_grupos_diabetes(df_global)
        
        print("\n" + "="*60)
        print("=== PIPELINE POBLACIONAL DE GRUPOS COMPLETADO CON ÉXITO ===")
        print("Las imágenes se han clasificado en la estructura jerárquica de la carpeta 'Grupos'")
        print("="*60)
        
    except Exception as e:
        print(f"\n[ERROR CRÍTICO] El motor de análisis de grupos falló durante la ejecución: {e}")

In [ ]:
if __name__ == "__main__":
    # Activación nativa al lanzar el script independiente
    main_poblacional()

### C2. Análisis Univariante

**📋 PLAN DE DESARROLLO Y ANÁLISIS DE SERIES TEMPORALES**
________________________________________________________________________________________

Este apartado establece una estrategia estructurada para auditar la estructura estadística de la señal de glucosa, identificando patrones de memoria larga, periodicidades circadianas y criterios de estabilización matemática.

---

**🔄 Fases del Flujo de Trabajo Analítico**
__________________________________________________

Para ejecutar un diagnóstico exhaustivo de la serie, se sugiere seguir el siguiente itinerario técnico secuencial:

**1. Inspección Visual de Tendencias y Dispersión**
*   Evaluar la serie temporal en bruto tanto en su extensión completa como desagregada por jornadas independientes.
*   *Objetivo:* Determinar si la media aritmética y la variabilidad (varianza) experimentan oscilaciones macroscópicas significativas a lo largo del tiempo.

**2. Estimación de Memoria Estacional por Autocorrelación Global**
*   Calcular la función de autocorrelación (ACF) extendiendo el horizonte hasta un rango amplio de 1440 retardos (*lags*).
*   *Objetivo:* Identificar de forma empírica la presencia de ciclos o periodicidades repetitivas de carácter diario (24 horas).

**3. Evaluación de Dependencia Directa Local**
*   Acotar la función de autocorrelación parcial (PACF) a un horizonte temporal notablemente inferior, fijando el límite entre 240 y 360 retardos.
*   *Objetivo:* Aislar la dependencia directa local entre observaciones contiguas, mitigando la introducción de ruido de fondo o correlaciones espurias indirectas.

**4. Descomposición Estructural de la Señal**
*   Implementar un algoritmo de descomposición STL (*Seasonal and Trend decomposition using Loess*) configurando un periodo estacional diario.
*   *Objetivo:* Aislar matemáticamente las componentes de tendencia a largo plazo, la estacionalidad periódica y el residuo estocástico, una aproximación de alta utilidad en el análisis de glucosa continua.

**5. Contrastes Formales de Estacionariedad**
*   Aplicar de forma simultánea e integrada los tests estadísticos de Dickey-Fuller Aumentado (ADF) y KPSS (*Kwiatkowski-Phillips-Schmidt-Shin*).
*   *Objetivo:* Evaluar formalmente las propiedades estocásticas tanto de la serie original como de sus subsiguientes versiones transformadas.

**6. Modelado de Operadores de Diferenciación**
*   Aplicar un operador de diferenciación simple ($\Delta Y_t$) ante evidencias de tendencias o persistencia extrema, y una diferenciación estacional indexada a 1440 minutos si la impronta del ritmo circadiano es dominante.

**7. Diagnóstico de la Serie Transformada**
*   Reevaluar las curvas ACF y PACF sobre los residuos resultantes de las transformaciones previas.
*   *Objetivo:* Comprobar matemáticamente si la serie depurada converge de forma efectiva hacia un comportamiento estacionario.

---

**🎯 Criterios de Interpretación en la Práctica Biomédica**
__________________________________________________

Al procesar los resultados analíticos, se deben contrastar las trazas frente a los siguientes patrones de comportamiento:

*   **Comportamiento de la ACF en Bruto:** Si los coeficientes muestran un decaimiento hiperbólico sumamente lento acompañado de picos armónicos cíclicos cada 1440 minutos, se confirma que la serie original no es estacionaria y posee una fuerte estacionalidad.
*   **Impacto de la Diferenciación:** Si tras inyectar una diferencia regular (1 minuto) o estacional (24 horas) la curva ACF experimenta un truncamiento abrupto cayendo rápidamente a cero, se valida que la estrategia de estabilización avanza en la dirección correcta.
*   **Estructura Local en la PACF:** Si la curva parcial exhibe únicamente un número reducido y aislado de picos significativos en los rezagos iniciales cortos, el comportamiento autorregresivo local puede modelarse matemáticamente de forma eficiente con pocos términos.
*   **Persistencia de Inestabilidad:** Si a pesar de las transformaciones lineales la serie no estabiliza sus momentos estadísticos, se deduce que el problema no radica exclusivamente en la raíz unitaria, sino en una marcada heterogeneidad estructural gatillada por perturbaciones exógenas (ingestas alimentarias, picos de actividad física o cronobiología adaptativa).

---

**📌 Conclusión y Estrategia de Modelado**
__________________________________________________

En síntesis, se recomienda utilizar la **ACF** como el instrumento de diagnóstico principal para mapear la memoria larga y las dependencias cíclicas globales del dataset, reservando la **PACF** para caracterizar la dependencia local inmediata y guiar la selección del orden del modelo. 

Para abordar el criterio de estacionariedad, la ruta óptima consiste en auditar la señal nativa, someterla a descomposición STL y operadores de diferenciación estacional, y validar formalmente la estructura de los residuos mediante contrastes ADF/KPSS antes de proceder a la parametrización de cualquier modelo predictivo formal.


#### C2.1 Estacionalidad

##### C2.1.1 Autocorrelación (ACF) y Autocorrelación Parcial (PACF)

La autocorrelación examina la dependencia entre valores de la serie en diferentes retrasos (lags). Útil para identificar patrones repetitivos.

La autocorrelación parcial permite entender la relación entre una serie temporal y sus rezagos, eliminando el efecto de los rezagos intermedios.

Para modificar los lags y hacer que el análisis sea más flexible, podemos usar la función acf de statsmodels en lugar de pd.plotting.autocorrelation_plot, ya que esta función permite especificar el número de lags que deseamos calcular y visualizar.

Debemos tener en cuenta que cada lag representa 1 minuto y que se visualiza un día completo más un pequeño margen (1440 lags ≈ 24 horas):

In [ ]:
if False:
    def analizar_autocorrelacion(df, variable, archivo, lags=1440, out_dir=r'C:\Users\regin\Documents\GitHub\TFG-pruebas\Analysis\CGMacros\Autocorrelación'):
        """
        Analiza y grafica la autocorrelación de la variable dada, marcando cada hora.
        Guarda la imagen en la carpeta especificada.
        """
        nombre_base = os.path.splitext(os.path.basename(archivo))[0]
        titulo = f"Autocorrelación {variable} {nombre_base}"

        if df[variable].notna().sum() > 0:
            os.makedirs(out_dir, exist_ok=True)
            fig, ax = plt.subplots(figsize=(10, 6))
            plot_acf(df[variable].dropna(), lags=lags, ax=ax)
            ax.set_title(titulo)
            ax.set_xlabel("Lags (horas)")
            ax.set_ylabel("Autocorrelación")
            xticks = np.arange(0, lags + 1, 60)
            ax.set_xticks(xticks)
            ax.set_xticklabels([f"{tick // 60} h" for tick in xticks], rotation=45, ha='right')
            for tick in xticks:
                ax.axvline(x=tick, color='gray', linestyle='--', linewidth=0.7)
            plt.tight_layout()
            filename = os.path.join(out_dir, f"{nombre_base}_autocorrelacion_{variable}.png")
            fig.savefig(filename, dpi=300)
            mostrar_y_liberar_figura(fig)
            print(f"[OK] Guardado: {filename}")
        else:
            print(f"No hay suficientes datos válidos para el análisis de autocorrelación de {variable}.")

    def analizar_autocorrelacion_parcial(df, variable, archivo, lags=120, out_dir=r'C:\Users\regin\Documents\GitHub\TFG-pruebas\Analysis\CGMacros\Autocorrelación Parcial'):
        """
        Analiza y grafica la autocorrelación parcial de la variable dada, marcando cada hora.
        Guarda la imagen en la carpeta especificada.
        """
        nombre_base = os.path.splitext(os.path.basename(archivo))[0]
        titulo = f"Autocorrelación Parcial {variable} {nombre_base}"

        if df[variable].notna().sum() > 0:
            os.makedirs(out_dir, exist_ok=True)
            fig, ax = plt.subplots(figsize=(10, 6))
            plot_pacf(df[variable].dropna(), lags=lags, ax=ax)
            ax.set_title(titulo)
            ax.set_xlabel("Lags (horas)")
            ax.set_ylabel("Autocorrelación Parcial")
            xticks = np.arange(0, lags + 1, 60)
            ax.set_xticks(xticks)
            ax.set_xticklabels([f"{tick // 60} h" for tick in xticks], rotation=45, ha='right')
            for tick in xticks:
                ax.axvline(x=tick, color='gray', linestyle='--', linewidth=0.7)
            plt.tight_layout()
            filename = os.path.join(out_dir, f"{nombre_base}_autocorrelacion_parcial_{variable}.png")
            fig.savefig(filename, dpi=300)
            mostrar_y_liberar_figura(fig)
            print(f"[OK] Guardado: {filename}")
        else:
            print(f"No hay suficientes datos válidos para el análisis de autocorrelación parcial de {variable}.")

    def procesar_archivos(carpeta_archivos, lags=1440):
        """
        Procesa todos los archivos CSV en la carpeta, y analiza autocorrelación y autocorrelación parcial para 'glucose'.
        El nombre base del archivo se usa para título y nombre de imagen.
        """
        for archivo in os.listdir(carpeta_archivos):
            if archivo.endswith('.csv'):
                ruta_archivo = os.path.join(carpeta_archivos, archivo)
                try:
                    df = pd.read_csv(ruta_archivo)
                    print(f"\nProcesando {archivo}...")
                    analizar_autocorrelacion(df, 'glucose_dexcom', archivo, lags=lags)
                    analizar_autocorrelacion(df, 'glucose_libre', archivo, lags=lags)
                    analizar_autocorrelacion_parcial(df, 'glucose_dexcom', archivo, lags=120)
                    analizar_autocorrelacion_parcial(df, 'glucose_libre', archivo, lags=120)
                except Exception as e:
                    print(f"Error procesando el archivo {archivo}: {e}")

def analizar_autocorrelacion(df, variable, archivo, lags=1440, out_dir=r'C:\Users\regin\Documents\GitHub\TFG-pruebas\Analysis\CGMacros\Autocorrelación'):
    """
    Calcula y grafica la función de autocorrelación (ACF) marcando líneas de referencia horarias.

    Verifica la existencia de registros válidos para la columna especificada, genera un gráfico 
    de autocorrelación global acotado por el número de lags indicado y configura marcas visuales 
    e intervalos en el eje horizontal cada 60 unidades (1 hora) para facilitar la lectura temporal.

    Args:
        df (pd.DataFrame): DataFrame que contiene la serie temporal bajo estudio.
        variable (str): Nombre de la columna de interés sobre la que se calculará la ACF.
        archivo (str): Ruta completa o nombre del fichero de origen para extraer metadatos de título.
        lags (int, opcional): Cantidad máxima de retardos a evaluar en la serie. Por defecto es 1440.
        out_dir (str, opcional): Ruta del sistema para almacenar la imagen resultante. 
            Por defecto apunta a la carpeta local de Autocorrelación del repositorio.

    Returns:
        None: La función no retorna ningún objeto, exporta la gráfica (.png) directamente al disco.

    Recomendaciones:
        - Asegurar la importación previa de 'plot_acf' desde statsmodels.graphics.tsaplots en el entorno.
        - Verificar que los datos no contengan strings o valores vacíos continuos antes de aplicar 
          la limpieza por dropna para evitar muestras insuficientes.
    """
    # === 1. Aislamiento del Nombre Base y Configuración del Título ===
    nombre_base = os.path.splitext(os.path.basename(archivo))[0]
    titulo = f"Autocorrelación {variable} {nombre_base}"

    # === 2. Validación de Datos y Creación del Lienzo ===
    if df[variable].notna().sum() > 0:
        os.makedirs(out_dir, exist_ok=True)
        fig, ax = plt.subplots(figsize=(10, 6))
        
        # Renderizado de la curva ACF sobre el eje configurado
        plot_acf(df[variable].dropna(), lags=lags, ax=ax)
        
        # Personalización estética de las etiquetas
        ax.set_title(titulo)
        ax.set_xlabel("Lags (horas)")
        ax.set_ylabel("Autocorrelación")
        
        # === 3. Configuración Específica del Eje Temporal ===
        # Definición de marcas equidistantes cada 60 minutos (1 hora) en el eje X
        xticks = np.arange(0, lags + 1, 60)
        ax.set_xticks(xticks)
        ax.set_xticklabels([f"{tick // 60} h" for tick in xticks], rotation=45, ha='right')
        
        # Inyección de rejillas verticales grises discontinuas sobre las marcas horarias
        for tick in xticks:
            ax.axvline(x=tick, color='gray', linestyle='--', linewidth=0.7)
            
        plt.tight_layout()
        
        # === 4. Exportación Física del Artefacto y Cierre ===
        filename = os.path.join(out_dir, f"{nombre_base}_autocorrelacion_{variable}.png")
        fig.savefig(filename, dpi=300)
        mostrar_y_liberar_figura(fig)
        print(f"[OK] Guardado: {filename}")
    else:
        print(f"No hay suficientes datos válidos para el análisis de autocorrelación de {variable}.")


def analizar_autocorrelacion_parcial(df, variable, archivo, lags=120, out_dir=r'C:\Users\regin\Documents\GitHub\TFG-pruebas\Analysis\CGMacros\Autocorrelación Parcial'):
    """
    Calcula y grafica la función de autocorrelación parcial (PACF) marcando referencias horarias.

    Verifica la existencia de registros numéricos no nulos, inicializa el subgráfico para evaluar 
    la PACF acotando el horizonte para capturar dependencias directas locales y añade cuadrículas 
    y etiquetas formateadas en horas cada 60 unidades.

    Args:
        df (pd.DataFrame): DataFrame que contiene la serie temporal bajo estudio.
        variable (str): Nombre de la columna de interés sobre la que se calculará la PACF.
        archivo (str): Ruta completa o nombre del fichero de origen para extraer metadatos de título.
        lags (int, opcional): Cantidad máxima de retardos locales a evaluar. Por defecto es 120.
        out_dir (str, opcional): Ruta del sistema para almacenar la imagen resultante. 
            Por defecto apunta a la carpeta local de Autocorrelación Parcial del repositorio.

    Returns:
        None: La función no retorna ningún objeto, exporta la gráfica (.png) directamente al disco.

    Recomendaciones:
        - Asegurar la importación previa de 'plot_pacf' desde statsmodels.graphics.tsaplots en el entorno.
        - Utilizar un número de lags moderado (como el valor por defecto de 120) para analizar la estructura 
          autorregresiva directa local sin introducir excesivo ruido gráfico.
    """
    # === 1. Aislamiento del Nombre Base y Configuración del Título ===
    nombre_base = os.path.splitext(os.path.basename(archivo))[0]
    titulo = f"Autocorrelación Parcial {variable} {nombre_base}"

    # === 2. Validación de Datos y Creación del Lienzo ===
    if df[variable].notna().sum() > 0:
        os.makedirs(out_dir, exist_ok=True)
        fig, ax = plt.subplots(figsize=(10, 6))
        
        # Renderizado de la curva PACF sobre el eje configurado
        plot_pacf(df[variable].dropna(), lags=lags, ax=ax)
        
        # Personalización estética de las etiquetas
        ax.set_title(titulo)
        ax.set_xlabel("Lags (horas)")
        ax.set_ylabel("Autocorrelación Parcial")
        
        # === 3. Configuración Específica del Eje Temporal ===
        # Definición de marcas equidistantes cada 60 minutos (1 hora) en el eje X
        xticks = np.arange(0, lags + 1, 60)
        ax.set_xticks(xticks)
        ax.set_xticklabels([f"{tick // 60} h" for tick in xticks], rotation=45, ha='right')
        
        # Inyección de rejillas verticales grises discontinuas sobre las marcas horarias
        for tick in xticks:
            ax.axvline(x=tick, color='gray', linestyle='--', linewidth=0.7)
            
        plt.tight_layout()
        
        # === 4. Exportación Física del Artefacto y Cierre ===
        filename = os.path.join(out_dir, f"{nombre_base}_autocorrelacion_parcial_{variable}.png")
        fig.savefig(filename, dpi=300)
        mostrar_y_liberar_figura(fig)
        print(f"[OK] Guardado: {filename}")
    else:
        print(f"No hay suficientes datos válidos para el análisis de autocorrelación parcial de {variable}.")


def procesar_archivos(carpeta_archivos, lags=1440):
    """
    Orquesta la ejecución por lotes analizando la autocorrelación regular y parcial para series de glucosa.

    Escanea de forma secuencial el directorio especificado filtrando exclusivamente documentos con 
    extensión `.csv`. Lee el contenido e invoca los módulos analíticos independientes tanto para el 
    sensor Dexcom como para el sensor Libre inyectando los parámetros del horizonte máximo.

    Args:
        carpeta_archivos (str): Ruta del directorio del sistema donde se alojan los ficheros CSV de entrada.
        lags (int, opcional): Horizonte máximo de retardos para el análisis ACF global. Por defecto es 1440.

    Returns:
        None: La función controla el flujo iterativo y la gestión de capturas preventivas de errores.

    Recomendaciones:
        - Verificar que los nombres de las columnas en los archivos de entrada contengan de forma obligatoria 
          'glucose_dexcom' y 'glucose_libre' para evitar advertencias de variables vacías en consola.
    """
    # === 1. Iteración Secuencial de Archivos en el Directorio ===
    for archivo in os.listdir(carpeta_archivos):
        if archivo.endswith('.csv'):
            ruta_archivo = os.path.join(carpeta_archivos, archivo)
            
            try:
                # Lectura e impresión del progreso en terminal
                df = pd.read_csv(ruta_archivo)
                print(f"\nProcesando {archivo}...")
                
                # === 2. Invocación de Módulos Analíticos de Autocorrelación ===
                # Análisis de Autocorrelación Regular (ACF Global)
                analizar_autocorrelacion(df, 'glucose_dexcom', archivo, lags=lags)
                analizar_autocorrelacion(df, 'glucose_libre', archivo, lags=lags)
                
                # Análisis de Autocorrelación Parcial (PACF Local acotado a 120)
                analizar_autocorrelacion_parcial(df, 'glucose_dexcom', archivo, lags=120)
                analizar_autocorrelacion_parcial(df, 'glucose_libre', archivo, lags=120)
                
            except Exception as e:
                # Captura preventiva para asegurar la continuidad del bucle ante fallos de lectura locales
                print(f"Error procesando el archivo {archivo}: {e}")


In [ ]:
def analizar_autocorrelacion(df, variable, archivo, lags=1440, out_dir=r'C:\Users\regin\Documents\GitHub\TFG-pruebas\Analysis\CGMacros\Autocorrelación'):
    """
    Analiza y grafica la autocorrelación de una variable de glucosa.

    La serie se submuestrea antes de calcular la ACF para conservar solo los
    puntos reales de cada fuente: 5 minutos para `glucose_dexcom` y 15 minutos
    para `glucose_libre`.

    Args:
        df (pd.DataFrame): DataFrame de entrada con la variable temporal.
        variable (str): Nombre de la columna a analizar.
        archivo (str): Ruta o nombre del archivo original, usado para el título
            y el nombre de salida.
        lags (int, optional): Número máximo de lags a representar en la ACF.
            Por defecto `1440`.
        out_dir (str, optional): Carpeta donde guardar la imagen generada.

    Returns:
        None: Guarda una figura PNG en disco y muestra un mensaje por consola.

    Ejemplo:
        >>> analizar_autocorrelacion(df, 'glucose_dexcom', 'CGMacros-001.csv')
        >>> analizar_autocorrelacion(df, 'glucose_libre', 'CGMacros-001.csv', lags=720)

    Recomendaciones:
        - Conviene ejecutar esta función sobre datos ya ordenados temporalmente.
        - Si la serie no contiene la columna indicada, la función no dibuja nada.
        - El eje X se interpreta en horas reales aunque el cálculo use una serie
          submuestreada.
    """
    nombre_base = os.path.splitext(os.path.basename(archivo))[0]
    titulo = f"Autocorrelación {variable} {nombre_base}"

    if variable not in df.columns:
        print(f"La variable {variable} no existe en el DataFrame.")
        return

    serie, paso_minutos = _submuestrear_serie_para_acf_pacf(df[variable], variable)
    lags_ajustados = _calcular_lags_ajustados(lags, paso_minutos, len(serie))

    if len(serie) > 0 and lags_ajustados > 0:
        os.makedirs(out_dir, exist_ok=True)
        fig, ax = plt.subplots(figsize=(10, 6))
        plot_acf(serie, lags=lags_ajustados, ax=ax)
        ax.set_title(titulo)
        _configurar_eje_lags(ax, lags_ajustados, paso_minutos, 'Autocorrelación')
        plt.tight_layout()
        filename = os.path.join(out_dir, f"{nombre_base}_autocorrelacion_{variable}.png")
        fig.savefig(filename, dpi=300)
        mostrar_y_liberar_figura(fig)
        print(f"[OK] Guardado: {filename}")
    else:
        print(f"No hay suficientes datos válidos para el análisis de autocorrelación de {variable}.")


def analizar_autocorrelacion_parcial(df, variable, archivo, lags=120, out_dir=r'C:\Users\regin\Documents\GitHub\TFG-pruebas\Analysis\CGMacros\Autocorrelación Parcial'):
    """
    Analiza y grafica la autocorrelación parcial de una variable de glucosa.

    Igual que en la ACF, se conserva solo el muestreo real de cada sensor:
    5 minutos para `glucose_dexcom` y 15 minutos para `glucose_libre`.

    Args:
        df (pd.DataFrame): DataFrame de entrada con la variable temporal.
        variable (str): Nombre de la columna a analizar.
        archivo (str): Ruta o nombre del archivo original, usado para el título
            y el nombre de salida.
        lags (int, optional): Número máximo de lags a representar en la PACF.
            Por defecto `120`.
        out_dir (str, optional): Carpeta donde guardar la imagen generada.

    Returns:
        None: Guarda una figura PNG en disco y muestra un mensaje por consola.

    Ejemplo:
        >>> analizar_autocorrelacion_parcial(df, 'glucose_dexcom', 'CGMacros-001.csv')
        >>> analizar_autocorrelacion_parcial(df, 'glucose_libre', 'CGMacros-001.csv', lags=60)

    Recomendaciones:
        - Para la PACF conviene usar menos lags que en la ACF, porque es más
          sensible al ruido cuando la serie es corta.
        - Si la columna no existe o tiene pocos datos válidos, la función avisa
          y no genera imagen.
    """
    nombre_base = os.path.splitext(os.path.basename(archivo))[0]
    titulo = f"Autocorrelación Parcial {variable} {nombre_base}"

    if variable not in df.columns:
        print(f"La variable {variable} no existe en el DataFrame.")
        return

    serie, paso_minutos = _submuestrear_serie_para_acf_pacf(df[variable], variable)
    lags_ajustados = _calcular_lags_ajustados(lags, paso_minutos, len(serie))

    if len(serie) > 0 and lags_ajustados > 0:
        os.makedirs(out_dir, exist_ok=True)
        fig, ax = plt.subplots(figsize=(10, 6))
        plot_pacf(serie, lags=lags_ajustados, ax=ax)
        ax.set_title(titulo)
        _configurar_eje_lags(ax, lags_ajustados, paso_minutos, 'Autocorrelación Parcial')
        plt.tight_layout()
        filename = os.path.join(out_dir, f"{nombre_base}_autocorrelacion_parcial_{variable}.png")
        fig.savefig(filename, dpi=300)
        mostrar_y_liberar_figura(fig)
        print(f"[OK] Guardado: {filename}")
    else:
        print(f"No hay suficientes datos válidos para el análisis de autocorrelación parcial de {variable}.")


def procesar_archivos(carpeta_archivos, lags=1440):
    """
    Procesa todos los archivos CSV de una carpeta y genera ACF y PACF.

    La función recorre los CSV, analiza `glucose_dexcom` y `glucose_libre`, y
    guarda una imagen por análisis y por archivo.

    Args:
        carpeta_archivos (str): Carpeta que contiene los archivos CSV a procesar.
        lags (int, optional): Número de lags para la ACF. La PACF usa un valor
            menor de forma fija para mantener el cálculo más estable.

    Returns:
        None: Solo imprime progreso por consola y guarda imágenes en disco.

    Ejemplo:
        >>> procesar_archivos(TRUSTED_DIR)
        >>> procesar_archivos(r'Data\Trusted\Completos', lags=720)

    Recomendaciones:
        - Asegúrate de que los CSV tengan columnas `glucose_dexcom` y/o
          `glucose_libre`.
        - Si el volumen de datos es grande, puede tardar bastante en completarse.
    """
    for archivo in os.listdir(carpeta_archivos):
        if archivo.endswith('.csv'):
            ruta_archivo = os.path.join(carpeta_archivos, archivo)
            try:
                df = pd.read_csv(ruta_archivo)
                print(f"\nProcesando {archivo}...")
                analizar_autocorrelacion(df, 'glucose_dexcom', archivo, lags=lags)
                analizar_autocorrelacion(df, 'glucose_libre', archivo, lags=lags)
                analizar_autocorrelacion_parcial(df, 'glucose_dexcom', archivo, lags=120)
                analizar_autocorrelacion_parcial(df, 'glucose_libre', archivo, lags=120)
            except Exception as e:
                print(f"Error procesando el archivo {archivo}: {e}")



def _obtener_paso_muestreo(variable):
    """
    Devuelve el salto en minutos según la fuente de glucosa.

    Args:
        variable (str): Nombre de la variable de glucosa.

    Returns:
        tuple[int, float]: Paso en minutos y frecuencia de muestreo por hora.

    Ejemplo:
        >>> _obtener_paso_muestreo('glucose_dexcom')
        (5, 12.0)
        >>> _obtener_paso_muestreo('glucose_libre')
        (15, 4.0)

    Recomendaciones:
        - Si se pasa otra variable, devuelve un muestreo neutro de 1 minuto.
    """
    if variable == 'glucose_dexcom':
        return 5
    if variable == 'glucose_libre':
        return 15
    return 1



def _submuestrear_serie_para_acf_pacf(serie, variable):
    """
    Conserva el primer dato y luego un dato cada N minutos según la variable.

    Args:
        serie (pd.Series): Serie original con NaNs posibles.
        variable (str): Nombre de la fuente de glucosa.

    Returns:
        tuple[pd.Series, int]: Serie reducida y paso en minutos empleado.

    Ejemplo:
        >>> reducida, paso = _submuestrear_serie_para_acf_pacf(df['glucose_dexcom'], 'glucose_dexcom')
        >>> paso
        5

    Recomendaciones:
        - La serie de entrada debe estar ordenada temporalmente o ya alineada.
        - Esta función elimina NaNs antes de submuestrear.
    """
    paso_minutos = _obtener_paso_muestreo(variable)
    serie_limpia = serie.dropna()
    if paso_minutos <= 1:
        return serie_limpia, 1
    return serie_limpia.iloc[::paso_minutos], paso_minutos



def _calcular_lags_ajustados(lags, paso_minutos, longitud_serie):
    """
    Convierte lags en minutos a lags de la serie submuestreada.

    Args:
        lags (int): Número de lags deseado en minutos reales.
        paso_minutos (int): Intervalo de muestreo efectivo de la serie.
        longitud_serie (int): Número de observaciones disponibles.

    Returns:
        int: Número de lags a usar sobre la serie submuestreada.

    Ejemplo:
        >>> _calcular_lags_ajustados(1440, 5, 200)
        199

    Recomendaciones:
        - Nunca devuelve más lags que observaciones menos una.
        - Si el muestreo real es de 15 minutos, un día completo equivale a 96 lags.
    """
    if longitud_serie < 2:
        return 0
    if paso_minutos <= 1:
        return min(lags, longitud_serie - 1)
    lags_ajustados = max(1, lags // paso_minutos)
    return min(lags_ajustados, longitud_serie - 1)



def _configurar_eje_lags(ax, lags_ajustados, paso_minutos, etiqueta_y):
    """
    Etiqueta el eje X en minutos reales aunque la serie esté submuestreada.

    Args:
        ax (matplotlib.axes.Axes): Eje donde dibujar las marcas de tiempo.
        lags_ajustados (int): Lags efectivos sobre la serie submuestreada.
        paso_minutos (int): Intervalo en minutos de la serie utilizada.
        etiqueta_y (str): Etiqueta para el eje Y.

    Returns:
        None: Modifica el eje recibido.

    Ejemplo:
        >>> _configurar_eje_lags(ax, 96, 15, 'Autocorrelación')

    Recomendaciones:
        - Las marcas se dibujan por horas reales, no por lags submuestreados.
    """
    if lags_ajustados <= 0:
        return
    paso_marcas = max(1, 60 // paso_minutos)
    xticks = np.arange(0, lags_ajustados + 1, paso_marcas)
    ax.set_xticks(xticks)
    ax.set_xticklabels([f"{int(round((tick * paso_minutos) / 60))} h" for tick in xticks], rotation=45, ha='right')
    for tick in xticks:
        ax.axvline(x=tick, color='gray', linestyle='--', linewidth=0.7)
    ax.set_xlabel('Lags (horas reales)')
    ax.set_ylabel(etiqueta_y)


La siguiente celda es computacionalmente costosa, aun habiendo reducido el número de datos a tener en cuenta (Dexcom - 5 minutos, Libre - 15 minutos). Aproximadamente se necesitan 35 minutos.

In [ ]:
def main():
    carpeta_archivos = TRUSTED_DIR
    procesar_archivos(carpeta_archivos)

if __name__ == "__main__":
    main()

In [ ]:
plt.close('all')

##### C2.1.2 Análisis Espectral: periodograma y espectro de Welch

**Qué son**: El periodograma estima la densidad espectral de potencia y localiza dónde se concentra la energía, pero es bastante ruidoso. Welch hace el mismo análisis por segmentos solapados y los promedia, así que suele dar una lectura más estable.

**Idea clave**: En este análisis no tratamos la serie como minuto a minuto. Aunque el flujo general puede venir en minutos, para el espectro usamos la cadencia real de cada sensor: Dexcom a 5 minutos y Libre a 15 minutos.

**Finalidad**: Esta celda analiza la señal real de glucosa de cada subserie, separando `glucose_dexcom` y `glucose_libre`, para comprobar si existe energía espectral compatible con ritmos circadianos (~24 h) o ultradianos (2–6 h) que expliquen patrones observados en la ACF/PACF antes de plantear modelos estacionales como SARIMA.

En el código, cada fuente se submuestrea a su intervalo natural y, a partir de esa serie real, se calculan dos estimadores complementarios: el periodograma clásico y Welch. El primero es más sensible pero más ruidoso; el segundo reduce la varianza y ayuda a confirmar si el pico es real.

**Lectura práctica**:
- Las frecuencias (eje X) se muestran en ciclos por hora (cph).
- Una frecuencia de 1 cph significa un patrón que se repite cada 1 hora.
- 0.5 (1/2) cph equivale a 2 horas.
- 1/24 cph equivale a un periodo de 24 horas.
- En la figura, Dexcom se limita a 0–6 cph y Libre a 0–2 cph para centrar la vista en el rango útil de cada sensor.

**Criterio del resumen**: el periodo dominante no se busca en todo el espectro, sino solo entre 1 y 24 horas, para evitar que frecuencias muy bajas dominen por tendencia lenta o deriva. Por eso el texto de salida indica explícitamente `Periodo dominante (rango de búsqueda: 1-24 h)` y la tabla final compara `Periodo P Dexcom`, `Periodo W Dexcom`, `Periodo P Libre` y `Periodo W Libre`.

**Implicación**:
- Si no aparece energía clara alrededor de 24 h, no conviene forzar un SARIMA estacional con:
  - En Dexcom `s=288` representan un día completo cuando la serie está muestreada cada 5 minutos.
  - Si la serie fuera realmente minuto a minuto, como nuestros datos crudos interpolados, el equivalente diario sería `s=1440`.
  - Para Libre, muestreado cada 15 minutos, un día completo serían `s=96`.
- Si aparece una concentración moderada cerca de 24 h, puede tener sentido probar los `s` anteriores con `D=0` y compararlo con un ARIMA sin estacionalidad usando AIC/BIC y validación temporal.
- Si predominan picos de 2–6 h, la señal puede estar más relacionada con dinámica postprandial, actividad o recuperación, lo que encaja mejor con modelos AR de corto alcance o ARIMAX con covariables exógenas.

En conjunto, este bloque busca diferenciar entre periodicidad fisiológica real y picos espurios del estimador, usando el periodograma como detección rápida y Welch como comprobación más robusta.

In [ ]:
def calcular_periodograma(serie, fs_per_hour=12.0):
    """
    Calcula el periodograma en ciclos por hora.

    Args:
        serie (pd.Series): Serie de valores ya filtrada y, si procede,
            submuestreada.
        fs_per_hour (float, optional): Frecuencia de muestreo por hora.
            Por defecto `12.0`.

    Returns:
        tuple[np.ndarray, np.ndarray]: Frecuencias en ciclos por hora y potencia
        espectral asociada.
    """
    x = serie.to_numpy(dtype=float)
    x = x - np.mean(x)
    n = len(x)
    if n < 2:
        return np.array([]), np.array([])
    X = rfft(x, n=n)
    Pxx = (1.0 / n) * np.abs(X) ** 2
    freqs_cph = rfftfreq(n, d=1.0 / fs_per_hour)
    return freqs_cph, Pxx


def calcular_welch(serie, fs_per_hour=12.0, nperseg=None, noverlap=None, window="hann"):
    """
    Calcula el espectro de Welch en ciclos por hora.

    Args:
        serie (pd.Series): Serie de valores ya filtrada y, si procede,
            submuestreada.
        fs_per_hour (float, optional): Frecuencia de muestreo por hora.
            Por defecto `12.0`.
        nperseg (int, optional): Tamaño del segmento para Welch.
        noverlap (int, optional): Solape entre segmentos.
        window (str, optional): Tipo de ventana para cada segmento.

    Returns:
        tuple[np.ndarray, np.ndarray]: Frecuencias en ciclos por hora y densidad
        espectral promedio.
    """
    x = serie.to_numpy(dtype=float)
    x = x - np.mean(x)
    if len(x) < 2:
        return np.array([]), np.array([])
    if nperseg is None:
        nperseg = min(len(x), max(8, int(fs_per_hour * 24)))
    if noverlap is None:
        noverlap = nperseg // 2
    noverlap = min(noverlap, max(0, nperseg - 1))
    f, Pxx = welch(x, fs=fs_per_hour, window=window, nperseg=nperseg, noverlap=noverlap, detrend="constant")
    return f, Pxx


def graficar_espectro(freqs_cph, P_per, f_welch, P_welch, nombre_base, variable, out_dir=r'C:\Users\regin\Documents\GitHub\TFG-pruebas\Analysis\CGMacros\Periodograma'):
    """
    Grafica de forma combinada la densidad espectral de potencia mediante el Periodograma y el método de Welch.

    Valida la existencia de vectores espectrales, proyecta las curvas en escala semilogarítmica (eje Y log), 
    añade líneas verticales de referencia para los ciclos armónicos principales y ajusta los límites 
    visuales de frecuencia según la naturaleza de muestreo de cada dispositivo sensor.

    Args:
        freqs_cph (np.ndarray): Vector de frecuencias en ciclos/hora estimado por el periodograma.
        P_per (np.ndarray): Vector de potencia estimado por el periodograma.
        f_welch (np.ndarray): Vector de frecuencias en ciclos/hora estimado por Welch.
        P_welch (np.ndarray): Vector de potencia estimado por Welch.
        nombre_base (str): Identificador base del sujeto utilizado para el etiquetado.
        variable (str): Nombre de la columna del sensor bajo análisis ('glucose_dexcom' o 'glucose_libre').
        out_dir (str, opcional): Carpeta física destino para almacenar la imagen. 
            Por defecto apunta al subdirectorio del repositorio local.

    Returns:
        None: La función no retorna ningún objeto, exporta el gráfico (.png) directamente a disco.

    Recomendaciones:
        - Verificar que los objetos pasados contengan valores finitos para evitar distorsiones en la escala logarítmica.
    """
    # === 1. Validación de Carpetas y Creación del Lienzo ===
    os.makedirs(out_dir, exist_ok=True)
    plt.figure(figsize=(10, 6))
    
    # Renderizado condicional de la traza del Periodograma basándose en el tamaño del array
    if getattr(freqs_cph, 'size', 0):
        plt.semilogy(freqs_cph, P_per, color="tab:blue", alpha=0.6, label="Periodograma")
    else:
        plt.text(0.5, 0.5, 'No hay periodograma disponible', ha='center', va='center', transform=plt.gca().transAxes)
        
    # Renderizado condicional de la traza del método de Welch basándose en el tamaño del array
    if getattr(f_welch, 'size', 0):
        plt.semilogy(f_welch, P_welch, color="tab:orange", alpha=0.8, label="Welch")
    else:
        plt.text(0.5, 0.4, 'No hay Welch disponible', ha='center', va='center', transform=plt.gca().transAxes)

    # === 2. Configuración Estética de Ciclos y Límites Espectrales ===
    # Inyección de marcas verticales grises para frecuencias biológicas (ej. 24h, 12h, 6h, etc.)
    marcas = [1.0 / 24.0, 1.0 / 12.0, 1.0 / 6.0, 1.0 / 3.0, 0.5, 1.0, 2.0]
    for m in marcas:
        plt.axvline(m, color="gray", linestyle="--", linewidth=0.8, alpha=0.7)
        
    # Ajuste dinámico del rango visual en base a la tasa de Nyquist característica de cada sensor
    if variable == 'glucose_dexcom':
        plt.xlim(0, 6.0)
    else:
        plt.xlim(0, 2.0)
        
    # Formateo de etiquetas informativas
    plt.xlabel("Frecuencia (ciclos/hora)")
    plt.ylabel("Densidad espectral de potencia (escala log)")
    plt.title(f"Espectro (Periodograma y Welch) - {nombre_base} - {variable}")
    plt.legend()
    plt.tight_layout()
    
    # === 3. Almacenamiento Físico y Liberación de Gráficos ===
    filename = os.path.join(out_dir, f"periodograma_{nombre_base}_{variable}.png")
    plt.savefig(filename, dpi=300)
    mostrar_y_liberar_figura(plt.gcf())
    print(f"[OK] Guardado: {filename}")


def _configuracion_muestreo_espectral(variable):
    """
    Define el intervalo nominal y la frecuencia horaria teórica según el sensor de glucosa continua.

    Args:
        variable (str): Selector del sensor bajo análisis ('glucose_dexcom' o 'glucose_libre').

    Returns:
        tuple (int, float): Una tupla conteniendo el paso en minutos y las muestras por hora (fs).

    Recomendaciones:
        - Utilizar estos coeficientes para homogeneizar la densidad espectral de potencia.
    """
    if variable == 'glucose_dexcom':
        return 5, 12.0
    if variable == 'glucose_libre':
        return 15, 4.0
    return 1, 60.0


def _submuestrear_serie_espectral(serie, variable):
    """
    Aísla las lecturas nativas reales submuestreando la serie continua para el cálculo espectral.

    Args:
        serie (pd.Series): Serie temporal continua con las mediciones de glucosa.
        variable (str): Nombre selector del dispositivo sensor analizado.

    Returns:
        tuple (pd.Series, float): Serie temporal depurada submuestreada y la tasa de muestreo horaria.

    Recomendaciones:
        - Aplicar este filtro para remover interpolaciones artificiales que distorsionen altas frecuencias.
    """
    paso_minutos, fs_per_hour = _configuracion_muestreo_espectral(variable)
    serie_limpia = serie.dropna().astype(float)
    if paso_minutos <= 1:
        return serie_limpia, fs_per_hour
    return serie_limpia.iloc[::paso_minutos], fs_per_hour


def _calcular_periodo_dominante(freqs_cph, potencia):
    """
    Identifica la frecuencia fundamental de máxima potencia y estima su periodo equivalente en horas.

    Construye vectores numéricos, descarta indeterminaciones o valores nulos y localiza el pico 
    máximo absoluto de densidad espectral para derivar el ciclo temporal dominante.

    Args:
        freqs_cph (np.ndarray o list): Vector de frecuencias evaluadas.
        potencia (np.ndarray o list): Vector de potencias estimadas asociadas.

    Returns:
        float o None: El periodo cíclico dominante expresado en horas, o None si no existen datos válidos.

    Recomendaciones:
        - Excluir la frecuencia cero (componente continua/media) para capturar ciclos verdaderamente oscilatorios.
    """
    # === 1. Inicialización y Depuración de Residuos ===
    freqs = np.asarray(freqs_cph, dtype=float)
    potencia = np.asarray(potencia, dtype=float)
    
    # Máscara lógica para descartar valores infinitos y remover la frecuencia cero
    mascara = np.isfinite(freqs) & np.isfinite(potencia) & (freqs > 0)
    if not np.any(mascara):
        return None
        
    freqs_validas = freqs[mascara]
    potencia_valida = potencia[mascara]
    if len(freqs_validas) == 0:
        return None
        
    # === 2. Búsqueda de la Frecuencia Fundamental ===
    freq_dominante = freqs_validas[np.argmax(potencia_valida)]
    if freq_dominante <= 0:
        return None
        
    # Estimación inversa para derivar el periodo temporal en horas
    return 1.0 / freq_dominante


def _formatear_periodo(periodo_horas):
    """
    Normaliza el string del periodo dominante para su correcta representación tabular.

    Args:
        periodo_horas (float o None): Valor numérico del periodo estimado en horas.

    Returns:
        str: Cadena de texto formateada con la unidad de medida correspondiente (min o h).
    """
    if periodo_horas is None:
        return 'N/D'
    if periodo_horas < 1:
        return f"{periodo_horas * 60:.0f} min"
    return f"{periodo_horas:.2f} h"


def procesar_archivos(carpeta_archivos, nperseg_welch=None, noverlap_welch=None, out_dir=r'C:\Users\regin\Documents\GitHub\TFG-pruebas\Analysis\CGMacros\Periodograma'):
    """
    Orquesta la ejecución por lotes estimando y comparando el análisis espectral para la cohorte.

    Escanea el directorio, lee los ficheros de forma ordenada y extrae las señales nativas reales. 
    Calcula el periodograma estándar y el estimador de Welch por sensor, invoca los módulos de dibujo 
    y consolida los periodos dominantes resultantes en un reporte tabular global (.csv).

    Args:
        carpeta_archivos (str): Directorio raíz del sistema donde se alojan los ficheros CSV de entrada.
        nperseg_welch (int, opcional): Longitud de cada segmento para el método de Welch. Por defecto es None.
        noverlap_welch (int, opcional): Puntos de solapamiento entre segmentos de Welch. Por defecto es None.
        out_dir (str, opcional): Carpeta de salida dedicada a almacenar las figuras y el reporte integrado.

    Returns:
        pd.DataFrame: Matriz consolidada de resumen comparativo de periodos dominantes para todos los sujetos.

    Recomendaciones:
        - Asegurar la carga de las rutinas externas 'calcular_periodograma' y 'calcular_welch' en el entorno.
        - Verificar que los ficheros dispongan de al menos 24 registros reales tras remover ceros y nulos.
    """
    # === 1. Inicialización y Preparación de Índices Tabulares ===
    os.makedirs(out_dir, exist_ok=True)
    resumen_periodos = []

    # === 2. Bucle Iterativo y Secuencial Organizado por Sujeto ===
    for archivo in sorted(os.listdir(carpeta_archivos)):
        if not archivo.endswith('.csv'):
            continue
        ruta_archivo = os.path.join(carpeta_archivos, archivo)
        
        try:
            df = pd.read_csv(ruta_archivo, parse_dates=['time'])
            print(f"\nProcesando {archivo}...")
            nombre_base = os.path.splitext(archivo)[0]
            
            # Estructuración base del registro fila para el consolidado tabular
            fila = {
                'Sujeto': nombre_base,
                'Periodo P Dexcom': 'N/D',
                'Periodo W Dexcom': 'N/D',
                'Periodo P Libre': 'N/D',
                'Periodo W Libre': 'N/D',
            }

            # === 3. Procesamiento Espectral Cruzado por Dispositivo Sensor ===
            for variable in ['glucose_dexcom', 'glucose_libre']:
                if variable not in df.columns:
                    continue
                    
                # Exclusión explícita de registros nulos y lecturas iguales a cero (artefactos)
                serie = df[variable].astype(float)
                serie = serie[(~serie.isna()) & (serie != 0)]
                
                # Submuestreo adaptativo para recuperar la tasa de registro original
                serie, fs_per_hour = _submuestrear_serie_espectral(serie, variable)
                if len(serie) < 24:
                    print(f"[WARN] Muy pocos datos válidos en {archivo} para {variable}")
                    continue

                # Estimaciones de Densidad Espectral de Potencia (Nativa y Welch)
                freqs_cph, P_per = calcular_periodograma(serie, fs_per_hour=fs_per_hour)
                f_welch, P_welch = calcular_welch(
                    serie,
                    fs_per_hour=fs_per_hour,
                    nperseg=nperseg_welch,
                    noverlap=noverlap_welch,
                )

                # Invocación del módulo unificado de graficación
                graficar_espectro(freqs_cph, P_per, f_welch, P_welch, nombre_base, variable, out_dir=out_dir)

                # Cálculo analítico de los ciclos temporales dominantes
                periodo_per = _calcular_periodo_dominante(freqs_cph, P_per)
                periodo_w = _calcular_periodo_dominante(f_welch, P_welch)

                # Distribución indexada de strings en la estructura fila
                if variable == 'glucose_dexcom':
                    fila['Periodo P Dexcom'] = _formatear_periodo(periodo_per)
                    fila['Periodo W Dexcom'] = _formatear_periodo(periodo_w)
                else:
                    fila['Periodo P Libre'] = _formatear_periodo(periodo_per)
                    fila['Periodo W Libre'] = _formatear_periodo(periodo_w)

            resumen_periodos.append(fila)
            
        except Exception as e:
            # Captura de errores locales para prevenir la interrupción del pipeline masivo
            print(f"Error procesando el archivo {archivo}: {e}")

    # === 4. Consolidación de Reportes y Exportación Final ===
    df_res = pd.DataFrame(resumen_periodos, columns=['Sujeto', 'Periodo P Dexcom', 'Periodo W Dexcom', 'Periodo P Libre', 'Periodo W Libre'])

    print('\nPeriodo dominante (rango de búsqueda: 1-24 h):')
    if not df_res.empty:
        print(df_res.to_string(index=False))
        df_res.to_csv(os.path.join(out_dir, 'resumen_periodos_dominantes_comparado.csv'), index=False)
        print('[OK] Resumen guardado')
    else:
        print('[WARN] No se generaron periodos dominantes.')

    return df_res


In [ ]:
def main():
    carpeta_archivos = r'C:\Users\regin\Documents\GitHub\TFG-pruebas\Data\CGMacros\Trusted'
    procesar_archivos(carpeta_archivos)

if __name__ == "__main__":
    main()

In [ ]:
plt.close('all')

##### C2.1.4 Descomposición STL

Utilizaremos STL en vez de seasonal_descompose. Para descomponer con mas de una estacionalidad a al vez se puede usar MSTL.

In [ ]:
def _configuracion_stl(variable):
    """
    Define los parámetros nominales de frecuencia y ventanas móviles para la descomposición STL.

    Args:
        variable (str): Selector del sensor bajo análisis ('glucose_dexcom' o 'glucose_libre').

    Returns:
        tuple (int, int, int, int): Una tupla conteniendo el paso en minutos, el periodo estacional, 
            la ventana estacional y la ventana de tendencia teóricas.
    """
    if variable == "glucose_dexcom":
        return 5, 288, 13, 361
    if variable == "glucose_libre":
        return 15, 96, 13, 145
    return 1, 1440, 13, 361


def _submuestrear_serie_stl(serie, variable):
    """
    Normaliza el eje temporal y reindexa la serie continua a su malla nativa regular antes de aplicar STL.

    Aísla los valores numéricos eliminando nulos y valida que el índice de entrada sea estrictamente temporal. 
    Construye un rango de fechas regular anclado a los extremos originales para mitigar el impacto de 
    los desfases microscópicos en la estimación de componentes.

    Args:
        serie (pd.Series): Serie continua con las mediciones de glucosa.
        variable (str): Nombre selector del dispositivo sensor analizado.

    Returns:
        tuple (pd.Series, int, int, int): Serie temporal reindexada y los parámetros estructurales de STL.

    Raises:
        ValueError: Si el índice del objeto Series de entrada no corresponde a un DatetimeIndex.

    Recomendaciones:
        - Aplicar este método de forma sistemática para asegurar que la frecuencia temporal del vector 
          sea homogénea antes de alimentar el motor de ajuste de Loess.
    """
    # === 1. Extracción de Parámetros de Configuración y Depuración ===
    paso_minutos, periodo, seasonal, trend = _configuracion_stl(variable)

    s = pd.to_numeric(serie, errors="coerce").dropna().astype(float)

    if not isinstance(s.index, pd.DatetimeIndex):
        raise ValueError("STL requiere un índice temporal DatetimeIndex")

    s = s.sort_index()

    if paso_minutos <= 1:
        return s, periodo, seasonal, trend

    # === 2. Construcción de la Malla Regular de Reindexación ===
    # Anclar la malla al primer valor disponible
    inicio = s.index[0]
    fin = s.index[-1]
    freq = f"{paso_minutos}min"

    malla = pd.date_range(start=inicio, end=fin, freq=freq)
    serie_muestreada = s.reindex(malla)

    return serie_muestreada, periodo, seasonal, trend


def _descomponer_stl(serie, periodo, seasonal=13, trend=None, robust=True):
    """
    Aplica el algoritmo de descomposición STL segmentando la serie en tramos contiguos exentos de NaN.

    Detecta de forma automática los bloques continuos sin registros perdidos para evitar desalineaciones 
    o desplazamientos artificiales en la componente estacional cíclica. Valida las dimensiones operativas 
    mínimas y las restricciones matemáticas del tamaño de ventana de tendencia antes de ejecutar el ajuste.

    Args:
        serie (pd.Series): Serie temporal regular indexada (DatetimeIndex) con posibles celdas vacías.
        periodo (int): Número de observaciones equivalentes a un ciclo estacional completo (24 horas).
        seasonal (int, opcional): Suavizado de la componente estacional. Por defecto es 13.
        trend (int, opcional): Suavizado de la componente de tendencia a largo plazo. Por defecto es None.
        robust (bool, opcional): Criterio de estimación robusta ante la presencia de valores atípicos. Por defecto es True.

    Returns:
        Resultado o None: Objeto contenedor con las series de las componentes aisladas (.observed, .trend, 
            .seasonal, .resid), o None si el volumen de datos válidos es inferior al umbral mínimo requerido.

    Raises:
        ValueError: Si la estructura indexada de entrada no corresponde a un DatetimeIndex.

    Recomendaciones:
        - Controlar que el valor de 'trend' sea impar y estrictamente mayor que 'periodo' para cumplir 
          con los requisitos de inicialización de la librería subyacente de statsmodels.
    """
    # === 1. Validación de Tipos y Asignación de Contenedores ===
    class Resultado:
        pass

    if not isinstance(serie.index, pd.DatetimeIndex):
        raise ValueError("STL requiere un índice temporal DatetimeIndex")

    serie = serie.sort_index()

    observed = serie.copy()
    trend_s = pd.Series(index=serie.index, dtype=float)
    seasonal_s = pd.Series(index=serie.index, dtype=float)
    resid_s = pd.Series(index=serie.index, dtype=float)

    # === 2. Segmentación Algorítmica de Bloques Contiguos ===
    mask = serie.notna()
    if mask.sum() < periodo * 2:
        return None

    grupos = (mask != mask.shift(fill_value=False)).cumsum()

    # === 3. Ajuste Iterativo de Loess por Tramo Aislado ===
    for _, idx in serie.groupby(grupos).groups.items():
        tramo = serie.loc[idx]
        tramo = tramo.dropna()

        if tramo.notna().sum() < periodo * 2:
            continue

        kwargs = {
            "period": periodo,
            "seasonal": seasonal,
            "robust": robust,
        }
        
        # Validación de las restricciones dimensionales de la ventana de tendencia
        if trend is not None:
            # STL exige trend impar y mayor que period
            if trend % 2 == 0 or trend <= periodo:
                trend_local = None
            else:
                trend_local = trend
            if trend_local is not None:
                kwargs["trend"] = trend_local

        # Invocación y ajuste del modelo STL
        res = STL(tramo, **kwargs).fit()

        # Reintegración de las series parciales resultantes en los vectores de salida globales
        observed.loc[tramo.index] = res.observed
        trend_s.loc[tramo.index] = res.trend
        seasonal_s.loc[tramo.index] = res.seasonal
        resid_s.loc[tramo.index] = res.resid

    # Instanciación y empaquetado final del objeto contenedor
    resultado = Resultado()
    resultado.observed = observed
    resultado.trend = trend_s
    resultado.seasonal = seasonal_s
    resultado.resid = resid_s
    return resultado


def _graficar_descomposicion_stl(result, nombre_base, variable, out_dir=STL_DIR):
    """
    Exporta un panel multiejas vertical con las cuatro componentes estructurales extraídas por STL.

    Args:
        result (Resultado): Objeto contenedor con las series calculadas del desglose de Loess.
        nombre_base (str): Identificador único del participante inyectado en nombres y títulos.
        variable (str): Nombre descriptivo de la columna bajo estudio.
        out_dir (str, opcional): Ruta del directorio físico para guardar la imagen. 

    Returns:
        None: La función graba la figura directamente en el sistema local y libera recursos de memoria.
    """
    # === 1. Configuración de Carpetas y Lienzo Multicapa ===
    os.makedirs(out_dir, exist_ok=True)

    fig, axes = plt.subplots(4, 1, figsize=(14, 9), sharex=True)
    
    # Proyecciones individuales de las series temporales estimadas
    axes[0].plot(result.observed, color="tab:blue", linewidth=1)
    axes[0].set_title("Observada")
    axes[1].plot(result.trend, color="tab:orange", linewidth=1)
    axes[1].set_title("Tendencia")
    axes[2].plot(result.seasonal, color="tab:green", linewidth=1)
    axes[2].set_title("Estacionalidad")
    axes[3].plot(result.resid, color="tab:red", linewidth=1)
    axes[3].axhline(0, color="gray", linestyle="--", linewidth=0.8)
    axes[3].set_title("Residuo")

    # === 2. Personalización Estética y Rejillas ===
    axes[3].set_xlabel("Tiempo")
    for ax in axes:
        ax.grid(True, linestyle="--", alpha=0.3)

    fig.suptitle(f"{nombre_base} - STL {variable}", y=0.995)
    fig.tight_layout()

    # === 3. Almacenamiento Físico del Gráfico Combinado ===
    nombre_salida = f"{nombre_base}_stl_{variable}.png"
    ruta_salida = os.path.join(out_dir, nombre_salida)
    fig.savefig(ruta_salida, dpi=300, bbox_inches="tight")
    mostrar_y_liberar_figura(fig)
    print(f"[OK] Guardado: {ruta_salida}")


# === Inicialización Preventiva de Carpetas para Ficheros de Texto ===
for _dir in STL_CSV_DIRS.values():
    os.makedirs(_dir, exist_ok=True)


def _crear_df_componente(serie, variable, componente, nombre_componente):
    """
    Estructura un DataFrame normalizado listo para serialización tabular a partir de vectores NumPy.

    Args:
        serie (pd.Series): Serie temporal base que proveerá el eje indexado de marcas de tiempo.
        variable (str): Nombre descriptivo del dispositivo sensor evaluado.
        componente (np.ndarray o list): Vector numérico correspondiente a la componente STL aislada.
        nombre_componente (str): Etiqueta descriptiva para nombrar la columna de salida de datos.

    Returns:
        pd.DataFrame: Matriz de datos formateada con las columnas 'time', 'variable' y la métrica asignada.
    """
    return pd.DataFrame(
        {
            "time": serie.index,
            "variable": variable,
            nombre_componente: np.asarray(componente),
        }
    )


def _guardar_componentes_stl(nombre_base, componentes):
    """
    Consolida y exporta a ficheros CSV de forma indexada los listados de componentes de Loess acumulados.

    Args:
        nombre_base (str): Identificador único del participante utilizado para nombrar el archivo de salida.
        componentes (dict): Diccionario indexado por tipo de componente que aloja listas de sub-DataFrames.

    Returns:
        None: La función escribe físicamente las tablas consolidadas en sus rutas fijas del sistema.

    Recomendaciones:
        - Garantizar la existencia previa del diccionario de rutas 'STL_CSV_DIRS' antes de realizar la invocación.
    """
    # === 1. Bucle Iterativo de Serialización Tabular ===
    for nombre_componente, frames in componentes.items():
        if not frames:
            continue

        # Concatenación de todas las subseries temporales y volcado estructurado a disco
        # Concatenación de todas las subseries temporales y volcado estructurado a disco
        df_componente = pd.concat(frames, ignore_index=True)
        ruta_salida = os.path.join(STL_CSV_DIRS[nombre_componente], f"{nombre_base}.csv")
        df_componente.to_csv(ruta_salida, index=False)
        print(f"[OK] Guardado CSV STL: {ruta_salida}")


def procesar_descomposicion_stl(
    carpeta_archivos=TRUSTED_DIR,
    variables=None,
    guardar_csv=True,
):
    """
    Orquesta la ejecución por lotes aplicando descomposición STL sobre las series temporales de la cohorte.

    Escanea de forma secuencial el directorio especificado, lee de manera cronológica cada fichero 
    y ejecuta el submuestreo adaptativo por sensor. Aplica el modelo robusto de Loess por tramos contiguos 
    para evitar sesgos por pérdidas, invoca el módulo de dibujo vertical y automatiza la exportación 
    de los reportes consolidados tanto tabulares como analíticos.

    Args:
        carpeta_archivos (str, opcional): Ruta del directorio del sistema donde se alojan los ficheros 
            CSV limpios de entrada. Por defecto apunta a la constante TRUSTED_DIR.
        variables (list, opcional): Lista con las cadenas de texto de las columnas de los sensores a evaluar. 
            Por defecto es None (inicializa ["glucose_dexcom", "glucose_libre"]).
        guardar_csv (bool, opcional): Criterio booleano para habilitar o deshabilitar la exportación física 
            de subseries tabulares individuales por componente. Por defecto es True.

    Returns:
        pd.DataFrame: Matriz de datos resumen con las varianzas muestrales estimadas de cada componente 
            para la totalidad de la muestra procesada de participantes.

    Recomendaciones:
        - Asegurar que constantes de entorno como 'TRUSTED_DIR', 'STL_DIR' y el diccionario 'VARS_POSTPRANDIAL' 
          estén previamente declaradas en el Notebook.
        - Verificar la disponibilidad previa de los ayudantes de submuestreo, descomposición por tramos 
          y grabado físico para prevenir detenciones en mitad de la ejecución por lotes.
    """
    # === 1. Inicialización y Validación de Argumentos Base ===
    if variables is None:
        variables = ["glucose_dexcom", "glucose_libre"]

    resumen = []

    # === 2. Bucle Iterativo Secuencial por Archivo de Participante ===
    for archivo in sorted(os.listdir(carpeta_archivos)):
        if not archivo.endswith(".csv"):
            continue

        ruta_archivo = os.path.join(carpeta_archivos, archivo)

        try:
            # Lectura y forzado de ordenación cronológica estricta en base al índice temporal
            df = pd.read_csv(ruta_archivo, parse_dates=["time"])
            if "time" not in df.columns:
                print(f"[WARN] {archivo} no tiene columna time.")
                continue

            df = df.set_index("time").sort_index()
            nombre_base = os.path.splitext(archivo)

            # Inicialización de acumuladores locales para almacenamiento tabular por componente
            componentes = {
                "tendencia": [],
                "estacionalidad": [],
                "residuo": [],
            }

            # === 3. Procesamiento Interno de Descomposición por Sensor ===
            for variable in variables:
                if variable not in df.columns:
                    continue

                # Submuestreo y extracción adaptativa de parámetros de Loess para el sensor activo
                serie, periodo, seasonal, trend = _submuestrear_serie_stl(df[variable], variable)

                # Validación dimensional del volumen mínimo de registros no vacíos
                if serie.notna().sum() < periodo * 2:
                    print(f"[WARN] {nombre_base} / {variable}: datos insuficientes para STL.")
                    continue

                # Ajuste del estimador por tramos continuos sin interpolación artificial
                result = _descomponer_stl(
                    serie,
                    periodo=periodo,
                    seasonal=seasonal,
                    trend=trend,
                    robust=True,
                )
                if result is None:
                    print(f"[WARN] {nombre_base} / {variable}: no se pudo ajustar STL.")
                    continue

                # Invocación del generador de paneles multi-ejes estáticos
                _graficar_descomposicion_stl(result, nombre_base, variable)

                # Inyección condicional de sub-DataFrames formateados en los acumuladores locales
                if guardar_csv:
                    componentes["tendencia"].append(
                        _crear_df_componente(serie, variable, result.trend, "tendencia")
                    )
                    componentes["estacionalidad"].append(
                        _crear_df_componente(serie, variable, result.seasonal, "estacionalidad")
                    )
                    componentes["residuo"].append(
                        _crear_df_componente(serie, variable, result.resid, "residuo")
                    )

                # Registro de estadísticas descriptivas y varianzas de componentes para la tabla maestra
                resumen.append(
                    {
                        "archivo": archivo,
                        "variable": variable,
                        "n_obs": len(serie),
                        "periodo": periodo,
                        "seasonal": seasonal,
                        "trend": trend,
                        "var_observada": float(np.nanvar(result.observed)),
                        "var_tendencia": float(np.nanvar(result.trend)),
                        "var_estacional": float(np.nanvar(result.seasonal)),
                        "var_residuo": float(np.nanvar(result.resid)),
                    }
                )

            # === 4. Persistencia Tabular por Sujeto ===
            if guardar_csv:
                _guardar_componentes_stl(nombre_base, componentes)

        except Exception as e:
            # Captura de excepciones locales para blindar el bucle iterativo general
            print(f"Error procesando STL en {archivo}: {e}")

    # === 5. Consolidación de Métricas Maetras de Salida ===
    df_resumen = pd.DataFrame(resumen)
    if not df_resumen.empty:
        print("[OK] STL completado para todos los sujetos.")
    else:
        print("[WARN] No se generaron descomposiciones STL.")

    return df_resumen


In [ ]:
def main():
    return procesar_descomposicion_stl(
        carpeta_archivos=TRUSTED_DIR,
        variables=["glucose_dexcom", "glucose_libre"],
    )


if __name__ == "__main__":
    main()

###### C2.1.4.1 Análisis de los residuos: estacionariedad

In [ ]:
def _obtener_paso_muestreo(variable):
    """
    Devuelve el intervalo nominal en minutos característico según el dispositivo sensor de glucosa.

    Args:
        variable (str): Selector del sensor bajo análisis ('glucose_dexcom' o 'glucose_libre').

    Returns:
        int: Cantidad de minutos correspondiente al paso de muestreo nativo del dispositivo.
    """
    if variable == 'glucose_dexcom':
        return 5
    if variable == 'glucose_libre':
        return 15
    return 1


def _preparar_serie_estacionariedad_stl(df_grupo, variable):
    """
    Prepara e indexa de forma regular la serie temporal del residuo STL para los contrastes estadísticos.

    Valida la presencia de la columna de residuos, normaliza el eje temporal forzando una estructura 
    de tipo DatetimeIndex ordenada, aísla las mediciones numéricas nativas descartando ceros y nulos, 
    y reindexa la serie sobre una rejilla regular equidistante según la frecuencia teórica del sensor.

    Args:
        df_grupo (pd.DataFrame): DataFrame filtrado por grupo que contiene la serie analítica.
        variable (str): Nombre selector de la variable o dispositivo bajo estudio.

    Returns:
        pd.Series: Serie numérica submuestreada, indexada regularmente y limpia de nulos, lista 
            para la ejecución de pruebas estadísticas de raíces unitarias.

    Raises:
        ValueError: Si la estructura de entrada carece tanto de una columna 'time' como de un DatetimeIndex.

    Recomendaciones:
        - Asegurar que la columna 'residuo' provenga de una descomposición STL previa válida para evitar 
          distorsiones o series vacías en el retorno.
        - Verificar que los tipos de datos sean convertibles a formato numérico de forma coercitiva antes 
          de invocar el módulo.
    """
    # === 1. Validación Estructural e Ingesta de Datos ===
    # Verificación de que las columnas necesarias existen en este fragmento
    if 'residuo' not in df_grupo.columns:
        return pd.Series(dtype=float)

    # === 2. Normalización del Eje Temporal ===
    if 'time' in df_grupo.columns:
        data = df_grupo.copy()
        data['time'] = pd.to_datetime(data['time'], errors='coerce')
        data = data.dropna(subset=['time']).sort_values('time').set_index('time')
    elif isinstance(df_grupo.index, pd.DatetimeIndex):
        data = df_grupo.sort_index().copy()
    else:
        raise ValueError("El DataFrame debe tener una columna 'time' o un DatetimeIndex.")

    # === 3. Depuración Nutricional y Aislamiento de Residuos ===
    # Trabajamos sobre la columna 'residuo' en lugar del nombre de la variable
    serie = pd.to_numeric(data['residuo'], errors='coerce')
    serie = serie[(serie.notna()) & (serie != 0)].copy()
    if serie.empty:
        return pd.Series(dtype=float)

    # === 4. Submuestreo y Reindexación Regular ===
    paso_minutos = _obtener_paso_muestreo(variable)
    inicio = serie.index[0]
    fin = serie.index[-1]
    rejilla = pd.date_range(start=inicio, end=fin, freq=f"{paso_minutos}min")

    serie_submuestreada = serie.reindex(rejilla)
    serie_submuestreada = serie_submuestreada.dropna()
    return serie_submuestreada


def adf_test(series):
    """
    Realiza el contraste estadístico formal de Dickey-Fuller Aumentado (ADF) para evaluar raíces unitarias.

    Depura la serie abstrayendo ceros y nulos, valida que se cuente con un volumen muestral mínimo 
    operativo (al menos 3 observaciones) y ejecuta la prueba formal devolviendo un diccionario estructurado 
    con los estadísticos, p-valores, valores críticos y el criterio de estacionariedad (p < 0.05).

    Args:
        series (pd.Series o array-like): Vector numérico continuo de la serie temporal que se va a evaluar.

    Returns:
        dict: Diccionario estructurado con las métricas cuantitativas del contraste estadístico 
            ('ADF Statistic', 'p-value', 'Critical Values', 'Stationary') y notas aclaratorias si los datos son insuficientes.

    Recomendaciones:
        - Asegurar la importación previa de 'adfuller' desde statsmodels.tsa.stattools en el entorno.
        - Utilizar esta función preferiblemente sobre series estacionarizadas o residuos de descomposición 
          para contrastar la eliminación efectiva de componentes deterministas.
    """
    # === 1. Depuración Numérica de la Serie ===
    series = pd.to_numeric(series, errors='coerce')
    series = series[(series != 0) & (~series.isna())]
    
    # === 2. Validación de Umbral Muestral Mínimo ===
    if len(series) < 3:
        return {
            'ADF Statistic': None, 'p-value': None, 'Critical Values': None,
            'Stationary': None, 'Observaciones': 'No hay suficientes datos válidos para el test'
        }

    # === 3. Ejecución del Contraste de Raíz Unitaria ===
    result = adfuller(series)
    return {
        'ADF Statistic': result[0],
        'p-value': result[1],
        'Critical Values': result[4],
        'Stationary': result[1] < 0.05
    }


def kpss_test(series):
    """
    Realiza el contraste estadístico formal de Kwiatkowski-Phillips-Schmidt-Shin (KPSS) para estacionariedad.

    Depura numéricamente el vector abstrayendo ceros y nulos, valida que se cumpla con el volumen 
    muestral mínimo operativo (al menos 3 observaciones) y ejecuta la prueba formal bajo la hipótesis 
    nula de estacionariedad en torno a una constante, devolviendo los estadísticos y p-valores estructurados.

    Args:
        series (pd.Series o array-like): Vector numérico continuo de la serie temporal que se va a evaluar.

    Returns:
        dict: Diccionario estructurado con las métricas cuantitativas del contraste estadístico 
            ('KPSS Statistic', 'p-value', 'Critical Values', 'Stationary') y notas aclaratorias si los datos son insuficientes.

    Recomendaciones:
        - Asegurar la importación previa de 'kpss' desde statsmodels.tsa.stattools en el entorno.
        - Recordar que en el test KPSS, un p-valor mayor a 0.05 (p > 0.05) significa que NO se rechaza 
          la hipótesis nula, lo que estadísticamente indica que la serie es estacionaria.
    """
    # === 1. Depuración Numérica de la Serie ===
    series = pd.to_numeric(series, errors='coerce')
    series = series[(series != 0) & (~series.isna())]
    
    # === 2. Validación de Umbral Muestral Mínimo ===
    if len(series) < 3:
        return {
            'KPSS Statistic': None, 'p-value': None, 'Critical Values': None,
            'Stationary': None, 'Observaciones': 'No hay suficientes datos válidos para el test'
        }

    # === 3. Ejecución del Contraste de Estacionariedad ===
    result = kpss(series, regression='c', nlags='auto')
    return {
        'KPSS Statistic': result[0],
        'p-value': result[1],
        'Critical Values': result[3],
        'Stationary': result[1] > 0.05
    }


def analyze_files_stl(folder_path, variables=('glucose_dexcom', 'glucose_libre')):
    """
    Orquesta el análisis por lotes evaluando la estacionariedad de los residuos STL de la cohorte.

    Escanea el directorio, lee los ficheros CSV correspondientes a las componentes y valida su estructura. 
    Agrupa los registros internamente por tipo de dispositivo sensor e invoca de forma secuencial 
    los contrastes formales de raíces unitarias (ADF) y estacionariedad (KPSS).

    Args:
        folder_path (str): Ruta de la carpeta del sistema donde se alojan los ficheros de residuos STL.
        variables (tuple, opcional): Tupla con los nombres selectores de los sensores a analizar. 
            Por defecto es ('glucose_dexcom', 'glucose_libre').

    Returns:
        dict: Estructura anidada de diccionarios que centraliza por archivo y sensor los resultados 
            estadísticos integrados de los tests ADF y KPSS.

    Recomendaciones:
        - Verificar la disponibilidad previa de las funciones de preparación de series y contrastes 
          ('_preparar_serie_estacionariedad_stl', 'adf_test', 'kpss_test') para mitigar excepciones en el bucle.
    """
    results = {}

    # === 1. Iteración Secuencial de Archivos en el Directorio ===
    for file in os.listdir(folder_path):
        if not file.endswith('.csv'):
            continue

        ruta = os.path.join(folder_path, file)
        try:
            df = pd.read_csv(ruta)
        except Exception as e:
            print(f"No se pudo leer {file}: {e}")
            continue

        # Validación estructural de las columnas requeridas provenientes del volcado de componentes
        if 'variable' not in df.columns or 'residuo' not in df.columns:
            print(f"El archivo {file} no contiene las columnas 'variable' o 'residuo'.")
            continue

        file_results = {}
        
        # === 2. Procesamiento Interno Grupal Segregado por Sensor ===
        # Agrupamos por la columna 'variable' (glucose_dexcom y glucose_libre)
        for variable, grupo in df.groupby('variable'):
            if variable not in variables:
                continue

            try:
                # Submuestreo adaptativo y ejecución de los contrastes formales
                serie = _preparar_serie_estacionariedad_stl(grupo, variable)
                file_results[variable] = {
                    'ADF': adf_test(serie),
                    'KPSS': kpss_test(serie)
                }
            except Exception as e:
                # Captura y estructuración local del error para no interrumpir el mapeo
                error_dict = {
                    'Statistic': None, 'p-value': None, 'Critical Values': None,
                    'Stationary': None, 'Observaciones': f'Error: {e}'
                }
                file_results[variable] = {
                    'ADF': error_dict.copy(),
                    'KPSS': error_dict.copy()
                }

        # Almacenamiento indexado en el contenedor global si el archivo arrojó métricas
        if file_results:
            results[file] = file_results

    return results


def create_results_table(results):
    """
    Transforma el diccionario estructurado de resultados en un DataFrame plano normalizado.

    Args:
        results (dict): Diccionario anidado que centraliza las métricas estadísticas de los archivos.

    Returns:
        pd.DataFrame: Matriz tabular ordenada con las columnas de identificación y los estadísticos 
            y p-valores de las pruebas ADF y KPSS.
    """
    data = []
    # === 1. Desenvolvimiento Iterativo de las Estructuras Anidadas ===
    for file, result in results.items():
        # Formateo estético del identificador del participante removiendo extensiones
        file_name = file.replace('_', ' ').replace('.csv', '')
        for variable, tests in result.items():
            adf_result = tests['ADF']
            kpss_result = tests['KPSS']
            
            # Construcción estructurada del registro fila plano
            data.append({
                'Archivo': file_name,
                'Variable': variable,
                'ADF Statistic': adf_result.get('ADF Statistic'),
                'ADF p-value': adf_result.get('p-value'),
                'ADF Estacionaria': adf_result.get('Stationary'),
                'KPSS Statistic': kpss_result.get('KPSS Statistic'),
                'KPSS p-value': kpss_result.get('p-value'),
                'KPSS Estacionaria': kpss_result.get('Stationary')
            })
    return pd.DataFrame(data)


def format_value(value):
    """
    Formatea de manera estricta cadenas, booleanos y números de alta precisión decimal para su reporte gráfico.

    Args:
        value (any): Objeto o variable escalar que se desea formatear.

    Returns:
        str: Cadena de texto normalizada (ej. "Sí"/"No" para booleanos o precisión a 10 decimales para floats).
    """
    if isinstance(value, bool): return "Sí" if value else "No"
    if isinstance(value, float): return f"{value:.10f}"
    return value


def plot_results_as_image(results_df):
    """
    Renderiza la matriz de datos de contrastes formalizada en una tabla gráfica estática y exporta el PNG.

    Calcula dimensiones dinámicas del lienzo en función del volumen de filas del DataFrame, 
    abstrae los ejes de coordenadas y construye una tabla de alta resolución con rejillas finas, 
    guardando el resultado final en el subdirectorio de residuos.

    Args:
        results_df (pd.DataFrame): DataFrame plano con las métricas consolidadas de la cohorte.

    Returns:
        None: La función graba la figura directamente en el almacenamiento del sistema y la proyecta.

    Recomendaciones:
        - Asegurar que la constante de ruta 'RESIDUOS_ESTACIONARIEDAD_DIR' se encuentre previamente 
          declarada en el Notebook para mitigar fallos del sistema operativo.
    """
    # === 1. Validación de Ingesta y Configuración del Lienzo ===
    if results_df.empty:
        print("No hay resultados para mostrar."); return
    os.makedirs(RESIDUOS_ESTACIONARIEDAD_DIR, exist_ok=True)
    output_path = os.path.join(RESIDUOS_ESTACIONARIEDAD_DIR, 'resultados_adf_kpss.png')
    
    # Estimación de dimensiones proporcionales para evitar colisiones de texto por filas masivas
    n_rows = len(results_df)
    fig_h = max(2.2, 0.45 + n_rows * 0.32)
    fig, ax = plt.subplots(figsize=(12, fig_h))
    ax.axis('off')
    
    # === 2. Construcción de la Tabla de Matplotlib ===
    table = ax.table(
        cellText=[[format_value(cell) for cell in row] for row in results_df.values],
        colLabels=results_df.columns, cellLoc='center', loc='center', bbox=[0.01, 0.02, 0.98, 0.88]
    )
    
    # Estandarización tipográfica interna del objeto Tabla
    table.auto_set_font_size(False); table.set_fontsize(8.5); table.scale(1, 0.92)
    
    # === 3. Ajustes Estéticos de Celdas y Bordes Finos ===
    for (i, j), cell in table.get_celld().items():
        if i == 0: cell.set_text_props(weight='bold')
        cell.set_edgecolor('black'); cell.set_facecolor('white')
        cell.set_text_props(ha='center', va='center'); cell.set_linewidth(0.4)
        
    # Anotaciones de título y márgenes geométricos estrechos
    fig.text(0.5, 0.965, 'Resultados de Pruebas ADF y KPSS (Residuos STL)', ha='center', va='top', fontsize=12)
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
    
    # === 4. Grabado Físico en Disco y Despliegue ===
    fig.savefig(output_path, dpi=300, bbox_inches='tight', pad_inches=0.02)
    plt.show()
    print(f"Imagen guardada en: {output_path}")


In [ ]:
def main():
    folder_path = RESIDUOS_DIR
    
    # CAMBIO CLAVE: Usamos la nueva función adaptada para STL.
    # Pasamos la tupla de los valores que toma la columna 'variable' ('glucose_dexcom', 'glucose_libre')
    results = analyze_files_stl(folder_path, variables=('glucose_dexcom', 'glucose_libre'))
    
    # Muestra un breve resumen por archivo en stdout
    for file, result in results.items():
        print(f'Resultados para {file}:')
        for variable, tests in result.items():
            print(f'  {variable}:')
            print(f'    ADF: {tests["ADF"]}')
            print(f'    KPSS: {tests["KPSS"]}')
            
    # Construye la tabla plana y la guarda como CSV
    results_table = create_results_table(results)
    os.makedirs(RESIDUOS_ESTACIONARIEDAD_DIR, exist_ok=True)
    csv_path = os.path.join(RESIDUOS_ESTACIONARIEDAD_DIR, 'resumen_adf_kpss.csv')
    results_table.to_csv(csv_path, index=False)
    print(f'CSV guardado en: {csv_path}')
    
    # Genera la imagen compacta con la función existente
    try:
        plot_results_as_image(results_table)
    except Exception as e:
        print(f'Error generando la imagen: {e}')

if __name__ == '__main__':
    main()


###### C2.1.4.2 Análisis de los residuos: estacionalidad ACF/PACF

In [ ]:
def analizar_autocorrelacion_stl(df_grupo, variable, archivo, lags=1440, out_dir=RESIDUOS_ACF_DIR):
    """
    Calcula y grafica la función de autocorrelación (ACF) aplicada sobre los residuos de la descomposición STL.

    Valida la presencia obligatoria de la columna de residuos, fuerza la normalización del eje temporal 
    mediante un índice cronológico de tipo datetime, realiza un submuestreo regular adaptativo y genera 
    un gráfico ACF acotado y ajustado por la frecuencia nativa del sensor correspondiente.

    Args:
        df_grupo (pd.DataFrame): DataFrame filtrado que contiene las componentes de la descomposición del sensor.
        variable (str): Selector del sensor o dispositivo bajo análisis ('glucose_dexcom' o 'glucose_libre').
        archivo (str): Nombre o ruta del archivo de origen utilizado para extraer metadatos de título.
        lags (int, opcional): Horizonte máximo teórico de retardos en minutos. Por defecto es 1440.
        out_dir (str, opcional): Ruta física del sistema para almacenar la imagen. Por defecto apunta 
            a la constante RESIDUOS_ACF_DIR.

    Returns:
        None: La función no retorna ningún objeto, exporta el gráfico (.png) directamente al almacenamiento local.

    Recomendaciones:
        - Asegurar que las constantes globales de ruta como 'RESIDUOS_ACF_DIR' y los ayudantes de eje y desfase 
          ('_submuestrear_serie_para_acf_pacf', '_calcular_lags_ajustados', '_configurar_eje_lags') se encuentren 
          definidos previamente en el Notebook.
    """
    # === 1. Aislamiento del Nombre Base y Configuración del Título ===
    nombre_base = os.path.splitext(os.path.basename(archivo))[0]
    titulo = f"Autocorrelación Residuos STL - {variable} {nombre_base}"

    # === 2. Validación de la Presencia de Componentes ===
    # CAMBIO CLAVE: Comprobamos que la columna residuo exista en el grupo filtrado
    if 'residuo' not in df_grupo.columns:
        print(f"La columna 'residuo' no existe en el DataFrame.")
        return

    # === 3. Normalización y Aseguramiento del Índice Temporal ===
    if 'time' in df_grupo.columns:
        df_grupo = df_grupo.copy()
        df_grupo['time'] = pd.to_datetime(df_grupo['time'], errors='coerce')
        df_grupo = df_grupo.dropna(subset=['time']).sort_values('time').set_index('time')

    # === 4. Submuestreo Regular y Ajuste de Retardos (Lags) ===
    # Pasamos la columna 'residuo' en lugar de la variable ausente en las columnas
    serie, paso_minutos = _submuestrear_serie_para_acf_pacf(df_grupo['residuo'], variable)
    lags_ajustados = _calcular_lags_ajustados(lags, paso_minutos, len(serie))

    # === 5. Construcción y Configuración del Gráfico ACF ===
    if len(serie) > 0 and lags_ajustados > 0:
        os.makedirs(out_dir, exist_ok=True)
        fig, ax = plt.subplots(figsize=(10, 6))
        
        # Renderizado de la curva ACF estadística
        plot_acf(serie, lags=lags_ajustados, ax=ax)
        ax.set_title(titulo)
        
        # Formateo y muestreo del eje horizontal basado en intervalos de tiempo reales
        _configurar_eje_lags(ax, lags_ajustados, paso_minutos, 'Autocorrelación')
        plt.tight_layout()
        
        # === 6. Almacenamiento Físico y Liberación de Memoria ===
        filename = os.path.join(out_dir, f"{nombre_base}_autocorrelacion_{variable}.png")
        fig.savefig(filename, dpi=300)
        mostrar_y_liberar_figura(fig)
        print(f"[OK] Guardado: {filename}")
    else:
        print(f"No hay suficientes datos válidos para el análisis de autocorrelación de {variable}.")


def analizar_autocorrelacion_parcial_stl(df_grupo, variable, archivo, lags=120, out_dir=RESIDUOS_PACF_DIR):
    """
    Calcula y grafica la función de autocorrelación parcial (PACF) aplicada sobre los residuos de la descomposición STL.

    Valida la presencia obligatoria de la columna de residuos, fuerza la normalización del eje temporal 
    mediante un índice cronológico de tipo datetime, realiza un submuestreo regular adaptativo y genera 
    un gráfico PACF acotado para capturar dependencias directas locales ajustado a la frecuencia del sensor.

    Args:
        df_grupo (pd.DataFrame): DataFrame filtrado que contiene las componentes de la descomposición del sensor.
        variable (str): Selector del sensor o dispositivo bajo análisis ('glucose_dexcom' o 'glucose_libre').
        archivo (str): Nombre o ruta del archivo de origen utilizado para extraer metadatos de título.
        lags (int, opcional): Horizonte máximo de retardos locales en minutos. Por defecto es 120.
        out_dir (str, opcional): Ruta física del sistema para almacenar la imagen. Por defecto apunta 
            a la constante RESIDUOS_PACF_DIR.

    Returns:
        None: La función no retorna ningún objeto, exporta la gráfica (.png) directamente al disco.

    Recomendaciones:
        - Asegurar que las constantes globales de ruta como 'RESIDUOS_PACF_DIR' y los ayudantes de eje 
          ('_submuestrear_serie_para_acf_pacf', '_calcular_lags_ajustados', '_configurar_eje_lags') se encuentren 
          definidos previamente en el Notebook.
    """
    # === 1. Aislamiento del Nombre Base y Configuración del Título ===
    nombre_base = os.path.splitext(os.path.basename(archivo))[0]
    titulo = f"Autocorrelación Parcial Residuos STL - {variable} {nombre_base}"

    # === 2. Validación de la Presencia de Componentes ===
    # CAMBIO CLAVE: Comprobamos que la columna residuo exista en el grupo filtrado
    if 'residuo' not in df_grupo.columns:
        print(f"La columna 'residuo' no existe en el DataFrame.")
        return

    # === 3. Normalización y Aseguramiento del Índice Temporal ===
    if 'time' in df_grupo.columns:
        df_grupo = df_grupo.copy()
        df_grupo['time'] = pd.to_datetime(df_grupo['time'], errors='coerce')
        df_grupo = df_grupo.dropna(subset=['time']).sort_values('time').set_index('time')

    # === 4. Submuestreo Regular y Ajuste de Retardos Locales (Lags) ===
    # Pasamos la columna 'residuo' en lugar de la variable ausente en las columnas
    serie, paso_minutos = _submuestrear_serie_para_acf_pacf(df_grupo['residuo'], variable)
    lags_ajustados = _calcular_lags_ajustados(lags, paso_minutos, len(serie))

    # === 5. Construcción y Configuración del Gráfico PACF ===
    if len(serie) > 0 and lags_ajustados > 0:
        os.makedirs(out_dir, exist_ok=True)
        fig, ax = plt.subplots(figsize=(10, 6))
        
        # Renderizado de la curva PACF estadística
        plot_pacf(serie, lags=lags_ajustados, ax=ax)
        ax.set_title(titulo)
        
        # Formateo y muestreo del eje horizontal basado en intervalos de tiempo reales
        _configurar_eje_lags(ax, lags_ajustados, paso_minutos, 'Autocorrelación Parcial')
        plt.tight_layout()
        
        # === 6. Almacenamiento Físico y Liberación de Memoria ===
        filename = os.path.join(out_dir, f"{nombre_base}_autocorrelacion_parcial_{variable}.png")
        fig.savefig(filename, dpi=300)
        mostrar_y_liberar_figura(fig)
        print(f"[OK] Guardado: {filename}")
    else:
        print(f"No hay suficientes datos válidos para el análisis de autocorrelación parcial de {variable}.")


def procesar_archivos_stl(carpeta_archivos, lags=1440, variables=('glucose_dexcom', 'glucose_libre')):
    """
    Orquesta la ejecución por lotes aplicando análisis de autocorrelación regular y parcial sobre residuos STL.

    Escanea de forma secuencial el directorio de salida de componentes, valida la estructura tabular 
    requerida y agrupa los registros internamente por tipo de dispositivo sensor (Dexcom/Libre) para invocar 
    los módulos independientes de cálculo e informe gráfico.

    Args:
        carpeta_archivos (str): Ruta de la carpeta del sistema donde se alojan los ficheros de residuos STL.
        lags (int, opcional): Horizonte máximo de retardos para el análisis regular global. Por defecto es 1440.
        variables (tuple, opcional): Tupla con los nombres selectores de los sensores a procesar de forma aislada.

    Returns:
        None: La función controla el flujo iterativo y la gestión de capturas preventivas de errores.

    Recomendaciones:
        - Verificar la disponibilidad previa de los scripts 'analizar_autocorrelacion_stl' y 
          'analizar_autocorrelacion_parcial_stl' para mitigar fallos en mitad de la ejecución masiva.
    """
    # === 1. Iteración Secuencial de Archivos en el Directorio ===
    for archivo in os.listdir(carpeta_archivos):
        if archivo.endswith('.csv'):
            ruta_archivo = os.path.join(carpeta_archivos, archivo)
            try:
                df = pd.read_csv(ruta_archivo)
                print(f"\nProcesando {archivo}...")

                # CAMBIO CLAVE: Validación de la estructura del STL
                if 'variable' not in df.columns or 'residuo' not in df.columns:
                    print(f"El archivo {archivo} no contiene las columnas 'variable' o 'residuo'.")
                    continue

                # === 2. Procesamiento Grupal Segregado por Sensor ===
                # Agrupamos por los valores internos de la columna 'variable'
                for variable, grupo in df.groupby('variable'):
                    if variable not in variables:
                        continue
                    
                    # Llamamos a las nuevas funciones pasando el fragmento filtrado (grupo)
                    analizar_autocorrelacion_stl(grupo, variable, archivo, lags=lags)
                    analizar_autocorrelacion_parcial_stl(grupo, variable, archivo, lags=120)

            except Exception as e:
                # Captura preventiva para asegurar la continuidad del pipeline si falla un archivo específico
                print(f"Error procesando el archivo {archivo}: {e}")


In [ ]:
def main():
    carpeta_archivos = RESIDUOS_DIR
    procesar_archivos_stl(carpeta_archivos)

if __name__ == "__main__":
    main()

#### C2.2 Estacionariedad

En datos de glucosa es frecuente que la estacionariedad no se cumpla en la serie original, porque la media puede variar según la hora del día o por episodios de comida o ejercicio.

Por eso suele ser útil estudiar primero la serie original, y luego estudiar también una posible transformación por diferenciación, eliminación de tendencias, análisis por ventanas temporales más pequeñas o estudiar la serie desestacionalizada si procede.
En glucosa intersticial es infrecuente requerir una diferenciación d>1.

##### C2.2.1 ADF y KPSS

**Estacionariedad** 

Una serie es estacionaria si sus propiedades estadísticas (media, varianza, autocovarianza) no cambian a lo largo del tiempo, es decir, se mantienen aproximadamente constantes, o al menos dependensólo del desfase temporal y no del instante concreto. Esto es importante porque muchas técnicas de análisis de series temporales asumen que los datos son estacionarios.

---

**Prueba ADF (Augmented Dickey-Fuller)**

Objetivo: Determinar si una serie temporal tiene una raíz unitaria, lo que implica que no es estacionaria.
La idea es comprobar si los valores presentes dependen demasiado del pasado como para que la serie "vuelva" a una medida estable.

Hipótesis:
  - Hipótesis nula (H0): La serie tiene una raíz unitaria (no es estacionaria).
  - Hipótesis alternativa (H1): La serie no tiene una raíz unitaria (es estacionaria).

Interpretación: Si el p-valor es bajo (por ejemplo, menos de 0.05), se rechaza la hipótesis nula, sugiriendo que la serie es estacionaria.

---

**Prueba KPSS (Kwiatkowski-Phillips-Schmidt-Shin)**

Objetivo: Evaluar si una serie temporal es estacionaria en torno a una media o una tendencia determinista.
La descomposición conceptual es que la serie se compone de una tendencia determinista, estacionariedad y ruido. KPSS analiza si la parte estacionaria realmente se comporta como tal. Si las desviaciones acumuladas son demasiado grandes, concluye que la serie no es estacionaria, pues sugiere que no está estabilizada.

Hipótesis:
  - Hipótesis nula (H0): La serie es estacionaria.
  - Hipótesis alternativa (H1): La serie no es estacionaria.

Interpretación: Si el p-valor es bajo (por ejemplo, menos de 0.05), se rechaza la hipótesis nula, indicando que la serie no es estacionaria. Esta prueba es más sensible a la presencia de tendencias y estacionalidad.

---

**Cómo interpretar los resultados**
- Si ADF no rechaza H0 y/o KPSS rechaza H0: evidencia de no estacionariedad. Corresponde aplicar diferenciación (∇^d),u otras técnicas, para lograr estacionariedad en media antes de ajustar posibles modelos ARMA; o aplicar modelos ARIMA(p,d,q) con d mínimo que estabilice la serie.
- Si ADF rechaza y KPSS no rechaza: evidencia de estacionariedad.
- En casos ambiguos, se decide observando ACF/PACF de la serie (y, si es necesario, de la diferenciada) y la estabilidad de la media.
  - Ninguno rechaza H0: la evidencia es débil, caso ambiguo pues la serie puede estar cerca del límite o requerir más tratamiento.
  - Ambas rechazan H0: la serie puede tener una estructura compleja, rupturas de estructura, cambios de régimen o unaserie mas espefcifiada para el test. 

---

https://www.statsmodels.org/dev/examples/notebooks/generated/stationarity_detrending_adf_kpss.html

In [ ]:
def _obtener_paso_muestreo(variable):
    """
    Devuelve el intervalo nominal en minutos característico según el dispositivo sensor de glucosa.

    Args:
        variable (str): Selector del sensor bajo análisis ('glucose_dexcom' o 'glucose_libre').

    Returns:
        int: Cantidad de minutos correspondiente al paso de muestreo nativo del dispositivo.
    """
    if variable == 'glucose_dexcom':
        return 5
    if variable == 'glucose_libre':
        return 15
    return 1


def _preparar_serie_estacionariedad(df, variable):
    """
    Prepara e indexa de forma regular una serie temporal tomando puntos reales anclados al inicio.

    Valida la presencia de la columna del sensor, normaliza el eje temporal forzando una estructura 
    de tipo DatetimeIndex ordenada, aísla las mediciones numéricas nativas descartando ceros y nulos, 
    y construye una rejilla regular equidistante acotada a la frecuencia teórica del dispositivo 
    tomando como origen el primer timestamp válido de la muestra.

    Args:
        df (pd.DataFrame): DataFrame que contiene el histórico de registros temporales continuos.
        variable (str): Nombre selector de la columna del sensor bajo análisis ('glucose_dexcom' o 'glucose_libre').

    Returns:
        pd.Series: Serie numérica submuestreada, indexada regularmente y limpia de nulos, lista 
            para la ejecución de pruebas estadísticas de raíces unitarias.

    Raises:
        ValueError: Si la estructura de entrada carece tanto de una columna 'time' como de un DatetimeIndex.

    Recomendaciones:
        - Utilizar este método de preparación previo a los tests ADF/KPSS para asegurar que la frecuencia 
          temporal de la serie sea homogénea y evitar distorsiones por interpolaciones artificiales.
    """
    # === 1. Validación Estructural e Ingesta de Datos ===
    if variable not in df.columns:
        return pd.Series(dtype=float)

    # === 2. Normalización del Eje Temporal ===
    if 'time' in df.columns:
        data = df.copy()
        data['time'] = pd.to_datetime(data['time'], errors='coerce')
        data = data.dropna(subset=['time']).sort_values('time').set_index('time')
    elif isinstance(df.index, pd.DatetimeIndex):
        data = df.sort_index().copy()
    else:
        raise ValueError("El DataFrame debe tener una columna 'time' o un DatetimeIndex.")

    # === 3. Depuración Numérica y Aislamiento de Mediciones ===
    serie = pd.to_numeric(data[variable], errors='coerce')
    serie = serie[(serie.notna()) & (serie != 0)].copy()
    if serie.empty:
        return pd.Series(dtype=float)

    # === 4. Submuestreo y Reindexación Regular Anclada ===
    paso_minutos = _obtener_paso_muestreo(variable)
    inicio = serie.index[0]
    fin = serie.index[-1]
    rejilla = pd.date_range(start=inicio, end=fin, freq=f"{paso_minutos}min")

    serie_submuestreada = serie.reindex(rejilla)
    serie_submuestreada = serie_submuestreada.dropna()
    return serie_submuestreada


def adf_test(series):
    """
    Realiza el contraste estadístico formal de Dickey-Fuller Aumentado (ADF) para evaluar raíces unitarias.

    Depura la serie abstrayendo ceros y nulos, valida que se cuente con un volumen muestral mínimo 
    operativo (al menos 3 observaciones) y ejecuta la prueba formal devolviendo un diccionario estructurado 
    con los estadísticos, p-valores, valores críticos y el criterio de estacionariedad (p < 0.05).

    Args:
        series (pd.Series o array-like): Vector numérico continuo de la serie temporal que se va a evaluar.

    Returns:
        dict: Diccionario estructurado con las métricas cuantitativas del contraste estadístico 
            ('ADF Statistic', 'p-value', 'Critical Values', 'Stationary') y notas aclaratorias si los datos son insuficientes.

    Recomendaciones:
        - Asegurar la importación previa de 'adfuller' desde statsmodels.tsa.stattools en el entorno.
        - Utilizar esta función preferiblemente sobre series regularizadas o residuos de descomposición 
          para contrastar la eliminación de componentes deterministas.
    """
    # === 1. Depuración Numérica de la Serie ===
    series = pd.to_numeric(series, errors='coerce')
    series = series[(series != 0) & (~series.isna())]
    
    # === 2. Validación de Umbral Muestral Mínimo ===
    if len(series) < 3:
        return {
            'ADF Statistic': None,
            'p-value': None,
            'Critical Values': None,
            'Stationary': None,
            'Observaciones': 'No hay suficientes datos válidos para el test'
        }

    # === 3. Ejecución del Contraste de Raíz Unitaria ===
    result = adfuller(series)
    return {
        'ADF Statistic': result[0],
        'p-value': result[1],
        'Critical Values': result[4],
        'Stationary': result[1] < 0.05
    }


def kpss_test(series):
    """
    Realiza el contraste estadístico formal de Kwiatkowski-Phillips-Schmidt-Shin (KPSS) para estacionariedad.

    Depura numéricamente el vector abstrayendo ceros y nulos, valida que se cumpla con el volumen 
    muestral mínimo operativo (al menos 3 observaciones) y ejecuta la prueba formal bajo la hipótesis 
    nula de estacionariedad en torno a una constante, devolviendo los estadísticos y p-valores estructurados.

    Args:
        series (pd.Series o array-like): Vector numérico continuo de la serie temporal que se va a evaluar.

    Returns:
        dict: Diccionario estructurado con las métricas cuantitativas del contraste estadístico 
            ('KPSS Statistic', 'p-value', 'Critical Values', 'Stationary') y notas aclaratorias si los datos son insuficientes.

    Recomendaciones:
        - Asegurar la importación previa de 'kpss' desde statsmodels.tsa.stattools en el entorno.
        - Recordar que en el test KPSS, un p-valor mayor a 0.05 (p > 0.05) significa que NO se rechaza 
          la hipótesis nula, lo que estadísticamente indica que la serie es estacionaria.
    """
    # === 1. Depuración Numérica de la Serie ===
    series = pd.to_numeric(series, errors='coerce')
    series = series[(series != 0) & (~series.isna())]
    
    # === 2. Validación de Umbral Muestral Mínimo ===
    if len(series) < 3:
        return {
            'KPSS Statistic': None,
            'p-value': None,
            'Critical Values': None,
            'Stationary': None,
            'Observaciones': 'No hay suficientes datos válidos para el test'
        }

    # === 3. Ejecución del Contraste de Estacionariedad ===
    result = kpss(series, regression='c', nlags='auto')
    return {
        'KPSS Statistic': result[0],
        'p-value': result[1],
        'Critical Values': result[3],
        'Stationary': result[1] > 0.05
    }


def analyze_files(folder_path, variables=('glucose_dexcom', 'glucose_libre')):
    """
    Orquesta el análisis por lotes evaluando la estacionariedad de las variables brutas de la cohorte.

    Escanea el directorio, lee los ficheros CSV correspondientes a las series nativas y valida su estructura. 
    Itera de forma cruzada por tipo de dispositivo sensor e invoca de forma secuencial los contrastes 
    formales de raíces unitarias (ADF) y estacionariedad (KPSS).

    Args:
        folder_path (str): Ruta de la carpeta del sistema donde se alojan los ficheros CSV base.
        variables (tuple, opcional): Tupla con los nombres selectores de los sensores a analizar. 
            Por defecto es ('glucose_dexcom', 'glucose_libre').

    Returns:
        dict: Estructura anidada de diccionarios que centraliza por archivo y sensor los resultados 
            estadísticos integrados de los tests ADF y KPSS.

    Recomendaciones:
        - Verificar la disponibilidad previa de las funciones de preparación de series y de contrastes 
          ('_preparar_serie_estacionariedad', 'adf_test', 'kpss_test') para mitigar excepciones en el bucle.
    """
    results = {}

    # === 1. Iteración Secuencial de Archivos en el Directorio ===
    for file in os.listdir(folder_path):
        if not file.endswith('.csv'):
            continue

        ruta = os.path.join(folder_path, file)
        try:
            df = pd.read_csv(ruta)
        except Exception as e:
            print(f"No se pudo leer {file}: {e}")
            continue

        file_results = {}
        
        # === 2. Procesamiento Interno Segregado por Sensor ===
        for variable in variables:
            if variable not in df.columns:
                continue

            try:
                # Submuestreo adaptativo anclado y ejecución simultánea de los contrastes formales
                serie = _preparar_serie_estacionariedad(df, variable)
                file_results[variable] = {
                    'ADF': adf_test(serie),
                    'KPSS': kpss_test(serie)
                }
            except Exception as e:
                # Captura estructural local con claves explícitas ante fallos de preparación
                file_results[variable] = {
                    'ADF': {
                        'ADF Statistic': None,
                        'p-value': None,
                        'Critical Values': None,
                        'Stationary': None,
                        'Observaciones': f'Error preparando la serie: {e}'
                    },
                    'KPSS': {
                        'KPSS Statistic': None,
                        'p-value': None,
                        'Critical Values': None,
                        'Stationary': None,
                        'Observaciones': f'Error preparando la serie: {e}'
                    }
                }

        # Almacenamiento indexado en el contenedor global si el archivo arrojó métricas
        if file_results:
            results[file] = file_results

    return results


def create_results_table(results):
    """
    Transforma el diccionario estructurado de resultados de variables en un DataFrame plano normalizado.

    Args:
        results (dict): Diccionario anidado que centraliza las métricas estadísticas de los archivos.

    Returns:
        pd.DataFrame: Matriz tabular ordenada con las columnas de identificación y los estadísticos 
            y p-valores de las pruebas ADF y KPSS.
    """
    data = []
    
    # === 1. Desenvolvimiento Iterativo de las Estructuras Anidadas ===
    for file, result in results.items():
        # Formateo estético del identificador del participante removiendo extensiones
        file_name = file.replace('_', ' ').replace('.csv', '')
        for variable, tests in result.items():
            adf_result = tests['ADF']
            kpss_result = tests['KPSS']
            
            # Construcción estructurada del registro fila plano usando accesos por corchetes directos
            data.append({
                'Archivo': file_name,
                'Variable': variable,
                'ADF Statistic': adf_result['ADF Statistic'],
                'ADF p-value': adf_result['p-value'],
                'ADF Estacionaria': adf_result['Stationary'],
                'KPSS Statistic': kpss_result['KPSS Statistic'],
                'KPSS p-value': kpss_result['p-value'],
                'KPSS Estacionaria': kpss_result['Stationary']
            })

    results_df = pd.DataFrame(data)
    return results_df


def format_value(value):
    """
    Formatea de manera estricta cadenas, booleanos y números de alta precisión decimal para su reporte gráfico.

    Args:
        value (any): Objeto o variable escalar que se desea formatear.

    Returns:
        str: Cadena de texto normalizada (ej. "Sí"/"No" para booleanos o precisión a 10 decimales para floats).
    """
    if isinstance(value, bool):
        return "Sí" if value else "No"
    if isinstance(value, float):
        return f"{value:.10f}"
    return value


def plot_results_as_image(results_df):
    """
    Renderiza la matriz de datos de contrastes de variables brutas en una tabla gráfica estática y exporta el PNG.

    Calcula dimensiones dinámicas del lienzo en función del volumen de filas del DataFrame, 
    abstrae los ejes de coordenadas y construye una tabla de alta resolución con rejillas finas, 
    guardando el resultado final en el subdirectorio de estacionariedad general.

    Args:
        results_df (pd.DataFrame): DataFrame plano con las métricas consolidadas de la cohorte.

    Returns:
        None: La función graba la figura directamente en el almacenamiento del sistema y la proyecta.

    Recomendaciones:
        - Asegurar que la constante de ruta 'ESTACIONARIEDAD_DIR' se encuentre previamente 
          declarada en el Notebook para mitigar fallos del sistema operativo.
    """
    # === 1. Validación de Ingesta y Configuración del Lienzo ===
    if results_df.empty:
        print("No hay resultados para mostrar.")
        return

    os.makedirs(ESTACIONARIEDAD_DIR, exist_ok=True)
    output_path = os.path.join(ESTACIONARIEDAD_DIR, 'resultados_adf_kpss.png')

    # Reducir el espacio vertical: altura por fila más pequeña y márgenes más ajustados
    n_rows = len(results_df)
    fig_h = max(2.2, 0.45 + n_rows * 0.32)
    fig, ax = plt.subplots(figsize=(12, fig_h))
    ax.axis('off')

    # === 2. Construcción de la Tabla de Matplotlib ===
    table = ax.table(
        cellText=[[format_value(cell) for cell in row] for row in results_df.values],
        colLabels=results_df.columns,
        cellLoc='center',
        loc='center',
        bbox=[0.01, 0.02, 0.98, 0.88]
    )
    
    # Estandarización tipográfica interna del objeto Tabla
    table.auto_set_font_size(False)
    table.set_fontsize(8.5)
    table.scale(1, 0.92)
    
    # === 3. Ajustes Estéticos de Celdas y Bordes Finos ===
    for (i, j), cell in table.get_celld().items():
        if i == 0:
            cell.set_text_props(weight='bold')
        cell.set_edgecolor('black')
        cell.set_facecolor('white')
        cell.set_text_props(ha='center', va='center')
        cell.set_linewidth(0.4)
        
    # Anotaciones de título y márgenes geométricos estrechos
    fig.text(0.5, 0.965, 'Resultados de Pruebas ADF y KPSS', ha='center', va='top', fontsize=12)
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
    
    # === 4. Grabado Físico en Disco y Despliegue ===
    fig.savefig(output_path, dpi=300, bbox_inches='tight', pad_inches=0.02)
    plt.show()
    print(f"Imagen guardada en: {output_path}")


In [ ]:
def main():
    folder_path = TRUSTED_DIR
    # Ejecuta los tests sobre los archivos de la carpeta de confianza
    results = analyze_files(folder_path)
    # Muestra un breve resumen por archivo en stdout
    for file, result in results.items():
        print(f'Resultados para {file}:')
        for variable, tests in result.items():
            print(f'  {variable}:')
            print(f'    ADF: {tests["ADF"]}')
            print(f'    KPSS: {tests["KPSS"]}')
    # Construye la tabla plana y la guarda como CSV
    results_table = create_results_table(results)
    os.makedirs(ESTACIONARIEDAD_DIR, exist_ok=True)
    csv_path = os.path.join(ESTACIONARIEDAD_DIR, 'resumen_adf_kpss.csv')
    results_table.to_csv(csv_path, index=False)
    print(f'CSV guardado en: {csv_path}')
    # Genera la imagen compacta con la función existente
    try:
        plot_results_as_image(results_table)
    except Exception as e:
        print(f'Error generando la imagen: {e}')

if __name__ == '__main__':
    main()

In [ ]:
plt.close('all')

## D. MODELIZACIÓN

En esta parte se construyen modelos predictivos y se evalúa su capacidad de generalización sobre los datos procesados.


In [ ]:
def cargar_y_unificar_dataset(
    ruta_carpeta_csv: str, 
    mapeo_diabetes: Dict[str, str]
) -> pd.DataFrame:
    """
    Lee, etiqueta y concatena los archivos CSV individuales de cada sujeto.

    Carga de forma secuencial todos los archivos CSV presentes en un directorio,
    extrae el identificador del usuario a partir del nombre del archivo, elimina
    los sufijos de subseries temporales (ej: '_serie1'), aplica el mapeo estático 
    de su grupo metabólico (condición de diabetes) y unifica toda la información 
    en un único DataFrame estructurado cronológicamente.

    Args:
        ruta_carpeta_csv (str): Ruta absoluta o relativa al directorio que 
            contiene los archivos CSV individuales (ej: 'datos_sujetos/').
        mapeo_diabetes (Dict[str, str]): Diccionario de control que vincula 
            el ID del sujeto con su estado clínico (ej: 'No Diabético').

    Returns:
        pd.DataFrame: DataFrame global unificado, indexado numéricamente y 
            ordenado jerárquicamente por 'user_id' y tiempo.

    Recommendations:
        - Asegúrese de que la primera parte del nombre del archivo (antes del '_')
          coincida exactamente con las claves del diccionario `mapeo_diabetes`.
        - Las categorías se configuran con un orden biológico lógico para 
          facilitar las divisiones de los árboles en XGBoost.

    Example:
        >>> MAPEO = {'CGMacros-018': 'No Diabético'}
        >>> df_global = cargar_y_unificar_dataset('data/', MAPEO)
        >>> print(df_global['grupo_metabolico'].dtype)
        category
    """
    lista_dataframes: List[pd.DataFrame] = []
    
    # Buscar todos los archivos .csv en la carpeta especificada
    patron_busqueda = os.path.join(ruta_carpeta_csv, "*.csv")
    archivos_csv = glob.glob(patron_busqueda)
    
    if not archivos_csv:
        raise FileNotFoundError(f"No se encontraron archivos CSV en la ruta: {ruta_carpeta_csv}")
        
    for ruta_archivo in archivos_csv:
        # Extraer el nombre del archivo sin extensión (ej: 'CGMacros-018_serie1')
        nombre_archivo = os.path.basename(ruta_archivo)
        nombre_sin_extension = os.path.splitext(nombre_archivo)[0]
        
        # Separar por '_' y quedarse con la primera parte (ej: 'CGMacros-018')
        user_id_base = nombre_sin_extension.split('_')[0]
        
        # Leer el archivo CSV individual (ya contiene la columna 'user_id' internamente)
        df_sujeto = pd.read_csv(ruta_archivo)
        
        # Inyectar la variable de control clínico usando el ID base limpio
        df_sujeto['grupo_metabolico'] = mapeo_diabetes[user_id_base]
        
        lista_dataframes.append(df_sujeto)
        
    # Concatenar todos los sujetos en una única matriz tabular
    df_global = pd.concat(lista_dataframes, ignore_index=True)
    
    # --- DIAGNÓSTICO PREVENTIVO ---
    # Eliminamos columnas que estén casi vacías y no presentes en todos los sujetos
    # Eliminamos también columnas que no nos interesan (amount consumed, image_path)
    columnas_a_eliminar = ['recordindex', 'sugar', 'amount consumed ', 'amount_consumed', 'image_path']
    df_global = df_global.drop(columns=[c for c in columnas_a_eliminar if c in df_global.columns])

    # Formalizar el formato de la variable temporal
    df_global['time'] = pd.to_datetime(df_global['time'])
    
    # --- ESTANDARIZACIÓN DE VARIABLES CATEGÓRICAS EN EL VECTOR DE ESTADOS ---
    # Convertimos 'user_id' a tipo categórico para evitar texto plano (object)
    df_global['user_id'] = df_global['user_id'].astype('category')
    
    # Definimos el espacio cerrado y ordenado para el grupo metabólico de tu BD
    categorias_diabetes = ['No Diabético', 'Prediabético', 'Diabético']
    df_global['grupo_metabolico'] = pd.Categorical(
        df_global['grupo_metabolico'], 
        categories=categorias_diabetes, 
        ordered=True
    )
    
    # Ordenar jerárquicamente: primero por sujeto, y dentro de cada sujeto por tiempo
    df_global = df_global.sort_values(by=['user_id', 'time']).reset_index(drop=True)
    
    return df_global


# Invocar la función para obtener el DataFrame global
df_global = cargar_y_unificar_dataset(TRUSTED_DIR, MAPEO_DIABETES)

In [ ]:
df_global

Dado que el algoritmo elegido para la modelización es XGBoost, se ha optado por preservar los valores ausentes (NaN) en la variable de frecuencia cardíaca (heart_rate). Los algoritmos basados en árboles de gradiente aumentado gestionan la dispersión de forma nativa asignando una dirección por defecto a los datos faltantes basada en la ganancia de información. Esto evita la introducción de sesgos o ruido artificial derivados de una interpolación lineal en una señal biológica de alta variabilidad.

In [ ]:
def inicializar_pipeline_temporal_real(df: pd.DataFrame) -> pd.DataFrame:
    """
    Prepara el DataFrame global aislando el target futuro y preservando nulos reales.

    Ordena el dataset cronológicamente, calcula la variable objetivo desfasada
    60 minutos hacia el futuro de forma aislada por usuario para evitar filtraciones
    temporales (Data Leakage) y preserva los valores ausentes (NaN) biológicos.

    Args:
        df (pd.DataFrame): DataFrame global unificado procedente de la función
            de carga de datos.

    Returns:
        pd.DataFrame: DataFrame modificado con la columna de la variable
            objetivo 'target_glucose_60m' calculada.

    Recommendations:
        - No se aplica interpolación en la frecuencia cardíaca (heart_rate) ni en 
          los macronutrientes, permitiendo que XGBoost gestione la dispersión 
          de forma nativa mediante su algoritmo de división por defecto.
        - Las últimas 60 filas de cada sujeto tendrán un target NaN y se eliminarán
          justo antes de entrenar el modelo.

    Example:
        >>> df_preparado = inicializar_pipeline_temporal_real(df_global)
        >>> print(df_preparado['target_glucose_60m'].iloc[0])
        112.5
    """
    df_proc = df.copy()
    
    # 1. Definir horizonte matemático (60 minutos = 60 filas)
    FILAS_HORIZONTE = 60 
    
    # 2. Generar el Target G_{t+60} aislado por sujeto usando un shift negativo
    df_proc['target_glucose_60m'] = df_proc.groupby('user_id', observed=False)['glucose_dexcom'].shift(-FILAS_HORIZONTE)
    
    # 3. Rellenar con 0 las columnas de eventos puntuales y Fitbit
    # Al no estar interpolados en Fitbit, un NaN significa 'sin actividad registrada en ese minuto'
    #columnas_a_poner_cero = [
    #    'calories_activity', 'mets', 'heart_rate', 
    #    'calories', 'carbs', 'protein', 'fat', 'fiber'
    #]
    #df_proc[columnas_a_poner_cero] = df_proc[columnas_a_poner_cero].fillna(0.0)
    
    return df_proc

### D1. Ingeniería de características (Feature Engineering)

VARIABLES CREADAS PARA EL VECTOR DE ESTADOS:

BLOQUE 1: VARIABLES DE CONTROL ESTADÍSTICAS Y TEMPORALES
1. Variables categóricas del sujeto: ID del sujeto
2. Variables categóricas del sujeto: grupo metabólico
3. Codificación cíclica del tiempo

BLOQUE 2: RECONSTRUCCIÓN DEL ESPACIO DE ESTADOS - AUTORREGRESIÓN Y DINÁMICA INTERIOR
1. Vectores de retraso (lag features)
2. Derivadas numéricas discretas (aproximación por diferencias finitas): velocidad instantánea
3. Derivadas numéricas discretas (aproximación por diferencias finitas): aceleración instantánea

BLOQUE 3: OPERADORES DE CONTEXTO HISTÓRICO - ESTADISTICAS MÓVILES
1. Media móvil
2. Desviación estándar móvil
3. Coeficiente de variación local
4. Intensidad media del movimiento: media de METS acumulados (ej: última media hora)
5. Energía total quemada: suma de kcal quemadas acumuladas (ej: últimas 2h)
6. MAGE móvil
7. MODD móvil
8. Eficiencia metabólica: ratio glucosa (velocidad) /METS
9. Interacciones con variables estáticas: CH efectivo x HbA1c (podemos incluir la HbA1c inicial como una columna más para cada sujeto)
10. Inercia glucémica corta: AUC reciente (ej: últimas 2h)
11. iAUC

BLOQUE 4: MODELIZACIÓN DE PERTURBACIONES EXÓGENAS - NUTRICIÓN Y DEPORTE
1. Ventana de deuda de oxígeno o aclaramiento acumulado: EWMA de METS (ej: ventana muy amplia de 12h) - Función de decaimiento temporal (parámetro de decaimiento rápido) - sistema dinámico acumulativo con entrada de flujo kcal
2. Función de decaimiento temporal CH: disponibilidad de nutrientes en sangre - parámetro para 2h de digestión
3.  Función de decaimiento temporal grasas, proteína, fibra: disponibilidad de nutrientes en sangre - parámetro lento (ej: para 4h de digestión)
4.  Ratio de complejidad estático (y ratio estático con lags)
5.  Ratio de complejidad dinámico con funciones de decaimiento
6.  Índice de retardo gástrico estático (e índice de retardo gástrico con lags)
7.  Índice de retardo gástrico dinámico con funciones de decaimiento

In [ ]:
#=======================================================================
# BLOQUE 4: VARIABLES EXÓGENAS DE PERTURBACIONES NUTRICIONALES Y DEPORTE
#=======================================================================

@njit # Acelerar el bucle for nativo de numpy

def _bucle_tanque_nativo(valores_brutos: np.ndarray, alpha: float) -> np.ndarray:
    """
    Bucle interno optimizado para calcular el decaimiento exponencial.
    """
    valores_decaidos = np.zeros(len(valores_brutos))
    acumulado = 0.0
    for i in range(len(valores_brutos)):
        # Ecuación de decaimiento exponencial + nueva ingesta
        acumulado = acumulado * (1.0 - alpha) + valores_brutos[i]
        valores_decaidos[i] = acumulado
    return valores_decaidos

def calcular_tanque_vectorizado(
    df: pd.DataFrame, 
    columna_nutriente: str, 
    alpha: float
) -> pd.Series:
    """
    Simula el tanque de almacenamiento fisiológico aislando el cálculo por sujeto.

    Utiliza un bucle optimizado a nivel de array de NumPy para calcular el 
    decaimiento exponencial de los impulsos, procesando de forma aislada cada 
    usuario (user_id) para evitar fugas de información biológica entre sujetos.

    La ecuación diferencial discretizada que rige el sistema por cada t es:
        X_efectivo(t) = X_efectivo(t-1) * (1 - alpha) + X_bruto(t)

    Args:
        df (pd.DataFrame): DataFrame global unificado que contiene las series 
            temporales y la columna 'user_id'.
        columna_nutriente (str): Nombre de la columna con los gramos brutos 
            del nutriente (ej: 'carbs'). Los NaNs se tratan como 0.0 temporalmente.
        alpha (float): Tasa de aclaramiento/absorción por unidad de tiempo (0 < alpha < 1).

    Returns:
        pd.Series: Serie continua con los gramos efectivamente activos en el 
            organismo en cada instante t, alineada con el DataFrame original.

    Recommendations:
        - Esta función sustituye la entrada de arrays planos por el DataFrame completo 
          para garantizar la integridad de las fronteras entre distintos sujetos.
        - Para una digestión estimada de 2 horas (carbohidratos), se recomienda 
          un alpha aproximado de 0.015.
        - Para una digestión estimada de 4 horas (grasas/proteínas), se recomienda
          un alpha aproximado de 0.006.

    Example:
        >>> import pandas as pd
        >>> df_test = pd.DataFrame({'user_id': ['A', 'A', 'B'], 'carbs': [0.0, 50.0, 0.0]})
        >>> df_test['valores_efectivos'] = calcular_tanque_vectorizado(df_test, 'carbs', alpha=0.1)
        >>> print(df_test['valores_efectivos'].tolist())
        [0.0, 50.0, 0.0]
    """
    # Creamos una copia de la columna rellenando nulos con 0.0 únicamente para el cálculo 
    # matemático del tanque (sin alterar los NaNs originales del DataFrame principal)
    valores_limpios = df[columna_nutriente].copy().fillna(0.0).values
    
    # Creamos un DataFrame temporal ligero para operar por grupos de forma eficiente
    df_temporal = pd.DataFrame({
        'user_id': df['user_id'],
        'valores_brutos': valores_limpios
    })
    
    # Aplicamos la función del tanque aislando estrictamente por cada usuario
    # transformando los datos a arrays de numpy antes del bucle
    valores_efectivos = df_temporal.groupby('user_id', observed=False)['valores_brutos'].transform(
        lambda x: _bucle_tanque_nativo(x.values, alpha)
    )
    
    return valores_efectivos


def construir_bloque_exogeno_simplificado(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula componentes estáticos y rezagos (lags) de los ratios de comidas.

    Calcula los ratios de complejidad e índice de retardo gástrico únicamente 
    en los instantes donde existe una ingesta registrada de carbohidratos. 
    Posteriormente, genera sus rezagos temporales (lags) de forma aislada por 
    usuario para evitar fugas de información en el modelo poblacional.

    Args:
        df (pd.DataFrame): DataFrame que debe contener obligatoriamente las columnas 
            'user_id', 'carbs', 'fat', 'protein' y 'fiber' indexadas temporalmente.

    Returns:
        pd.DataFrame: Copia del DataFrame original con las nuevas columnas de 
            ratios puntuales y sus respectivos rezagos (lags) temporales calculados.

    Recommendations:
        - Esta función preserva los NaNs biológicos en los minutos donde no hay ingesta.
        - Los ratios se calculan con un factor de estabilidad (epsilon) solo si carbs > 0
          para evitar que comidas con 0 carbohidratos generen valores infinitos.

    Example:
        >>> import pandas as pd
        >>> import numpy as np
        >>> data = {'user_id': ['A']*3, 'carbs': [50.0, np.nan, np.nan], 'fat': [10.0, np.nan, np.nan], 'protein': [20.0, np.nan, np.nan], 'fiber': [5.0, np.nan, np.nan]}
        >>> df = pd.DataFrame(data)
        >>> construir_bloque_exogeno_simplificado(df)['ratio_complejidad_estatico'].iloc[0] > 0
        True
    """
    df_res = df.copy()
    epsilon = 1e-5
    
    # Inicializamos las columnas de ratios completamente con NaN (espacio seguro)
    df_res['ratio_complejidad_estatico'] = np.nan
    df_res['indice_retardo_gastrico_estatico'] = np.nan
    
    # Creamos una máscara que detecta ingestas reales de carbohidratos
    # Al usar .fillna(0) solo para la condición, nos protegemos de los NaNs biológicos
    mascara_comida_real = df_res['carbs'].fillna(0) > 0
    
    # Calcular los ratios estáticos puntuales ÚNICAMENTE donde aplica la máscara
    df_res.loc[mascara_comida_real, 'ratio_complejidad_estatico'] = (
        (df_res.loc[mascara_comida_real, 'fat'] + df_res.loc[mascara_comida_real, 'protein']) / 
        (df_res.loc[mascara_comida_real, 'carbs'] + epsilon)
    )
    
    df_res.loc[mascara_comida_real, 'indice_retardo_gastrico_estatico'] = (
        (df_res.loc[mascara_comida_real, 'fat'] + df_res.loc[mascara_comida_real, 'fiber']) / 
        (df_res.loc[mascara_comida_real, 'carbs'] + epsilon)
    )

    # Lags de las Variables Estáticas (hace 15, 30, 45, 60 min)
    # Al desplazar un perfil lleno de NaNs, los lags mantendrán la estructura de NaNs perfectamente
    minutos_lag = [15, 30, 45, 60]
    for lag in minutos_lag:
        df_res[f'ratio_complejidad_estatico_lag_{lag}'] = (
            df_res.groupby('user_id', observed=False)['ratio_complejidad_estatico'].shift(lag)
        )
        df_res[f'indice_retardo_estatico_lag_{lag}'] = (
            df_res.groupby('user_id', observed=False)['indice_retardo_gastrico_estatico'].shift(lag)
        )

    return df_res


def calcular_aclaramiento_ejercicio_acumulado(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula el efecto residual de la actividad física a largo plazo usando EWMA.

    Implementa un sistema dinámico acumulativo a partir de los METs de Fitbit. 
    Para evitar distorsiones en sujetos con pérdidas masivas de señal (NaNs extremos),
    restringe el arrastre (forward fill) a un máximo de 15 minutos; si la pérdida 
    persiste, colapsa de forma segura al nivel basal de reposo (10.0 METs).

    Args:
        df (pd.DataFrame): DataFrame que contiene la columna bruta de 'mets'.

    Returns:
        pd.DataFrame: DataFrame modificado con la nueva columna 
            'mets_ewma_12h' calculada de forma aislada por usuario.

    Recommendations:
        - Esta función protege el vector de estados frente a sesgos artificiales de 
          actividad física prolongada causados por fallos masivos del sensor Fitbit.
        - El límite de 15 minutos actúa como ventana de confianza para ráfagas cortas 
          de pérdida de señal.

    Example:
        >>> import pandas as pd
        >>> data = {'user_id': ['A']*20, 'mets': [80.0] + [None]*19}
        >>> df_test = pd.DataFrame(data)
        >>> df_out = calcular_aclaramiento_ejercicio_acumulado(df_test)
        >>> # Comprobar que tras 15 mins decae hacia el basal de 10.0
        >>> df_out['mets_ewma_12h'].iloc[-1] < 80.0
        True
    """
    df_res = df.copy()
    
    # Parámetros de control analítico
    VENTANA_12H_MINUTOS = 720
    LIMITE_ARRASTRE_MINUTOS = 15
    VALOR_BASAL_REPOSO = 10.0  # Escala x10 de nuetros datos
    
    # Función interna para curar el sensor aislando por sujeto sin inventar datos
    def preparar_mets_sensor(grupo: pd.Series) -> pd.Series:
        # Arrastra el último valor conocido pero MÁXIMO durante 15 minutos consecutivos
        ffill_limitado = grupo.ffill(limit=LIMITE_ARRASTRE_MINUTOS)
        # Los NaNs que queden tras pasar los 15 minutos se consideran reposo seguro (10.0)
        return ffill_limitado.fillna(VALOR_BASAL_REPOSO)
    
    # Aplicar la curación protectora por sujeto
    mets_curados = df_res.groupby('user_id', observed=False)['mets'].transform(preparar_mets_sensor)
    
    # Creamos un DataFrame ligero para ejecutar el EWMA sin riesgo de contaminación
    df_temporal = pd.DataFrame({
        'user_id': df_res['user_id'],
        'mets_calculo': mets_curados
    })
    
    # Calcular la EWMA de 12 horas sobre los METs controlados
    df_res['mets_ewma_12h'] = df_temporal.groupby('user_id', observed=False)['mets_calculo'].transform(
        lambda x: x.ewm(span=VENTANA_12H_MINUTOS, adjust=False).mean()
    )
    
    return df_res


In [ ]:
#=================================================================================
# BLOQUE 2: RECONSTRUCCIÓN DEL ESPACIO DE ESTADOS DE LA GLUCOSA (DINÁMICA INTERNA)
#=================================================================================

def construir_bloque_espacio_estados(df: pd.DataFrame) -> pd.DataFrame:
    """
    Implementa las variables del Bloque 2 asociadas a la inercia interna de la glucosa.

    Calcula los vectores de retraso (lag features) en puntos estratégicos del 
    pasado y aproxima numéricamente las derivadas del sistema homeostático 
    (velocidad y aceleración instantánea) mediante operadores de diferencias finitas 
    vectorizados de alto rendimiento.

    La aproximación de la velocidad (primera derivada) se formaliza como:
        v_t = (G_t - G_{t-5}) / 5 (mg/dL por minuto)

    La aproximación de la aceleración (segunda derivada) se formaliza como:
        a_t = (v_t - v_{t-5}) / 5 (mg/dL por minuto cuadrado)

    Args:
        df (pd.DataFrame): DataFrame que contiene la columna 'glucose_dexcom' 
            limpia e indexada temporalmente.

    Returns:
        pd.DataFrame: DataFrame original enriquecido con los vectores de retraso,
            la velocidad y la aceleración calculadas.

    Recommendations:
        - Esta función optimiza el rendimiento eliminando llamadas redundantes a groupby 
          y operaciones lambda pesadas, aprovechando los lags ya calculados en memoria.
        - Las primeras filas de cada sujeto contendrán valores nulos de forma correcta 
          debido al desplazamiento del histórico.

    Example:
        >>> import pandas as pd
        >>> data = {'user_id': ['A']*10, 'glucose_dexcom': [100.0]*10}
        >>> df_test = pd.DataFrame(data)
        >>> df_out = construir_bloque_espacio_estados(df_test)
        >>> 'glucose_vel_instantanea' in df_out.columns
        True
    """
    df_res = df.copy()
    
    # --- 1. Vectores de Retraso (Lag Features) ---
    # Seleccionamos pasos temporales clave en minutos (1 fila = 1 minuto)
    minutos_lag_glucose = [5, 10, 15, 30, 45, 60]
    
    for lag in minutos_lag_glucose:
        df_res[f'glucose_dexcom_lag_{lag}'] = (
            df_res.groupby('user_id', observed=False)['glucose_dexcom'].shift(lag)
        )
    
    # --- 2. Velocidad Instantánea (Primera Derivada Discreta) ---
    # En lugar de usar transform + lambda, aprovechamos 'glucose_dexcom_lag_5' que ya está 
    # calculado y aislado por sujeto. Esto es instantáneo y vectorizado en memoria.
    df_res['glucose_vel_instantanea'] = (
        df_res['glucose_dexcom'] - df_res['glucose_dexcom_lag_5']
    ) / 5.0
    
    # --- 3. Aceleración Instantánea (Segunda Derivada Discreta) ---
    # Para calcular la aceleración de forma segura por sujeto, generamos un rezago de 5 minutos 
    # de la velocidad recién creada, garantizando el aislamiento con un único groupby eficiente.
    vel_lag_5 = df_res.groupby('user_id', observed=False)['glucose_vel_instantanea'].shift(5)
    
    df_res['glucose_acc_instantanea'] = (
        df_res['glucose_vel_instantanea'] - vel_lag_5
    ) / 5.0
    
    return df_res

In [ ]:
#==================================================================
# BLOQUE 3: OPERADORES DE CONTEXTO HISTÓRICO Y ESTADÍSTICAS MÓVILES
#==================================================================

@njit # Acelerar el bucle for nativo de numpy

def _calcular_mage_ventana_numba(valores: np.ndarray) -> float:
    """
    Algoritmo matemático interno acelerado por Numba para calcular MAGE.

    Mide la media aritmética de las diferencias entre picos y valles, filtrando 
    aquellas excursiones que superan 1 desviación estándar del array actual.

    Args:
        valores (np.ndarray): Array unidimensional de glucosa (sin nulos).

    Returns:
        float: Valor numérico de MAGE para la ventana analizada.
    """
    n = len(valores)
    if n < 3:  # Se necesitan al menos 3 puntos para detectar un cambio de pendiente
        return 0.0
        
    # Calcular desviación estándar nativa en el array
    std_ventana = np.std(valores)
    if std_ventana == 0.0:
        return 0.0
        
    # Identificar puntos de inflexión (picos y valles locales)
    puntos_inflexion = []
    
    # Clonamos la lógica de diferencias finitas de forma indexada para Numba
    for i in range(1, n - 1):
        pendiente_izq = valores[i] - valores[i-1]
        pendiente_der = valores[i+1] - valores[i]
        
        # Si hay un cambio de signo en el producto de pendientes, es un extremo local
        # Se usa un umbral decimal flotante muy pequeño en lugar de np.sign por compatibilidad
        if (pendiente_izq > 0 and pendiente_der < 0) or (pendiente_izq < 0 and pendiente_der > 0):
            puntos_inflexion.append(valores[i])
            
    n_extremos = len(puntos_inflexion)
    if n_extremos < 2:
        return 0.0
        
    # Calcular amplitudes entre extremos consecutivos y filtrar por desviación estándar
    suma_amplitudes = 0.0
    conteo_validas = 0
    
    for j in range(1, n_extremos):
        amplitud = abs(puntos_inflexion[j] - puntos_inflexion[j-1])
        if amplitud > std_ventana:
            suma_amplitudes += amplitud
            conteo_validas += 1
            
    if conteo_validas == 0:
        return 0.0
        
    return suma_amplitudes / conteo_validas

def _calcular_mage_ventana(serie_glucosa: pd.Series) -> float:
    """
    Función envoltura (wrapper) compatible con el método .rolling() de Pandas.

    Args:
        serie_glucosa (pd.Series): Subserie temporal de glucosa de la ventana actual.

    Returns:
        float: Valor numérico de MAGE para la ventana.
    """
    # Limpiamos nulos y extraemos el array subyacente para el motor de Numba
    valores = serie_glucosa.dropna().to_numpy()
    return _calcular_mage_ventana_numba(valores)


def extraer_dict_hba1c_desde_bio(ruta_bio_csv: str) -> Dict[str, float]:
    """
    Carga bio.csv y genera un mapeo del nivel de HbA1c indexado por ID de sujeto.

    Lee el archivo analítico y demográfico externo del dataset, normaliza la 
    columna entera 'subject' transformándola al formato de identificador global 
    'CGMacros-XXX' y mapea dichos identificadores con sus respectivos valores 
    de laboratorio de Hemoglobina Glicosilada (A1c PDL (Lab)).

    Args:
        ruta_bio_csv (str): Ruta absoluta o relativa hacia el archivo 'bio.csv'.

    Returns:
        Dict[str, float]: Diccionario con el par clave-valor correspondientes 
            a {ID_Sujeto_Limpio (str): Valor_HbA1c (float)}.

    Recommendations:
        - Al inyectar este diccionario en tu DataFrame global, asegúrese de limpiar 
          los sufijos de subseries (ej: '_serie1') de la columna 'user_id' antes 
          de aplicar el mapeo para evitar nulos artificiales (NaN).
        - Asegúrese de que la columna 'A1c PDL (Lab)' no contenga espacios extras 
          en la cabecera del archivo fuente.

    Example:
        >>> RUTA = "datos_clinicos/bio.csv"
        >>> hba1c_dict = extraer_dict_hba1c_desde_bio(RUTA)
        >>> print('CGMacros-001' in hba1c_dict)
        True
    """
    # Cargar el CSV analítico de laboratorio
    df_bio = pd.read_csv(ruta_bio_csv)
    
    # Normalizar los nombres de las columnas eliminando espacios en blanco invisibles
    df_bio.columns = df_bio.columns.str.strip()
    
    # Construir la transformación topológica del ID: int -> 'CGMacros-00X'
    # .astype(str).str.zfill(3) convierte el entero 1 en '001', el 45 en '045'
    df_bio['user_id_mapeado'] = 'CGMacros-' + df_bio['subject'].astype(str).str.zfill(3)
    
    # Convertir explícitamente el valor de HbA1c a float, asegurando la limpieza de nulos clínicos
    df_bio['A1c PDL (Lab)'] = pd.to_numeric(df_bio['A1c PDL (Lab)'], errors='coerce')
    
    # Extraer el par estructurado convirtiéndolo a un diccionario nativo de Python
    dict_hba1c = dict(zip(df_bio['user_id_mapeado'], df_bio['A1c PDL (Lab)']))
    
    return dict_hba1c


def construir_bloque_contexto_historico(
    df: pd.DataFrame, 
    hba1c_iniciales: dict
) -> pd.DataFrame:
    """
    Implementa las variables de contexto histórico y estadísticas móviles del Bloque 3.

    Calcula operadores estadísticos móviles basándose en ventanas de tiempo de 
    30 minutos, 2 horas y 24 horas, integrando métricas de variabilidad glucémica 
    clínica (MAGE, MODD, iAUC) e interacciones metabólicas estáticas de forma 
    aislada por sujeto para evitar fugas de información.

    Args:
        df (pd.DataFrame): DataFrame que ya contiene el Bloque 2 y Bloque 4 calculados.
        hba1c_iniciales (dict): Diccionario con los niveles de HbA1c iniciales por sujeto 
            (ej: {'CGMacros-001': 5.4, 'CGMacros-002': 6.8}).

    Returns:
        pd.DataFrame: DataFrame original enriquecido con las variables del Bloque 3.

    Recommendations:
        - Todas las ventanas móviles se encapsulan dentro de transformaciones agrupadas 
          por usuario para asegurar que las fronteras individuales queden protegidas.
        - El cálculo de MODD requiere la existencia de datos en el instante t y t-1440. 
          Los primeros días de cada sujeto darán nulo de forma natural en esta métrica.
    """
    df_res = df.copy()
    epsilon = 1e-5
    
    # Definición de ventanas expresadas en minutos (1 fila = 1 minuto debido a la interpolación)
    W_30M = 30
    W_2H = 120
    W_24H = 1440
    
    # --- 1. Inyectar columna de HbA1c Estática por Sujeto (Protección contra sufijos _serie) ---
    # Extraemos el string base limpio en memoria sin alterar la columna original del DataFrame
    user_id_limpio = df_res['user_id'].astype(str).str.split('_').str[0]
    df_res['hba1c_inicial'] = user_id_limpio.map(hba1c_iniciales).astype(float)
    
    # --- 2. Estadísticos Móviles Estándar de Glucosa (Ventana 2 Horas) ---
    df_res['glucose_media_2h'] = df_res.groupby('user_id', observed=False)['glucose_dexcom'].transform(
        lambda x: x.rolling(window=W_2H, min_periods=15).mean()
    )
    df_res['glucose_std_2h'] = df_res.groupby('user_id', observed=False)['glucose_dexcom'].transform(
        lambda x: x.rolling(window=W_2H, min_periods=15).std()
    )
    # Coeficiente de Variación Local (CV_W)
    df_res['glucose_cv_2h'] = (df_res['glucose_std_2h'] / (df_res['glucose_media_2h'] + epsilon)) * 100.0
    
    # --- 3. Intensidad de Movimiento y Gasto Energético (Fitbit) ---
    df_res['mets_media_30m'] = df_res.groupby('user_id', observed=False)['mets'].transform(
        lambda x: x.rolling(window=W_30M, min_periods=1).mean()
    )
    df_res['calories_suma_2h'] = df_res.groupby('user_id', observed=False)['calories_activity'].transform(
        lambda x: x.rolling(window=W_2H, min_periods=1).sum()
    )
    
    # --- 4. Eficiencia Metabólica (Ratio velocidad / METs) ---
    df_res['eficiencia_metabolica_ratio'] = (
        df_res['glucose_vel_instantanea'] / (df_res['mets_media_30m'] + epsilon)
    )
    
    # --- 5. Interacciones con Variables Estáticas ---
    # Dejado pendiente: Se usará la variable bruta 'carbs' provisionalmente hasta 
    # la implementación de la optimización dinámica de CH_efectivo en el lazo de Optuna.
    # df_res['interaccion_CH_HbA1c'] = df_res['carbs'].fillna(0.0) * df_res['hba1c_inicial']
    
    # --- 6. Integrales Geométricas: AUC reciente e iAUC (Ventana 2 Horas) ---
    df_res['auc_total_2h'] = df_res.groupby('user_id', observed=False)['glucose_dexcom'].transform(
        lambda x: x.rolling(window=W_2H, min_periods=1).sum()
    )
    # iAUC: Área incremental por encima de la línea base (el valor de glucosa hace 2 horas)
    df_res['iauc_incremental_2h'] = df_res.groupby('user_id', observed=False)['glucose_dexcom'].transform(
        lambda x: (x - x.shift(W_2H)).clip(lower=0).rolling(window=W_2H, min_periods=1).sum()
    )
    
    # --- 7. Métricas de Variabilidad Avanzadas (MAGE y MODD móviles) ---
    # MAGE móvil empleando la envoltura optimizada compatible con Numba
    df_res['mage_movil_2h'] = df_res.groupby('user_id', observed=False)['glucose_dexcom'].transform(
        lambda x: x.rolling(window=W_2H, min_periods=30).apply(_calcular_mage_ventana, raw=False)
    )
    
    # MODD móvil: Diferencia absoluta entre la glucosa de ahora y la de hace exactamente 24h
    glucosa_ayer = df_res.groupby('user_id', observed=False)['glucose_dexcom'].shift(W_24H)
    diferencia_interdiaria = np.abs(df_res['glucose_dexcom'] - glucosa_ayer)
    
    # Forzamos el groupby para evitar que el promedio móvil mezcle sujetos en los extremos
    df_temporal_modd = pd.DataFrame({'user_id': df_res['user_id'], 'diff': diferencia_interdiaria})
    df_res['modd_movil_2h'] = df_temporal_modd.groupby('user_id', observed=False)['diff'].transform(
        lambda x: x.rolling(window=W_2H, min_periods=15).mean()
    )
    
    return df_res

Para una fase de ampliación, el archivo bio.csv contiene variables con un peso fisiológico importante para enriquecer el modelo global (como el IMC (BMI), los Triglicéridos o los niveles de Insulina basal).

El siguiente es fundamental porque aborda la topología del tiempo y la representación de espacios cualitativos. No podemos introducir la hora como un número lineal (del 0 al 23) porque para el modelo la hora 23 y la hora 00 estarían matemáticamente en extremos opuestos, cuando en la realidad circadiana están pegadas. Al proyectar la hora sobre un círculo unitario utilizando funciones trigonométricas (Seno y Coseno), preservamos la continuidad temporal de la sensibilidad a la insulina.

In [ ]:
#=========================================================
# BLOQUE 1: VARIABLES DE CONTROL ESTADÍSTICAS Y TEMPORALES 
#=========================================================

def construir_bloque_control_temporal(df: pd.DataFrame) -> pd.DataFrame:
    """
    Implementa las variables de control del Bloque 1 (cíclicas y categóricas).

    Transforma la variable de tiempo 'time' en proyecciones trigonométricas 
    bidimensionales (seno y coseno) para modelar adecuadamente el ritmo 
    circadiano del metabolismo. Adicionalmente, formatea las variables 
    cualitativas para su consumo nativo en algoritmos de machine learning.

    Las ecuaciones de transformación cíclica para la hora del día son:
        H_sin = sin(2 * pi * H / 24)
        H_cos = cos(2 * pi * H / 24)

    Args:
        df (pd.DataFrame): DataFrame que contiene la columna 'time' en formato 
            datetime, 'user_id' y 'grupo_metabolico'.

    Returns:
        pd.DataFrame: DataFrame original enriquecido con las codificaciones 
            cíclicas y las columnas categóricas formateadas.

    Recommendations:
        - Asegúrese de que la columna 'time' haya sido casteada previamente a 
          tipo datetime mediante `pd.to_datetime`.
        - Las variables de grupo_metabolico y user_id mantienen el tipado categórico 
          fijado al inicio del pipeline para asegurar consistencia en SHAP.

    Example:
        >>> import pandas as pd
        >>> import numpy as np
        >>> fechas = pd.date_range('2026-01-01', periods=5, freq='H')
        >>> df_test = pd.DataFrame({'time': fechas, 'user_id': ['A']*5, 'grupo_metabolico': ['Diabético']*5})
        >>> df_test['grupo_metabolico'] = df_test['grupo_metabolico'].astype('category')
        >>> df_out = construir_bloque_control_temporal(df_test)
        >>> df_out['dia_semana'].dtype.name == 'category'
        True
    """
    df_res = df.copy()
    
    # --- 1. Codificación Cíclica del Tiempo (Ritmo Circadiano) ---
    # Extraemos la hora exacta en formato flotante (incluyendo minutos para máxima precisión)
    hora_flotante = df_res['time'].dt.hour + (df_res['time'].dt.minute / 60.0)
    
    # Proyección sobre el círculo unitario (periodo de 24 horas)
    df_res['hora_sin'] = np.sin(2.0 * np.pi * hora_flotante / 24.0)
    df_res['hora_cos'] = np.cos(2.0 * np.pi * hora_flotante / 24.0)
    
    # --- 2. Formateo y Respeto de Variables Categóricas ---
    # Convertimos el día de la semana a categórico para evitar jerarquías numéricas artificiales (0-6)
    df_res['dia_semana'] = df_res['time'].dt.dayofweek.astype('category')
    
    # Nos aseguramos de mantener el tipo de dato de las columnas categóricas base si por alguna razón se perdieron,
    # pero sin sobreescribir sus propiedades si ya venían configuradas desde el cargador de datos.
    if df_res['user_id'].dtype.name != 'category':
        df_res['user_id'] = df_res['user_id'].astype('category')
        
    if df_res['grupo_metabolico'].dtype.name != 'category':
        # Mapeo de seguridad por si no venía ordenado
        categorias_diabetes = ['No Diabético', 'Prediabético', 'Diabético']
        df_res['grupo_metabolico'] = pd.Categorical(
            df_res['grupo_metabolico'], 
            categories=categorias_diabetes, 
            ordered=True
        )
    
    return df_res

In [ ]:
#============================================================
# CÓDIGO INTEGRADOR DE LOS BLOQUES DE INGENIERÍA DE VARIABLES 
#============================================================

# --- PIPELINE DE CONSTRUCCIÓN DEL DATASET ---

# Paso Inicial: Construir de forma automatizada el diccionario HbA1c desde bio.csv
HBA1C_DICT = extraer_dict_hba1c_desde_bio(RUTA_BIO_CSV)

# Script Principal - Construcción del Dataset del TFG

# 1. Cargar datos
df_base = cargar_y_unificar_dataset(TRUSTED_DIR, MAPEO_DIABETES)

# 2. Inicializar pipeline y target (a 60 minutos vista)
df_pipeline = inicializar_pipeline_temporal_real(df_base)

# 3. Aplicar Bloque 4 (Perturbaciones Exógenas: Tanques de Macronutrientes y Ratios)
df_features = construir_bloque_exogeno_simplificado(df_pipeline)
df_features = calcular_aclaramiento_ejercicio_acumulado(df_features)

# 4. Aplicar Bloque 2 (Espacio de Estados / Derivadas: Vectores de retraso de glucosa, Velocidad y Aceleración)
df_features = construir_bloque_espacio_estados(df_features)

# 5. Aplicar Bloque 3 (Contexto Histórico / Estadísticos / HbA1c)
df_features = construir_bloque_contexto_historico(df_features, HBA1C_DICT)

# 6. Aplicar Bloque 1 (Control Estadístico y Tiempo Cíclico)
df_final = construir_bloque_control_temporal(df_features)

# Guardar o pasar directamente a la fase de partición
print(f"Ingeniería finalizada correctamente. Columnas totales: {len(df_final.columns)}")

### D2. Validación

#### D2.1 Métricas para la validación

**Métricas para validación y evaluación** 

- **Métrica de Entrenamiento (Loss Function / Función de Pérdida)**: El modelo (XGBoost) solo puede mirar una única métrica durante su entrenamiento. Esta métrica debe ser diferenciable (debe tener derivadas continuas) para que el algoritmo de descenso de gradiente pueda calcular hacia dónde mover los pesos del árbol. En nuestro caso, la elegida de forma nativa es el **MSE** (Error Cuadrático Medio), cuya raíz es el **RMSE**.
- **Métricas de Evaluación Post-hoc**: El resto de las métricas (**MAE, R², MARD y Clarke**) se calculan a posteriori, una vez que el modelo ya ha generado las predicciones en el conjunto de test. Sirven exclusivamente para evaluar el rendimiento desde diferentes ángulos (estadísticos y clínicos).

**Descrpción nuevas métricas:**
- **MARD**: Diferencia Relativa Absoluta Media, métrica estándar en la industria de sensores CGM que mide el error porcentual promedio ponderado por el valor real. Es altamente sensible a desviaciones en rangos de hipoglucemia.
- **CLARKE**: Análisis de zonas de riesgo clínico según la rejilla de Clarke (EGA, Error Grid Analysis).

In [ ]:
#=======================================================================
# MÉTRICAS AÑADIDAS: MARD Y CLARKE
#=======================================================================

# MARD: Diferencia Relativa Absoluta Media
# CLARKE: Análisis de zonas de riesgo clínico según la rejilla de Clarke (EGA)

def calcular_mard(y_real: np.ndarray, y_pred: np.ndarray) -> float:
    """
    Calcula la Diferencia Relativa Absoluta Media (MARD).

    Métrica estándar en la industria de sensores CGM que mide el error porcentual
    promedio ponderado por el valor real. Es altamente sensible a desviaciones 
    en rangos de hipoglucemia.

    Args:
        y_real (np.ndarray): Vector de valores reales de glucosa (mg/dL).
        y_pred (np.ndarray): Vector de valores predichos de glucosa (mg/dL).

    Returns:
        float: Valor porcentual de MARD.

    Recommendations:
        - Asegúrese de que no existan valores iguales a cero en y_real para evitar 
          indeterminaciones (fisiológicamente la glucosa nunca es cero).
    """
    # Evitar divisiones por cero por seguridad matemática
    y_real_safe = np.where(y_real == 0, 1e-5, y_real)
    diferencias_relativas = np.abs(y_real_safe - y_pred) / y_real_safe
    
    return float(np.mean(diferencias_relativas) * 100.0)


def clarke_zone_oficial(real: float, pred: float) -> int:
    """
    Clasificación matemática exacta y canónica de la rejilla de Clarke (1987).

    Returns:
        0 -> Zona A (Verde)
        1 -> Zona B (Amarillo)
        2 -> Zona C (Naranja)
        3 -> Zona D (Rojo)
        4 -> Zona E (Morado)
    """
    # 1. REGIÓN A (Exactitud Clínica Estricta)
    if (real <= 70 and pred <= 70) or (0.8 * real <= pred <= 1.2 * real):
        return 0

    # 2. REGIÓN E (Peligro Vital por Tratamiento Inverso)
    elif (real <= 70 and pred >= 180) or (real >= 180 and pred <= 70):
        return 4

    # 3. REGIÓN D (Fallo Crítico en la Detección de Extremos)
    elif (
        (real < 70 and 70 < pred < 180)
        or (real >= 240 and 70 < pred <= real - 110)
        or (70 < real < 180 and pred <= 70 and real - pred > 110)
    ):
        return 3

    # 4. REGIÓN C (Sobre-correcciones Innecesarias)
    elif (
        (70 < real <= 130 and pred > real + 110)
        or (130 < real <= 180 and pred <= (7.0 / 5.0) * real - 182)
    ):
        return 2

    # 5. REGIÓN B (Errores Benignos / Desviaciones Clínicamente Seguras)
    else:
        return 1
    

def analizar_zonas_clarke(y_real: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    """
    Calcula la distribución porcentual de observaciones en las distintas zonas de la
    Clarke Error Grid a partir de valores reales y predichos de glucosa.

    La clasificación de cada punto se realiza mediante la función `clarke_zone_oficial`,
    que asigna una etiqueta discreta (0-4) correspondiente a las zonas A-E.

    Args:
        y_real : np.ndarray
            Valores reales de glucosa en sangre (mg/dL).
        y_pred : np.ndarray
            Valores predichos por el modelo (mg/dL).

    Returns:
        Dict[str, float]
            Diccionario con los porcentajes de puntos en cada zona:

            - 'Zona_A': Porcentaje de puntos en zona A (exactitud clínica).
            - 'Zona_B': Porcentaje de puntos en zona B (desviaciones benignas).
            - 'Zona_C': Porcentaje de puntos en zona C (sobre-correcciones).
            - 'Zona_D': Porcentaje de puntos en zona D (fallos críticos).
            - 'Zona_E': Porcentaje de puntos en zona E (errores peligrosos).
            - 'Zona_A_Plus_B': Suma de zonas A y B (rendimiento clínicamente aceptable).
            - 'Zonas_Criticas_CDE': Suma de zonas C, D y E (errores relevantes).

    Notes:
        - Si no hay datos de entrada, todas las métricas se devuelven a 0.0.
        - La función elimina automáticamente valores no finitos (NaN o inf).
        - La coherencia con la representación gráfica depende de la implementación
        de `clarke_zone_oficial`.

    Examples:
        >>> import numpy as np
        >>> y_real = np.array([100, 150, 200])
        >>> y_pred = np.array([105, 160, 190])
        >>> analizar_zonas_clarke(y_real, y_pred)
        {
            'Zona_A': 66.67,
            'Zona_B': 33.33,
            'Zona_A_Plus_B': 100.0,
            'Zona_C': 0.0,
            'Zona_D': 0.0,
            'Zona_E': 0.0,
            'Zonas_Criticas_CDE': 0.0
        }

    Recommendation:
        - Asegurar que `clarke_zone_oficial` utiliza la misma definición de zonas
        que la utilizada en la visualización gráfica para evitar inconsistencias.
        - Para grandes volúmenes de datos, considerar vectorización completa o
        compilación (Numba) para mejorar rendimiento.
    """
    total_puntos = len(y_real)
    if total_puntos == 0:
        return {
            'Zona_A': 0.0,
            'Zona_B': 0.0,
            'Zona_A_Plus_B': 0.0,
            'Zona_C': 0.0,
            'Zona_D': 0.0,
            'Zona_E': 0.0,
            'Zonas_Criticas_CDE': 0.0
        }

    r = np.asarray(y_real, dtype=float)
    p = np.asarray(y_pred, dtype=float)

    # limpieza opcional (evita NaNs)
    mask = np.isfinite(r) & np.isfinite(p)
    r = r[mask]
    p = p[mask]

    # clasificación
    zonas = np.fromiter(
        (clarke_zone_oficial(ri, pi) for ri, pi in zip(r, p)),
        dtype=np.int8,
        count=len(r)
    )

    conteos = np.bincount(zonas, minlength=5)

    pct_a = conteos[0] / len(r) * 100
    pct_b = conteos[1] / len(r) * 100
    pct_c = conteos[2] / len(r) * 100
    pct_d = conteos[3] / len(r) * 100
    pct_e = conteos[4] / len(r) * 100

    return {
        'Zona_A': pct_a,
        'Zona_B': pct_b,
        'Zona_A_Plus_B': pct_a + pct_b,
        'Zona_C': pct_c,
        'Zona_D': pct_d,
        'Zona_E': pct_e,
        'Zonas_Criticas_CDE': pct_c + pct_d + pct_e
    }

La Zona A de Clarke se redefine como la unión de un criterio absoluto de seguridad clínica en hipoglucemia (≤70 mg/dL) y un criterio relativo de precisión del modelo (±20%), eliminando la dependencia exclusiva del umbral como frontera de partición global.

#### D2.2 LOSO (Leave-One-Subject-Out) - Modelos globales / grupos metabólicos

Como nuestra variable objetivo es continua (target_glucose_60m), plantearemos un problema de regresión supervisada. 

El algoritmo entrenará con 44 sujetos y testeará con el sujeto 45, repitiendo el proceso en un bucle (loop) para obtener el error promedio real y evitar el Data Leakage. 

Para que un XGBoost funcione correctamente debemos limpiar las filas que tengan nulos en el target. Las últimas 60 filas de cada paciente no tienen un "futuro a 60 minutos" registrado, por lo que no sirven para entrenar y hay que eliminarlas.

Además, diseñemos la validación LOSO con la optimización (de Optuna) integrada para que empiece a buscar los parámetros perfectos para las funciones de decaimiento de las variables que definimos anteriormente.
Los parámetros de los tanques dinámicos no se fijaron, sino que se trataron como parámetros latentes optimizados mediante optimización bayesiana en el espacio in-sample de cada pliegue, garantizando que el paciente de test fuera 100% ciego a la calibración biológica.

In [ ]:
#=======================================================================
# VALIDACIÓN LOSO CON OPTIMIZACIÓN DE HIPERPARÁMETROS (OPTUNA + XGBOOST)
#=======================================================================

# Desactivar los mensajes de log repetitivos de Optuna para mantener la consola limpia
optuna.logging.set_verbosity(optuna.logging.WARNING)

def preparar_matrices_entrenamiento(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.Series]:
    """
    Elimina filas sin target y filtra estrictamente las columnas no predictoras.

    Quita del dataset las colas de tiempo de cada sujeto que carecen de variable 
    objetivo y purga la matriz X eliminando rutas de imágenes y metadatos textuales, 
    preservando temporalmente el eje cronológico 'time' y las variables categóricas 
    para la posterior segmentación y cálculo dinámico en la validación LOSO.

    Args:
        df (pd.DataFrame): DataFrame final con todas las variables calculadas de la ingeniería.

    Returns:
        Tuple[pd.DataFrame, pd.Series]: Tupla conteniendo:
            1. X (pd.DataFrame): Matriz de características que conserva 'time' e identificadores.
            2. y (pd.Series): Vector objetivo de glucosa a 60 minutos vista.

    Recommendations:
        - La columna 'time' se mantiene deliberadamente en la matriz X devuelta para 
          permitir que la función LOSO alinee cronológicamente las trayectorias de test.
        - Las columnas categóricas ('user_id' y 'grupo_metabolico') preservan su tipado 
          nativo fijado en la Fase 1.
    """
    # 1. Eliminar registros donde el target sea NaN (obligatorio para la regresión del horizonte futuro)
    # Hacemos una copia profunda para aislar el proceso en memoria
    df_limpio = df.dropna(subset=['target_glucose_60m']).copy()
    
    # 2. Lista explícita de exclusiones textuales o irrelevantes para los árboles
    # Retiramos 'time' de esta lista para no destruir el eje cronológico del sujeto
    columnas_excluidas = [
        'target_glucose_60m', 
        'meal_type', 'image_path', 
        'amount consumed', 'amount_consumed'
    ]
    
    # Evaluamos de forma segura la existencia de las columnas a descartar en el DF de origen
    columnas_a_dropear = [col for col in columnas_excluidas if col in df_limpio.columns]
    
    # 3. Construir Matriz X y Vector y manteniendo el tipado estructural
    X = df_limpio.drop(columns=columnas_a_dropear).reset_index(drop=True)
    y = df_limpio['target_glucose_60m'].reset_index(drop=True)
    
    return X, y


def ejecutar_loso_con_optimizacion_fisiologica(
    df_final_eng: pd.DataFrame, 
    num_trials_optuna: int = 15
) -> Tuple[Dict[str, Any], pd.DataFrame, Dict[str, xgb.XGBRegressor], Dict[str, pd.DataFrame]]:
    """
    Ejecuta validación LOSO optimizando parámetros de absorción e hiperparámetros.

    En cada fold (sujeto ciego), Optuna busca la tasa de vaciado gástrico óptima 
    para los carbohidratos y componentes lentos basándose estrictamente en el 
    conjunto de entrenamiento de los sujetos restantes, eliminando el Data Leakage.

    Args:
        df_final_eng (pd.DataFrame): DataFrame global ordenado con la ingeniería base.
        num_trials_optuna (int, optional): Intentos de optimización por fold. Defaults to 15.

    Returns:
        Tuple: Contiene el reporte_consolidado, el DataFrame de trayectorias, 
               un diccionario con los modelos entrenados por fold (clave: sujeto_test) 
               y las matrices X_train correspondientes para la Fase 3 (SHAP).
    """
    # 1. Limpieza obligatoria del target antes de separar sujetos y ordenación jerárquica
    df_modelado = df_final_eng.dropna(subset=['target_glucose_60m']).copy()
    df_modelado = df_modelado.sort_values(by=['user_id', 'time']).reset_index(drop=True)
    sujetos_unicos = df_modelado['user_id'].unique()
    epsilon = 1e-5
    
    # Contenedores para las 5 métricas en Test Ciego
    resultados_rmse, resultados_mae, resultados_r2, resultados_mard, resultados_clarke_ab = [], [], [], [], []
    lista_df_predicciones: List[pd.DataFrame] = []
    
    # SHAP: Diccionarios para resguardar el entorno explicativo por fold
    modelos_poblacionales_loso = {}
    matrices_train_poblacionales_loso = {}
    
    # Columnas que XGBoost no debe ver en ningún modelo (metadatos/tiempo)
    # Conservamos obligatoriamente 'user_id' y 'grupo_metabolico' con su tipo category
    cols_drop_regresor = ['time', 'target_glucose_60m', 'meal_type', 'image_path']

    print(f"Iniciando Validación LOSO con Optimización Híbrida para {len(sujetos_unicos)} sujetos...")
    
    for idx, sujeto_test in enumerate(sujetos_unicos):
        print(f"\n[{idx+1}/{len(sujetos_unicos)}] Optimizando y evaluando sujeto ciego: {sujeto_test}")
        
        # Segmentación estricta del pliegue (Fold)
        df_train = df_modelado[df_modelado['user_id'] != sujeto_test].copy().reset_index(drop=True)
        df_test = df_modelado[df_modelado['user_id'] == sujeto_test].copy().reset_index(drop=True)
        
        # --- Función Objetivo Interna de Optuna ---
        def objetivo_hibrido(trial: optuna.Trial) -> float:
            alpha_ch = trial.suggest_float('alpha_ch', 0.005, 0.05)
            alpha_lento = trial.suggest_float('alpha_lento', 0.002, 0.02)
            
            params_xgb = {
                'n_estimators': trial.suggest_int('n_estimators', 50, 150), # Rango sutilmente acotado para velocidad
                'max_depth': trial.suggest_int('max_depth', 3, 6),
                'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.12, log=True),
                'subsample': trial.suggest_float('subsample', 0.7, 1.0),
                'tree_method': 'hist', 
                'enable_categorical': True, # <--- ACTIVADO NATIVAMENTE PARA VARIABLE POBLACIONAL
                'n_jobs': -1
            }
            
            # OPTIMIZACIÓN DE RENDIMIENTO 
            # Ejecutamos el bucle nativo de numpy aislado por sujeto de forma ultra-veloz
            # Evitamos lambdas redundantes aprovechando las propiedades vectorizadas estructuradas
            ch_tr = df_train.groupby('user_id', observed=False)['carbs'].transform(lambda x: _bucle_tanque_nativo(x.fillna(0.0).to_numpy(), alpha_ch))
            fat_tr = df_train.groupby('user_id', observed=False)['fat'].transform(lambda x: _bucle_tanque_nativo(x.fillna(0.0).to_numpy(), alpha_lento))
            pro_tr = df_train.groupby('user_id', observed=False)['protein'].transform(lambda x: _bucle_tanque_nativo(x.fillna(0.0).to_numpy(), alpha_lento))
            fib_tr = df_train.groupby('user_id', observed=False)['fiber'].transform(lambda x: _bucle_tanque_nativo(x.fillna(0.0).to_numpy(), alpha_lento))
            
            # Ensamblar matriz X_train temporal del trial
            X_train_tmp = df_train.drop(columns=cols_drop_regresor, errors='ignore').copy()
            X_train_tmp['CH_efectivo'] = ch_tr
            X_train_tmp['ratio_complejidad_dinamico'] = (fat_tr + pro_tr) / (ch_tr + epsilon)
            X_train_tmp['indice_retardo_gastrico_dinamico'] = (fat_tr + fib_tr) / (ch_tr + epsilon)
            X_train_tmp['interaccion_CH_HbA1c'] = ch_tr * df_train['hba1c_inicial'].astype(float)
            
            y_train_tmp = df_train['target_glucose_60m']
            
            # LOGÍSTICA MULTI-SUJETO
            # La división 80/20 in-sample debe respetar de forma estricta la estructura secuencial 
            # interna de cada sujeto presente en el set de entrenamiento, en lugar de cortar el DataFrame global.
            indices_train_int = []
            indices_val_int = []
            
            for u in df_train['user_id'].unique():
                idx_u = df_train[df_train['user_id'] == u].index
                corte_u = int(len(idx_u) * 0.8)
                indices_train_int.extend(idx_u[:corte_u])
                indices_val_int.extend(idx_u[corte_u:])
                
            modelo_int = xgb.XGBRegressor(**params_xgb)
            modelo_int.fit(X_train_tmp.iloc[indices_train_int], y_train_tmp.iloc[indices_train_int])
            
            preds_val = modelo_int.predict(X_train_tmp.iloc[indices_val_int])
            return float(np.sqrt(mean_squared_error(y_train_tmp.iloc[indices_val_int], preds_val)))

        # Ejecutar la optimización bayesiana para este pliegue poblacional
        estudio = optuna.create_study(direction="minimize")
        estudio.optimize(objetivo_hibrido, n_trials=num_trials_optuna)
        mejores_parametros_hibridos = estudio.best_params
        
        # --- RECONSTRUCCIÓN FINAL CON LOS PARÁMETROS GANADORES EN AMBOS BLOQUES ---
        best_ch = mejores_parametros_hibridos['alpha_ch']
        best_len = mejores_parametros_hibridos['alpha_lento']
        
        for df_bloque in [df_train, df_test]:
            ch_ef = df_bloque.groupby('user_id', observed=False)['carbs'].transform(lambda x: _bucle_tanque_nativo(x.fillna(0.0).to_numpy(), best_ch))
            fat_ef = df_bloque.groupby('user_id', observed=False)['fat'].transform(lambda x: _bucle_tanque_nativo(x.fillna(0.0).to_numpy(), best_len))
            pro_ef = df_bloque.groupby('user_id', observed=False)['protein'].transform(lambda x: _bucle_tanque_nativo(x.fillna(0.0).to_numpy(), best_len))
            fib_ef = df_bloque.groupby('user_id', observed=False)['fiber'].transform(lambda x: _bucle_tanque_nativo(x.fillna(0.0).to_numpy(), best_len))
            
            df_bloque['CH_efectivo'] = ch_ef
            df_bloque['ratio_complejidad_dinamico'] = (fat_ef + pro_ef) / (ch_ef + epsilon)
            df_bloque['indice_retardo_gastrico_dinamico'] = (fat_ef + fib_ef) / (ch_ef + epsilon)
            df_bloque['interaccion_CH_HbA1c'] = ch_ef * df_bloque['hba1c_inicial'].astype(float)
            
        # Generar las matrices finales purgadas para alimentar al XGBoost definitivo del fold
        X_train = df_train.drop(columns=cols_drop_regresor, errors='ignore')
        y_train = df_train['target_glucose_60m']
        X_test = df_test.drop(columns=cols_drop_regresor, errors='ignore')
        y_test = df_test['target_glucose_60m']

        # Las matrices purgadas X_train, y_train, X_test e y_test 
        # ya se calculan al final de la primera parte usando 'cols_drop_regresor' de forma limpia.
        
        # Extraer parámetros de árbol ganadores entrenando el XGBoost definitivo del fold
        params_arbol_ganador = {k: v for k, v in mejores_parametros_hibridos.items() if k not in ['alpha_ch', 'alpha_lento']}
        params_arbol_ganador.update({'tree_method': 'hist', 'enable_categorical': True, 'n_jobs': -1})
        
        modelo_final = xgb.XGBRegressor(**params_arbol_ganador)
        modelo_final.fit(X_train, y_train)
        
        # --- SHAP POBLACIONAL ---
        # Almacenamos el modelo entrenado y su matriz en los diccionarios usando el sujeto ciego como clave.
        # Esto nos permitirá en la Fase 3 explicar el modelo de este pliegue exacto.
        modelos_poblacionales_loso[sujeto_test] = modelo_final
        matrices_train_poblacionales_loso[sujeto_test] = X_train
        
        # --- EVALUACIÓN SOBRE EL SUJETO TEST CIEGO ---
        predicciones = modelo_final.predict(X_test)
        y_test_np = y_test.to_numpy()
        
        # Calcular las 5 métricas seleccionadas utilizando tus funciones optimizadas y sklearn
        rmse_v = np.sqrt(mean_squared_error(y_test, predicciones))
        mae_v = mean_absolute_error(y_test, predicciones)
        r2_v = r2_score(y_test, predicciones)
        mard_v = calcular_mard(y_test_np, predicciones)
        clarke_v = analizar_zonas_clarke(y_test_np, predicciones)['Zona_A_Plus_B']
        
        # Guardar en contenedores globales para promediar la población al final
        resultados_rmse.append(rmse_v)
        resultados_mae.append(mae_v)
        resultados_r2.append(r2_v)
        resultados_mard.append(mard_v)
        resultados_clarke_ab.append(clarke_v)
        
        # Almacenar físicamente las predicciones de este sujeto de test respetando el DF final alineado
        df_preds_sujeto = pd.DataFrame({
            'time': df_test['time'],
            'user_id': sujeto_test,
            'Real': y_test_np,
            'Predicho': predicciones
        })
        lista_df_predicciones.append(df_preds_sujeto)

        print(f"      [GANADOR FOLD] Alpha Carbs: {best_ch:.4f} | Alpha Lento: {best_len:.4f}")
        print(f"      [MÉTRICAS TEST] RMSE={rmse_v:.1f} | MAE={mae_v:.1f} | R2={r2_v:.2f} | MARD={mard_v:.1f}% | Clarke={clarke_v:.1f}%")
        
    # Concatenar todos los históricos de predicciones en un único tablón masivo de la población
    df_predicciones_totales = pd.concat(lista_df_predicciones, ignore_index=True)
    
    reporte_final = {
        'sujetos_evaluados': list(sujetos_unicos),
        'rmse_m': np.mean(resultados_rmse),
        'mae_m': np.mean(resultados_mae),
        'r2_m': np.mean(resultados_r2),
        'mard_m': np.mean(resultados_mard),
        'clarke_m': np.mean(resultados_clarke_ab),
        
        # Almacenar el diccionario de hiperparámetros del último pliegue procesado
        'mejores_parametros': mejores_parametros_hibridos
    }

    print("\n================ VALIDACIÓN LOSO COMPLETADA ================")
    print(f"Métricas Promedio Colectivas -> "
          f"RMSE: {reporte_final['rmse_m']:.1f} | "
          f"MAE: {reporte_final['mae_m']:.1f} | "
          f"R²: {reporte_final['r2_m']:.2f} | "
          f"MARD: {reporte_final['mard_m']:.1f}% | "
          f"Clarke A+B: {reporte_final['clarke_m']:.1f}%")
    
    # Devolvemos el reporte, las trayectorias, y las estructuras completas de modelos y matrices para SHAP
    return reporte_final, df_predicciones_totales, modelos_poblacionales_loso, matrices_train_poblacionales_loso


In [ ]:
#================================================
# IMPLEMENTACIÓN VALIDACIÓN LOSO CON OPTIMIZACIÓN
#================================================

# Inicializamos contenedores globales dedicados para la persistencia del entorno poblacional
modelos_poblacionales_loso = {}
matrices_train_poblacionales_loso = {}
diccionario_curvas_poblacionales = None # Almacenará el tablón consolidado df_preds_global

print("Arrancando validación cruzada poblacional LOSO (Leave-One-Subject-Out)...")

# Desempaquetamos los 4 elementos devueltos por la función optimizada
(
    reporte_global_hibrido, 
    df_preds_global, 
    modelos_poblacionales_loso, 
    matrices_train_poblacionales_loso
) = ejecutar_loso_con_optimizacion_fisiologica(
    df_final_eng=df_final, 
    num_trials_optuna=15
)

# Imprimir el resultado definitivo listo para las tablas de tu manuscrito
print("\n>>> RESULTADOS CONSOLIDADOS DEL MODELO GLOBAL (ALPHAS OPTIMIZADOS) <<<")
print(f"RMSE Promedio: {reporte_global_hibrido['rmse_m']:.2f} mg/dL")
print(f"MAE Promedio:  {reporte_global_hibrido['mae_m']:.2f} mg/dL")
print(f"R² Promedio:   {reporte_global_hibrido['r2_m']:.4f}")
print(f"MARD Promedio: {reporte_global_hibrido['mard_m']:.2f}%")
print(f"Clarke (A+B):  {reporte_global_hibrido['clarke_m']:.2f}%")

# [CORRECCIÓN CRÍTICA] Registramos el diccionario de modelos nativos para que la Fase 3 (SHAP) pueda consumirlos
REGISTRO_MODELOS['general'] = modelos_poblacionales_loso

# Almacenamos el tablón de trayectorias en su variable correspondiente para tus funciones de gráficas
diccionario_curvas_poblacionales = df_preds_global

print("\n[OK] Modelos poblacionales y matrices de entrenamiento registrados en memoria para SHAP.")


In [ ]:
#===================================
# GUARDADO MODELO POBLACIONAL GLOBAL
#===================================

def serializar_entorno_poblacional(
    reporte: Dict[str, Any], 
    curvas: pd.DataFrame, 
    modelos: Dict[str, xgb.XGBRegressor], 
    matrices: Dict[str, pd.DataFrame]
) -> None:
    """
    Serializa y almacena en disco todos los artefactos del modelado poblacional LOSO.

    Guarda de forma estructurada las métricas colectivas, el tablón completo de 
    predicciones de test, los modelos definitivos indexados por fold y las matrices 
    exactas de entrenamiento, asegurando el entorno analítico para la fase de SHAP.

    Args:
        reporte (dict): Diccionario 'reporte_global_hibrido'.
        curvas (pd.DataFrame): Tablón masivo 'df_preds_global'.
        modelos (dict): Diccionario 'modelos_poblacionales_loso'.
        matrices (dict): Diccionario 'matrices_train_poblacionales_loso'.
    """
    # Crear el directorio base por seguridad si no existiera
    os.makedirs(os.path.dirname(METRICAS_POB_PKL_DIR), exist_ok=True)
    
    # 1. Guardar reporte numérico consolidado
    with open(METRICAS_POB_PKL_DIR, 'wb') as f:
        pickle.dump(reporte, f)

    # 2. Guardar tablón masivo de curvas de test (Real vs Predicho de toda la población)
    with open(CURVAS_POB_PKL_DIR, 'wb') as f:
        pickle.dump(curvas, f)
        
    # 3. Guardar objetos binarios de los modelos XGBoost de cada fold (Esencial para SHAP)
    with open(MODELOS_POB_PKL_DIR, 'wb') as f:
        pickle.dump(modelos, f)
        
    # 4. Guardar matrices de entrenamiento exactas que vio cada árbol (Esencial para SHAP)
    with open(MATRICES_POB_PKL_DIR, 'wb') as f:
        pickle.dump(matrices, f)

    print("[OK] ¡Entorno poblacional completo serializado y guardado con éxito!")

In [ ]:
# --- INVOCACIÓN DEL GUARDADO ---
serializar_entorno_poblacional(
    reporte=reporte_global_hibrido,
    curvas=df_preds_global,
    modelos=modelos_poblacionales_loso,
    matrices=matrices_train_poblacionales_loso
)

In [ ]:
#================================
# CARGA MODELO POBLACIONAL GLOBAL
#================================

def cargar_entorno_poblacional_explicativo() -> Tuple[Dict[str, Any], pd.DataFrame, Dict[str, xgb.XGBRegressor], Dict[str, pd.DataFrame]]:
    """Recupera desde disco todos los artefactos necesarios para la fase de SHAP Poblacional.

    Garantiza que los modelos nativos de XGBoost y las dimensiones de sus matrices 
    de entrenamiento clonadas se carguen de forma idéntica sin desalineaciones.

    Returns:
        tuple: (reporte_pob, curvas_pob, modelos_pob, matrices_pob)
    """
    # 1. Cargar reporte de métricas colectivas
    with open(METRICAS_POB_PKL_DIR, 'rb') as f:
        reporte = pickle.load(f)

    # 2. Cargar el tablón unificado de trayectorias continuas
    with open(CURVAS_POB_PKL_DIR, 'rb') as f:
        curvas = pickle.load(f)
        
    # 3. Cargar los objetos de los modelos por fold
    with open(MODELOS_POB_PKL_DIR, 'rb') as f:
        modelos = pickle.load(f)
        
    # 4. Cargar las matrices de entrenamiento correspondientes
    with open(MATRICES_POB_PKL_DIR, 'rb') as f:
        matrices = pickle.load(f)

    print("[OK] Entorno poblacional recuperado. Modelos y espacios de variables listos para SHAP.")
    return reporte, curvas, modelos, matrices

In [ ]:
# --- INVOCACIÓN DE LA CARGA ---
(
    reporte_global_hibrido, 
    df_preds_global, 
    modelos_poblacionales_loso, 
    matrices_train_poblacionales_loso
) = cargar_entorno_poblacional_explicativo()

# Dejamos los modelos registrados para que SHAP los encuentre indexando por sujeto ciego
REGISTRO_MODELOS['general'] = modelos_poblacionales_loso
diccionario_curvas_poblacionales = df_preds_global

**Modificación para modelizar por Grupo Metabólico:** 

Para entrenar y evaluar los modelos segmentados por grupo metabólico (no diaéticos vs prediabéticos vs diabéticos), sólo tha que filtrar el df_final antes de pasarlo a las matrices. Por ejemplo:

In [ ]:
#=================================================================
# HELPER DE GUARDADO Y CARGA DE MODELOS POBLACIONALES POR COHORTES
#=================================================================

def serializar_entorno_poblacional_cohortes(
    nombre_cohorte: str,
    reporte: Dict[str, Any], 
    curvas: pd.DataFrame, 
    modelos: Dict[str, xgb.XGBRegressor], 
    matrices: Dict[str, pd.DataFrame]
) -> None:
    """
    Serializa y almacena en disco los artefactos de una cohorte metabólica específica.

    Genera archivos binarios independientes utilizando el nombre de la cohorte 
    como sufijo para evitar que se sobreescriban los experimentos entre grupos.

    Args:
        nombre_cohorte (str): Etiqueta identificativa de la cohorte (ej: 'diabeticos').
        reporte (dict): Diccionario de métricas.
        curvas (pd.DataFrame): Tablón de predicciones.
        modelos (dict): Diccionario de modelos por fold.
        matrices (dict): Diccionario de matrices X_train.
    """
    # Construcción dinámica de rutas basadas en el nombre de la cohorte
    metricas_dir = os.path.join(MODELIZACION_DIR, f"reportes_loso_{nombre_cohorte}.pkl")
    curvas_dir = os.path.join(MODELIZACION_DIR, f"curvas_loso_{nombre_cohorte}.pkl")
    modelos_dir = os.path.join(MODELIZACION_DIR, f"modelos_xgboost_{nombre_cohorte}.pkl")
    matrices_dir = os.path.join(MODELIZACION_DIR, f"matrices_train_{nombre_cohorte}.pkl")
    
    os.makedirs(os.path.dirname(metricas_dir), exist_ok=True)
    
    # Volcado de datos binarios
    with open(metricas_dir, 'wb') as f: pickle.dump(reporte, f)
    with open(curvas_dir, 'wb') as f: pickle.dump(curvas, f)
    with open(modelos_dir, 'wb') as f: pickle.dump(modelos, f)
    with open(matrices_dir, 'wb') as f: pickle.dump(matrices, f)

    print(f"[OK] ¡Entorno de la cohorte '{nombre_cohorte}' guardado con éxito en disco!")


def cargar_entorno_poblacional_cohortes(
    nombre_cohorte: str
) -> Tuple[Dict[str, Any], pd.DataFrame, Dict[str, xgb.XGBRegressor], Dict[str, pd.DataFrame]]:
    """
    Recupera desde disco todos los artefactos necesarios para la fase de SHAP de una cohorte.

    Garantiza que los modelos nativos de XGBoost y las dimensiones de sus matrices 
    de entrenamiento clonadas se carguen de forma idéntica sin desalineaciones.

    Args:
        nombre_cohorte (str): Etiqueta identificativa de la cohorte (ej: 'diabeticos').

    Returns:
        tuple: (reporte, curvas, modelos, matrices)
    """
    # Reconstrucción exacta de las rutas binarias específicas de la cohorte
    metricas_dir = os.path.join(MODELIZACION_DIR, f"reportes_loso_{nombre_cohorte}.pkl")
    curvas_dir = os.path.join(MODELIZACION_DIR, f"curvas_loso_{nombre_cohorte}.pkl")
    modelos_dir = os.path.join(MODELIZACION_DIR, f"modelos_xgboost_{nombre_cohorte}.pkl")
    matrices_dir = os.path.join(MODELIZACION_DIR, f"matrices_train_{nombre_cohorte}.pkl")
    
    # Lectura secuencial de los archivos binarios pkl
    with open(metricas_dir, 'rb') as f: reporte = pickle.load(f)
    with open(curvas_dir, 'rb') as f: curvas = pickle.load(f)
    with open(modelos_dir, 'rb') as f: modelos = pickle.load(f)
    with open(matrices_dir, 'rb') as f: matrices = pickle.load(f)

    print(f"[OK] Entorno de la cohorte '{nombre_cohorte}' recuperado con éxito.")
    return reporte, curvas, modelos, matrices

In [ ]:
#===================================================================
# IMPLEMENTACIÓN VALIDACIÓN LOSO CON OPTIMIZACIÓN: MODELO DIABÉTICOS
#===================================================================

# 1. Filtrar datos exclusivos de la cohorte
df_diabeticos = df_final[df_final['grupo_metabolico'] == 'Diabético']

# 2. Inicializar contenedores de resguardo en memoria
modelos_diab_loso = {}
matrices_train_diab_loso = {}

print("Arrancando validación LOSO exclusiva para pacientes Diabéticos...")

# 3. Lanzar la validación con los 4 desempaquetados obligatorios
(
    reporte_diab_hibrido, 
    df_preds_diab, 
    modelos_diab_loso, 
    matrices_train_diab_loso
) = ejecutar_loso_con_optimizacion_fisiologica(
    df_final_eng=df_diabeticos, 
    num_trials_optuna=15
)

# 4. Imprimir resultados consolidados
print("\n>>> RESULTADOS CONSOLIDADOS DEL MODELO DIABÉTICOS (ALPHAS OPTIMIZADOS) <<<")
print(f"RMSE Promedio: {reporte_diab_hibrido['rmse_m']:.2f} mg/dL")
print(f"MAE Promedio:  {reporte_diab_hibrido['mae_m']:.2f} mg/dL")
print(f"R² Promedio:   {reporte_diab_hibrido['r2_m']:.4f}")
print(f"MARD Promedio: {reporte_diab_hibrido['mard_m']:.2f}%")
print(f"Clarke (A+B):  {reporte_diab_hibrido['clarke_m']:.2f}%")

# 5. Registrar en las variables globales de memoria del enrutador
if REGISTRO_MODELOS['metabolico'] is None:
    REGISTRO_MODELOS['metabolico'] = {}
REGISTRO_MODELOS['metabolico']['Diabético'] = modelos_diab_loso

In [ ]:
# ==============================================
# GUARDADO MODELO POBLACIONAL COHORTE DIABÉTICOS
# ==============================================

# 6. INVOCACIÓN DE LA FUNCIÓN DE GUARDADO COMPACTA
serializar_entorno_poblacional_cohortes(
    nombre_cohorte="diabeticos",
    reporte=reporte_diab_hibrido,
    curvas=df_preds_diab,
    modelos=modelos_diab_loso,
    matrices=matrices_train_diab_loso
)


In [ ]:
# ===========================================
# CARGA MODELO POBLACIONAL COHORTE DIABÉTICOS
# ===========================================
(
    reporte_diab_hibrido, 
    df_preds_diab, 
    modelos_diab_recuperados, 
    matrices_diab_recuperadas
) = cargar_entorno_poblacional_cohortes(nombre_cohorte="diabeticos")

# Aseguramos la existencia del subdiccionario 'metabolico'
if 'metabolico' not in REGISTRO_MODELOS or REGISTRO_MODELOS['metabolico'] is None:
    REGISTRO_MODELOS['metabolico'] = {}

# Registramos los modelos nativos recuperados para que SHAP Poblacional pueda localizarlos por fold
REGISTRO_MODELOS['metabolico']['Diabético'] = modelos_diab_recuperados

# Opcional: Si necesitas registrar el tablón de curvas continuas de test para tus gráficas:
diccionario_curvas_poblacionales_diab = df_preds_diab

In [ ]:
#======================================================================
# IMPLEMENTACIÓN VALIDACIÓN LOSO CON OPTIMIZACIÓN: MODELO NO DIABÉTICOS
#======================================================================

df_no_diabeticos = df_final[df_final['grupo_metabolico'] == 'No Diabético']

# 1. Inicializamos contenedores locales dedicados para resguardar esta cohorte en memoria
modelos_no_diab_loso = {}
matrices_train_no_diab_loso = {}

print("Filtrando cohorte y arrancando validación LOSO exclusiva para pacientes No Diabéticos...")

# 2. Desempaquetamos los 4 elementos devueltos por la función optimizada
(
    reporte_no_diab_hibrido, 
    df_preds_no_diab, 
    modelos_no_diab_loso, 
    matrices_train_no_diab_loso
) = ejecutar_loso_con_optimizacion_fisiologica(
    df_final_eng=df_no_diabeticos, 
    num_trials_optuna=15
)

# 3. Imprimir los resultados consolidados de la cohorte para tu manuscrito
print("\n>>> RESULTADOS CONSOLIDADOS DEL MODELO NO DIABÉTICOS (ALPHAS OPTIMIZADOS) <<<")
print(f"RMSE Promedio: {reporte_no_diab_hibrido['rmse_m']:.2f} mg/dL")
print(f"MAE Promedio:  {reporte_no_diab_hibrido['mae_m']:.2f} mg/dL")
print(f"R² Promedio:   {reporte_no_diab_hibrido['r2_m']:.4f}")
print(f"MARD Promedio: {reporte_no_diab_hibrido['mard_m']:.2f}%")
print(f"Clarke (A+B):  {reporte_no_diab_hibrido['clarke_m']:.2f}%")

# 4. Control de inicialización y enrutamiento inteligente de estructuras
if 'metabolico' not in REGISTRO_MODELOS or REGISTRO_MODELOS['metabolico'] is None:
    REGISTRO_MODELOS['metabolico'] = {}

# Registramos el diccionario de modelos nativos por fold para que la Fase 3 (SHAP) los localice
REGISTRO_MODELOS['metabolico']['No Diabético'] = modelos_no_diab_loso

In [ ]:
# =================================================
# GUARDADO MODELO POBLACIONAL COHORTE NO DIABÉTICOS
# =================================================

# 5. INVOCACIÓN DE LA FUNCIÓN DE GUARDADO COMPACTA REUTILIZABLE
serializar_entorno_poblacional_cohortes(
    nombre_cohorte="no_diabeticos",
    reporte=reporte_no_diab_hibrido,
    curvas=df_preds_no_diab,
    modelos=modelos_no_diab_loso,
    matrices=matrices_train_no_diab_loso
)

In [ ]:
# ==============================================
# CARGA MODELO POBLACIONAL COHORTE NO DIABÉTICOS
# ==============================================

(
    reporte_no_diab_hibrido, 
    df_preds_no_diab, 
    modelos_no_diab_recuperados, 
    matrices_no_diab_recuperadas
) = cargar_entorno_poblacional_cohortes(nombre_cohorte="no_diabeticos")

REGISTRO_MODELOS['metabolico']['No Diabético'] = modelos_no_diab_recuperados

In [ ]:
#======================================================================
# IMPLEMENTACIÓN VALIDACIÓN LOSO CON OPTIMIZACIÓN: MODELO PREDIABÉTICOS
#======================================================================

df_prediabeticos = df_final[df_final['grupo_metabolico'] == 'Prediabético']

# 1. Inicializamos contenedores locales dedicados para resguardar esta cohorte en memoria
modelos_prediab_loso = {}
matrices_train_prediab_loso = {}

print("Filtrando cohorte y arrancando validación LOSO exclusiva para pacientes Prediabéticos...")

# 2. Desempaquetamos los 4 elementos devueltos por la función optimizada
(
    reporte_prediab_hibrido, 
    df_preds_prediab, 
    modelos_prediab_loso, 
    matrices_train_prediab_loso
) = ejecutar_loso_con_optimizacion_fisiologica(
    df_final_eng=df_prediabeticos, 
    num_trials_optuna=15
)

# 3. Imprimir los resultados consolidados de la cohorte para tu manuscrito
print("\n>>> RESULTADOS CONSOLIDADOS DEL MODELO PREDIABÉTICOS (ALPHAS OPTIMIZADOS) <<<")
print(f"RMSE Promedio: {reporte_prediab_hibrido['rmse_m']:.2f} mg/dL")
print(f"MAE Promedio:  {reporte_prediab_hibrido['mae_m']:.2f} mg/dL")
print(f"R² Promedio:   {reporte_prediab_hibrido['r2_m']:.4f}")
print(f"MARD Promedio: {reporte_prediab_hibrido['mard_m']:.2f}%")
print(f"Clarke (A+B):  {reporte_prediab_hibrido['clarke_m']:.2f}%")

# 4. Control de inicialización y enrutamiento inteligente de estructuras
if 'metabolico' not in REGISTRO_MODELOS or REGISTRO_MODELOS['metabolico'] is None:
    REGISTRO_MODELOS['metabolico'] = {}

# Registramos el diccionario de modelos nativos por fold para que la Fase 3 (SHAP) los localice
REGISTRO_MODELOS['metabolico']['Prediabético'] = modelos_prediab_loso

In [ ]:
# Código de inspección rápida de categorías reales
print("--- Categorías únicas detectadas en tu dataset ---")
print(df_final['grupo_metabolico'].unique().tolist())
print("\n--- Tipos de datos oficiales de la columna ---")
print(df_final['grupo_metabolico'].dtype)

In [ ]:
# =================================================
# GUARDADO MODELO POBLACIONAL COHORTE PREDIABÉTICOS
# =================================================

# 5. INVOCACIÓN DE LA FUNCIÓN DE GUARDADO COMPACTA REUTILIZABLE
serializar_entorno_poblacional_cohortes(
    nombre_cohorte="prediabeticos",
    reporte=reporte_prediab_hibrido,
    curvas=df_preds_prediab,
    modelos=modelos_prediab_loso,
    matrices=matrices_train_prediab_loso
)

In [ ]:
# ==============================================
# CARGA MODELO POBLACIONAL COHORTE PREDIABÉTICOS
# ==============================================

(
    reporte_prediab_hibrido, 
    df_preds_prediab, 
    modelos_prediab_recuperados, 
    matrices_prediab_recuperadas
) = cargar_entorno_poblacional_cohortes(nombre_cohorte="prediabeticos")

REGISTRO_MODELOS['metabolico']['Prediabético'] = modelos_prediab_recuperados

#### D2.3 Validación Walk-Forward (Time-Series Split) - Modelos individuales

Queremos evaluar el modelo a través de múltiples pliegues (folds) temporales:
- Fold 1: Entrena con los días 1-4 → Evalúa en el día 5.
- Fold 2: Entrena con los días 1-5 → Evalúa en el día 6.
- Fold 3: Entrena con los días 1-6 → Evalúa en el día 7.

In [ ]:
#===============================================================================
# VALIDACIÓN WALK-FORWARD CON OPTIMIZACIÓN DE HIPERPARÁMETROS (OPTUNA + XGBOOST)
#===============================================================================

def ejecutar_walk_forward_individual_hibrido(
    df_sujeto: pd.DataFrame, 
    num_folds: int = 5, 
    num_trials_optuna: int = 15
) -> Tuple[Dict[str, Any], pd.DataFrame, xgb.XGBRegressor, pd.DataFrame]:
    """
    Ejecuta una validación Walk-Forward individual optimizando alphas e hiperparámetros.

    Segmenta cronológicamente los registros de un único usuario. En cada fold 
    temporal, optimiza de forma bayesiana las tasas de absorción y los parámetros 
    del árbol basándose en el pasado, preserva la inercia biológica en el test, 
    y emite las métricas junto al último modelo entrenado para la fase de SHAP.

    Args:
        df_sujeto (pd.DataFrame): DataFrame con la ingeniería base del sujeto.
        num_folds (int, optional): Número de particiones temporales. Defaults to 5.
        num_trials_optuna (int, optional): Iteraciones de búsqueda de Optuna. Defaults to 15.

    Returns:
        Tuple[Dict[str, Any], pd.DataFrame, xgb.XGBRegressor, pd.DataFrame]:
            - reporte_individual (Dict): Métricas promedio y mejores parámetros.
            - df_predicciones_individuales (pd.DataFrame): Serie continua ['time', 'Real', 'Predicho'].
            - modelo_final_temporal (xgb.XGBRegressor): Último modelo entrenado del fold final.
            - X_train (pd.DataFrame): Matriz exacta de entrenamiento que vio el último modelo.
    """
    # 1. Limpieza inicial estricta del horizonte futuro y ordenación cronológica
    df_limpio = df_sujeto.dropna(subset=['target_glucose_60m']).copy()
    df_limpio = df_limpio.sort_values(by='time').reset_index(drop=True)
    epsilon = 1e-5
    
    # Inicialización de contenedores para almacenar los resultados de cada fold temporal
    resultados_rmse: List[float] = []
    resultados_mae: List[float] = []
    resultados_r2: List[float] = []
    resultados_mard: List[float] = []
    resultados_clarke_ab: List[float] = []
    
    # Lista para acumular los trozos de futuro ciego evaluados en cada pliegue
    lista_df_predicciones: List[pd.DataFrame] = []
    
    # Configurar el particionador temporal de Scikit-Learn
    tscv = TimeSeriesSplit(n_splits=num_folds)
    
    # Eliminamos variables constantes e identificadores para el modelo individual
    cols_drop = [
        'time', 'target_glucose_60m', 'meal_type', 'image_path',
        'user_id', 'grupo_metabolico', 'hba1c_inicial'
    ]
    
    # Definimos variables externas para extraer el último modelo entrenado del bucle
    ultimo_modelo = None
    ultima_matriz_x = None
    
    # Bucle cronológico del Walk-Forward
    for fold, (train_index, test_index) in enumerate(tscv.split(df_limpio)):
        df_train_raw = df_limpio.iloc[train_index].copy()
        
        # --- Función de Optimización Bayesiana Interna para este Bloque Histórico ---
        def objetivo_interno_individual(trial: optuna.Trial) -> float:
            # Optuna sugiere las alphas del trial (parámetros biológicos in-sample)
            alpha_ch = trial.suggest_float('alpha_ch', 0.005, 0.05)
            alpha_lento = trial.suggest_float('alpha_lento', 0.002, 0.02)
            
            # Optuna sugiere hiperparámetros del XGBoost
            params_xgb = {
                'n_estimators': trial.suggest_int('n_estimators', 50, 200),
                'max_depth': trial.suggest_int('max_depth', 3, 6),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
                'subsample': trial.suggest_float('subsample', 0.7, 1.0),
                'tree_method': 'hist', 
                'enable_categorical': True, 
                'n_jobs': -1
            }
            
            # Uso directo del bucle nativo optimizado con numpy arrays
            ch_tr = _bucle_tanque_nativo(df_train_raw['carbs'].fillna(0.0).to_numpy(), alpha_ch)
            fat_tr = _bucle_tanque_nativo(df_train_raw['fat'].fillna(0.0).to_numpy(), alpha_lento)
            pro_tr = _bucle_tanque_nativo(df_train_raw['protein'].fillna(0.0).to_numpy(), alpha_lento)
            fib_tr = _bucle_tanque_nativo(df_train_raw['fiber'].fillna(0.0).to_numpy(), alpha_lento)
            
            # Construir la matriz X temporal agregando las columnas dinámicas calculadas
            X_train_tmp = df_train_raw.drop(columns=cols_drop, errors='ignore').copy()
            X_train_tmp['CH_efectivo'] = ch_tr
            X_train_tmp['ratio_complejidad_dinamico'] = (fat_tr + pro_tr) / (ch_tr + epsilon)
            X_train_tmp['indice_retardo_gastrico_dinamico'] = (fat_tr + fib_tr) / (ch_tr + epsilon)
            X_train_tmp['interaccion_CH_HbA1c'] = ch_tr * df_train_raw['hba1c_inicial'].astype(float)

            y_train_tmp = df_train_raw['target_glucose_60m']
            
            # Validación temporal interna rápida (80% entrenamiento histórico, 20% validación histórica)
            corte_interno = int(len(X_train_tmp) * 0.8)
            modelo_int = xgb.XGBRegressor(**params_xgb)
            modelo_int.fit(X_train_tmp.iloc[:corte_interno], y_train_tmp.iloc[:corte_interno])
            
            preds_val = modelo_int.predict(X_train_tmp.iloc[corte_interno:])
            return float(np.sqrt(mean_squared_error(y_train_tmp.iloc[corte_interno:], preds_val)))

        # Ejecutar el estudio de optimización sobre el pasado del sujeto
        optuna.logging.set_verbosity(optuna.logging.WARNING)
        estudio = optuna.create_study(direction="minimize")
        estudio.optimize(objetivo_interno_individual, n_trials=num_trials_optuna)
        mejores_params = estudio.best_params
        
        # --- [MEMORIA TEMPORAL ANTES DE DIVIDIR] ---
        best_ch = mejores_params['alpha_ch']
        best_len = mejores_params['alpha_lento']
        
        # Calculamos los tanques sobre la serie completa para preservar la inercia en la frontera
        ch_completo = _bucle_tanque_nativo(df_limpio['carbs'].fillna(0.0).to_numpy(), best_ch)
        fat_completo = _bucle_tanque_nativo(df_limpio['fat'].fillna(0.0).to_numpy(), best_len)
        pro_completo = _bucle_tanque_nativo(df_limpio['protein'].fillna(0.0).to_numpy(), best_len)
        fib_completo = _bucle_tanque_nativo(df_limpio['fiber'].fillna(0.0).to_numpy(), best_len)
        
        # Inyectar las variables biológicas optimizadas sobre la totalidad del histórico del sujeto
        df_bloque_completo = df_limpio.copy()
        df_bloque_completo['CH_efectivo'] = ch_completo
        df_bloque_completo['ratio_complejidad_dinamico'] = (fat_completo + pro_completo) / (ch_completo + epsilon)
        df_bloque_completo['indice_retardo_gastrico_dinamico'] = (fat_completo + fib_completo) / (ch_completo + epsilon)
        df_bloque_completo['interaccion_CH_HbA1c'] = ch_completo * df_bloque_completo['hba1c_inicial'].astype(float)
        
        # Extraer ahora sí los conjuntos definitivos perfectamente alineados en sus índices
        df_train_final = df_bloque_completo.iloc[train_index]
        df_test_final = df_bloque_completo.iloc[test_index]
        
        X_train = df_train_final.drop(columns=cols_drop, errors='ignore')
        y_train = df_train_final['target_glucose_60m']
        X_test = df_test_final.drop(columns=cols_drop, errors='ignore')
        y_test = df_test_final['target_glucose_60m']
        
        # Entrenar el XGBoost final del pliegue con los hiperparámetros ganadores
        params_xgb_ganador = {k: v for k, v in mejores_params.items() if k not in ['alpha_ch', 'alpha_lento']}
        params_xgb_ganador.update({'tree_method': 'hist', 'enable_categorical': True, 'n_jobs': -1})
        
        modelo_final_temporal = xgb.XGBRegressor(**params_xgb_ganador)
        modelo_final_temporal.fit(X_train, y_train)
        
        # --- EVALUACIÓN EN EL FUTURO CIEGO DEL INDIVIDUO ---
        predicciones = modelo_final_temporal.predict(X_test)
        y_test_np = y_test.to_numpy()
        
        # Acumular métricas resultantes
        resultados_rmse.append(np.sqrt(mean_squared_error(y_test, predicciones)))
        resultados_mae.append(mean_absolute_error(y_test, predicciones))
        resultados_r2.append(r2_score(y_test, predicciones))
        resultados_mard.append(calcular_mard(y_test_np, predicciones))
        resultados_clarke_ab.append(analizar_zonas_clarke(y_test_np, predicciones)['Zona_A_Plus_B'])
        
        # Almacenar las predicciones de esta ventana de futuro ciego alineada
        df_preds_fold = pd.DataFrame({
            'time': df_test_final['time'],
            'Real': y_test_np,
            'Predicho': predicciones
        })
        lista_df_predicciones.append(df_preds_fold)
        
        # Guardamos en la variable externa el modelo y matriz del fold final
        ultimo_modelo = modelo_final_temporal
        ultima_matriz_x = X_train
        
    # Concatenar todos los bloques de futuro evaluados en una serie de tiempo continua
    df_predicciones_individuales = pd.concat(lista_df_predicciones, ignore_index=True)
    
    # 5. Retornar el informe unificado del paciente
    reporte_individual = {
        'rmse_promedio': np.mean(resultados_rmse),
        'mae_promedio': np.mean(resultados_mae),
        'r2_promedio': np.mean(resultados_r2),
        'mard_promedio': np.mean(resultados_mard),
        'clarke_ab_promedio': np.mean(resultados_clarke_ab),
        'mejores_parametros': mejores_params
    }
    
    return reporte_individual, df_predicciones_individuales, ultimo_modelo, ultima_matriz_x

In [ ]:
#========================================================
# IMPLEMENTACIÓN VALIDACIÓN WALK-FORWARD CON OPTIMIZACIÓN
#========================================================

# Contenedores para almacenar el rendimiento, curvas, modelos y matrices de entrenamiento
reportes_hibridos_individuales = {}
diccionario_curvas_individuales = {}

# Mismas dimensiones SHAP: Diccionarios dedicados para almacenar los objetos finales de cada usuario
modelos_finales_individuales = {}
matrices_train_individuales = {}

# Extraer la lista de usuarios únicos del dataset final
todos_los_usuarios = df_final['user_id'].unique()

print(f"Arrancando entrenamiento de {len(todos_los_usuarios)} Modelos Individuales mediante Walk-Forward...")

for usuario in todos_los_usuarios:
    # 1. Aislar los datos exclusivos del sujeto actual
    df_usuario = df_final[df_final['user_id'] == usuario].copy()
    
    # 2. Correr la validación temporal Walk-Forward (ej: 5 pliegues cronológicos)
    # Recibe los 4 elementos de retorno correspondientes
    reporte_individual, df_curvas_ind, mod_final, x_train_final = ejecutar_walk_forward_individual_hibrido(
        df_sujeto=df_usuario, 
        num_folds=5, 
        num_trials_optuna=12  # Un número equilibrado para evitar sobrecargar la CPU
    )
    
    # 3. Almacenar los resultados indexados por su ID
    reportes_hibridos_individuales[usuario] = reporte_individual
    diccionario_curvas_individuales[usuario] = df_curvas_ind
    
    # Almacenamos el modelo y su espacio geométrico para consumirlos directamente en la Fase 3 con SHAP
    modelos_finales_individuales[usuario] = mod_final
    matrices_train_individuales[usuario] = x_train_final
    
    print(
        f"   [OK] {usuario} finalizado -> "
        f"RMSE: {reporte_individual['rmse_promedio']:.1f} mg/dL | "
        f"MAE: {reporte_individual['mae_promedio']:.1f} mg/dL | "
        f"R²: {reporte_individual['r2_promedio']:.2f} | "
        f"MARD: {reporte_individual['mard_promedio']:.1f}% | "
        f"Clarke A+B: {reporte_individual['clarke_ab_promedio']:.1f}%"
    )

# Convertir el diccionario final a un DataFrame para análisis
df_benchmarking_individual = pd.DataFrame(reportes_hibridos_individuales).T

# Separar la columna de parámetros para que queden solo las 5 métricas numéricas
df_tabla_metricas = df_benchmarking_individual.drop(columns=['mejores_parametros'], errors='ignore').copy()

# Nombrar las columnas del DataFrame de forma elegante para su posterior análisis o exportación
df_tabla_metricas.columns = ['RMSE Promedio', 'MAE Promedio', 'R² Promedio', 'MARD Promedio', 'Clarke A+B Promedio']

print("\n>>> PIPELINE DE MODELADO INDIVIDUAL COMPLETADO CON ÉXITO <<<")

In [ ]:
REGISTRO_MODELOS['individual'] = modelos_finales_individuales

Hacemos un guardado de los resultados del modelo individual para no tener que correr el modelo cada vez que abramos el notebook

In [ ]:
#=================================
# GUARDADO DE MODELOS INDIVIDUALES
#=================================

# Suponiendo que tenemos definidas estas constantes en tu entorno:
# METRICAS_PKL_DIR = "ruta/metricas_individuales.pkl"
# CURVAS_PKL_DIR = "ruta/curvas_individuales.pkl"
# MODELOS_PKL_DIR = "ruta/modelos_xgboost_individuales.pkl"
# MATRICES_PKL_DIR = "ruta/matrices_train_individuales.pkl"

def serializar_entorno_individual(
    reportes: dict, 
    curvas: dict, 
    modelos: dict, 
    matrices: dict
) -> None:
    """Serializa y almacena en disco todos los artefactos del modelado individual.

    Guarda de forma estructurada tanto las métricas clínicas y trayectorias de test,
    como los objetos nativos de XGBoost y sus espacios de variables de entrenamiento,
    garantizando la persistencia necesaria para la posterior fase de SHAP.

    Args:
        reportes (dict): Diccionario 'reportes_hibridos_individuales'.
        curvas (dict): Diccionario 'diccionario_curvas_individuales'.
        modelos (dict): Diccionario 'modelos_finales_individuales'.
        matrices (dict): Diccionario 'matrices_train_individuales'.

    Recommendations:
        - Asegúrese de que los directorios de destino tengan permisos de escritura.
        - No altere el orden de los diccionarios al recargarlos en la Fase 3.
    """
    # 1. Guardar el diccionario de reportes numéricos y parámetros de Optuna
    with open(METRICAS_PKL_DIR, 'wb') as f:
        pickle.dump(reportes, f)

    # 2. Guardar el diccionario de curvas y trayectorias continuas de test
    with open(CURVAS_PKL_DIR, 'wb') as f:
        pickle.dump(curvas, f)
        
    # 3. [CRÍTICO PARA SHAP] Guardar los objetos de los modelos entrenados
    with open(MODELOS_PKL_DIR, 'wb') as f:
        pickle.dump(modelos, f)
        
    # 4. [CRÍTICO PARA SHAP] Guardar las matrices exactas con las que entrenó cada modelo
    with open(MATRICES_PKL_DIR, 'wb') as f:
        pickle.dump(matrices, f)

    print("[OK] ¡Entorno completo serializado de forma segura! Modelos y Matrices listos para SHAP.")

# Invocación directa para guardar todo
serializar_entorno_individual(
    reportes=reportes_hibridos_individuales,
    curvas=diccionario_curvas_individuales,
    modelos=modelos_finales_individuales,
    matrices=matrices_train_individuales
)

**Carga de resultados de un modelo guardado** (Sin entrenar nada):

Comentar toda la celda del bucle de entrenamiento y ejecutar simplemente el siguiente bloque para revivir las variables.

In [ ]:
#==============================
# CARGA DE MODELOS INDIVIDUALES
#==============================

def cargar_entorno_individual_explicativo() -> tuple:
    """Carga desde disco todos los artefactos necesarios para la fase de SHAP.

    Recupera instantáneamente los reportes de métricas, las curvas de predicción,
    los objetos nativos de los modelos XGBoost y sus respectivas matrices de 
    entrenamiento clonadas exactamente en el mismo espacio dimensional.

    Returns:
        tuple: Contiene los cuatro diccionarios cargados en orden:
            (reportes, curvas, modelos, matrices)
            
    Recommendations:
        - Asegúrese de que las rutas relativas o absolutas de los archivos .pkl 
          coincidan exactamente con las utilizadas en la función de serialización.
    """
    # 1. Cargar reportes de rendimiento e hiperparámetros de Optuna
    with open(METRICAS_PKL_DIR, 'rb') as f:
        reportes = pickle.load(f)

    # 2. Cargar curvas y trayectorias continuas de test para tus gráficas temporalizadas
    with open(CURVAS_PKL_DIR, 'rb') as f:
        curvas = pickle.load(f)
        
    # 3. Cargar los objetos de los modelos XGBoost entrenados (Esencial para SHAP)
    with open(MODELOS_PKL_DIR, 'rb') as f:
        modelos = pickle.load(f)
        
    # 4. Cargar las matrices exactas de características (Esencial para SHAP)
    with open(MATRICES_PKL_DIR, 'rb') as f:
        matrices = pickle.load(f)

    print("[OK] Entorno explicativo recuperado con éxito. Modelos y espacios dimensionales listos.")
    return reportes, curvas, modelos, matrices


# --- INVOCACIÓN Y CONFIGURACIÓN DEL ENRUTADOR ---
(
    reportes_hibridos_individuales, 
    diccionario_curvas_individuales, 
    modelos_finales_individuales, 
    matrices_train_individuales
) = cargar_entorno_individual_explicativo()

# Registramos los modelos nativos en el diccionario centralizado para que 
# el motor de SHAP pueda acceder a ellos indexando por el ID del usuario
REGISTRO_MODELOS['individual'] = modelos_finales_individuales

### D3. Visualización de resultados

In [ ]:
#================================
# GRÁFICAS PREDICCIÓN VS REALIDAD
#================================

def graficar_prediccion_vs_realidad(
    user_id: str,
    tipo_modelo: str,
    ventana_minutos: int = 180,
    fecha_inicio: Optional[str] = None,
    fecha_fin: Optional[str] = None
) -> None:
    """
    Extrae dinámicamente las trayectorias del registro global aplicando enrutamiento clínico.

    Consulta las variables globales de curvas y mapeos de todas las cohortes.
    Recorta la ventana de observación solicitada para un usuario concreto, genera
    un lienzo de alta resolución (500 DPI) y lo exporta a disco liberando la RAM.

    Args:
        user_id (str): Identificador del usuario (ej: 'CGMacros-013').
        tipo_modelo (str): Enfoque del modelo ('individual', 'general' o 'metabolico').
        ventana_minutos (int, optional): Ventana temporal por defecto. Defaults to 180.
        fecha_inicio (Optional[str], optional): Fecha de inicio del filtro. Defaults to None.
        fecha_fin (Optional[str], optional): Fecha de fin del filtro. Defaults to None.

    Returns:
        None: Guarda el gráfico en disco y utiliza el helper para liberar memoria RAM.
    """
    tipo_modelo_limpio = tipo_modelo.lower().strip()
    df_fuente = pd.DataFrame()
    
    # Declarar el acceso a las variables y estructuras del entorno global
    global REGISTRO_MODELOS
    global MAPEO_DIABETES
    global diccionario_curvas_individuales  # Curvas del enfoque individual
    global df_preds_global                  # Curvas del enfoque poblacional general
    global df_preds_diab                    # Curvas cohorte Diabéticos
    global df_preds_no_diab                 # Curvas cohorte No Diabéticos
    global df_preds_prediab                 # Curvas cohorte Prediabéticos
    
    # --- 1. EXTRACCIÓN Y ENRUTAMIENTO INTELIGENTE DE LAS MATRICES DE CURVAS ---
    if tipo_modelo_limpio == 'individual':
        if diccionario_curvas_individuales is None:
            print("[ADVERTENCIA] El diccionario de curvas individuales está vacío. Cargue el archivo binario.")
            return
        if user_id not in diccionario_curvas_individuales:
            raise KeyError(f"El usuario {user_id} no se encuentra en las curvas individuales.")
        df_fuente = diccionario_curvas_individuales[user_id]
        
    elif tipo_modelo_limpio == 'general':
        # Enrutamos hacia el DataFrame global continuo recuperado de la validación LOSO
        if df_preds_global is None:
            print("[ADVERTENCIA] El DataFrame de predicciones globales está vacío.")
            return
        df_fuente = df_preds_global[df_preds_global['user_id'] == user_id]
        
    elif tipo_modelo_limpio == 'metabolico':
        if user_id not in MAPEO_DIABETES:
            raise KeyError(f"El usuario {user_id} no está registrado en el diccionario global MAPEO_DIABETES.")
        
        grupo_clinico = MAPEO_DIABETES[user_id]   # Devuelve: 'Diabético', 'Prediabético' o 'No Diabético'
        
        # Enrutamiento hacia las curvas específicas de cada modelo metabólico independiente
        if grupo_clinico == 'Diabético':
            df_fuente = df_preds_diab[df_preds_diab['user_id'] == user_id] if df_preds_diab is not None else pd.DataFrame()
        elif grupo_clinico in ['Prediabético']:
            df_fuente = df_preds_prediab[df_preds_prediab['user_id'] == user_id] if df_preds_prediab is not None else pd.DataFrame()
        elif grupo_clinico == 'No Diabético':
            df_fuente = df_preds_no_diab[df_preds_no_diab['user_id'] == user_id] if df_preds_no_diab is not None else pd.DataFrame()
            
        if df_fuente.empty:
            print(f"[ADVERTENCIA] No se han encontrado datos de curvas para el usuario {user_id} en la cohorte '{grupo_clinico}'.")
            return
    else:
        raise ValueError("El parámetro 'tipo_modelo' debe ser estrictamente: 'individual', 'general' o 'metabolico'.")

    if df_fuente.empty:
        print(f"[ADVERTENCIA] No se hallaron registros de test para {user_id} en el enfoque '{tipo_modelo_limpio}'.")
        return

    # --- 3. PROCESAMIENTO TEMPORAL Y ALINEACIÓN DE ÍNDICES ---
    df_plot = df_fuente.copy()
    df_plot['time'] = pd.to_datetime(df_plot['time'])
    df_plot = df_plot.sort_values(by='time').reset_index(drop=True)
    
    # --- 4. RECORTE DE LA VENTANA DE OBSERVACIÓN ---
    if fecha_inicio is not None and fecha_fin is not None:
        f_inicio = pd.to_datetime(fecha_inicio)
        f_fin = pd.to_datetime(fecha_fin)
        df_sub = df_plot[(df_plot['time'] >= f_inicio) & (df_plot['time'] <= f_fin)].copy()
        
        if not df_sub.empty:
            minutos_reales = int((df_sub['time'].max() - df_sub['time'].min()).total_seconds() / 60.0)
        else:
            minutos_reales = ventana_minutos
    else:
        df_sub = df_plot.head(ventana_minutos).copy()
        minutos_reales = ventana_minutos

    if df_sub.empty:
        print(f"[ADVERTENCIA] El intervalo de fechas solicitado no contiene datos para {user_id}. Gráfica omitida.")
        return

    # --- 5. LIENZO Y RENDERIZADO CON RESOLUCIÓN PROFESIONAL ---
    fig, ax = plt.subplots(figsize=(15, 6), dpi=500)
    
    # Señal Real (Referencia del sensor)
    ax.plot(
        df_sub['time'], df_sub['Real'], 
        label='Glucosa Real (Dexcom G6)', 
        color="#379bff", linewidth=2.5, linestyle='-'
    )
    
    # Señal Predicha (Salida a t+60m)
    ax.plot(
        df_sub['time'], df_sub['Predicho'], 
        label='Predicción XGBoost (t + 60 min)', 
        color="#6d0c8a", linewidth=2.0, linestyle='--'
    )
    
    # Detalles estéticos
    ax.set_title(f"Trayectoria Glucémica ({tipo_modelo_limpio.capitalize()}): {user_id}", fontsize=14, pad=15, fontweight='bold')
    ax.set_xlabel("Estampa Temporal (Minuto a Minuto)", fontsize=11, labelpad=10)
    ax.set_ylabel("Glucosa Intersticial (mg/dL)", fontsize=11, labelpad=10)
    ax.grid(True, linestyle=':', alpha=0.6, color='gray')
    ax.legend(loc='upper left', fontsize=10, frameon=True)
    plt.setp(ax.get_xticklabels(), rotation=15)
    fig.tight_layout()
    
    # --- 6. ALMACENAMIENTO AUTOMATIZADO EN DISCO ---
    ruta_directorio = PREDICCIONES_DIR
    os.makedirs(ruta_directorio, exist_ok=True)
    
    if fecha_inicio is not None and fecha_fin is not None:
        f_inicio_limpia = str(fecha_inicio).replace("-", "").replace(":", "").replace(" ", "_")
        f_fin_limpia = str(fecha_fin).replace("-", "").replace(":", "").replace(" ", "_")
        cadena_fechas = f"{f_inicio_limpia}_{f_fin_limpia}"
    else:
        cadena_fechas = "inicio"
        
    nombre_archivo = f"{user_id}_{tipo_modelo_limpio}_{minutos_reales}_{cadena_fechas}.png"
    ruta_guardado_completa = os.path.join(ruta_directorio, nombre_archivo)
    
    fig.savefig(ruta_guardado_completa, dpi=500, bbox_inches='tight')
    
    # --- 7. LIBERACIÓN DEL LIENZO MEDIANTE TU HELPER ---
    mostrar_y_liberar_figura(fig)
    print(f"[OK] Gráfica exportada con éxito: {nombre_archivo}")


In [ ]:
#===========================
# GRÁFICAS MÉTRICA DE CLARKE
#===========================

def graficar_cuadricula_clarke_sujeto(
    user_id: str,
    tipo_modelo: str = 'individual'
) -> None:
    """Genera la cuadrícula de Clarke con armonización de color RGB y visibilidad adaptativa.

    Construye un fondo discretizado por malla densa sincronizando los colores de los 
    puntos, textos y regiones con transparencias atenuadas. Aplica un control dinámico 
    que oculta las etiquetas de las zonas que quedan fuera del zoom del paciente.
    """
    global REGISTRO_MODELOS
    global diccionario_curvas_individuales
    global MAPEO_DIABETES
    global CLARKE_DIR

    # ==================================================
    # 1. RECUPERACIÓN AUTOMATIZADA DE TRAYECTORIAS
    # ==================================================
    tipo_modelo_limpio = tipo_modelo.lower().strip()

    if tipo_modelo_limpio == 'individual':
        if 'diccionario_curvas_individuales' not in globals():
            print("[ERROR] El objeto 'diccionario_curvas_individuales' no se encuentra en memoria.")
            return
        df_fuente = diccionario_curvas_individuales[user_id]
    elif tipo_modelo_limpio == 'general':
        df_fuente = REGISTRO_MODELOS['general'][REGISTRO_MODELOS['general']["user_id"] == user_id]
    else:
        grupo_clinico = MAPEO_DIABETES[user_id]
        df_fuente = REGISTRO_MODELOS['metabolico'][grupo_clinico][REGISTRO_MODELOS['metabolico'][grupo_clinico]["user_id"] == user_id]

    if df_fuente.empty:
        print(f"[ADVERTENCIA] No se localizaron trayectorias de test para el sujeto {user_id}.")
        return

    y_real = df_fuente["Real"].to_numpy()
    y_pred = df_fuente["Predicho"].to_numpy()

    # ==================================================
    # 2. CÓMPUTO DEL ZOOM AUTOMÁTICO CUADRADO ADAPTATIVO
    # ==================================================
    max_detectado = max(np.max(y_real), np.max(y_pred))
    limite_max = int(np.ceil(max_detectado + 20))
    limite_max = max(160, limite_max) 

    # ==================================================
    # 3. COMPILACIÓN DE LA REJILLA POR MALLA DENSA LÓGICA
    # ==================================================
    resolucion = 700  
    x = np.linspace(0, limite_max, resolucion)
    y = np.linspace(0, limite_max, resolucion)
    X, Y = np.meshgrid(x, y)
    Z = np.vectorize(clarke_zone_oficial)(X, Y).astype(np.uint8)

    for i in range(resolucion):
        for j in range(resolucion):
            Z[i, j] = clarke_zone_oficial(X[i, j], Y[i, j])

    # Clasificación e índices de acierto
    zonas = np.fromiter(
        (clarke_zone_oficial(r, p) for r, p in zip(y_real, y_pred)),
        dtype=np.int8,
        count=len(y_real)
    )
    conteo = {'A': np.sum(zonas == 0), 'B': np.sum(zonas == 1), 'C': np.sum(zonas == 2), 'D': np.sum(zonas == 3), 'E': np.sum(zonas == 4)}
    total = len(zonas)

    if total == 0:
        pct = {k: 0.0 for k in ['A','B','C','D','E']}
    else:
        conteo = {
            'A': np.sum(zonas == 0),
            'B': np.sum(zonas == 1),
            'C': np.sum(zonas == 2),
            'D': np.sum(zonas == 3),
            'E': np.sum(zonas == 4)
        }
        pct = {k: 100 * v / total for k, v in conteo.items()}

    # ==================================================
    # 4. SISTEMA DE PALETAS DE COLOR AR_MONIZADAS (REQUISITO 3)
    # ==================================================
    # Diccionario central de códigos HEX institucionales (Puntos y textos)
    # 0->A (Verde), 1->B (Amarillo oscuro/Oliva), 2->C (Naranja), 3->D (Rojo), 4->E (Morado)
    dict_colores = {
            0: "#2B580F",
            1: "#699830",
            2: "#aeac37",
            3: "#A86224",
            4: "#660000" 
        }
    
    colores_array = np.array([dict_colores[0], dict_colores[1], dict_colores[2], dict_colores[3], dict_colores[4]])
    cmap = ListedColormap(colores_array)

    # ==================================================
    # 5. CONSTRUCCIÓN DEL LIENZO GRÁFICO ISOMÉTRICO
    # ==================================================
    fig, ax = plt.subplots(figsize=(8, 8), dpi=150)
    
    # Fondo con transparencia alta (alpha=0.08) para lograr un degradado pastel muy suave
    ax.contourf(X, Y, Z, levels=np.arange(-0.5, 5.5, 1), cmap=cmap, alpha=0.08, antialiased=False)
    # Líneas divisorias estables
    ax.contour(X, Y, Z, levels=np.arange(0.5, 5, 1), colors='black', linewidths=0.8, alpha=0.4)

    # Diagonal ideal de control (y = x)
    ax.plot([0, limite_max], [0, limite_max], '--', color='black', linewidth=1.0, alpha=0.4, zorder=3)

    # Inyección del scatter con el color exacto saturado de su zona
    ax.scatter(y_real, y_pred, c=colores_array[zonas], s=5, alpha=0.70, edgecolors='none', zorder=5)

    # ==================================================
    # 6. ROTULACIÓN QUIRÚRGICA CON CONTROL DE VISIBILIDAD (REQUISITOS 1 Y 2)
    # ==================================================
    # Diccionario de coordenadas absolutas fijas en mg/dL basadas en los baricentros de Clarke
    coordenadas_etiquetas = {
        'A': (130, 130, f"A\n({pct['A']:.1f}%)", dict_colores[0]),
        'B_sup': (110, 180, f"B\n({pct['B']:.1f}%)", dict_colores[1]),
        'B_inf': (180, 100, f"B\n({pct['B']:.1f}%)", dict_colores[1]),
        'C_sup': (90, 220, f"C\n({pct['C']:.1f}%)", dict_colores[2]),
        'C_inf': (155, 20, f"C\n({pct['C']:.1f}%)", dict_colores[2]),
        'D_izq': (35, 125, f"D\n({pct['D']:.1f}%)", dict_colores[3]),
        'D_der': (245, 80, f"D\n({pct['D']:.1f}%)", dict_colores[3]),
        'E_sup': (35, 220, f"E\n({pct['E']:.1f}%)", dict_colores[4]),
        'E_inf': (220, 35, f"E\n({pct['E']:.1f}%)", dict_colores[4])
    }

    # Bucle inteligente de renderizado de textos con control de desbordamiento por zoom
    for clave, (cx, cy, texto, col) in coordenadas_etiquetas.items():
        # [CONTROL DE VISIBILIDAD] El texto se inyecta SOLO si cae dentro del recuadro actual
        if cx < limite_max and cy < limite_max:
            ax.text(
                cx, cy, texto, 
                fontsize=10, 
                fontweight='bold', 
                color=col,  # Mismo color exacto saturado que el punto
                ha='center', 
                va='center',
                zorder=12
            )

    # Cuadro formal clínico de la métrica unificada (Zona A + Zona B)
    ax.text(12, 12, f"Zona A+B = {pct['A'] + pct['B']:.2f}%", fontsize=11, fontweight='bold',
            bbox=dict(facecolor='white', alpha=0.95, boxstyle='round,pad=0.4', edgecolor='gray', linewidth=0.5), zorder=15)

    # ==================================================
    # 7. MAQUETACIÓN E ISOMETRÍA FORMAL
    # ==================================================
    ax.set_xlim(0, limite_max)
    ax.set_ylim(0, limite_max)
    ax.set_aspect('equal') 

    ax.set_xlabel("Glucosa de Referencia Real (mg/dL)", fontsize=11, labelpad=10)
    ax.set_ylabel("Glucosa Predicha por XGBoost (mg/dL)", fontsize=11, labelpad=10)
    ax.set_title(f"Clarke Error Grid Analysis ({tipo_modelo_limpio.capitalize()}): {user_id}", fontsize=13, pad=15, fontweight='bold')
    ax.grid(linestyle=':', alpha=0.3, color='gray')
    plt.tight_layout()

    # ==================================================
    # 8. EXPORTACIÓN FÍSICA AUTOMATIZADA
    # ==================================================
    ruta_salida = globals().get('CLARKE_DIR', r"C:\Users\regin\Documents\GitHub\TFG-pruebas\Modeling\CGMacros")
    os.makedirs(ruta_salida, exist_ok=True)
    
    nombre_archivo = f"clarke_grid_{user_id}_{tipo_modelo_limpio}.png"
    ruta_guardado = os.path.join(ruta_salida, nombre_archivo)
    
    fig.savefig(ruta_guardado, dpi=300, bbox_inches='tight')
    
    display(fig)
    plt.close(fig)
    print(f"[OK] Cuadrícula con diseño estético avanzado exportada con éxito en: {nombre_archivo}")


Visualización del modelo individual:

In [ ]:
user = 'CGMacros-009'

# LLAMADA 1: Ver cómo predice el modelo personalizado (Individual) de este usuario

# Al no pasar fechas, el script guarda la imagen automáticamente como "CGMacros-013_5760.png"
# Si inyectamos las fechas, el modelo ignora ventana_minutos y recorta el intervalo exacto
# Nota: Asegurarnos de elegir fechas que estén dentro del rango de test de ese usuario

graficar_prediccion_vs_realidad(
    user_id=user,
    tipo_modelo='individual',
    ventana_minutos=5760
    #fecha_inicio='2024-01-27 09:00:00', # Día y hora de inicio exactos '2020-05-08 08:00:00'
    #fecha_fin='2024-02-07 22:00:00'     # Día y hora de fin exactos
)

In [ ]:
for i in range(1, 50):
    user = f"CGMacros-{i:03d}"
    
    try:
        graficar_prediccion_vs_realidad(
            user_id=user,
            tipo_modelo='individual',
            ventana_minutos=5760
        )
    except Exception as e:
        print(f"Error en {user}: {e}")

In [ ]:
user = 'CGMacros-013'

# Renderizar y guardar la cuadrícula de Clarke para el modelo individual de este sujeto
graficar_cuadricula_clarke_sujeto(user_id=user, tipo_modelo='individual')

In [ ]:
for i in range(1, 50):
    user = f"CGMacros-{i:03d}"

    try:
        graficar_cuadricula_clarke_sujeto(
            user_id=user,
            tipo_modelo='individual'
        )
    except Exception as e:
        print(f"Error en {user}: {e}")

Visualización del modelo grupal general:

In [ ]:
user = 'CGMacros-013'

# LLAMADA 2: Ver cómo predice el modelo General de este mismo usuario en el mismo periodo

# Al no pasar fechas, el script guarda la imagen automáticamente como "CGMacros-013_5760.png"
# Si inyectamos las fechas, el modelo ignora ventana_minutos y recorta el intervalo exacto
# Nota: Asegurarnos de elegir fechas que estén dentro del rango de test de ese usuario

graficar_prediccion_vs_realidad(
    user_id=user,
    tipo_modelo='general',    # <--- Él sabe aplicar el filtro df_preds_global[... == user]
    ventana_minutos=5760,
    fecha_inicio='2024-01-27 09:00:00', # Día y hora de inicio exactos '2020-05-08 08:00:00'
    fecha_fin='2024-02-07 22:00:00'     # Día y hora de fin exactos
)

In [ ]:
for i in range(1, 50):
    user = f"CGMacros-{i:03d}"
    
    try:
        graficar_prediccion_vs_realidad(
            user_id=user,
            tipo_modelo='general',
            ventana_minutos=5760
        )
    except Exception as e:
        print(f"Error en {user}: {e}")

Visualización del modelo por grupo metabólico:

In [ ]:
user = 'CGMacros-013'

# LLAMADA 3: Ve que es diabético, entra en REGISTRO_MODELOS['metabolico']['Diabético'] y lo aísla

# Al no pasar fechas, el script guarda la imagen automáticamente como "CGMacros-013_5760.png"
# Si inyectamos las fechas, el modelo ignora ventana_minutos y recorta el intervalo exacto
# Nota: Asegurarnos de elegir fechas que estén dentro del rango de test de ese usuario

graficar_prediccion_vs_realidad(
    user_id=user,
    tipo_modelo='metabolico',    # <--- Él sabe aplicar el filtro df_preds_global[... == user]
    ventana_minutos=5760,
    fecha_inicio='2024-01-27 09:00:00', # Día y hora de inicio exactos '2020-05-08 08:00:00'
    fecha_fin='2024-02-07 22:00:00'     # Día y hora de fin exactos
)

In [ ]:
for i in range(1, 50):
    user = f"CGMacros-{i:03d}"
    
    try:
        graficar_prediccion_vs_realidad(
            user_id=user,
            tipo_modelo='metabolico',
            ventana_minutos=5760
        )
    except Exception as e:
        print(f"Error en {user}: {e}")

 Podemos elegir dos ventanas temporales distintas de un mismo usuario y visualizar ambas comparativas:
 - Una ventana en **régimen basal** (Ayuno/Sueño): Donde ver cómo la predicción individual se mantiene plana, estable y pegada a la realidad (demostrando un error absoluto bajísimo en ausencia de perturbaciones).
 - Una ventana en **régimen posprandial** (Comida): Donde ver el impacto de las funciones de decaimiento e índices clínicos añadidos al vector de estados. Veremos si el XGBoost copia bien la pendiente de subida tras la ingesta y si capta el efecto de amortiguación de la insulina durante la bajada.

### D4. Explicabilidad (XAI): SHAP

In [ ]:
# ===================================================================================
# SHAP POR GRUPOS (SUCONJUNTO DE LOS DATOS TOTALES PARA EVITAR DESBORDAMIENTO DE RAM)
# ===================================================================================

def calcular_y_guardar_shap(
    df_final_eng: pd.DataFrame,
    best_alpha_ch: float,
    best_alpha_lento: float,
    params_xgb_ganador: dict,
    nombre_experimento: str = "global",
    num_muestras_shap: int = 5000
) -> None:
    """Calcula los valores SHAP para el modelo y exporta los gráficos de explicabilidad.

    Reconstruye la matriz de características X óptima utilizando las mejores alphas 
    fisiológicas descubiertas, entrena el modelo XGBoost definitivo y calcula 
    la importancia matemática de las variables mediante TreeExplainer sobre un 
    subconjunto representativo. Exporta el Summary Plot a disco.

    Args:
        df_final_eng (pd.DataFrame): DataFrame original con la ingeniería base.
        best_alpha_ch (float): Tasa de absorción de carbohidratos óptima descubierta.
        best_alpha_lento (float): Tasa de absorción lenta óptima descubierta.
        params_xgb_ganador (dict): Hiperparámetros óptimos del árbol de XGBoost.
        nombre_experimento (str, optional): Etiqueta para el nombre del archivo 
            ('global', 'diabetico', 'no_diabetico'). Defaults to "global".
        num_muestras_shap (int, optional): Número de filas aleatorias para aproximar 
            los valores SHAP sin saturar la RAM. Defaults to 5000.

    Returns:
        None: Guarda los gráficos en el repositorio local y libera los lienzos.

    Recommendations:
        - Ejecute esta función al final del pipeline de modelización, utilizando 
          las mejores combinaciones arrojadas por tu bucle de Optuna.
    """
    df_modelado = df_final_eng.dropna(subset=['target_glucose_60m']).copy()
    epsilon = 1e-5
    
    print(f"\n[SHAP] Inicializando cálculo de explicabilidad para enfoque: {nombre_experimento}...")
    
    # 1. Reconstrucción exacta de las columnas dinámicas óptimas ganadoras
    ch_ef = df_modelado.groupby('user_id', observed=False)['carbs'].transform(lambda x: calcular_tanque_vectorizado(x.to_numpy(), best_alpha_ch))
    fat_ef = df_modelado.groupby('user_id', observed=False)['fat'].transform(lambda x: calcular_tanque_vectorizado(x.to_numpy(), best_alpha_lento))
    pro_ef = df_modelado.groupby('user_id', observed=False)['protein'].transform(lambda x: calcular_tanque_vectorizado(x.to_numpy(), best_alpha_lento))
    fib_ef = df_modelado.groupby('user_id', observed=False)['fiber'].transform(lambda x: calcular_tanque_vectorizado(x.to_numpy(), best_alpha_lento))
    
    df_modelado['CH_efectivo'] = ch_ef
    df_modelado['ratio_complejidad_dinamico'] = (fat_ef + pro_ef) / (ch_ef + epsilon)
    df_modelado['indice_retardo_gastrico_dinamico'] = (fat_ef + fib_ef) / (ch_ef + epsilon)
    df_modelado['interaccion_CH_HbA1c'] = ch_ef * df_modelado['hba1c_inicial']
    
    # 2. Purgar columnas no predictoras para ensamblar X e y
    cols_drop = ['time', 'target_glucose_60m', 'meal_type', 'image_path']
    X_completa = df_modelado.drop(columns=cols_drop, errors='ignore')
    y_completa = df_modelado['target_glucose_60m']
    
    # 3. Entrenar el Modelo Definitivo con la estructura óptima
    print("[SHAP] Entrenando el modelo XGBoost definitivo...")
    params_xgb_ganador.update({'tree_method': 'hist', 'enable_categorical': True, 'n_jobs': -1})
    modelo_final = xgb.XGBRegressor(**params_xgb_ganador)
    modelo_final.fit(X_completa, y_completa)
    
    # 4. Submuestreo estadístico para el cálculo de SHAP (Evitar desbordamiento de RAM)
    # Seleccionamos muestras aleatorias representativas manteniendo la estructura temporal
    if len(X_completa) > num_muestras_shap:
        X_submuestra = X_completa.sample(n=num_muestras_shap, random_state=42)
    else:
        X_submuestra = X_completa
        
    # 5. Inicializar el TreeExplainer nativo de SHAP optimizado para árboles de decisión
    print("[SHAP] Calculando valores de Shapley mediante TreeExplainer...")
    explainer = shap.TreeExplainer(modelo_final)
    shap_values = explainer(X_submuestra)
    
    # --- 6. GENERACIÓN Y EXPORTACIÓN DE GRÁFICOS ACADÉMICOS ---
    ruta_directorio = SHAP_DIR
    os.makedirs(ruta_directorio, exist_ok=True)
    
    # Gráfico 1: SHAP Summary Plot (Gráfico de abejas / Beeswarm)
    # Es el gráfico más potente: muestra el impacto y la dirección (positivo/negativo) de cada variable
    plt.figure(figsize=(12, 8), dpi=150)
    shap.plots.beeswarm(shap_values, max_display=20, show=False)
    plt.title(f"Importancia y Atribución SHAP: Enfoque {nombre_experimento.capitalize()}", fontsize=14, pad=15, fontweight='bold')
    plt.tight_layout()
    
    nombre_grafico = f"shap_summary_{nombre_experimento.lower()}.png"
    ruta_guardado = os.path.join(ruta_directorio, nombre_grafico)
    plt.savefig(ruta_guardado, dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"[OK] Módulo SHAP finalizado. Gráfico exportado en: {ruta_guardado}")

In [ ]:
# ===========================================
# SHAP POR GRUPOS (100% DE LOS DATOS TOTALES)
# ===========================================

def calcular_shap_poblacional_completo(
    df_final_eng: pd.DataFrame,
    reporte_modelo: Dict[str, Any],
    nombre_experimento: str = "global"
) -> None:
    """
    Calcula los valores SHAP poblacionales exactos sobre el 100% de la cohorte usando la API nativa de C++.

    Mantiene la consistencia del espacio de variables categóricas nativas fijado en las fases previas,
    delegando el cálculo aditivo directamente en el núcleo de XGBoost mediante 'pred_contribs=True'
    para preservar el orden jerárquico biológico de los grupos metabólicos.

    Args:
        df_final_eng (pd.DataFrame): DataFrame global ordenado con la ingeniería base.
        reporte_modelo (Dict[str, Any]): Diccionario de resultados de la validación LOSO 
            que aloja el subdiccionario 'mejores_parametros'.
        nombre_experimento (str, optional): Identificador descriptivo para etiquetar la cohorte. 
            Defaults to "global".
    """
    # 1. Clonación estructural y depuración de registros sin variable objetivo futura (t+60)
    df_modelado = df_final_eng.dropna(subset=['target_glucose_60m']).copy()
    df_modelado = df_modelado.sort_values(by=['user_id', 'time']).reset_index(drop=True)
    epsilon = 1e-5
    
    # 2. Desempaquetado automatizado de la parametrización óptima inyectada por Optuna
    params_hibridos = reporte_modelo['mejores_parametros']
    best_alpha_ch = params_hibridos['alpha_ch']
    best_alpha_lento = params_hibridos['alpha_lento']
    params_xgb = {k: v for k, v in params_hibridos.items() if k not in ['alpha_ch', 'alpha_lento']}
    
    print(f"\n[SHAP POBLACIONAL NATIVO] Procesando el 100% de la cohorte: '{nombre_experimento}'...")
    
    # 3. Reconstrucción acelerada usando el bucle de numpy aislado por sujeto
    ch_ef = df_modelado.groupby('user_id', observed=False)['carbs'].transform(lambda x: _bucle_tanque_nativo(x.fillna(0.0).to_numpy(), best_alpha_ch))
    fat_ef = df_modelado.groupby('user_id', observed=False)['fat'].transform(lambda x: _bucle_tanque_nativo(x.fillna(0.0).to_numpy(), best_alpha_lento))
    pro_ef = df_modelado.groupby('user_id', observed=False)['protein'].transform(lambda x: _bucle_tanque_nativo(x.fillna(0.0).to_numpy(), best_alpha_lento))
    fib_ef = df_modelado.groupby('user_id', observed=False)['fiber'].transform(lambda x: _bucle_tanque_nativo(x.fillna(0.0).to_numpy(), best_alpha_lento))
    
    df_modelado['CH_efectivo'] = ch_ef
    df_modelado['ratio_complejidad_dinamico'] = (fat_ef + pro_ef) / (ch_ef + epsilon)
    df_modelado['indice_retardo_gastrico_dinamico'] = (fat_ef + fib_ef) / (ch_ef + epsilon)
    df_modelado['interaccion_CH_HbA1c'] = ch_ef * df_modelado['hba1c_inicial'].astype(float)
    
    # 4. Mismas dimensiones: Exclusión de marcas temporales e identificadores nominales puros
    # Mantenemos 'grupo_metabolico' y 'dia_semana' como variables categóricas nativas dentro de X
    cols_drop = ['time', 'target_glucose_60m', 'meal_type', 'image_path', 'user_id']
    X_completa = df_modelado.drop(columns=cols_drop, errors='ignore').copy()
    y_completa = df_modelado['target_glucose_60m']
        
    # 5. Ajuste del estimador definitivo homogéneo de XGBoost con soporte categórico activado
    # Forzamos enable_categorical=True para que coincida con la Fase 2 Poblacional
    params_xgb.update({'tree_method': 'hist', 'n_jobs': -1, 'enable_categorical': True})
    modelo_final = xgb.XGBRegressor(**params_xgb)
    modelo_final.fit(X_completa, y_completa)
    
    # --- EXTRACCIÓN DE APORTES MEDIANTE CONTRIBUCIONES NATIVAS DE C++ ---
    booster = modelo_final.get_booster()
    dtrain = xgb.DMatrix(X_completa, label=y_completa, enable_categorical=True)
    contribuciones = booster.predict(dtrain, pred_contribs=True)
    
    shap_data = contribuciones[:, :-1]  
    base_value = contribuciones[0, -1]   
    
    # Pasamos 'X_completa' directo (sin .values) para que SHAP lea los metadatos de las categorías
    shap_values_obj = shap.Explanation(
        values=shap_data,
        base_values=np.full(len(X_completa), base_value),
        data=X_completa,
        feature_names=list(X_completa.columns)
    )
    
    # 6. Renderizado estético y guardado físico en disco
    ruta_directorio = SHAP_DIR
    os.makedirs(ruta_directorio, exist_ok=True)
    
    fig, ax = plt.subplots(figsize=(12, 8), dpi=300)
    shap.plots.beeswarm(shap_values_obj, max_display=20, show=False)
    plt.title(f"Atribución SHAP Poblacional Colectiva (100% Datos): Enfoque {nombre_experimento.upper()}", fontsize=13, pad=15, fontweight='bold')
    plt.tight_layout()
    
    nombre_archivo = f"shap_summary_{nombre_experimento.lower()}_completo.png"
    plt.savefig(os.path.join(ruta_directorio, nombre_archivo), dpi=150, bbox_inches='tight')
    
    # Liberación del lienzo en RAM
    mostrar_y_liberar_figura(fig)
    print(f"[OK] Gráfico SHAP Poblacional Colectivo exportado con éxito en: {nombre_archivo}")


In [ ]:
# =============================================================
# EJECUCIÓN SHAP POR GRUPOS (100% DE LOS DATOS DE CADA COHORTE)
# =============================================================

# Ejemplo de llamada para el SHAP Colectivo Poblacional al 100%
calcular_shap_poblacional_completo(
    df_final_eng=df_final, 
    reporte_modelo=reporte_global_hibrido, 
    nombre_experimento="global"
)

In [ ]:
# =========================================================================
# EJECUCIÓN SHAP POR GRUPOS METABÓLICOS (100% DE LOS DATOS DE CADA COHORTE)
# =========================================================================

# 1. EJECUCIÓN GRUPO GLOBAL (Toda la población junta)
calcular_shap_poblacional_completo(
    df_final_eng=df_final, 
    reporte_modelo=reporte_global_hibrido, 
    nombre_experimento="global"
)

# 2. EJECUCIÓN COHORTE NO DIABÉTICOS
df_no_diabeticos = df_final[df_final['grupo_metabolico'] == 'No Diabético']

calcular_shap_poblacional_completo(
    df_final_eng=df_no_diabeticos,
    reporte_modelo=reporte_no_diab_hibrido, 
    nombre_experimento="no_diabeticos"
)

# 3. EJECUCIÓN COHORTE PREDIABÉTICOS
df_prediabeticos = df_final[df_final['grupo_metabolico'] == 'Prediabético']

calcular_shap_poblacional_completo(
    df_final_eng=df_prediabeticos,
    reporte_modelo=reporte_prediab_hibrido, 
    nombre_experimento="prediabeticos"
)

# 4. EJECUCIÓN COHORTE DIABÉTICOS
df_diabeticos = df_final[df_final['grupo_metabolico'] == 'Diabético']

calcular_shap_poblacional_completo(
    df_final_eng=df_diabeticos,
    reporte_modelo=reporte_diab_hibrido, 
    nombre_experimento="diabeticos"
)

In [ ]:
# ===========================================
# SHAP INDIVIDUAL (100% DE LOS DATOS TOTALES)
# ===========================================

def calcular_shap_individual_completo(
    df_final_eng: pd.DataFrame,
    user_target: str,
    reporte_usuario: Dict[str, Any]
) -> None:
    """Calcula y exporta la firma fisiológica SHAP exacta del 100% de los datos de un único paciente.

    Aísla cronológicamente los registros del individuo, elimina variables constantes de control 
    con entropía nula y computa los aportes marginales exactos mediante la API de bajo nivel 
    de XGBoost. Exporta de forma automatizada un set de tres gráficos complementarios:
    impacto global (Beeswarm), umbral dinámico (Scatter) y explicación local en pico (Waterfall).

    Args:
        df_final_eng (pd.DataFrame): DataFrame global ordenado con la ingeniería base.
        user_target (str): Identificador del sujeto a evaluar (ej: 'CGMacros-001').
        reporte_usuario (Dict[str, Any]): Diccionario de resultados del bucle Walk-Forward 
            personalizado de este usuario específico.
    """
    # 1. Segmentación del subconjunto temporal exclusivo de este usuario
    df_usuario = df_final_eng[df_final_eng['user_id'] == user_target].copy()
    df_modelado = df_usuario.dropna(subset=['target_glucose_60m']).copy()
    epsilon = 1e-5
    
    # 2. Desempaquetado analítico de los hiperparámetros óptimos e hiperalphas personalizadas
    params_hibridos = reporte_usuario['mejores_parametros']
    best_alpha_ch = params_hibridos['alpha_ch']
    best_alpha_lento = params_hibridos['alpha_lento']
    params_xgb = {k: v for k, v in params_hibridos.items() if k not in ['alpha_ch', 'alpha_lento']}
    
    print(f"\n[SHAP INDIVIDUAL BAJO NIVEL] Procesando el 100% de los datos del sujeto: {user_target}...")
    
    # 3. Reconstrucción acelerada usando la función interna nativa de NumPy
    ch_ef = _bucle_tanque_nativo(df_modelado['carbs'].fillna(0.0).to_numpy(), best_alpha_ch)
    fat_ef = _bucle_tanque_nativo(df_modelado['fat'].fillna(0.0).to_numpy(), best_alpha_lento)
    pro_ef = _bucle_tanque_nativo(df_modelado['protein'].fillna(0.0).to_numpy(), best_alpha_lento)
    fib_ef = _bucle_tanque_nativo(df_modelado['fiber'].fillna(0.0).to_numpy(), best_alpha_lento)
    
    df_modelado['CH_efectivo'] = ch_ef
    df_modelado['ratio_complejidad_dinamico'] = (fat_ef + pro_ef) / (ch_ef + epsilon)
    df_modelado['indice_retardo_gastrico_dinamico'] = (fat_ef + fib_ef) / (ch_ef + epsilon)
    df_modelado['interaccion_CH_HbA1c'] = ch_ef * df_modelado['hba1c_inicial'].astype(float)
    
    # Purga de constantes locales (Mismo espacio vectorial exacto de la Fase 2)
    cols_drop = [
        'time', 'target_glucose_60m', 'meal_type', 'image_path', 
        'user_id', 'grupo_metabolico', 'hba1c_inicial'
    ]
    X_completa = df_modelado.drop(columns=cols_drop, errors='ignore').copy()
    y_completa = df_modelado['target_glucose_60m']
    
    # 4. Ajuste del regresor local calibrado con soporte de variables categóricas activado
    params_xgb.update({'tree_method': 'hist', 'n_jobs': -1, 'enable_categorical': True})
    modelo_individual = xgb.XGBRegressor(**params_xgb)
    modelo_individual.fit(X_completa, y_completa)

    # --- EXTRACCIÓN DE APORTES MEDIANTE PRED_CONTRIBS EN C++ ---
    booster = modelo_individual.get_booster()
    dtrain = xgb.DMatrix(X_completa, label=y_completa, enable_categorical=True)
    contribuciones = booster.predict(dtrain, pred_contribs=True)
    shap_data = contribuciones[:, :-1]
    base_value = contribuciones[0, -1]

    # Recomponer la firma distributiva en el objeto de visualización oficial
    shap_values_obj = shap.Explanation(
        values=shap_data,
        base_values=np.full(len(X_completa), base_value),
        data=X_completa,
        feature_names=list(X_completa.columns)
    )
    
    # Asegurar la existencia del directorio de destino
    ruta_directorio = SHAP_DIR
    os.makedirs(ruta_directorio, exist_ok=True)
    
    # =========================================================================
    # GRÁFICO 1: IMPACTO GLOBAL (BEESWARM)
    # =========================================================================
    fig1, ax1 = plt.subplots(figsize=(12, 8), dpi=300)
    shap.plots.beeswarm(shap_values_obj, max_display=20, show=False)
    plt.title(f"Firma Metabólica SHAP Personalizada (Global): Sujeto {user_target}", fontsize=13, pad=15, fontweight='bold')
    plt.tight_layout()
    
    nombre_beeswarm = f"shap_1_beeswarm_individual_{user_target}.png"
    plt.savefig(os.path.join(ruta_directorio, nombre_beeswarm), dpi=150, bbox_inches='tight')
    mostrar_y_liberar_figura(fig1)
    
    # =========================================================================
    # GRÁFICO 2: UMBRAL DINÁMICO DE GLUCOSA (SCATTER / DEPENDENCE)
    # =========================================================================
    fig2, ax2 = plt.subplots(figsize=(8, 5), dpi=300)
    
    # Recomponemos localmente un objeto de explicación simplificado
    # con arrays puros de NumPy (.values) para la columna seleccionada. Esto evita que 
    # SHAP intente desempaquetar Series de Pandas y soluciona el error del '.reshape()'.
    col_idx = X_completa.columns.get_loc("glucose_libre")
    
    shap_value_single = shap.Explanation(
        values=shap_data[:, col_idx],
        base_values=np.full(len(X_completa), base_value),
        data=X_completa["glucose_libre"].values,  # <--- Array puro de NumPy garantiza compatibilidad con .reshape()
        feature_names="glucose_libre"
    )
    
    # Invocamos la función pasando el objeto compatible procesado en memoria continua
    shap.plots.scatter(
        shap_value_single, 
        ax=ax2, 
        show=False
    )
    
    plt.title(f"Umbral Homeostático SHAP (Dinámico): Sujeto {user_target}", fontsize=11, pad=12, fontweight='bold')
    plt.grid(True, linestyle=':', alpha=0.5)
    plt.tight_layout()
    
    nombre_scatter = f"shap_2_dependence_individual_{user_target}.png"
    plt.savefig(os.path.join(ruta_directorio, nombre_scatter), dpi=150, bbox_inches='tight')
    mostrar_y_liberar_figura(fig2)

    # =========================================================================
    # GRÁFICO 3: EXPLICACIÓN LOCAL EN PICO CLÍNICO (WATERFALL)
    # =========================================================================
    fila_pico_entera = X_completa['glucose_libre'].reset_index(drop=True).idxmax()
    
    # Inicializamos una figura estándar limpia
    fig3, ax3 = plt.subplots(figsize=(10, 6), dpi=300)
    
    # Pintamos la cascada nativa en su forma más simple y estable
    shap.plots.waterfall(shap_values_obj[fila_pico_entera], max_display=10, show=False)
    
    # Título limpio con un pad estándar para dar un poco de aire
    plt.title(f"Explicación Local SHAP (Pico Glucémico Máximo): Sujeto {user_target}", fontsize=11, pad=15, fontweight='bold')
    plt.tight_layout()
    
    # Guardado directo recortando bordes sobrantes automáticamente
    nombre_waterfall = f"shap_3_waterfall_pico_individual_{user_target}.png"
    plt.savefig(os.path.join(ruta_directorio, nombre_waterfall), dpi=150, bbox_inches='tight')
    
    # Invocación de tu helper de control de memoria RAM
    mostrar_y_liberar_figura(fig3)

In [ ]:
# =============================================================
# EJECUCIÓN SHAP INDIVIDUAL (100% DE LOS DATOS DE CADA COHORTE)
# =============================================================

shap.initjs()  # Renderizado interactivo de SHAP en notebooks
user = 'CGMacros-009'

# Ejemplo de llamada para extraer la fisiología del Sujeto 001 individual al 100%
calcular_shap_individual_completo(
    df_final_eng=df_final,
    user_target=user,
    reporte_usuario=reportes_hibridos_individuales[user]
)


Dentro de los momentos donde el sujeto está digiriendo alimentos, ¿cómo pondera su modelo individual el impacto de las variables para predecir la glucosa futura?

El modelo se entrena con todos los datos del sujeto (tanto periodos de ayuno, sueño como comidas) porque el árbol necesita aprender el contraste fisiológico general. Sin embargo, la explicación de SHAP a continuación se ejecuta exclusivamente sobre las ventanas de comida.

In [ ]:
# ======================================
# SHAP INDIVIDUAL (DATOS POSTPRANDIALES)
# ======================================

def calcular_shap_ventanas_comida(
    df_final_eng: pd.DataFrame,
    user_target: str,
    reporte_usuario: Dict[str, Any]
) -> None:
    """Calcula y exporta el gráfico SHAP exclusivo para los periodos posprandiales de un usuario.

    Aísla las ventanas cronológicas de absorción activa de 3 horas basándose en los eventos 
    de ingesta registrados en la columna 'carbs', garantizando la estabilidad del cálculo, 
    respetando el tipado de variables categóricas y eliminando constantes locales.

    Args:
        df_final_eng (pd.DataFrame): DataFrame global ordenado con la ingeniería base.
        user_target (str): Identificador del sujeto bajo estudio (ej: 'CGMacros-009').
        reporte_usuario (Dict[str, Any]): Diccionario de salida de la validación 
            Walk-Forward del usuario con la clave 'mejores_parametros'.
    """
    df_usuario = df_final_eng[df_final_eng['user_id'] == user_target].copy()
    df_modelado = df_usuario.dropna(subset=['target_glucose_60m']).copy()
    epsilon = 1e-5
    
    # 1. Comprobación inicial de seguridad adaptada a los NaNs biológicos reales
    carbs_rellenos = df_modelado['carbs'].fillna(0.0)
    if (carbs_rellenos == 0.0).all():
        print(f"[ADVERTENCIA CRÍTICA] El usuario {user_target} no tiene ningún evento de comida registrado en todo su historial. Gráfica posprandial omitida.")
        return
        
    # 2. Desempaquetado analítico de hiperparámetros e hiperalphas personalizadas
    params_hibridos = reporte_usuario['mejores_parametros']
    best_alpha_ch = params_hibridos['alpha_ch']
    best_alpha_lento = params_hibridos['alpha_lento']
    params_xgb = {k: v for k, v in params_hibridos.items() if k not in ['alpha_ch', 'alpha_lento']}
    
    print(f"\n[SHAP POSPRANDIAL TEMPORAL] Reconstruyendo dinámica de nutrientes para: {user_target}...")
    
    # 3. Reconstrucción exacta mediante la función matemática de bajo nivel
    ch_ef = _bucle_tanque_nativo(df_modelado['carbs'].fillna(0.0).to_numpy(), best_alpha_ch)
    fat_ef = _bucle_tanque_nativo(df_modelado['fat'].fillna(0.0).to_numpy(), best_alpha_lento)
    pro_ef = _bucle_tanque_nativo(df_modelado['protein'].fillna(0.0).to_numpy(), best_alpha_lento)
    fib_ef = _bucle_tanque_nativo(df_modelado['fiber'].fillna(0.0).to_numpy(), best_alpha_lento)
    
    df_modelado['CH_efectivo'] = ch_ef
    df_modelado['ratio_complejidad_dinamico'] = (fat_ef + pro_ef) / (ch_ef + epsilon)
    df_modelado['indice_retardo_gastrico_dinamico'] = (fat_ef + fib_ef) / (ch_ef + epsilon)
    df_modelado['interaccion_CH_HbA1c'] = ch_ef * df_modelado['hba1c_inicial'].astype(float)
    
    # 4. Añadida 'hba1c_inicial' a la purga de constantes locales muertas
    cols_drop = [
        'time', 'target_glucose_60m', 'meal_type', 'image_path', 
        'user_id', 'grupo_metabolico', 'hba1c_inicial'
    ]
    X_completa = df_modelado.drop(columns=cols_drop, errors='ignore').copy()
    y_completa = df_modelado['target_glucose_60m']
    
    # 5. Forzar enable_categorical=True para dar soporte a la columna 'dia_semana'
    params_xgb.update({'tree_method': 'hist', 'n_jobs': -1, 'enable_categorical': True})
    modelo_individual = xgb.XGBRegressor(**params_xgb)
    modelo_individual.fit(X_completa, y_completa)
    
    # --- [MÁSCARA QUIRÚRGICA TEMPORAL POSPRANDIAL] ---
    # Marcamos los índices basados en la serie de carbs rellena generada en el paso 1
    indices_comida = np.where(carbs_rellenos.to_numpy() > 0)[0]
    
    # Construimos un conjunto de índices expandido que cubra los siguientes 180 minutos (3 horas) tras cada comida
    indices_posprandiales = set()
    for idx in indices_comida:
        for t in range(0, 180):  
            if idx + t < len(X_completa):
                indices_posprandiales.add(idx + t)
                
    X_posprandial = X_completa.iloc[list(indices_posprandiales)].copy()
    
    if X_posprandial.empty:
        print(f"[ADVERTENCIA] No se consiguieron delimitar ventanas temporales posprandiales para el usuario {user_target}.")
        return
        
    print(f"[SHAP] Calculando SHAP posprandial nativo sobre {len(X_posprandial)} minutos tras las ingestas...")
    
    # 6. Cálculo de contribuciones nativas de C++ sobre el bloque de comida con soporte categórico activo
    booster = modelo_individual.get_booster()
    dtest = xgb.DMatrix(X_posprandial, enable_categorical=True)
    contribuciones = booster.predict(dtest, pred_contribs=True)
    
    shap_data = contribuciones[:, :-1]
    base_value = contribuciones[0, -1]
    
    # Pasamos 'X_posprandial' directo en data (sin .values) para preservar los metadatos categóricos
    shap_values_obj = shap.Explanation(
        values=shap_data,
        base_values=np.full(len(X_posprandial), base_value),
        data=X_posprandial,
        feature_names=list(X_posprandial.columns)
    )
    
    # 7. Renderizado estético y uso de tu helper de control de memoria RAM
    ruta_directorio = SHAP_DIR
    os.makedirs(ruta_directorio, exist_ok=True)
    
    fig, ax = plt.subplots(figsize=(12, 8), dpi=150)
    shap.plots.beeswarm(shap_values_obj, max_display=20, show=False)
    plt.title(f"Firma SHAP en Ventanas de Comida Posprandial: Sujeto {user_target}", fontsize=13, pad=15, fontweight='bold')
    plt.tight_layout()
    
    nombre_archivo = f"shap_summary_individual_{user_target}_posprandial.png"
    plt.savefig(os.path.join(ruta_directorio, nombre_archivo), dpi=150, bbox_inches='tight')
    
    # Invocación del helper: Renderiza en el notebook y destruye el lienzo inmediatamente
    mostrar_y_liberar_figura(fig)
    print(f"[OK] Gráfica SHAP Posprandial exportada con éxito: {nombre_archivo}")


In [ ]:
user = 'CGMacros-009'

calcular_shap_ventanas_comida(
    df_final_eng=df_final,
    user_target=user,
    reporte_usuario=reportes_hibridos_individuales[user]
)

En el Modelo Global, SHAP nos da la importancia de variables del "humano promedio". En cambio, si calculamos SHAP para el Modelo Individual de un sujeto concreto, el gráfico nos desvelará la fisiología exacta e hiperpersonalizada de ese individuo.

Por ejemplo:
- Al sacar el SHAP del Sujeto A, vemos que la variable reina tras los lags es CH_efectivo (es una persona muy sensible a la comida).
- Al sacar el SHAP del Sujeto B, puede que la variable más importante abajo de la glucosa sea mets_ewma_12h (el ejercicio tiene un impacto contrarregulador masivo en su cuerpo).

Para la memoria del proyecto, comparar el gráfico SHAP Global frente al gráfico SHAP de dos individuos con comportamientos metabólicos opuestos puede aportar valor.